# Streaming row-wise log-sum-exp

The log-sum-exp $\mathrm{lse}(x) = \log \sum_j e^{x_j}$ is the normaliser behind softmax and cross-entropy. For a 128k-token vocabulary a row does not fit in one block, so the kernel must **stream** over it: one read of each element, keeping a running maximum and a running sum that is rescaled whenever the maximum grows.

Write `logsumexp_rows(x)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor of shape `(M, N)`, $M, N \ge 1$, with $N$ possibly much larger than the block.
- Entries are finite or `-inf`. A row may be **entirely** `-inf`.

## Output
- `float32` tensor of shape `(M,)` with $\mathrm{lse}$ of each row. An all-`-inf` row gives `-inf` (never NaN).

## Requirements
- One program per row, a loop over chunks of `BLOCK` columns, each element loaded once.
- Keep **lane-wise** running state `m` (max) and `s` (sum of $e^{x - m}$) of shape `[BLOCK]`, updated per chunk with
  $$m' = \max(m, x), \qquad s' = s\, e^{m - m'} + e^{x - m'}$$
  then combine the lanes after the loop: $M = \max_\ell m_\ell$, $S = \sum_\ell s_\ell\, e^{m_\ell - M}$, result $M + \log S$.
- Whenever a quantity like $-\infty - (-\infty)$ would appear, the corresponding term must count as 0.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def logsumexp_rows(x):
    """x: (M, N) float32 CUDA tensor, entries finite or -inf. Return (M,) float32 log-sum-exp per row."""
    raise NotImplementedError


@triton.jit
def _lse_kernel(x_ptr, out_ptr, N, stride_row, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[0.0, 0.0], [1.0, 2.0]], "dtype": "float32"}], "e": {"__nd__": [0.6931471824645996, 2.3132617473602295], "dtype": "float32"}}, {"a": [{"__nd__": [[1000.0, 1000.0, 1000.0], [-1000.0, -1000.0, -1000.0]], "dtype": "float32"}], "e": {"__nd__": [1001.0986328125, -998.9013671875], "dtype": "float32"}}, {"a": [{"__nd__": [[{"__f__": "-inf"}, {"__f__": "-inf"}], [{"__f__": "-inf"}, 3.0]], "dtype": "float32"}], "e": {"__nd__": [{"__f__": "-inf"}, 3.0], "dtype": "float32"}}, {"a": [{"__nd__": [[{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, -0.781517505645752, 1.126706838607788, -0.9632192850112915, -0.0031623332761228085, -1.7071576118469238, -2.040504217147827, -0.6065481305122375, 1.6816766262054443, 0.05615920200943947, 1.0493011474609375, 1.7172199487686157, 0.3093709349632263, -0.8698796033859253, 0.07232974469661713, -0.5490259528160095, 0.5026586055755615, -0.6030825972557068, 1.3677724599838257, 1.1007410287857056, 0.22255517542362213, 0.09442239254713058, 0.48028287291526794, -0.054195232689380646, 0.19079002737998962, 0.9757041335105896, -0.6836785674095154, -0.40156081318855286, -0.2136152982711792, -2.253920316696167, 1.3495157957077026, -0.31225407123565674, 1.5123406648635864, -2.0350565910339355, -2.128565788269043, -0.3649080991744995, 0.06765211373567581, 0.9905010461807251, 0.8258947730064392, 1.3813579082489014, 0.08663376420736313, 2.5803472995758057, 0.8653749823570251, -1.7690213918685913, -0.8160562515258789, -0.8318923115730286, -0.11388222873210907, -0.7023590207099915, 0.2648245692253113, 1.2755409479141235, -1.5520251989364624, -0.6458618640899658, 1.8219490051269531, 1.7989935874938965, 1.913770079612732, 0.28737789392471313, 1.3129174709320068, 0.45132821798324585, -1.6752477884292603, -0.6664184927940369, 0.7313655018806458, 0.10799900442361832, -0.9286052584648132, -1.032508134841919, 0.4251745939254761, 0.24360942840576172, -0.57334303855896, 1.5732908248901367, -0.5044539570808411, -0.4841218888759613, 0.025827458128333092, -0.08818510919809341, -0.14138495922088623, -2.0579943656921387, 2.451486825942993, 0.6147351264953613, 0.3743853271007538, -0.6113747358322144, -0.14171752333641052, 0.46673065423965454, -0.6579427123069763, 0.8320793509483337, 0.6747430562973022, -0.31794944405555725, -0.751824140548706, 0.7660699486732483, 2.36812424659729, 0.2561940550804138, 1.0190222263336182, -0.11954464018344879, -0.8150767087936401, 1.1972007751464844, -0.29418331384658813, 2.24135160446167, -0.6413514614105225, -1.5420787334442139, 0.32414498925209045, 0.8507000207901001, 0.8727014064788818, 0.11258368194103241, -1.2312688827514648], [{"__f__": "-inf"}, -1.0184887647628784, {"__f__": "-inf"}, -0.5589251518249512, {"__f__": "-inf"}, -0.27085334062576294, {"__f__": "-inf"}, 1.9998067617416382, {"__f__": "-inf"}, 0.5170678496360779, {"__f__": "-inf"}, 1.0153615474700928, {"__f__": "-inf"}, 1.163699984550476, {"__f__": "-inf"}, -0.8517117500305176, {"__f__": "-inf"}, 0.7934653162956238, {"__f__": "-inf"}, 1.716383695602417, {"__f__": "-inf"}, 0.48341384530067444, {"__f__": "-inf"}, 0.8444262742996216, {"__f__": "-inf"}, -0.5707187056541443, {"__f__": "-inf"}, -0.5526652336120605, {"__f__": "-inf"}, -0.8537525534629822, {"__f__": "-inf"}, -1.2542436122894287, {"__f__": "-inf"}, 0.9365442395210266, {"__f__": "-inf"}, -0.043098900467157364, {"__f__": "-inf"}, 0.6958320140838623, {"__f__": "-inf"}, -0.21735450625419617, {"__f__": "-inf"}, -0.47733721137046814, {"__f__": "-inf"}, -0.1129312589764595, {"__f__": "-inf"}, 1.1680049896240234, {"__f__": "-inf"}, 0.9357380270957947, {"__f__": "-inf"}, -0.1141018345952034, {"__f__": "-inf"}, -1.2495012283325195, {"__f__": "-inf"}, 0.0934840589761734, {"__f__": "-inf"}, 1.8214917182922363, {"__f__": "-inf"}, 1.7093394994735718, {"__f__": "-inf"}, -0.6120632886886597, {"__f__": "-inf"}, 0.19146037101745605, {"__f__": "-inf"}, 0.7138845920562744, {"__f__": "-inf"}, 0.019981790333986282, {"__f__": "-inf"}, -0.0705459713935852, {"__f__": "-inf"}, 0.21143025159835815, {"__f__": "-inf"}, 0.8092516660690308, {"__f__": "-inf"}, 0.06792890280485153, {"__f__": "-inf"}, 1.0944414138793945, {"__f__": "-inf"}, 0.829684317111969, {"__f__": "-inf"}, -0.23794710636138916, {"__f__": "-inf"}, 0.9081556797027588, {"__f__": "-inf"}, -0.19417808949947357, {"__f__": "-inf"}, -1.9427920579910278, {"__f__": "-inf"}, 0.1887802928686142, {"__f__": "-inf"}, -1.3604587316513062, {"__f__": "-inf"}, -1.0277488231658936, {"__f__": "-inf"}, -0.7042838931083679, {"__f__": "-inf"}, 0.9564189314842224, {"__f__": "-inf"}, -0.010110634379088879, {"__f__": "-inf"}, -0.2663789987564087, {"__f__": "-inf"}, 2.176848888397217, {"__f__": "-inf"}, -0.47151342034339905, {"__f__": "-inf"}, -0.13982842862606049, {"__f__": "-inf"}, -1.3477221727371216, {"__f__": "-inf"}, 1.183670997619629, {"__f__": "-inf"}, 1.5622646808624268, {"__f__": "-inf"}, -1.4909965991973877, {"__f__": "-inf"}, -0.6705626845359802, {"__f__": "-inf"}, -0.7907170653343201, {"__f__": "-inf"}, 0.7961927056312561, {"__f__": "-inf"}, -0.2886977791786194, {"__f__": "-inf"}, 0.2814508080482483, {"__f__": "-inf"}, -0.9768092036247253, {"__f__": "-inf"}, 1.5979273319244385, {"__f__": "-inf"}, -0.2736221253871918, {"__f__": "-inf"}, 1.319510817527771, {"__f__": "-inf"}, 1.3828115463256836, {"__f__": "-inf"}, -0.6434986591339111, {"__f__": "-inf"}, -1.5795894861221313, {"__f__": "-inf"}, 0.8086578845977783, {"__f__": "-inf"}, 0.4555204212665558, {"__f__": "-inf"}, -0.392645925283432, {"__f__": "-inf"}, 0.2408585399389267, {"__f__": "-inf"}, 1.1015301942825317, {"__f__": "-inf"}, 1.9552407264709473, {"__f__": "-inf"}, 1.7714214324951172, {"__f__": "-inf"}, -0.39231735467910767, {"__f__": "-inf"}, 0.8033680319786072, {"__f__": "-inf"}, 0.7408158183097839, {"__f__": "-inf"}, -0.13765719532966614, {"__f__": "-inf"}, -0.64333575963974, {"__f__": "-inf"}, -1.8565982580184937, {"__f__": "-inf"}, 0.12041637301445007, {"__f__": "-inf"}, -0.8143920302391052, {"__f__": "-inf"}, 0.719982385635376, {"__f__": "-inf"}, -0.9871837496757507, {"__f__": "-inf"}, 0.25358256697654724, {"__f__": "-inf"}, 0.7264899015426636, {"__f__": "-inf"}, 1.6957792043685913, {"__f__": "-inf"}, -1.065488576889038, {"__f__": "-inf"}, -1.3662536144256592, {"__f__": "-inf"}, -0.7214717864990234, {"__f__": "-inf"}, 0.027547648176550865, {"__f__": "-inf"}, 0.1290251910686493, {"__f__": "-inf"}, 0.46967440843582153, {"__f__": "-inf"}, -0.3504354953765869, {"__f__": "-inf"}, 0.4683256149291992, {"__f__": "-inf"}, -0.9271500706672668, {"__f__": "-inf"}, -1.545451283454895, {"__f__": "-inf"}, 0.08667578548192978, {"__f__": "-inf"}, -1.2795581817626953, {"__f__": "-inf"}, 1.1742222309112549, {"__f__": "-inf"}, -1.1742799282073975, {"__f__": "-inf"}, 1.0729987621307373, {"__f__": "-inf"}, -0.4344438314437866, {"__f__": "-inf"}, 0.30342429876327515, {"__f__": "-inf"}, 1.7782871723175049, {"__f__": "-inf"}, 0.07167766988277435, {"__f__": "-inf"}, 1.1279739141464233, {"__f__": "-inf"}, 1.4110437631607056, {"__f__": "-inf"}, -0.10187806934118271, {"__f__": "-inf"}, 0.23562592267990112, {"__f__": "-inf"}, -0.2500196695327759, {"__f__": "-inf"}, 2.4668195247650146, {"__f__": "-inf"}, 1.6180423498153687, {"__f__": "-inf"}, 1.5716160535812378, {"__f__": "-inf"}, -1.2877835035324097, {"__f__": "-inf"}, 0.4857978820800781, {"__f__": "-inf"}, -0.03721648454666138, {"__f__": "-inf"}, 1.4782873392105103, {"__f__": "-inf"}, -1.2994948625564575, {"__f__": "-inf"}, -0.25454795360565186, {"__f__": "-inf"}, 0.9689344763755798, {"__f__": "-inf"}, -0.5687971115112305, {"__f__": "-inf"}, -0.12438467890024185, {"__f__": "-inf"}, -0.8821952939033508, {"__f__": "-inf"}, 0.37567681074142456, {"__f__": "-inf"}, 1.7429826259613037, {"__f__": "-inf"}, -0.3227407932281494, {"__f__": "-inf"}, 1.2275201082229614, {"__f__": "-inf"}, 0.37395811080932617, {"__f__": "-inf"}, -0.9466121792793274, {"__f__": "-inf"}, -0.6795979738235474, {"__f__": "-inf"}, 0.43697673082351685, {"__f__": "-inf"}, 1.2402700185775757, {"__f__": "-inf"}, -0.6187254190444946, {"__f__": "-inf"}, -0.6807674169540405, {"__f__": "-inf"}, -2.3518834114074707, {"__f__": "-inf"}, 1.1844197511672974, {"__f__": "-inf"}, 0.5211671590805054, {"__f__": "-inf"}, -0.5241538286209106, {"__f__": "-inf"}, -0.22564828395843506, {"__f__": "-inf"}, -0.03323300927877426, {"__f__": "-inf"}, -0.22818298637866974, {"__f__": "-inf"}, -0.699068009853363, {"__f__": "-inf"}, -0.929375171661377, {"__f__": "-inf"}, -0.5539638996124268, {"__f__": "-inf"}, 1.3601123094558716, {"__f__": "-inf"}, -1.0117592811584473, {"__f__": "-inf"}, 1.203709602355957, {"__f__": "-inf"}, -1.5879905223846436, {"__f__": "-inf"}, -0.3227812349796295, {"__f__": "-inf"}, -0.7083536982536316, {"__f__": "-inf"}, -0.9777945876121521, {"__f__": "-inf"}, -0.07482218742370605, {"__f__": "-inf"}, -1.8857500553131104, {"__f__": "-inf"}, 1.2549749612808228, {"__f__": "-inf"}, -1.5422126054763794, {"__f__": "-inf"}, -1.7514705657958984, {"__f__": "-inf"}, -0.7517598867416382, {"__f__": "-inf"}, 1.237052321434021, {"__f__": "-inf"}, -1.1584023237228394, {"__f__": "-inf"}, -1.0200161933898926, {"__f__": "-inf"}, -0.8555309772491455, {"__f__": "-inf"}, 0.5671093463897705, {"__f__": "-inf"}, -2.1694657802581787, {"__f__": "-inf"}, -1.242626667022705, {"__f__": "-inf"}, -1.0765284299850464, {"__f__": "-inf"}, 0.10304699838161469, {"__f__": "-inf"}, -0.338349848985672, {"__f__": "-inf"}, 0.8199916481971741, {"__f__": "-inf"}, -0.14117947220802307, {"__f__": "-inf"}, -0.5089758038520813, {"__f__": "-inf"}, 0.9448360204696655, {"__f__": "-inf"}, 1.2507400512695312, {"__f__": "-inf"}, 0.15852943062782288, {"__f__": "-inf"}, 0.3414141833782196, {"__f__": "-inf"}, -0.7117667198181152, {"__f__": "-inf"}, -1.333154559135437, {"__f__": "-inf"}, 2.4323813915252686, {"__f__": "-inf"}, -1.6010059118270874, {"__f__": "-inf"}, 0.17365725338459015, {"__f__": "-inf"}, 1.642374873161316, {"__f__": "-inf"}, -0.7316282987594604, {"__f__": "-inf"}, 0.3425194323062897, {"__f__": "-inf"}, 1.7227295637130737, {"__f__": "-inf"}, -1.9701327085494995, {"__f__": "-inf"}, -0.039851658046245575, {"__f__": "-inf"}, -0.07282532006502151, {"__f__": "-inf"}, 0.09860065579414368, {"__f__": "-inf"}, -1.9035446643829346, {"__f__": "-inf"}, -0.7357117533683777, {"__f__": "-inf"}, -1.404550313949585, {"__f__": "-inf"}, -0.3585866093635559, {"__f__": "-inf"}, -0.0694618970155716, {"__f__": "-inf"}, -0.09939122200012207, {"__f__": "-inf"}, 0.18237896263599396, {"__f__": "-inf"}, -0.30533748865127563, {"__f__": "-inf"}, 0.9187588095664978, {"__f__": "-inf"}, -0.44629034399986267, {"__f__": "-inf"}, -0.15205524861812592, {"__f__": "-inf"}, -1.1491998434066772, {"__f__": "-inf"}, 0.6008893847465515, {"__f__": "-inf"}, 1.477660059928894, {"__f__": "-inf"}, -0.15596874058246613, {"__f__": "-inf"}, -0.1670079529285431, {"__f__": "-inf"}, -1.2909183502197266, {"__f__": "-inf"}, -0.6647499203681946, {"__f__": "-inf"}, 2.1317808628082275, {"__f__": "-inf"}, -1.2829151153564453, {"__f__": "-inf"}, 1.3496474027633667, {"__f__": "-inf"}, -1.291317105293274, {"__f__": "-inf"}, -1.2880886793136597, {"__f__": "-inf"}, -0.10701286792755127, {"__f__": "-inf"}, 0.31833940744400024, {"__f__": "-inf"}, 0.37818005681037903, {"__f__": "-inf"}, 0.8739501237869263, {"__f__": "-inf"}, -0.16136419773101807, {"__f__": "-inf"}, -0.3109370768070221, {"__f__": "-inf"}, -0.3153008818626404, {"__f__": "-inf"}, 1.0395313501358032, {"__f__": "-inf"}, 0.8266658782958984, {"__f__": "-inf"}, 1.8066469430923462, {"__f__": "-inf"}, -3.0664844512939453, {"__f__": "-inf"}, 0.479562371969223, {"__f__": "-inf"}, 1.5370111465454102, {"__f__": "-inf"}, -0.4887831509113312, {"__f__": "-inf"}, 0.1747276335954666, {"__f__": "-inf"}, -0.5215981006622314, {"__f__": "-inf"}, 1.0315132141113281, {"__f__": "-inf"}, 0.1514960378408432, {"__f__": "-inf"}, -0.6448983550071716, {"__f__": "-inf"}, 0.7964668273925781, {"__f__": "-inf"}, 1.4567872285842896, {"__f__": "-inf"}, 0.15266071259975433, {"__f__": "-inf"}, -1.4153341054916382, {"__f__": "-inf"}, -0.35453060269355774, {"__f__": "-inf"}, -0.2955986559391022, {"__f__": "-inf"}, -2.1362295150756836, {"__f__": "-inf"}, -2.1011412143707275, {"__f__": "-inf"}, -0.22645631432533264, {"__f__": "-inf"}, 0.20902281999588013, {"__f__": "-inf"}, 0.6072528958320618, {"__f__": "-inf"}, -0.006514693144708872, {"__f__": "-inf"}, 0.41087594628334045, {"__f__": "-inf"}, 1.05104660987854, {"__f__": "-inf"}, 0.8848180770874023, {"__f__": "-inf"}, -0.7225026488304138, {"__f__": "-inf"}, 0.3244721293449402, {"__f__": "-inf"}, -0.2903190851211548, {"__f__": "-inf"}, -0.1508958488702774, {"__f__": "-inf"}, -2.095688819885254, {"__f__": "-inf"}, -1.0721970796585083, {"__f__": "-inf"}, 0.38151946663856506, {"__f__": "-inf"}, 0.6382202506065369, {"__f__": "-inf"}, -0.5360683798789978, {"__f__": "-inf"}, 1.1637986898422241, {"__f__": "-inf"}, -0.3254782259464264, {"__f__": "-inf"}, 0.20343683660030365, {"__f__": "-inf"}, 0.9434775114059448, {"__f__": "-inf"}, -1.014994740486145, {"__f__": "-inf"}, 0.9756955504417419, {"__f__": "-inf"}, 1.8438769578933716, {"__f__": "-inf"}, -0.681064248085022, {"__f__": "-inf"}, -0.35219404101371765, {"__f__": "-inf"}, 0.34033167362213135, {"__f__": "-inf"}, -0.38191625475883484, {"__f__": "-inf"}, 0.21430620551109314, {"__f__": "-inf"}, 0.3618394434452057, {"__f__": "-inf"}, 0.5207235813140869, {"__f__": "-inf"}, 1.3067703247070312, {"__f__": "-inf"}, 0.024130387231707573, {"__f__": "-inf"}, -1.1167570352554321, {"__f__": "-inf"}, -1.5732687711715698, {"__f__": "-inf"}, 1.3315364122390747, {"__f__": "-inf"}, 0.3272286653518677, {"__f__": "-inf"}, 0.47545313835144043, {"__f__": "-inf"}, -1.891380786895752, {"__f__": "-inf"}, -1.3692246675491333, {"__f__": "-inf"}, 0.20213274657726288, {"__f__": "-inf"}, 0.4977342486381531, {"__f__": "-inf"}, -0.9337007403373718, {"__f__": "-inf"}, 0.1574544757604599, {"__f__": "-inf"}, -0.1869732290506363, {"__f__": "-inf"}, 1.267552137374878, {"__f__": "-inf"}, 1.6917437314987183, {"__f__": "-inf"}, 0.2158326357603073, {"__f__": "-inf"}, -0.1444683074951172, {"__f__": "-inf"}, -0.2510125935077667, {"__f__": "-inf"}, -0.18427643179893494, {"__f__": "-inf"}, 0.2158830612897873, {"__f__": "-inf"}, 0.999337911605835, {"__f__": "-inf"}, 0.832640528678894, {"__f__": "-inf"}, 0.013073347508907318, {"__f__": "-inf"}, -0.9543712139129639, {"__f__": "-inf"}, 1.1667534112930298, {"__f__": "-inf"}, 0.9156278371810913, {"__f__": "-inf"}, 1.1160436868667603, {"__f__": "-inf"}, 0.048328910022974014, {"__f__": "-inf"}, 0.10333245992660522, {"__f__": "-inf"}, -0.8922365307807922, {"__f__": "-inf"}, -2.087963819503784, {"__f__": "-inf"}, 0.6959990859031677, {"__f__": "-inf"}, -1.684399962425232, {"__f__": "-inf"}, -1.0473569631576538, {"__f__": "-inf"}, -0.959793210029602, {"__f__": "-inf"}, 1.154924988746643, {"__f__": "-inf"}, -0.06479770690202713, {"__f__": "-inf"}, 2.098653793334961, {"__f__": "-inf"}, -0.8211389780044556, {"__f__": "-inf"}, 0.04181194677948952, {"__f__": "-inf"}, -0.1493178755044937, {"__f__": "-inf"}, 2.367239236831665, {"__f__": "-inf"}, -0.47089919447898865, {"__f__": "-inf"}, -0.2506568431854248, {"__f__": "-inf"}, -0.8803347945213318, {"__f__": "-inf"}, -0.38139042258262634, {"__f__": "-inf"}, 1.035429835319519, {"__f__": "-inf"}, 0.39657798409461975, {"__f__": "-inf"}, -0.0724635124206543, {"__f__": "-inf"}, 0.6088682413101196, {"__f__": "-inf"}, 0.5119999647140503, {"__f__": "-inf"}, -0.3540497422218323, {"__f__": "-inf"}, 0.0011909493478015065, {"__f__": "-inf"}, 0.32867366075515747, {"__f__": "-inf"}, 0.16937503218650818, {"__f__": "-inf"}, 0.8383763432502747, {"__f__": "-inf"}, 0.23325936496257782, {"__f__": "-inf"}, 0.9332694411277771, {"__f__": "-inf"}, -0.6170122623443604, {"__f__": "-inf"}, 0.8158351182937622, {"__f__": "-inf"}, -0.44799885153770447, {"__f__": "-inf"}, -1.960042953491211, {"__f__": "-inf"}, 0.12207945436239243, {"__f__": "-inf"}, -1.8079866170883179, {"__f__": "-inf"}, 0.501421332359314, {"__f__": "-inf"}, -1.3654539585113525, {"__f__": "-inf"}, 0.35076218843460083, {"__f__": "-inf"}, 0.8885220885276794, {"__f__": "-inf"}, -0.226316437125206, {"__f__": "-inf"}, 0.9783335328102112, {"__f__": "-inf"}, 0.3032347559928894, {"__f__": "-inf"}, -0.668805718421936, {"__f__": "-inf"}, -0.8357987403869629, {"__f__": "-inf"}, 0.10191008448600769, {"__f__": "-inf"}, 0.6675599813461304, {"__f__": "-inf"}, -0.5705645680427551, {"__f__": "-inf"}, 0.6084129214286804, {"__f__": "-inf"}, 2.141448974609375, {"__f__": "-inf"}, -0.16235557198524475, {"__f__": "-inf"}, 1.1121636629104614, {"__f__": "-inf"}, -1.6396980285644531, {"__f__": "-inf"}, -0.5571956634521484, {"__f__": "-inf"}, -0.26233381032943726, {"__f__": "-inf"}, 1.148213505744934, {"__f__": "-inf"}, 1.1152538061141968, {"__f__": "-inf"}, 0.22659309208393097, {"__f__": "-inf"}, -1.1112169027328491, {"__f__": "-inf"}, -0.00040965445805341005, {"__f__": "-inf"}, -0.23813261091709137, {"__f__": "-inf"}, 0.48703888058662415, {"__f__": "-inf"}, -0.9254855513572693, {"__f__": "-inf"}, -0.9759354591369629, {"__f__": "-inf"}, 0.4758020043373108, {"__f__": "-inf"}, 1.7307921648025513, {"__f__": "-inf"}, 0.07579861581325531, {"__f__": "-inf"}, 0.2042585015296936, {"__f__": "-inf"}, 0.10137693583965302, {"__f__": "-inf"}, -0.3210904002189636, {"__f__": "-inf"}, 0.3873254656791687, {"__f__": "-inf"}, 0.25833678245544434, {"__f__": "-inf"}, 2.2495386600494385, {"__f__": "-inf"}, 0.48256412148475647, {"__f__": "-inf"}, -0.6149365901947021, {"__f__": "-inf"}, 0.6443282961845398, {"__f__": "-inf"}, -1.042686939239502, {"__f__": "-inf"}, -0.18597595393657684, {"__f__": "-inf"}, 0.12131335586309433, {"__f__": "-inf"}, 1.7629410028457642, {"__f__": "-inf"}, -1.703382134437561, {"__f__": "-inf"}, 0.10267815738916397, {"__f__": "-inf"}, 0.7074564695358276, {"__f__": "-inf"}, 1.3643473386764526, {"__f__": "-inf"}, -1.1768745183944702, {"__f__": "-inf"}, 0.35378536581993103, {"__f__": "-inf"}, -0.7573941349983215, {"__f__": "-inf"}, -1.1053476333618164, {"__f__": "-inf"}, 0.8278039693832397, {"__f__": "-inf"}, 0.15727674961090088, {"__f__": "-inf"}, -0.03677097335457802, {"__f__": "-inf"}, 0.5036249160766602, {"__f__": "-inf"}, 0.07629937678575516, {"__f__": "-inf"}, 0.6720653176307678, {"__f__": "-inf"}, 0.9726060628890991, {"__f__": "-inf"}, -1.1897002458572388, {"__f__": "-inf"}, 0.45069971680641174, {"__f__": "-inf"}, -0.19711753726005554, {"__f__": "-inf"}, -0.47617819905281067, {"__f__": "-inf"}, 2.7907793521881104, {"__f__": "-inf"}, 0.13741527497768402, {"__f__": "-inf"}, -0.25766459107398987, {"__f__": "-inf"}, 0.563946008682251, {"__f__": "-inf"}, 0.830203652381897, {"__f__": "-inf"}, 0.44498923420906067, {"__f__": "-inf"}, -0.1443452686071396, {"__f__": "-inf"}, -0.3484450578689575, {"__f__": "-inf"}, 0.34161919355392456, {"__f__": "-inf"}, 0.8952907919883728, {"__f__": "-inf"}, -0.6002809405326843, {"__f__": "-inf"}, -0.5434569120407104, {"__f__": "-inf"}, -0.7249583005905151, {"__f__": "-inf"}, 0.17865067720413208, {"__f__": "-inf"}, 0.6361718773841858, {"__f__": "-inf"}, 2.856106758117676, {"__f__": "-inf"}, -0.23784074187278748, {"__f__": "-inf"}, -0.3903369903564453, {"__f__": "-inf"}, -0.6485607624053955, {"__f__": "-inf"}, 1.9117621183395386, {"__f__": "-inf"}, 1.1307923793792725, {"__f__": "-inf"}, -1.089968204498291, {"__f__": "-inf"}, -0.8596488237380981, {"__f__": "-inf"}, 0.36007949709892273, {"__f__": "-inf"}, -1.0809428691864014, {"__f__": "-inf"}, -0.1633368283510208, {"__f__": "-inf"}, -0.8268574476242065, {"__f__": "-inf"}, 0.3686165511608124, {"__f__": "-inf"}, 2.0109198093414307, {"__f__": "-inf"}, -0.04234478622674942, {"__f__": "-inf"}, -0.39276739954948425, {"__f__": "-inf"}, 0.8481754660606384, {"__f__": "-inf"}, -1.102107286453247, {"__f__": "-inf"}, -0.7659491896629333, {"__f__": "-inf"}, -0.20899657905101776, {"__f__": "-inf"}, -0.289043128490448, {"__f__": "-inf"}, 0.9750761985778809, {"__f__": "-inf"}, 0.16107365489006042, {"__f__": "-inf"}, -0.44460225105285645, {"__f__": "-inf"}, -1.750348448753357, {"__f__": "-inf"}, 0.585134744644165, {"__f__": "-inf"}, 0.38859522342681885, {"__f__": "-inf"}, 1.5008755922317505, {"__f__": "-inf"}, -0.4559735655784607, {"__f__": "-inf"}, -1.0648528337478638, {"__f__": "-inf"}, -0.6540673971176147, {"__f__": "-inf"}, -2.4209850380429998e-05, {"__f__": "-inf"}, -1.4942634105682373, {"__f__": "-inf"}, -0.2449069321155548, {"__f__": "-inf"}, -0.08954036980867386, {"__f__": "-inf"}, 0.7262442111968994, {"__f__": "-inf"}, 0.7181771397590637, {"__f__": "-inf"}, 0.4821120798587799, {"__f__": "-inf"}, -1.253129482269287, {"__f__": "-inf"}, 0.21938113868236542, {"__f__": "-inf"}, 0.6393017172813416, {"__f__": "-inf"}, 0.2777009606361389, {"__f__": "-inf"}, 0.5019185543060303, {"__f__": "-inf"}, -0.7202538251876831, {"__f__": "-inf"}, -0.4654763340950012, {"__f__": "-inf"}, -0.654046893119812, {"__f__": "-inf"}, 0.5101465582847595, {"__f__": "-inf"}, 1.8017693758010864, {"__f__": "-inf"}, 1.9507709741592407, {"__f__": "-inf"}, 0.9904482960700989, {"__f__": "-inf"}, 0.6092390418052673, {"__f__": "-inf"}, 1.6451773643493652, {"__f__": "-inf"}, 1.0664424896240234, {"__f__": "-inf"}, 1.2697384357452393, {"__f__": "-inf"}, -1.228411316871643, {"__f__": "-inf"}, -0.41896191239356995, {"__f__": "-inf"}, -1.2675856351852417, {"__f__": "-inf"}, 1.0186278820037842, {"__f__": "-inf"}, 0.6829168796539307, {"__f__": "-inf"}, -0.7008967399597168, {"__f__": "-inf"}, -1.7521623373031616, {"__f__": "-inf"}, 0.8690363168716431, {"__f__": "-inf"}, 1.9731751680374146, {"__f__": "-inf"}, 1.0977789163589478, {"__f__": "-inf"}, 0.10228242725133896, {"__f__": "-inf"}, 0.018756864592432976, {"__f__": "-inf"}, -0.5908299088478088, {"__f__": "-inf"}, 0.640724778175354, {"__f__": "-inf"}, -1.0946769714355469, {"__f__": "-inf"}, 1.3295292854309082, {"__f__": "-inf"}, 1.1623152494430542, {"__f__": "-inf"}, 0.6460853219032288, {"__f__": "-inf"}, 0.7179118990898132, {"__f__": "-inf"}, -0.9933621287345886, {"__f__": "-inf"}, 0.3372160792350769, {"__f__": "-inf"}, -0.8512228727340698, {"__f__": "-inf"}, 1.3348729610443115, {"__f__": "-inf"}, 0.3809121251106262, {"__f__": "-inf"}, -0.37149578332901, {"__f__": "-inf"}, -1.9041250944137573, {"__f__": "-inf"}, -1.4248900413513184, {"__f__": "-inf"}, 0.6022531986236572, {"__f__": "-inf"}, -0.024522820487618446, {"__f__": "-inf"}, -0.6155544519424438, {"__f__": "-inf"}, -0.7303169965744019, {"__f__": "-inf"}, -0.29751071333885193, {"__f__": "-inf"}, 0.37450820207595825, {"__f__": "-inf"}, 1.0765454769134521, {"__f__": "-inf"}, -0.6694222092628479, {"__f__": "-inf"}, -1.8298351764678955, {"__f__": "-inf"}, 0.4469994902610779, {"__f__": "-inf"}, 0.539318323135376, {"__f__": "-inf"}, -0.47252193093299866, {"__f__": "-inf"}, -0.2982392907142639, {"__f__": "-inf"}, 2.196784496307373, {"__f__": "-inf"}, 1.2383264303207397, {"__f__": "-inf"}, -0.4874134659767151, {"__f__": "-inf"}, -0.25751808285713196, {"__f__": "-inf"}, -0.3062484860420227, {"__f__": "-inf"}, 0.31152579188346863, {"__f__": "-inf"}, 0.23664024472236633, {"__f__": "-inf"}, 0.3361339271068573, {"__f__": "-inf"}, -1.1299632787704468, {"__f__": "-inf"}, 1.306781530380249, {"__f__": "-inf"}, 1.2319140434265137, {"__f__": "-inf"}, -2.2938053607940674, {"__f__": "-inf"}, 0.8819135427474976, {"__f__": "-inf"}, -0.07623136043548584, {"__f__": "-inf"}, -1.0901856422424316, {"__f__": "-inf"}, 0.2611364424228668, {"__f__": "-inf"}, -1.437781572341919, {"__f__": "-inf"}, 1.1237881183624268, {"__f__": "-inf"}, -0.4245574474334717, {"__f__": "-inf"}, -0.006552574690431356, {"__f__": "-inf"}, 0.02845979668200016, {"__f__": "-inf"}, -0.7649444937705994, {"__f__": "-inf"}, -0.16005122661590576, {"__f__": "-inf"}, 0.3993232846260071, {"__f__": "-inf"}, 1.180415153503418, {"__f__": "-inf"}, 1.5395253896713257, {"__f__": "-inf"}, -1.180469036102295, {"__f__": "-inf"}, 0.34295618534088135, {"__f__": "-inf"}, 0.7436990737915039, {"__f__": "-inf"}, 1.7328684329986572, {"__f__": "-inf"}, 0.6978336572647095, {"__f__": "-inf"}, 0.9429875016212463, {"__f__": "-inf"}, 0.9138637781143188, {"__f__": "-inf"}, -0.6032848954200745, {"__f__": "-inf"}, 1.2529011964797974, {"__f__": "-inf"}, -0.5613913536071777, {"__f__": "-inf"}, -2.520872116088867, {"__f__": "-inf"}, 0.1571771651506424, {"__f__": "-inf"}, -0.02414381317794323, {"__f__": "-inf"}, 0.7758445739746094, {"__f__": "-inf"}, 0.7459942698478699, {"__f__": "-inf"}, -0.590307354927063, {"__f__": "-inf"}, -0.4550234377384186, {"__f__": "-inf"}, 0.09710979461669922, {"__f__": "-inf"}, -2.1867899894714355, {"__f__": "-inf"}, -1.3010435104370117, {"__f__": "-inf"}, -0.1323482245206833, {"__f__": "-inf"}, -0.8608723282814026, {"__f__": "-inf"}, 0.14979100227355957, {"__f__": "-inf"}, 1.1783910989761353, {"__f__": "-inf"}, -0.4680992662906647, {"__f__": "-inf"}, 0.4296410381793976, {"__f__": "-inf"}, -0.6936913132667542, {"__f__": "-inf"}, -1.968440294265747, {"__f__": "-inf"}, -0.04329080507159233, {"__f__": "-inf"}, 0.28737831115722656, {"__f__": "-inf"}, 1.4068245887756348, {"__f__": "-inf"}, -1.3301692008972168, {"__f__": "-inf"}, 0.47833186388015747, {"__f__": "-inf"}, -0.611441969871521, {"__f__": "-inf"}, 0.5804266333580017, {"__f__": "-inf"}, -1.4131481647491455, {"__f__": "-inf"}, -0.8880813121795654, {"__f__": "-inf"}, 0.3854743540287018, {"__f__": "-inf"}, 0.6032086610794067, {"__f__": "-inf"}, -0.26615145802497864, {"__f__": "-inf"}, -1.4250239133834839, {"__f__": "-inf"}, 1.2980066537857056, {"__f__": "-inf"}, 0.3842678368091583, {"__f__": "-inf"}, -1.204788088798523, {"__f__": "-inf"}, -0.6804965734481812, {"__f__": "-inf"}, -0.6459720134735107, {"__f__": "-inf"}, 0.7052304148674011, {"__f__": "-inf"}, -0.5766420364379883, {"__f__": "-inf"}, 0.24913206696510315, {"__f__": "-inf"}, 0.6424113512039185, {"__f__": "-inf"}, -1.4752347469329834, {"__f__": "-inf"}, 0.15874400734901428, {"__f__": "-inf"}, 0.15295051038265228, {"__f__": "-inf"}, -0.4734576940536499, {"__f__": "-inf"}, -0.8595858812332153, {"__f__": "-inf"}, 0.038983169943094254, {"__f__": "-inf"}, -0.9303428530693054, {"__f__": "-inf"}, -0.27747610211372375, {"__f__": "-inf"}, 0.5119975209236145, {"__f__": "-inf"}, 0.12350836396217346, {"__f__": "-inf"}, -0.24461635947227478, {"__f__": "-inf"}, -0.719589114189148, {"__f__": "-inf"}, -0.4862928092479706, {"__f__": "-inf"}, 0.34655922651290894, {"__f__": "-inf"}, 1.0731538534164429, {"__f__": "-inf"}, -0.13066941499710083, {"__f__": "-inf"}, 0.01957579329609871, {"__f__": "-inf"}, 0.6825734972953796, {"__f__": "-inf"}, 0.5099731087684631, {"__f__": "-inf"}, 0.7590435743331909, {"__f__": "-inf"}, 0.122983917593956, {"__f__": "-inf"}, 1.843164086341858, {"__f__": "-inf"}, -0.8155293464660645, {"__f__": "-inf"}, -0.4708772897720337, {"__f__": "-inf"}, -0.8399676084518433, {"__f__": "-inf"}, -1.0093923807144165, {"__f__": "-inf"}, 0.8294666409492493, {"__f__": "-inf"}, 0.5825086236000061, {"__f__": "-inf"}, 1.1424708366394043, {"__f__": "-inf"}, 1.3458054065704346, {"__f__": "-inf"}, 1.638311743736267, {"__f__": "-inf"}, 0.6612764596939087, {"__f__": "-inf"}, -2.0474112033843994, {"__f__": "-inf"}, -0.05720864608883858, {"__f__": "-inf"}, 0.35635972023010254, {"__f__": "-inf"}, 0.07585285604000092, {"__f__": "-inf"}, -0.4687565863132477, {"__f__": "-inf"}, 0.1699233502149582, {"__f__": "-inf"}, -0.8072566986083984, {"__f__": "-inf"}, -0.4830152690410614, {"__f__": "-inf"}, -0.014767823740839958, {"__f__": "-inf"}, -0.9522486329078674, {"__f__": "-inf"}, -0.07127664983272552, {"__f__": "-inf"}, 0.38203588128089905, {"__f__": "-inf"}, -0.4575950801372528, {"__f__": "-inf"}, -1.1557401418685913, {"__f__": "-inf"}, -0.41935473680496216, {"__f__": "-inf"}, 0.6053182482719421, {"__f__": "-inf"}, -1.9951916933059692, {"__f__": "-inf"}, -0.5934111475944519, {"__f__": "-inf"}, 1.6255370378494263, {"__f__": "-inf"}, -0.8149253726005554, {"__f__": "-inf"}, 0.2724381387233734, {"__f__": "-inf"}, -0.5594436526298523, {"__f__": "-inf"}, 0.2870243489742279, {"__f__": "-inf"}, 0.47066938877105713, {"__f__": "-inf"}, 0.7792325615882874, {"__f__": "-inf"}, -1.6482030153274536, {"__f__": "-inf"}, -0.5224185585975647, {"__f__": "-inf"}, -0.13129808008670807, {"__f__": "-inf"}, -0.36059918999671936, {"__f__": "-inf"}, -1.9642823934555054, {"__f__": "-inf"}, 0.6046394109725952, {"__f__": "-inf"}, -0.004244479816406965, {"__f__": "-inf"}, -0.39741724729537964, {"__f__": "-inf"}, -0.32548806071281433, {"__f__": "-inf"}, 0.43667927384376526, {"__f__": "-inf"}, -1.4385771751403809, {"__f__": "-inf"}, 0.3399907946586609, {"__f__": "-inf"}, 1.8504917621612549, {"__f__": "-inf"}, -0.21290451288223267, {"__f__": "-inf"}, 0.6534970998764038, {"__f__": "-inf"}, 0.19306987524032593, {"__f__": "-inf"}, -0.5030452609062195, {"__f__": "-inf"}, 1.1429625749588013, {"__f__": "-inf"}, -0.25954416394233704, {"__f__": "-inf"}, -0.8175050616264343, {"__f__": "-inf"}, 1.0027927160263062, {"__f__": "-inf"}, 0.5292907953262329, {"__f__": "-inf"}, 2.0407047271728516, {"__f__": "-inf"}, -0.8404493927955627, {"__f__": "-inf"}, -0.29262876510620117, {"__f__": "-inf"}, -2.0582637786865234, {"__f__": "-inf"}, -0.35988810658454895, {"__f__": "-inf"}, 0.43550220131874084, {"__f__": "-inf"}, -0.20139062404632568, {"__f__": "-inf"}, 1.282346487045288, {"__f__": "-inf"}, 0.19513364136219025, {"__f__": "-inf"}, -0.19494150578975677, {"__f__": "-inf"}, 1.0765458345413208, {"__f__": "-inf"}, -0.4530275762081146, {"__f__": "-inf"}, 0.30319511890411377, {"__f__": "-inf"}, 1.3623874187469482, {"__f__": "-inf"}, 1.2809419631958008, {"__f__": "-inf"}, -0.21262824535369873, {"__f__": "-inf"}, -0.43411579728126526, {"__f__": "-inf"}, -1.6431455612182617, {"__f__": "-inf"}, 1.0907424688339233, {"__f__": "-inf"}, 0.045437075197696686, {"__f__": "-inf"}, 1.1516679525375366, {"__f__": "-inf"}, -1.7062383890151978, {"__f__": "-inf"}, -0.037368293851614, {"__f__": "-inf"}, 0.5134666562080383, {"__f__": "-inf"}, -0.39200636744499207, {"__f__": "-inf"}, -0.6731124520301819, {"__f__": "-inf"}, 0.40609633922576904, {"__f__": "-inf"}, 0.9066272974014282, {"__f__": "-inf"}, -0.3625313341617584, {"__f__": "-inf"}, -0.47476962208747864, {"__f__": "-inf"}, 0.9447537660598755, {"__f__": "-inf"}, -0.953292727470398, {"__f__": "-inf"}, -0.1320226788520813, {"__f__": "-inf"}, -0.7504201531410217, {"__f__": "-inf"}, -0.38203656673431396, {"__f__": "-inf"}, 1.118306279182434, {"__f__": "-inf"}, 0.9113028645515442, {"__f__": "-inf"}, 0.23566435277462006, {"__f__": "-inf"}, 0.37357738614082336, {"__f__": "-inf"}, -0.11550194025039673, {"__f__": "-inf"}, 0.5149913430213928, {"__f__": "-inf"}, -0.17552703619003296, {"__f__": "-inf"}, 0.10675184428691864, {"__f__": "-inf"}, -0.746303915977478, {"__f__": "-inf"}, -2.353645086288452, {"__f__": "-inf"}, -1.0152586698532104, {"__f__": "-inf"}, -1.2430763244628906, {"__f__": "-inf"}, 1.7108911275863647, {"__f__": "-inf"}, -0.7454031705856323, {"__f__": "-inf"}, -0.2257726490497589, {"__f__": "-inf"}, -2.127208709716797, {"__f__": "-inf"}, -0.772098183631897, {"__f__": "-inf"}, 0.18823231756687164, {"__f__": "-inf"}, 0.10338157415390015, {"__f__": "-inf"}, 0.8724315166473389, {"__f__": "-inf"}, 1.797432541847229, {"__f__": "-inf"}, -1.7289987802505493, {"__f__": "-inf"}, -0.13901907205581665, {"__f__": "-inf"}, -0.437626451253891, {"__f__": "-inf"}, 2.3804495334625244, {"__f__": "-inf"}, 1.3641514778137207, {"__f__": "-inf"}, 0.06405660510063171, {"__f__": "-inf"}, -1.3975553512573242, {"__f__": "-inf"}, 0.14246223866939545, {"__f__": "-inf"}, 0.43734320998191833, {"__f__": "-inf"}, 0.38255101442337036, {"__f__": "-inf"}, 0.412141352891922, {"__f__": "-inf"}, 0.9105792045593262, {"__f__": "-inf"}, -0.7053970694541931, {"__f__": "-inf"}, 1.7653143405914307, {"__f__": "-inf"}, 1.5666999816894531, {"__f__": "-inf"}, 0.19310960173606873, {"__f__": "-inf"}, -0.7643128037452698, {"__f__": "-inf"}, -0.8182781338691711, {"__f__": "-inf"}, -0.012844293378293514, {"__f__": "-inf"}, 2.1046500205993652, {"__f__": "-inf"}, -0.535022497177124, {"__f__": "-inf"}, -0.6824066638946533, {"__f__": "-inf"}, 1.2530710697174072, {"__f__": "-inf"}, -1.3128581047058105, {"__f__": "-inf"}, -0.46881115436553955, {"__f__": "-inf"}, 0.5126317143440247, {"__f__": "-inf"}, 0.3491878807544708, {"__f__": "-inf"}, -0.08738643676042557, {"__f__": "-inf"}, -0.5193564891815186, {"__f__": "-inf"}, -1.2765121459960938, {"__f__": "-inf"}, 0.08812452852725983, {"__f__": "-inf"}, 0.9467167854309082, {"__f__": "-inf"}, 1.0459349155426025, {"__f__": "-inf"}, -0.1801270693540573, {"__f__": "-inf"}, -1.2463868856430054, {"__f__": "-inf"}, -0.9530056118965149, {"__f__": "-inf"}, 1.140249252319336, {"__f__": "-inf"}, 1.2734951972961426, {"__f__": "-inf"}, -0.6166313290596008, {"__f__": "-inf"}, -0.20753398537635803, {"__f__": "-inf"}, -1.402363896369934, {"__f__": "-inf"}, -0.40943485498428345, {"__f__": "-inf"}, -1.1323875188827515, {"__f__": "-inf"}, 2.3356735706329346, {"__f__": "-inf"}, 0.16582553088665009, {"__f__": "-inf"}, 0.5293778777122498, {"__f__": "-inf"}, -0.8978193402290344, {"__f__": "-inf"}, -1.6030654907226562, {"__f__": "-inf"}, 2.407010555267334, {"__f__": "-inf"}, 1.6471459865570068, {"__f__": "-inf"}, 1.7909903526306152, {"__f__": "-inf"}, -1.4615906476974487, {"__f__": "-inf"}, 1.874476432800293, {"__f__": "-inf"}, -0.10721790790557861, {"__f__": "-inf"}, -1.4581464529037476, {"__f__": "-inf"}, 0.9851119518280029, {"__f__": "-inf"}, -0.09623034298419952, {"__f__": "-inf"}, 0.2947326600551605, {"__f__": "-inf"}, -0.9995104074478149, {"__f__": "-inf"}, 0.6610327959060669, {"__f__": "-inf"}, -0.5745011568069458, {"__f__": "-inf"}, 0.7319493293762207, {"__f__": "-inf"}, -1.188371181488037, {"__f__": "-inf"}, 1.4418742656707764, {"__f__": "-inf"}, 0.7496854662895203, {"__f__": "-inf"}, -0.5733516216278076, {"__f__": "-inf"}, 0.13542194664478302, {"__f__": "-inf"}, -0.35447409749031067, {"__f__": "-inf"}, 0.7360162734985352, {"__f__": "-inf"}, 0.46946951746940613, {"__f__": "-inf"}, -1.1521105766296387, {"__f__": "-inf"}, 0.7037740349769592, {"__f__": "-inf"}, 0.1583780199289322, {"__f__": "-inf"}, 2.062964916229248, {"__f__": "-inf"}, -0.060960639268159866, {"__f__": "-inf"}, 1.1186785697937012, {"__f__": "-inf"}, 1.9210494756698608, {"__f__": "-inf"}, -0.3338260352611542, {"__f__": "-inf"}, 0.756619930267334, {"__f__": "-inf"}, 0.3226832151412964, {"__f__": "-inf"}, 0.5560852289199829, {"__f__": "-inf"}, -0.14532826840877533, {"__f__": "-inf"}, 0.42327961325645447, {"__f__": "-inf"}, -0.4362733066082001, {"__f__": "-inf"}, -1.4094640016555786, {"__f__": "-inf"}, 0.22909219563007355, {"__f__": "-inf"}, -2.1250555515289307, {"__f__": "-inf"}, -0.12879173457622528, {"__f__": "-inf"}, -0.07107172906398773, {"__f__": "-inf"}, 0.0387103445827961, {"__f__": "-inf"}, 0.5096757411956787, {"__f__": "-inf"}, -0.9496592879295349, {"__f__": "-inf"}, -1.0248011350631714, {"__f__": "-inf"}, -0.16473332047462463, {"__f__": "-inf"}, 0.5877938270568848, {"__f__": "-inf"}, 0.566443145275116, {"__f__": "-inf"}, 1.311527132987976, {"__f__": "-inf"}, 0.23662538826465607, {"__f__": "-inf"}, 0.8353569507598877, {"__f__": "-inf"}, 1.4140788316726685, {"__f__": "-inf"}, -0.886796772480011, {"__f__": "-inf"}, 0.37654972076416016, {"__f__": "-inf"}, 0.40611180663108826, {"__f__": "-inf"}, 1.2531390190124512, {"__f__": "-inf"}, -1.0806353092193604, {"__f__": "-inf"}, 0.31028276681900024, {"__f__": "-inf"}, 1.8836053609848022, {"__f__": "-inf"}, 0.7813805341720581, {"__f__": "-inf"}, -0.7585296630859375, {"__f__": "-inf"}, -0.8063679933547974, {"__f__": "-inf"}, -0.14807797968387604, {"__f__": "-inf"}, 0.22196954488754272, {"__f__": "-inf"}, 0.6759764552116394, {"__f__": "-inf"}, 1.622282862663269, {"__f__": "-inf"}, 0.8591530323028564, {"__f__": "-inf"}, 0.4289686977863312, {"__f__": "-inf"}, -1.5243138074874878, {"__f__": "-inf"}, 0.38003405928611755, {"__f__": "-inf"}, 0.8674342632293701, {"__f__": "-inf"}, -1.4071406126022339, {"__f__": "-inf"}, 2.239070177078247, {"__f__": "-inf"}, 0.35601529479026794, {"__f__": "-inf"}, 0.4448168873786926, {"__f__": "-inf"}, 3.5505404472351074, {"__f__": "-inf"}, 1.2722818851470947, {"__f__": "-inf"}, -0.2918980121612549, {"__f__": "-inf"}, -0.6202453374862671, {"__f__": "-inf"}, 1.3236671686172485, {"__f__": "-inf"}, 1.6753809452056885, {"__f__": "-inf"}, 0.6312750577926636, {"__f__": "-inf"}, -0.7799338698387146, {"__f__": "-inf"}, -0.4885495603084564, {"__f__": "-inf"}, 0.5016099810600281, {"__f__": "-inf"}, -0.058504242449998856, {"__f__": "-inf"}, 1.574791669845581, {"__f__": "-inf"}, 0.12745606899261475, {"__f__": "-inf"}, -1.1826410293579102, {"__f__": "-inf"}, 0.5474725961685181, {"__f__": "-inf"}, 0.5420634746551514, {"__f__": "-inf"}, -0.07401915639638901, {"__f__": "-inf"}, 1.308016300201416, {"__f__": "-inf"}, 0.18963085114955902, {"__f__": "-inf"}, -1.327994465827942, {"__f__": "-inf"}, -1.432309865951538, {"__f__": "-inf"}, -0.18276217579841614, {"__f__": "-inf"}, -1.054782748222351, {"__f__": "-inf"}, 1.343076467514038, {"__f__": "-inf"}, 0.3066347539424896, {"__f__": "-inf"}, 0.3666762709617615, {"__f__": "-inf"}, -0.6469232439994812, {"__f__": "-inf"}, -1.1264668703079224, {"__f__": "-inf"}, -0.3760025203227997, {"__f__": "-inf"}, 1.2130658626556396, {"__f__": "-inf"}, 0.6298661231994629, {"__f__": "-inf"}, -1.250694990158081, {"__f__": "-inf"}, -0.8390555381774902, {"__f__": "-inf"}, -0.1188158318400383, {"__f__": "-inf"}, -0.0947219654917717, {"__f__": "-inf"}, -1.4229562282562256, {"__f__": "-inf"}, -0.9212255477905273, {"__f__": "-inf"}, 0.03267185762524605, {"__f__": "-inf"}, 0.5932673215866089, {"__f__": "-inf"}, 0.8945361375808716, {"__f__": "-inf"}, -0.5443627834320068, {"__f__": "-inf"}, -1.2237577438354492, {"__f__": "-inf"}, 0.13743789494037628, {"__f__": "-inf"}, -0.9171178340911865, {"__f__": "-inf"}, -0.017510244622826576, {"__f__": "-inf"}, 0.5929837226867676, {"__f__": "-inf"}, 0.588448703289032, {"__f__": "-inf"}, 1.3397784233093262, {"__f__": "-inf"}, 1.4834461212158203, {"__f__": "-inf"}, 1.0256928205490112, {"__f__": "-inf"}, -1.4354451894760132, {"__f__": "-inf"}, -0.03355515003204346, {"__f__": "-inf"}, 0.656286895275116, {"__f__": "-inf"}, -1.7099742889404297, {"__f__": "-inf"}, -0.6431142091751099, {"__f__": "-inf"}, 0.6553511023521423, {"__f__": "-inf"}, -0.5873875021934509, {"__f__": "-inf"}, -0.7975363731384277, {"__f__": "-inf"}, 0.5072416067123413, {"__f__": "-inf"}, -0.1811928153038025, {"__f__": "-inf"}, 1.543885588645935, {"__f__": "-inf"}, -0.7034510970115662, {"__f__": "-inf"}, 2.909769296646118, {"__f__": "-inf"}, 1.404311180114746, {"__f__": "-inf"}, -1.5974981784820557, {"__f__": "-inf"}, 0.40691861510276794, {"__f__": "-inf"}, 0.47539353370666504, {"__f__": "-inf"}, 0.002148998901247978, {"__f__": "-inf"}, -1.7215548753738403, {"__f__": "-inf"}, 2.204847812652588, {"__f__": "-inf"}, -1.1336299180984497, {"__f__": "-inf"}, -1.5024914741516113, {"__f__": "-inf"}, 2.016195297241211, {"__f__": "-inf"}, -1.182982087135315, {"__f__": "-inf"}, 1.165592908859253, {"__f__": "-inf"}, -0.14357328414916992, {"__f__": "-inf"}, -2.308856248855591, {"__f__": "-inf"}, -0.6065680980682373, {"__f__": "-inf"}, 0.3518909513950348, {"__f__": "-inf"}, -0.3297997713088989, {"__f__": "-inf"}, 0.6998968720436096, {"__f__": "-inf"}, 0.1391880214214325, {"__f__": "-inf"}, 0.25633546710014343, {"__f__": "-inf"}, -0.7430131435394287, {"__f__": "-inf"}, -1.32436203956604, {"__f__": "-inf"}, 1.5220259428024292, {"__f__": "-inf"}, 0.3552003502845764, {"__f__": "-inf"}, -2.188948154449463, {"__f__": "-inf"}, 1.234950304031372, {"__f__": "-inf"}, -0.5142840147018433, {"__f__": "-inf"}, -0.5748049020767212, {"__f__": "-inf"}, -0.3187241852283478, {"__f__": "-inf"}, -0.6706148982048035, {"__f__": "-inf"}, -0.0760548859834671, {"__f__": "-inf"}, -2.5455710887908936, {"__f__": "-inf"}, -0.7571995854377747, {"__f__": "-inf"}, -0.35593029856681824, {"__f__": "-inf"}, -0.6405986547470093, {"__f__": "-inf"}, -0.6862429976463318, {"__f__": "-inf"}, -1.321633219718933, {"__f__": "-inf"}, -1.022551417350769, {"__f__": "-inf"}, -1.4511477947235107, {"__f__": "-inf"}, 0.9127172827720642, {"__f__": "-inf"}, -0.853212833404541, {"__f__": "-inf"}, -0.7056829929351807, {"__f__": "-inf"}, 1.5501623153686523, {"__f__": "-inf"}, 0.739793598651886, {"__f__": "-inf"}, 1.3502326011657715, {"__f__": "-inf"}, 1.6161566972732544, {"__f__": "-inf"}, -0.5958281755447388, {"__f__": "-inf"}, -0.19731110334396362, {"__f__": "-inf"}, -0.4170260727405548, {"__f__": "-inf"}, -0.0018597646849229932, {"__f__": "-inf"}, -0.13918930292129517, {"__f__": "-inf"}, -1.9850106239318848, {"__f__": "-inf"}, -0.05624749884009361, {"__f__": "-inf"}, -1.4347219467163086, {"__f__": "-inf"}, -0.832792341709137, {"__f__": "-inf"}, -0.2833198308944702, {"__f__": "-inf"}, -0.3253123164176941, {"__f__": "-inf"}, -0.6930993795394897, {"__f__": "-inf"}, 2.3097758293151855, {"__f__": "-inf"}, 0.322182834148407, {"__f__": "-inf"}, -1.1599278450012207, {"__f__": "-inf"}, 0.7054274678230286, {"__f__": "-inf"}, -0.46605682373046875, {"__f__": "-inf"}, 1.0035927295684814, {"__f__": "-inf"}, 0.16927486658096313, {"__f__": "-inf"}, -0.6874349117279053, {"__f__": "-inf"}, 0.33773180842399597, {"__f__": "-inf"}, 1.2171269655227661, {"__f__": "-inf"}, 0.3685130774974823, {"__f__": "-inf"}, 1.305492877960205, {"__f__": "-inf"}, -0.4368577301502228, {"__f__": "-inf"}, 0.7603883743286133, {"__f__": "-inf"}, 1.1615291833877563, {"__f__": "-inf"}, -1.3381396532058716, {"__f__": "-inf"}, 0.7216427326202393, {"__f__": "-inf"}, 0.14467290043830872, {"__f__": "-inf"}, -0.31997090578079224, {"__f__": "-inf"}, 0.3288778066635132, {"__f__": "-inf"}, -0.9100295901298523, {"__f__": "-inf"}, 0.4090782105922699, {"__f__": "-inf"}, 1.2709314823150635, {"__f__": "-inf"}, 1.529801607131958, {"__f__": "-inf"}, 0.8657345175743103, {"__f__": "-inf"}, 0.7457234263420105, {"__f__": "-inf"}, 0.3470625877380371, {"__f__": "-inf"}, 0.1829739660024643, {"__f__": "-inf"}, 2.0078439712524414, {"__f__": "-inf"}, -0.6939875483512878, {"__f__": "-inf"}, -1.0233169794082642, {"__f__": "-inf"}, -0.6823147535324097, {"__f__": "-inf"}, -1.385590672492981, {"__f__": "-inf"}, -0.3814418613910675, {"__f__": "-inf"}, -0.4051353633403778, {"__f__": "-inf"}, -1.0615251064300537, {"__f__": "-inf"}, 0.5500848889350891, {"__f__": "-inf"}, 0.6198712587356567, {"__f__": "-inf"}, 0.6096224188804626, {"__f__": "-inf"}, -0.051655497401952744, {"__f__": "-inf"}, 0.266745924949646, {"__f__": "-inf"}, 0.4208816587924957, {"__f__": "-inf"}, -1.7538996934890747, {"__f__": "-inf"}, -0.6497073769569397, {"__f__": "-inf"}, -1.773227572441101, {"__f__": "-inf"}, 0.19126400351524353, {"__f__": "-inf"}, 0.8112546801567078, {"__f__": "-inf"}, -0.49932411313056946, {"__f__": "-inf"}, 1.6514414548873901, {"__f__": "-inf"}, -0.6994465589523315, {"__f__": "-inf"}, 1.4401302337646484, {"__f__": "-inf"}, 1.425451636314392, {"__f__": "-inf"}, -0.2155853509902954, {"__f__": "-inf"}, -0.49755191802978516, {"__f__": "-inf"}, 0.1736108660697937, {"__f__": "-inf"}, -0.8871750235557556, {"__f__": "-inf"}, -0.39173805713653564, {"__f__": "-inf"}, -0.2129078358411789, {"__f__": "-inf"}, 0.3953252136707306, {"__f__": "-inf"}, -1.433894395828247, {"__f__": "-inf"}, 0.6566073894500732, {"__f__": "-inf"}, -0.12633761763572693, {"__f__": "-inf"}, -1.3936816453933716, {"__f__": "-inf"}, -1.0054851770401, {"__f__": "-inf"}, 1.1686896085739136, {"__f__": "-inf"}, 1.8668614625930786, {"__f__": "-inf"}, 0.4729079604148865, {"__f__": "-inf"}, -0.49858659505844116, {"__f__": "-inf"}, -0.7599255442619324, {"__f__": "-inf"}, 0.35348373651504517, {"__f__": "-inf"}, 0.9727408289909363, {"__f__": "-inf"}, 1.1282203197479248, {"__f__": "-inf"}, -1.8735772371292114, {"__f__": "-inf"}, -0.4432128965854645, {"__f__": "-inf"}, 0.6178740859031677, {"__f__": "-inf"}, 0.6735584139823914, {"__f__": "-inf"}, -0.2990979850292206, {"__f__": "-inf"}, -0.6867149472236633, {"__f__": "-inf"}, 0.2130356878042221, {"__f__": "-inf"}, 1.50713312625885, {"__f__": "-inf"}, 1.6749800443649292, {"__f__": "-inf"}, -0.5689507722854614, {"__f__": "-inf"}, 1.9691256284713745, {"__f__": "-inf"}, -1.8978677988052368, {"__f__": "-inf"}, -0.7321833968162537, {"__f__": "-inf"}, -1.2314326763153076, {"__f__": "-inf"}, 0.22968651354312897, {"__f__": "-inf"}, 0.3961753845214844, {"__f__": "-inf"}, 0.21889451146125793, {"__f__": "-inf"}, -1.3209666013717651, {"__f__": "-inf"}, 1.5792962312698364, {"__f__": "-inf"}, 0.6031094789505005, {"__f__": "-inf"}, 1.0260447263717651, {"__f__": "-inf"}, 0.14580902457237244, {"__f__": "-inf"}, 0.1372915357351303, {"__f__": "-inf"}, 0.3746272623538971, {"__f__": "-inf"}, 0.3950577974319458, {"__f__": "-inf"}, 0.49263039231300354, {"__f__": "-inf"}, -0.6824504733085632, {"__f__": "-inf"}, -0.7735494375228882, {"__f__": "-inf"}, -0.2066897600889206, {"__f__": "-inf"}, 0.7240104675292969, {"__f__": "-inf"}, 0.8426957130432129, {"__f__": "-inf"}, -2.2601640224456787, {"__f__": "-inf"}, 1.023162603378296, {"__f__": "-inf"}, 2.1318697929382324, {"__f__": "-inf"}, -0.010972298681735992, {"__f__": "-inf"}, -0.05416432395577431, {"__f__": "-inf"}, 0.9457464814186096, {"__f__": "-inf"}, 1.4675862789154053, {"__f__": "-inf"}, -0.3983084559440613, {"__f__": "-inf"}, 1.8338133096694946, {"__f__": "-inf"}, 1.6140360832214355, {"__f__": "-inf"}, -0.05455249920487404, {"__f__": "-inf"}, -0.6332985162734985, {"__f__": "-inf"}, 0.23424215614795685, {"__f__": "-inf"}, -0.25882625579833984, {"__f__": "-inf"}, 1.389542818069458, {"__f__": "-inf"}, 2.214151382446289, {"__f__": "-inf"}, 0.394875168800354, {"__f__": "-inf"}, -0.06925416737794876, {"__f__": "-inf"}, -0.9130293726921082, {"__f__": "-inf"}, 0.4563741683959961, {"__f__": "-inf"}, -1.3387540578842163, {"__f__": "-inf"}, -0.25635287165641785, {"__f__": "-inf"}, -0.5838838815689087, {"__f__": "-inf"}, 0.11821714043617249, {"__f__": "-inf"}, -0.2823869585990906, {"__f__": "-inf"}, 0.7471506595611572, {"__f__": "-inf"}, 0.17569145560264587, {"__f__": "-inf"}, -1.0771209001541138, {"__f__": "-inf"}, -0.24781523644924164, {"__f__": "-inf"}, -3.074908971786499, {"__f__": "-inf"}, 0.15069974958896637, {"__f__": "-inf"}, -1.3895952701568604, {"__f__": "-inf"}, 0.3092142343521118, {"__f__": "-inf"}, 0.8116085529327393, {"__f__": "-inf"}, -0.1081196516752243, {"__f__": "-inf"}, -2.4626028537750244, {"__f__": "-inf"}, 1.0020934343338013, {"__f__": "-inf"}, -0.14860044419765472, {"__f__": "-inf"}, -0.9030683636665344, {"__f__": "-inf"}, -0.30191701650619507, {"__f__": "-inf"}, -0.4176281690597534, {"__f__": "-inf"}, -1.200134515762329, {"__f__": "-inf"}, -0.9093834161758423, {"__f__": "-inf"}, -0.14844541251659393, {"__f__": "-inf"}, -0.8642789721488953, {"__f__": "-inf"}, -0.2789013087749481, {"__f__": "-inf"}, -1.4636682271957397, {"__f__": "-inf"}, 0.6441859006881714, {"__f__": "-inf"}, 1.154948115348816, {"__f__": "-inf"}, 0.36384138464927673, {"__f__": "-inf"}, 1.6400264501571655, {"__f__": "-inf"}, -0.011101443320512772, {"__f__": "-inf"}, -1.59878671169281, {"__f__": "-inf"}, -0.5187939405441284, {"__f__": "-inf"}, -0.0866319090127945, {"__f__": "-inf"}, -1.0139527320861816, {"__f__": "-inf"}, 1.2300690412521362, {"__f__": "-inf"}, -0.12482073903083801, {"__f__": "-inf"}, 0.5975925922393799, {"__f__": "-inf"}, -0.6156625747680664, {"__f__": "-inf"}, -0.9573548436164856, {"__f__": "-inf"}, 0.31927627325057983, {"__f__": "-inf"}, -0.5961734056472778, {"__f__": "-inf"}, -1.4091354608535767, {"__f__": "-inf"}, -1.4846434593200684, {"__f__": "-inf"}, 0.8503133654594421, {"__f__": "-inf"}, -0.22032716870307922, {"__f__": "-inf"}, -1.6728019714355469, {"__f__": "-inf"}, 0.539341926574707, {"__f__": "-inf"}, 0.5265356302261353, {"__f__": "-inf"}, -1.6949676275253296, {"__f__": "-inf"}, 2.169114112854004, {"__f__": "-inf"}, 0.6886752247810364, {"__f__": "-inf"}, -0.9171911478042603, {"__f__": "-inf"}, -0.25976377725601196, {"__f__": "-inf"}, -0.7981937527656555, {"__f__": "-inf"}, 1.3535338640213013, {"__f__": "-inf"}, 0.2581557333469391, {"__f__": "-inf"}, 1.599216341972351, {"__f__": "-inf"}, -1.635155200958252, {"__f__": "-inf"}, 0.3104408085346222, {"__f__": "-inf"}, -1.826346516609192, {"__f__": "-inf"}, -0.5535962581634521, {"__f__": "-inf"}, 1.8695447444915771, {"__f__": "-inf"}, -0.3002869784832001, {"__f__": "-inf"}, 1.884597897529602, {"__f__": "-inf"}, -1.9402486085891724, {"__f__": "-inf"}, -0.014700864441692829, {"__f__": "-inf"}, 0.04853550344705582, {"__f__": "-inf"}, 0.2842799723148346, {"__f__": "-inf"}, -0.46124696731567383, {"__f__": "-inf"}, -0.8256766200065613, {"__f__": "-inf"}, 0.9614735841751099, {"__f__": "-inf"}, -2.152461051940918, {"__f__": "-inf"}, 1.121213436126709, {"__f__": "-inf"}, -0.5083461999893188, {"__f__": "-inf"}, 0.5415846109390259, {"__f__": "-inf"}, -0.2721284031867981, {"__f__": "-inf"}, 0.2442476898431778, {"__f__": "-inf"}, 1.1272926330566406, {"__f__": "-inf"}, -0.8398743271827698, {"__f__": "-inf"}, -1.7881066799163818, {"__f__": "-inf"}, -1.0879793167114258, {"__f__": "-inf"}, -0.013451785780489445, {"__f__": "-inf"}, -1.6180052757263184, {"__f__": "-inf"}, -1.1171633005142212, {"__f__": "-inf"}, 1.5226258039474487, {"__f__": "-inf"}, 0.8033534288406372, {"__f__": "-inf"}, 0.8011569976806641, {"__f__": "-inf"}, 0.44643497467041016, {"__f__": "-inf"}, -0.5431615710258484, {"__f__": "-inf"}, 0.45582687854766846, {"__f__": "-inf"}, 0.8112953901290894, {"__f__": "-inf"}, -0.08210965245962143, {"__f__": "-inf"}, -2.099072217941284, {"__f__": "-inf"}, 0.8536061644554138, {"__f__": "-inf"}, -0.010280419141054153, {"__f__": "-inf"}, 0.8284920454025269, {"__f__": "-inf"}, 0.6570068597793579, {"__f__": "-inf"}, -0.11992251873016357, {"__f__": "-inf"}, 0.7093120217323303, {"__f__": "-inf"}, -0.3307735323905945, {"__f__": "-inf"}, -1.4225555658340454, {"__f__": "-inf"}, -0.17918634414672852, {"__f__": "-inf"}, 0.90601646900177, {"__f__": "-inf"}, -2.1137475967407227, {"__f__": "-inf"}, -1.5820573568344116, {"__f__": "-inf"}, -0.3187393248081207, {"__f__": "-inf"}, -0.7423120141029358, {"__f__": "-inf"}, 0.5540120005607605, {"__f__": "-inf"}, 1.2686530351638794, {"__f__": "-inf"}, -0.7970882654190063, {"__f__": "-inf"}, -0.1613210290670395, {"__f__": "-inf"}, 0.8801825642585754, {"__f__": "-inf"}, -0.49923238158226013, {"__f__": "-inf"}, -1.4837712049484253, {"__f__": "-inf"}, -1.301725149154663, {"__f__": "-inf"}, -0.1811903417110443, {"__f__": "-inf"}, -1.5122369527816772, {"__f__": "-inf"}, -0.5934948921203613, {"__f__": "-inf"}, 0.8215594291687012, {"__f__": "-inf"}, -0.47225382924079895, {"__f__": "-inf"}, 0.8773334622383118, {"__f__": "-inf"}, -1.7273231744766235, {"__f__": "-inf"}, 1.0052834749221802, {"__f__": "-inf"}, -0.3189263641834259, {"__f__": "-inf"}, 0.2633325457572937, {"__f__": "-inf"}, 0.5796754956245422, {"__f__": "-inf"}, 0.5188369750976562, {"__f__": "-inf"}, 0.893561601638794, {"__f__": "-inf"}, -0.397430419921875, {"__f__": "-inf"}, 0.027715127915143967, {"__f__": "-inf"}, 1.3704822063446045, {"__f__": "-inf"}, -0.3076222836971283, {"__f__": "-inf"}, 0.8446739315986633, {"__f__": "-inf"}, -0.74770587682724, {"__f__": "-inf"}, 0.5251597166061401, {"__f__": "-inf"}, -0.899270236492157, {"__f__": "-inf"}, 0.1577979028224945, {"__f__": "-inf"}, 1.4592435359954834, {"__f__": "-inf"}, 0.9733019471168518, {"__f__": "-inf"}, -0.4613094627857208, {"__f__": "-inf"}, 0.30035340785980225, {"__f__": "-inf"}, 0.10346224904060364, {"__f__": "-inf"}, -0.9510462880134583, {"__f__": "-inf"}, -0.1377267688512802, {"__f__": "-inf"}, -2.3773748874664307, {"__f__": "-inf"}, 2.0380313396453857, {"__f__": "-inf"}, 2.87485671043396, {"__f__": "-inf"}, 0.19147813320159912, {"__f__": "-inf"}, 0.6144981980323792, {"__f__": "-inf"}, 1.6322684288024902, {"__f__": "-inf"}, -0.521202564239502, {"__f__": "-inf"}, -0.3755978047847748, {"__f__": "-inf"}, -0.576674222946167, {"__f__": "-inf"}, -1.2097513675689697, {"__f__": "-inf"}, 0.23442427814006805, {"__f__": "-inf"}, -0.017945220693945885, {"__f__": "-inf"}, 0.4852231442928314, {"__f__": "-inf"}, -0.6861541271209717, {"__f__": "-inf"}, 0.7005665302276611, {"__f__": "-inf"}, -0.9003230929374695, {"__f__": "-inf"}, -0.234209343791008, {"__f__": "-inf"}, -0.08878661692142487, {"__f__": "-inf"}, -0.15636248886585236, {"__f__": "-inf"}, -1.3793524503707886, {"__f__": "-inf"}, 1.6369136571884155, {"__f__": "-inf"}, -1.598835825920105, {"__f__": "-inf"}, -0.3944509029388428, {"__f__": "-inf"}, -0.2493932843208313, {"__f__": "-inf"}, -2.039626121520996, {"__f__": "-inf"}, 1.0143095254898071, {"__f__": "-inf"}, -0.7606303691864014, {"__f__": "-inf"}, 0.39140456914901733, {"__f__": "-inf"}, 0.8058751225471497, {"__f__": "-inf"}, 0.4040883183479309, {"__f__": "-inf"}, 0.3886395990848541, {"__f__": "-inf"}, 0.891227126121521, {"__f__": "-inf"}, 0.8413550853729248, {"__f__": "-inf"}, -0.31835654377937317, {"__f__": "-inf"}, -0.052590031176805496, {"__f__": "-inf"}, 0.03992096334695816, {"__f__": "-inf"}, -1.462044358253479, {"__f__": "-inf"}, -0.8526747226715088, {"__f__": "-inf"}, 1.879117488861084, {"__f__": "-inf"}, -1.260949730873108, {"__f__": "-inf"}, 1.147182583808899, {"__f__": "-inf"}, 0.5216277241706848, {"__f__": "-inf"}, -0.2552844285964966, {"__f__": "-inf"}, -0.833158016204834, {"__f__": "-inf"}, 0.06089455634355545, {"__f__": "-inf"}, -2.158966064453125, {"__f__": "-inf"}, 0.10379746556282043, {"__f__": "-inf"}, -0.5182905793190002, {"__f__": "-inf"}, -1.2552530765533447, {"__f__": "-inf"}, -0.20641019940376282, {"__f__": "-inf"}, 1.9092984199523926, {"__f__": "-inf"}, 1.3951054811477661, {"__f__": "-inf"}, -0.6865208148956299, {"__f__": "-inf"}, 0.6928232908248901, {"__f__": "-inf"}, 1.084190845489502, {"__f__": "-inf"}, 0.04529919847846031, {"__f__": "-inf"}, -0.7179672718048096, {"__f__": "-inf"}, 1.5218634605407715, {"__f__": "-inf"}, 0.19089141488075256, {"__f__": "-inf"}, -0.20259523391723633, {"__f__": "-inf"}, 2.816589593887329, {"__f__": "-inf"}, -0.49495890736579895, {"__f__": "-inf"}, 0.7073925137519836, {"__f__": "-inf"}, 0.6022928953170776, {"__f__": "-inf"}, -0.3945365846157074, {"__f__": "-inf"}, -0.16803127527236938, {"__f__": "-inf"}, 0.4875323176383972, {"__f__": "-inf"}, -1.4025403261184692, {"__f__": "-inf"}, -1.0093835592269897, {"__f__": "-inf"}, -0.10675159096717834, {"__f__": "-inf"}, 0.9524750709533691, {"__f__": "-inf"}, -1.05696702003479, {"__f__": "-inf"}, -0.5415336489677429, {"__f__": "-inf"}, -0.4829573333263397, {"__f__": "-inf"}, -0.7993463277816772, {"__f__": "-inf"}, -0.31043142080307007, {"__f__": "-inf"}, -0.03928004950284958, {"__f__": "-inf"}, -1.618641972541809, {"__f__": "-inf"}, 0.9995518922805786, {"__f__": "-inf"}, 0.23581339418888092, {"__f__": "-inf"}, -0.48346227407455444, {"__f__": "-inf"}, 0.21770818531513214, {"__f__": "-inf"}, -1.0658901929855347, {"__f__": "-inf"}, 0.8611325621604919, {"__f__": "-inf"}, 2.1747703552246094, {"__f__": "-inf"}, -1.1487020254135132, {"__f__": "-inf"}, 2.8280441761016846, {"__f__": "-inf"}, -0.20675785839557648, {"__f__": "-inf"}, 1.1876634359359741, {"__f__": "-inf"}, 0.18671338260173798, {"__f__": "-inf"}, -1.0000276565551758, {"__f__": "-inf"}, -0.8196762800216675, {"__f__": "-inf"}, -2.1889894008636475, {"__f__": "-inf"}, -0.19856911897659302, {"__f__": "-inf"}, 0.20887050032615662, {"__f__": "-inf"}, 0.38308414816856384, {"__f__": "-inf"}, 1.0448247194290161, {"__f__": "-inf"}, 0.2863808274269104, {"__f__": "-inf"}, 0.5569362640380859, {"__f__": "-inf"}, -0.5066768527030945, {"__f__": "-inf"}, -1.0567835569381714, {"__f__": "-inf"}, -0.6742801070213318, {"__f__": "-inf"}, 0.13065370917320251, {"__f__": "-inf"}, -0.8361381888389587, {"__f__": "-inf"}, 0.007882463745772839, {"__f__": "-inf"}, 0.39627981185913086, {"__f__": "-inf"}, -0.4900519549846649, {"__f__": "-inf"}, 0.564247727394104, {"__f__": "-inf"}, 0.3984590470790863, {"__f__": "-inf"}, 0.9502319097518921, {"__f__": "-inf"}, 0.5775466561317444, {"__f__": "-inf"}, -0.2264994978904724, {"__f__": "-inf"}, -1.049934983253479, {"__f__": "-inf"}, -0.9605152010917664, {"__f__": "-inf"}, -0.9317734837532043, {"__f__": "-inf"}, 1.7739067077636719, {"__f__": "-inf"}, 0.010722112841904163, {"__f__": "-inf"}, 1.0883861780166626, {"__f__": "-inf"}, 0.6503353118896484, {"__f__": "-inf"}, 1.2849640846252441, {"__f__": "-inf"}, -0.7390024065971375, {"__f__": "-inf"}, -1.935837984085083, {"__f__": "-inf"}, -1.2498222589492798, {"__f__": "-inf"}, 0.637969970703125, {"__f__": "-inf"}, 0.9736451506614685, {"__f__": "-inf"}, 0.7375224232673645, {"__f__": "-inf"}, -0.03603072091937065, {"__f__": "-inf"}, 0.6737213134765625, {"__f__": "-inf"}, -2.334204912185669, {"__f__": "-inf"}, -0.44830387830734253, {"__f__": "-inf"}, -1.6163136959075928, {"__f__": "-inf"}, -0.30548974871635437, {"__f__": "-inf"}, 0.05670590326189995, {"__f__": "-inf"}, -0.9143040776252747, {"__f__": "-inf"}, -2.026378870010376, {"__f__": "-inf"}, -0.2769443690776825, {"__f__": "-inf"}, -0.6932074427604675, {"__f__": "-inf"}, 0.4047513008117676, {"__f__": "-inf"}, 0.8583757281303406, {"__f__": "-inf"}, 0.9914923906326294, {"__f__": "-inf"}, 1.310609221458435, {"__f__": "-inf"}, -0.7804996967315674, {"__f__": "-inf"}, -0.8779751658439636, {"__f__": "-inf"}, 0.35089391469955444, {"__f__": "-inf"}, -1.6377602815628052, {"__f__": "-inf"}, -0.4564994275569916, {"__f__": "-inf"}, -0.9036184549331665, {"__f__": "-inf"}, -2.072610378265381, {"__f__": "-inf"}, -0.3254818320274353, {"__f__": "-inf"}, 0.6332396268844604, {"__f__": "-inf"}, 0.6075770258903503, {"__f__": "-inf"}, 1.5069568157196045, {"__f__": "-inf"}, 1.6419633626937866, {"__f__": "-inf"}, -1.161836862564087, {"__f__": "-inf"}, -0.043761782348155975, {"__f__": "-inf"}, -1.1885088682174683, {"__f__": "-inf"}, 0.33484601974487305, {"__f__": "-inf"}, 0.980628252029419, {"__f__": "-inf"}, 1.3381363153457642, {"__f__": "-inf"}, 0.6370581984519958, {"__f__": "-inf"}, 0.7687059044837952, {"__f__": "-inf"}, 1.6319880485534668, {"__f__": "-inf"}, -1.7710635662078857, {"__f__": "-inf"}, -1.0947632789611816, {"__f__": "-inf"}, 0.07455074042081833, {"__f__": "-inf"}, 0.21939194202423096, {"__f__": "-inf"}, 1.3507362604141235, {"__f__": "-inf"}, 0.18939390778541565, {"__f__": "-inf"}, 0.15852001309394836, {"__f__": "-inf"}, 1.0454360246658325, {"__f__": "-inf"}, 0.17528215050697327, {"__f__": "-inf"}, 0.8217769265174866, {"__f__": "-inf"}, -0.161952406167984, {"__f__": "-inf"}, -0.5194015502929688, {"__f__": "-inf"}, 0.3894425332546234, {"__f__": "-inf"}, -1.0745469331741333, {"__f__": "-inf"}, 0.4039126932621002, {"__f__": "-inf"}, 0.7556086778640747, {"__f__": "-inf"}, 1.0669101476669312, {"__f__": "-inf"}, -1.648539662361145, {"__f__": "-inf"}, -1.314535140991211, {"__f__": "-inf"}, 0.2651357054710388, {"__f__": "-inf"}, -0.6454184651374817, {"__f__": "-inf"}, 0.7146685123443604, {"__f__": "-inf"}, 0.35730820894241333, {"__f__": "-inf"}, -1.4180418252944946, {"__f__": "-inf"}, -1.198647379875183, {"__f__": "-inf"}, 0.8285083770751953, {"__f__": "-inf"}, -0.01766456663608551, {"__f__": "-inf"}, 0.26167792081832886, {"__f__": "-inf"}, -0.3974379897117615, {"__f__": "-inf"}, 0.2072213888168335, {"__f__": "-inf"}, -0.13371379673480988, {"__f__": "-inf"}, 0.6598846912384033, {"__f__": "-inf"}, -1.165724515914917, {"__f__": "-inf"}, 0.2779606282711029, {"__f__": "-inf"}, 0.15062178671360016, {"__f__": "-inf"}, -0.874881386756897, {"__f__": "-inf"}, 0.8214787244796753, {"__f__": "-inf"}, -0.1386103481054306, {"__f__": "-inf"}, 0.7765745520591736, {"__f__": "-inf"}, -2.440110683441162, {"__f__": "-inf"}, 0.07038132101297379, {"__f__": "-inf"}, -1.4400452375411987, {"__f__": "-inf"}, -0.038069941103458405, {"__f__": "-inf"}, -0.8664109110832214, {"__f__": "-inf"}, 0.04992581903934479, {"__f__": "-inf"}, -0.3074679374694824, {"__f__": "-inf"}, 0.5024350881576538, {"__f__": "-inf"}, 1.4138500690460205, {"__f__": "-inf"}, 0.5988901257514954, {"__f__": "-inf"}, 0.027059264481067657, {"__f__": "-inf"}, 0.4952591359615326, {"__f__": "-inf"}, -1.5342462062835693, {"__f__": "-inf"}, 0.9415639638900757, {"__f__": "-inf"}, -0.17129981517791748, {"__f__": "-inf"}, -0.5149053335189819, {"__f__": "-inf"}, -0.3620770573616028, {"__f__": "-inf"}, -1.8583124876022339, {"__f__": "-inf"}, 1.2437498569488525, {"__f__": "-inf"}, -0.6984595656394958, {"__f__": "-inf"}, -0.12925782799720764, {"__f__": "-inf"}, -1.5765588283538818, {"__f__": "-inf"}, -1.577419638633728, {"__f__": "-inf"}, -1.6867177486419678, {"__f__": "-inf"}, -0.9854783415794373, {"__f__": "-inf"}, -2.6654090881347656, {"__f__": "-inf"}, -0.2231437712907791, {"__f__": "-inf"}, -0.005989604163914919, {"__f__": "-inf"}, -0.05387880653142929, {"__f__": "-inf"}, 0.2319667786359787, {"__f__": "-inf"}, 0.7666008472442627, {"__f__": "-inf"}, 0.8524263501167297, {"__f__": "-inf"}, 0.5951335430145264, {"__f__": "-inf"}, -1.4397214651107788, {"__f__": "-inf"}, 1.1222032308578491, {"__f__": "-inf"}, -1.0353668928146362, {"__f__": "-inf"}, -0.7883328795433044, {"__f__": "-inf"}, -0.6913573741912842, {"__f__": "-inf"}, 0.12660716474056244, {"__f__": "-inf"}, 0.2952670753002167, {"__f__": "-inf"}, 0.5501588582992554, {"__f__": "-inf"}, 0.005515996366739273, {"__f__": "-inf"}, 1.7400323152542114, {"__f__": "-inf"}, 0.0847441777586937, {"__f__": "-inf"}, 0.35461583733558655, {"__f__": "-inf"}, -1.4904224872589111, {"__f__": "-inf"}, -0.4330652952194214, {"__f__": "-inf"}, -0.05073348060250282, {"__f__": "-inf"}, -0.29606688022613525, {"__f__": "-inf"}, -0.22227893769741058, {"__f__": "-inf"}, 0.015316561795771122, {"__f__": "-inf"}, 0.6020402908325195, {"__f__": "-inf"}, -0.16982169449329376, {"__f__": "-inf"}, -0.9205906391143799, {"__f__": "-inf"}, 0.5387210845947266, {"__f__": "-inf"}, 1.6243313550949097, {"__f__": "-inf"}, -0.11647308617830276, {"__f__": "-inf"}, 0.10397743433713913, {"__f__": "-inf"}, -1.1941795349121094, {"__f__": "-inf"}, -0.03954041004180908, {"__f__": "-inf"}, 0.014170035719871521, {"__f__": "-inf"}, 1.202669382095337, {"__f__": "-inf"}, 0.3702281415462494, {"__f__": "-inf"}, 1.624699592590332, {"__f__": "-inf"}, 0.27259552478790283, {"__f__": "-inf"}, 0.30909645557403564, {"__f__": "-inf"}, -0.05742330476641655, {"__f__": "-inf"}, -0.1897757649421692], [{"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}, {"__f__": "-inf"}]], "dtype": "float32"}], "e": {"__nd__": [5.274785041809082, 7.8170576095581055, {"__f__": "-inf"}], "dtype": "float32"}}, {"a": [{"__nd__": [[7.5]], "dtype": "float32"}], "e": {"__nd__": [7.5], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.9251281023025513, 10.146960258483887, -0.14011873304843903, -7.835186004638672, -13.319524765014648, -3.614964723587036, 5.408312797546387, -2.058269500732422, 1.306666612625122, -11.00096607208252, -5.444718360900879, 15.86532974243164, -4.93571138381958, -0.3683689534664154, -5.3502516746521, -3.710658073425293, 5.677233695983887, -12.1173095703125, 8.407004356384277, 22.00157356262207, 7.813009262084961, 2.846426486968994, 17.33945083618164, -1.220294713973999, 12.092419624328613, -2.2930331230163574, -10.815991401672363, 2.232511043548584, -0.7856415510177612, 20.434276580810547, -1.1962096691131592, -21.547441482543945, 2.9069855213165283, -15.680883407592773, -21.958349227905273, -6.342038631439209, 11.4900484085083, -6.053372383117676, 7.13303804397583, 5.3127522468566895, 10.781890869140625, -2.5589160919189453, -7.6969757080078125, 2.9680802822113037, 6.246557235717773, 5.213322639465332, 16.867725372314453, -6.478789329528809, -5.386928558349609, 8.812212944030762, 10.587312698364258, -14.799395561218262, 1.4369170665740967, 20.1558895111084, -11.24056339263916, -7.966298580169678, -15.080806732177734, -12.70962905883789, -5.78474235534668, 7.20884370803833, 8.231562614440918, 4.3274688720703125, -11.30362319946289, 6.6805219650268555, -3.8988919258117676, -2.0144476890563965, -23.523719787597656, -3.1169185638427734, 6.851874351501465, 12.485429763793945, -6.729159832000732, 16.546077728271484, -7.97677755355835, 10.542730331420898, -1.0514987707138062, -8.163640022277832, 6.452322959899902, -4.42439079284668, -26.603271484375, 5.328849792480469, -13.369154930114746, -0.36146634817123413, -0.31143897771835327, 1.315554141998291, 2.5607686042785645, 27.282745361328125, -9.361084938049316, 2.1624722480773926, 8.720975875854492, -18.37490463256836, 11.037639617919922, -1.9377408027648926, -5.499398231506348, 0.36628320813179016, -14.576191902160645, -8.97886848449707, 15.93215560913086, 8.220694541931152, 25.28280258178711, -2.1305861473083496, 3.430481433868408, 1.324232816696167, -3.822293519973755, 5.499879837036133, 2.6290056705474854, 9.850970268249512, -5.620977401733398, 13.32677936553955, -3.286548137664795, -17.690872192382812, -5.442750930786133, 13.484235763549805, 8.662653923034668, -3.5146095752716064, 6.306191444396973, -4.549201488494873, 5.235283374786377, -1.0007975101470947, 13.06755256652832, -7.049208164215088, 8.055771827697754, -6.227464199066162, 9.139389038085938, -10.345276832580566, -13.646936416625977, -3.066592216491699, 23.815231323242188, -3.787260055541992, -10.400217056274414, 3.61743426322937, 0.15625950694084167, -15.213776588439941, -1.6790187358856201, -5.286145210266113, 6.05837869644165, -3.5704874992370605, -13.72393798828125, 1.7809885740280151, 29.97525405883789, -1.1637945175170898, -4.044743537902832, 6.442667007446289, 12.68099308013916, 17.98442840576172, -6.3156890869140625, 2.116542339324951, -27.947895050048828, 2.2240867614746094, -6.30810022354126, -10.690767288208008, -12.872474670410156, -0.8838272094726562, 0.3120708167552948, 4.395784854888916, -15.956945419311523, -15.722908020019531, -12.091012954711914, -2.8373730182647705, -3.0141470432281494, 4.9955220222473145, -1.3652055263519287, 2.0438613891601562, -1.999664068222046, -7.877132892608643, -2.8554728031158447, 16.085041046142578, -6.576498985290527, 25.619426727294922, -3.3183577060699463, -18.137094497680664, 10.316211700439453, -5.015303611755371, 9.951902389526367, 2.3508241176605225, 7.54942512512207, 6.067410469055176, 12.471782684326172, -9.734793663024902, -0.4764142334461212, 7.245940685272217, -1.6111876964569092, 14.344127655029297, 0.6944185495376587, -2.9088568687438965, -10.731806755065918, -3.79581880569458, 4.993408203125, 17.11526107788086, -2.8060922622680664, 10.979071617126465, -5.163948059082031, -17.766878128051758, -4.132556438446045, 12.954729080200195, -0.04803185909986496, -1.858534574508667, 8.105758666992188, 0.17884472012519836, -6.785018444061279, 5.734495162963867, -9.449261665344238, 10.3029146194458, -7.861381530761719, 9.69813346862793, -10.36467456817627, -3.164259195327759, -14.620187759399414, 5.073540210723877, -4.310634136199951, 0.4331064820289612, -14.983415603637695, -0.9249284863471985, 0.09920842200517654, -8.578484535217285, -4.8953752517700195, 3.7571051120758057, 13.924704551696777, 3.113464117050171, 20.501312255859375, -13.69839859008789, -6.908513069152832, -10.541181564331055, 17.569000244140625, 1.5711987018585205, -4.856711387634277, 17.785566329956055, 4.330204010009766, -22.718692779541016, -15.391755104064941, 11.732388496398926, -1.4708328247070312, -16.70761489868164, -9.446117401123047, 5.411487579345703, -3.3993992805480957, 8.250141143798828, -8.692829132080078, -11.346305847167969, 4.449918270111084, 20.590660095214844, 9.243099212646484, 8.354433059692383, -5.315952777862549, -17.562423706054688, 13.765606880187988, 6.201183795928955, -1.8750275373458862, -4.885867118835449, 11.361181259155273, -10.623126029968262, -8.33392333984375, -1.7390059232711792, -5.92446231842041, 0.6991024017333984, -1.074185848236084, 0.9872633814811707, -8.962913513183594, -13.568336486816406, -0.7600876688957214, -8.628429412841797, 8.47048282623291, -3.4297034740448, -8.08169174194336, -2.421051025390625, -1.7481729984283447, -8.482606887817383, -8.28200912475586, -5.925495147705078, -14.818408966064453, 0.36783266067504883, -6.457486629486084, 0.2980504333972931, -12.028802871704102, -2.5538113117218018, -7.346479415893555, 9.480813026428223, 5.96419620513916, -1.2451274394989014, 17.734189987182617, -10.846400260925293, 8.75754165649414, 1.8156753778457642, 18.471620559692383, -29.469562530517578, 17.738059997558594, -13.806624412536621, 12.529718399047852, -12.785433769226074, 6.217288017272949, -1.2525166273117065, 17.417009353637695, -13.879369735717773, -14.12628173828125, 6.084197044372559, 5.869374752044678, -15.135880470275879, -1.4181761741638184, 1.1833975315093994, 1.655981421470642, -6.637877464294434, 6.705045223236084, 12.930036544799805, -11.94812297821045, 13.720708847045898, -7.9294657707214355, 1.1692938804626465, -3.430666923522949, 2.2733678817749023, -10.818076133728027, -1.6048617362976074, 5.145477294921875, -2.6109232902526855, 14.699438095092773, -8.925320625305176, -2.5238897800445557, -2.492565631866455, 5.240415573120117, -8.278709411621094, -9.152155876159668, 3.6539804935455322, 1.6835899353027344, -4.614210605621338, 6.419029235839844, 2.126354455947876, 3.28080153465271, 4.8178229331970215, 18.128520965576172, 17.829740524291992, 6.664313316345215, -19.44593620300293, -5.4610466957092285, 12.679244041442871, 10.171174049377441, -6.370992183685303, 1.5648980140686035, 8.709470748901367, -0.9686892628669739, 1.9063749313354492, -2.5391311645507812, 1.0848199129104614, -2.9275643825531006, 10.907279014587402, -2.074878454208374, -1.4631149768829346, 7.305997848510742, -0.4087664484977722, 0.6158895492553711, -16.5080623626709, 1.3783519268035889, 0.96844083070755, 14.26703929901123, -7.594937801361084, -1.297865867614746, 9.02457046508789, -2.9634177684783936, -10.567536354064941, -2.1436872482299805, 11.008544921875, 4.266599655151367, 14.351543426513672, 22.6685733795166, -5.67242431640625, 4.641449928283691, -3.0397164821624756, -7.8952412605285645, 5.494489669799805, -20.939617156982422, 11.113896369934082, -3.4402804374694824, 6.328454494476318, 3.68160343170166, -8.136523246765137, -4.134529113769531, 20.644376754760742, -18.525941848754883, 2.6550862789154053, 7.461993217468262, -16.773971557617188, -14.861608505249023, -5.863469123840332, 6.431119918823242, 0.5560922026634216, 0.21254992485046387, -1.2041178941726685, 12.816925048828125, 8.14588737487793, -3.5858168601989746, -4.785613059997559, 7.948782444000244, -7.165216445922852, 7.595887184143066, -11.07374095916748, -7.389679908752441, 7.072113037109375, -0.08944009244441986, -0.3807738125324249, -9.966919898986816, 5.181068420410156, -19.580135345458984, -0.46179041266441345, -15.338159561157227, -9.19587230682373, -6.362157344818115, -13.822598457336426, -3.3900234699249268, -3.3913612365722656, 4.3910346031188965, 5.65242862701416, 3.0971696376800537, 15.514163970947266, -17.11518096923828, -0.11480933427810669, -0.628313422203064, -18.971158981323242, -1.4255666732788086, -10.551566123962402, -1.2796694040298462, -9.958751678466797, 4.656935691833496, -3.792450428009033, 6.528656959533691, -19.745237350463867, -3.1362967491149902, -22.30385971069336, 19.964492797851562, -13.559225082397461, -5.211034297943115, 5.387101173400879, -2.9168174266815186, 3.7780604362487793, -5.590435981750488, -0.48376184701919556, 1.4395713806152344, -8.141929626464844, -14.042377471923828, -13.051777839660645, -3.660386562347412, 0.4517972767353058, 8.73863410949707, 4.15326452255249, -8.83297348022461, -6.341166973114014, 19.532917022705078, -4.7906060218811035, 2.7699403762817383, 11.16909122467041, -5.942602157592773, -18.734193801879883, 28.1009578704834, 7.807492256164551, -3.117548942565918, -5.465002536773682, -0.04871974140405655, -4.023508071899414, -11.833263397216797, -12.58276081085205, -16.320594787597656, -4.747217178344727, 14.31812858581543, -0.17616668343544006, -7.831646919250488, -16.745908737182617, -2.9020872116088867, -3.506659984588623, -13.157404899597168, 0.6224912405014038, -16.928237915039062, 2.29617977142334, -7.948023796081543, -3.0643770694732666, 1.3772408962249756, 8.84101390838623, -16.296764373779297, -3.707526683807373, -6.924567699432373, -6.44184684753418, 11.125470161437988, 1.7308399677276611, 4.298990249633789, -8.97147274017334, 9.839526176452637, -3.8788986206054688, -13.037847518920898, -3.392110824584961, -16.066503524780273, 15.98021411895752, -13.762465476989746, 1.0976446866989136, 7.464842796325684, -1.046952486038208, -14.018040657043457, 11.085820198059082, -4.558644771575928, -13.582412719726562, 2.9907326698303223, 0.5035985112190247, 3.5743627548217773, -10.16501235961914, 16.728717803955078, 3.3712635040283203, 11.394533157348633, 8.157470703125, 10.926471710205078, -6.501382827758789, 6.216048717498779, -6.168272495269775, 9.640961647033691, 3.90095591545105, -13.525052070617676, 9.318425178527832, 6.187140464782715, 10.80766487121582, -0.8956809043884277, -6.732566833496094, -6.304616928100586, 15.741321563720703, -9.244024276733398, 2.9672536849975586, 10.616756439208984, 16.504512786865234, -21.64728355407715, 2.7623775005340576, 12.405476570129395, 18.853708267211914, 5.047318935394287, -5.169684410095215, -12.376087188720703, -5.443795204162598, 8.246535301208496, -12.47629165649414, 6.285460472106934, -10.839553833007812, 3.8124232292175293, -5.228629112243652, 13.556780815124512, -1.8148738145828247, -6.113109588623047, -16.009105682373047, 13.550779342651367, 1.586371660232544, -5.7234649658203125, 0.0846041664481163, 6.243107318878174, 5.400766849517822, -1.4046837091445923, -1.6481654644012451, -1.1424864530563354, 7.975262641906738, -0.8970844149589539, -0.5292500257492065, -4.511714458465576, 8.810863494873047, 4.52022647857666, 0.4554179906845093, 6.498584747314453, 4.046725749969482, -2.0953707695007324, -7.186040878295898, -2.1410131454467773, 2.589938163757324, -12.829503059387207, -14.547630310058594, -2.236499786376953, -8.833768844604492, -5.324708938598633, -10.044242858886719, 15.667515754699707, 22.52479362487793, 13.738799095153809, 16.58836555480957, 0.13734589517116547, 15.624719619750977, -18.0952205657959, -28.132732391357422, -3.9228222370147705, -8.597811698913574, 7.152073383331299, 0.056528352200984955, -0.44365230202674866, -4.6615166664123535, -2.4137210845947266, -0.7089715003967285, 0.0413244254887104, 15.331893920898438, 15.149798393249512, 4.071836948394775, -3.949188232421875, -7.77193546295166, -5.533303737640381, -6.513282775878906, -14.336047172546387, -7.404487609863281, 6.942831993103027, -7.296606063842773, -6.476740837097168, -5.711756706237793, -5.0658721923828125, 3.3308396339416504, -4.751500606536865, -1.5947140455245972, 12.486620903015137, 0.2874529957771301, 14.070554733276367, -8.970235824584961, -14.963635444641113, 0.35252130031585693, 3.8222405910491943, 16.754220962524414, 2.913555145263672, -8.06849193572998, 9.253889083862305, -11.851509094238281, 0.5684732794761658, 1.9778046607971191, -3.8088738918304443, -1.6538046598434448, 19.06885528564453, -7.43318510055542, 18.577756881713867, 9.120882034301758, 2.352466344833374, 7.320693016052246, 16.815998077392578, -14.87866497039795, -0.8705825805664062, -3.282130718231201, 0.9346134066581726, 4.106261730194092, -8.31977653503418, 1.94614839553833, -4.325799465179443, -17.46283721923828, -12.93320083618164, -13.59560489654541, -18.812458038330078, 6.749186038970947, 1.8159618377685547, -9.28109073638916, 18.400379180908203, -8.510350227355957, -3.928372383117676, -3.0923051834106445, -8.306497573852539, -9.348827362060547, 11.57461929321289, -9.040740013122559, -2.556584119796753, 1.2964175939559937, 10.000455856323242, -2.250807762145996, 0.18478931486606598, 6.153614521026611, 21.213207244873047, -11.959224700927734, -2.319732189178467, 6.803033828735352, 1.1319531202316284, -18.529735565185547, 11.04007339477539, 6.261510372161865, -0.7876936197280884, 12.832586288452148, 14.094552993774414, -5.5125627517700195, 3.524683952331543, -14.071526527404785, -9.587822914123535, -13.413325309753418, -5.29158878326416, -15.71658992767334, 16.63600730895996, -0.11868253350257874, -5.105721473693848, -4.773181915283203, -9.040857315063477, -9.572184562683105, -11.393054962158203, 10.815383911132812, -4.390199184417725, 6.752615451812744, 1.8332304954528809, 4.088345527648926, 4.012081146240234, -13.570599555969238, 18.319599151611328, 4.752182483673096, 9.247694969177246, 7.314597129821777, 21.49441146850586, -3.26119327545166, -22.74500846862793, -13.595906257629395, -6.205430030822754, 2.5825858116149902, 7.075325965881348, 9.716590881347656, 1.951831579208374, -14.561967849731445, 5.385432243347168, -5.103517532348633, -4.216028213500977, -16.383914947509766, -9.000799179077148, 4.2204084396362305, -4.7383599281311035, -20.23516845703125, -9.74160385131836, 10.044103622436523, -13.234959602355957, -15.130820274353027, -10.72547721862793, 11.645947456359863, -10.860156059265137, -12.868051528930664, 7.859679222106934, 8.186301231384277, -8.957917213439941, 4.63687801361084, -2.8064651489257812, 7.7148590087890625, -6.392406463623047, -4.445178508758545, -0.7845710515975952, 0.46602827310562134, -1.1058796644210815, 8.65872573852539, 11.521378517150879, -7.771773815155029, 10.749515533447266, 2.500980854034424, -4.573765754699707, 3.4099626541137695, -20.566234588623047, 0.9995705485343933, -2.2214362621307373, -0.8741199374198914, 10.268139839172363, -18.4040470123291, 0.2143116295337677, 0.032785799354314804, -3.8637640476226807, -12.76896858215332, 1.421095848083496, 1.180545687675476, 2.740272045135498, 19.83994483947754, 2.6313111782073975, -14.579259872436523, -23.668882369995117, -19.78838348388672, -5.692671775817871, -9.839909553527832, -4.64594841003418, 5.707881927490234, -0.9781925678253174, -6.996468544006348, -2.1148629188537598, 4.964448928833008, 12.716375350952148, 10.652080535888672, -22.925016403198242, 3.6538748741149902, 6.94334602355957, 11.850017547607422, 9.53264331817627, 2.2481603622436523, -3.1485743522644043, 3.950239419937134, 13.791976928710938, 18.708471298217773, 5.702090740203857, -0.936724841594696, 5.719395160675049, -8.598332405090332, -2.881377935409546, -8.039179801940918, -5.626768589019775, -4.422863483428955, -1.9901384115219116, 2.092031478881836, 2.134329080581665, 1.1689624786376953, -23.512990951538086, 0.31300729513168335, -9.383502006530762, 15.393343925476074, 3.085235118865967, -9.1207275390625, 5.770905494689941, -16.03705596923828, 5.311872482299805, -7.519634246826172, 4.439902305603027, -13.059528350830078, 1.3317773342132568, -3.871166706085205, 3.349271059036255, -6.033900260925293, 8.054662704467773, 11.994754791259766, 5.102322101593018, 8.407792091369629, 18.021766662597656, -14.509359359741211, -14.787637710571289, -1.9739784002304077, -4.6303019523620605, 4.0874433517456055, 1.86014723777771, -9.602027893066406, -13.013629913330078, 10.95812702178955, 9.09511661529541, -13.312793731689453, 2.326500415802002, 15.062285423278809, 6.08305025100708, 7.15362548828125, 2.379267692565918, -9.845911026000977, -3.096327781677246, -7.279921531677246, 10.015040397644043, -1.7271802425384521, -4.8437418937683105, 7.170095920562744, -1.307529330253601, 1.1842108964920044, -18.78968048095703, 5.621882438659668, -4.145604133605957, 1.4542241096496582, 2.6749696731567383, -6.263139724731445, -9.277144432067871, -11.032880783081055, 2.196542978286743, -5.851622104644775, 6.059909343719482, -8.618060111999512, 7.863675117492676, -7.6868486404418945, 4.310659408569336, 13.512582778930664, -15.36247730255127, -23.85987091064453, 10.484050750732422, 11.204475402832031, -8.507806777954102, -0.9304851293563843, -3.1117770671844482, 17.0903377532959, -2.9442169666290283, 8.003999710083008, -15.97464656829834, 20.169437408447266, -3.007744789123535, -0.6031442284584045, 7.713832378387451, 15.180988311767578, 3.029290199279785, -15.817748069763184, -11.977633476257324, -8.329862594604492, 8.262531280517578, -13.175301551818848, 11.94400405883789, -6.813807964324951, 15.270866394042969, 6.051292419433594, -4.434586048126221, 0.9736297130584717, 13.510902404785156, -2.951303482055664, 23.119794845581055, 21.116397857666016, -8.197880744934082, -22.360610961914062, 17.911779403686523, -7.810023307800293, -8.127198219299316, -20.02023696899414, 1.4119428396224976, 6.895813465118408, -3.063992500305176, -11.997565269470215, 8.80445671081543, -14.61143684387207, -6.715223789215088, 2.168325901031494, -6.792351722717285, 12.571867942810059, 22.312278747558594, -14.4414701461792, 22.40424346923828, 0.12432186305522919, 3.990830421447754, 22.52289581298828, 2.320288896560669, 5.1318840980529785, 0.10356535017490387, 4.88953971862793, -1.4646703004837036, 25.537490844726562, 7.499986171722412, 0.7862802743911743, 0.516174852848053, -10.49146556854248, 4.360184669494629, 9.420679092407227, 4.232948303222656, 6.793678283691406, -21.014606475830078, -10.594362258911133, 5.789148330688477, -8.045190811157227, -1.5284147262573242, 4.977458953857422, -8.729292869567871, -2.509705066680908, -19.776676177978516, -17.88783836364746, 2.390622138977051, -14.844123840332031, -6.247407913208008, -8.145659446716309, -6.799612998962402, 4.938109397888184, -5.932070732116699, -6.479800224304199, 1.746161699295044, -5.342012405395508, -10.896363258361816, 0.9700717926025391, 5.8984904289245605, 14.280149459838867, 1.2075642347335815, -0.20630371570587158, -8.778350830078125, -8.989392280578613, -7.398797988891602, -18.59765625, -15.374448776245117, 7.687011241912842, 10.39239501953125, -5.590822696685791, 11.340377807617188, 15.046095848083496, 10.714106559753418, 9.440963745117188, 4.022426128387451, -5.519437789916992, 0.619778573513031, -3.0707287788391113, 6.816988468170166, 11.760963439941406, 2.7292890548706055, -5.764118194580078, 14.908742904663086, -4.078304767608643, -4.500545978546143, 24.177291870117188, -12.383460998535156, -0.7981211543083191, -15.85804557800293, -1.9947267770767212, -11.707001686096191, 17.534297943115234, 10.281864166259766, -1.3369226455688477, 1.6179507970809937, 3.496642589569092, 1.7573256492614746, -9.651739120483398, -4.199172019958496, 0.018484659492969513, 0.16963434219360352, 1.8208014965057373, -15.074079513549805, 32.3509635925293, 3.4949147701263428, 7.518513202667236, -12.96653938293457, -3.163609504699707, 19.586204528808594, -6.873378753662109, -6.156705856323242, -8.270498275756836, -8.068329811096191, -17.0107479095459, -9.659294128417969, -10.596891403198242, -13.992448806762695, -2.0651187896728516, -14.311159133911133, -3.7126035690307617, 0.898246705532074, -3.4661707878112793, 2.6339001655578613, 5.383645057678223, 8.266857147216797, 25.167137145996094, -1.9575928449630737, -14.145760536193848, 3.476396083831787, -6.006752967834473, -13.625753402709961, -0.034764181822538376, -3.417813301086426, 26.12771224975586, -3.504701852798462, -12.571977615356445, 9.33674430847168, -5.668952465057373, -4.388282775878906, -6.882248878479004, 5.130250453948975, 2.2124836444854736, -0.011278065852820873, -5.918935775756836, -6.903438568115234, -17.740825653076172, 6.089156150817871, -17.313613891601562, -22.16201400756836, -13.053691864013672, 20.111083984375, 2.1121091842651367, 5.373998641967773, -4.929935455322266, -7.673264503479004, -11.213272094726562, 8.318985939025879, 0.6812298893928528, 3.932624340057373, 18.137704849243164, -18.037805557250977, -1.9149158000946045, -7.859362602233887, 6.118391990661621, 3.591437578201294, -0.3856721818447113, -7.632170677185059, -12.377656936645508, -9.7520170211792, -8.560787200927734, -1.2393608093261719, 4.307951927185059, 17.04624366760254, 21.49321746826172, -3.537810802459717, 7.478151798248291, 2.50072979927063, 1.1565905809402466, -1.6158785820007324, -8.35196590423584, -2.716020107269287, 15.487217903137207, -5.696315288543701, 6.379596710205078, -7.860009670257568, 7.312752723693848, -5.921850681304932, -2.095933198928833, -2.4683239459991455, -21.512664794921875, 4.320141792297363, -6.324272632598877, -7.965203285217285, 12.374037742614746, 6.297116279602051, -8.220205307006836, 4.202700138092041, -11.260162353515625, -9.56358814239502, 13.295626640319824, -21.538002014160156, 2.6794543266296387, 24.813343048095703, 3.304684638977051, -4.549044132232666, -3.3758034706115723, 7.905908107757568, -7.916627883911133, 3.861198902130127, 12.048866271972656, -0.6732758283615112, -1.005174160003662, 1.1197179555892944, 2.391798973083496, 5.602359771728516, -2.532038450241089, 13.805733680725098, -9.435912132263184, 4.2570013999938965, -1.7936697006225586, 17.4668025970459, -4.804486274719238, -4.863761901855469, -5.084218978881836, -5.35335111618042, -15.896023750305176, 3.2269339561462402, 3.889033794403076, -2.8510518074035645, -4.712228298187256, -7.311424732208252, -7.0631818771362305, -8.134198188781738, 7.324351787567139, 13.286604881286621, -1.2028615474700928, -0.1679786741733551, -16.15176010131836, 6.699920654296875, 5.682971954345703, -20.860198974609375, 2.756221294403076, -6.817839622497559, -14.576574325561523, 10.519784927368164, 5.775942325592041, -12.49240493774414, -10.170698165893555, -8.13005256652832, 9.96702766418457, -12.048965454101562, 3.710207462310791, -0.5491058826446533, 8.165639877319336, -6.575648784637451, -5.2313690185546875, -0.7369664311408997, 4.8985915184021, 5.143261432647705, -23.209379196166992, -7.663442611694336, -3.91947078704834, 10.691428184509277, 5.656142234802246, -15.038114547729492, 9.235002517700195, -13.665589332580566, -10.774185180664062, -4.962190628051758, 15.839224815368652, 3.9385387897491455, -0.7144618630409241, -2.102876663208008, 3.068128824234009, 12.939435958862305, 4.076177597045898, 5.82014799118042, 20.148723602294922, 15.34585952758789, 2.575007915496826, -15.528912544250488, -9.073954582214355, -8.497631072998047, -9.017753601074219, 2.7204718589782715, 4.997170925140381, -5.82695198059082, -0.1665244698524475, 1.661760687828064, -6.2491230964660645, 10.30367660522461, 8.308208465576172, 10.943817138671875, -0.9891569018363953, -3.94415020942688, 4.151774883270264, -12.45313549041748, 10.370567321777344, 13.432548522949219, 4.593864440917969, -12.466898918151855, 1.9893317222595215, 0.7160696387290955, 3.7408299446105957, 8.430660247802734, 6.524541854858398, -15.079996109008789, 11.013726234436035, -6.955029487609863, 18.28447151184082, -8.624479293823242, -6.385020732879639, 8.700410842895508, -16.974031448364258, -9.510275840759277, -0.4003486633300781, -9.346639633178711, -4.940003871917725, -15.058456420898438, 5.09133243560791, -15.10908317565918, -1.9189367294311523, 4.664412975311279, -2.1123383045196533, -2.5950047969818115, 0.38312476873397827, 11.237579345703125, -5.087974548339844, -3.5009238719940186, -7.778824806213379, 9.839099884033203, 2.9752116203308105, -0.46858400106430054, 0.3834455907344818, 7.022190570831299, 13.40949821472168, 9.389208793640137, 13.727104187011719, 2.2718052864074707, -0.06903649121522903, 10.718057632446289, 15.140134811401367, 19.473976135253906, 0.9496760368347168, 0.28321805596351624, -1.431833028793335, 3.0002763271331787, -16.459630966186523, -7.892330169677734, 3.8482487201690674, 13.888124465942383, -3.337937831878662, -9.998537063598633, 3.579524040222168, 16.00309181213379, -7.475504398345947, -4.910675048828125, 6.73992919921875, -13.271044731140137, 9.733566284179688, -20.787771224975586, 3.3049705028533936, -11.014555931091309, -13.078022956848145, -5.5209150314331055, 2.3026282787323, -6.806619167327881, 3.8828089237213135, -0.24262404441833496, 3.2523398399353027, 13.925275802612305, 7.803190231323242, 17.02214241027832, 10.620018005371094, -13.143056869506836, 3.247500419616699, 9.139091491699219, 6.881875514984131, 12.655023574829102, 4.054755210876465, 2.1371982097625732, 20.380859375, -15.439359664916992, 3.3664979934692383, 21.582027435302734, -3.4068048000335693, -5.499534606933594, 21.456653594970703, 0.6294384598731995, -13.284587860107422, 26.91889190673828, -5.926757335662842, -3.0413272380828857, 8.850805282592773, -3.212404251098633, 3.6384072303771973, -2.8895578384399414, 10.98936653137207, -2.059119939804077, 11.169557571411133, 1.7725647687911987, -8.043243408203125, -22.278099060058594, -13.085538864135742, 0.8115248680114746, 15.046826362609863, 5.589047431945801, -27.19913673400879, -7.844429969787598, 18.59890365600586, -1.2466434240341187, 12.027793884277344, -13.316296577453613, -11.52761459350586, -2.041741132736206, 8.519113540649414, 3.464651346206665, -4.203102111816406, -6.965034484863281, 6.290714263916016, 11.176898956298828, -12.304258346557617, -11.550399780273438, 8.716440200805664, 3.8831734657287598, 4.732141971588135, -0.04333995655179024, -9.806302070617676, 2.8896350860595703, -9.34205150604248, -26.876022338867188, -14.224687576293945, -1.7064282894134521, -7.071646690368652, 3.712782382965088, -6.39395809173584, 28.126522064208984, -0.6739138960838318, -2.7955234050750732, -13.142594337463379, 19.17965316772461, -8.97838306427002, 0.7340950965881348, -14.27171802520752, -6.948652744293213, -1.3587028980255127, 2.9039361476898193, -0.3112908601760864, 15.488250732421875, 22.484737396240234, -21.317960739135742, -10.303450584411621, 5.104902267456055, 3.795771837234497, 16.641002655029297, -12.29515552520752, -4.967071533203125, 5.971249580383301, -0.9377731084823608, -2.5467495918273926, -0.7282564640045166, 1.366099238395691, 2.7081942558288574, 3.0308890342712402, -2.0859475135803223, -9.840300559997559, -0.8750967979431152, 0.7239344120025635, 2.4255616664886475, -7.305147171020508, 9.731263160705566, -2.8824880123138428, -0.7224678993225098, -9.173694610595703, 3.097153663635254, -7.342565536499023, -5.0695319175720215, -2.971663236618042, 8.059152603149414, -2.50991153717041, -19.41368865966797, -2.029601573944092, -3.3181679248809814, 4.708876132965088, -7.476049423217773, -0.9761205911636353, 6.152688026428223, -9.486528396606445, -5.230472087860107, 10.049545288085938, 6.947178363800049, -4.41525936126709, 7.120294570922852, -2.537867546081543, -10.92896556854248, 6.192030429840088, -14.84676742553711, 26.321533203125, 15.389022827148438, -6.923072338104248, 13.454938888549805, 0.17790748178958893, -6.146220684051514, 6.883879661560059, -16.986305236816406, -9.695662498474121, -3.37910795211792, -15.341601371765137, -20.817672729492188, 7.708248615264893, 15.243705749511719, -3.4087870121002197, -2.1984121799468994, -4.9715189933776855, 14.473817825317383, 13.52399730682373, -14.682771682739258, 3.4823033809661865, -8.354402542114258, -0.8466925621032715, -2.7225444316864014, -10.841546058654785, 12.049219131469727, 3.674499988555908, -19.83207893371582, -15.631722450256348, 3.4973552227020264, -4.75096321105957, 3.3781943321228027, 1.7966176271438599, 1.2421319484710693, 7.367706775665283, 3.719916820526123, 4.924109935760498, -7.1402692794799805, 4.81223201751709, 12.279845237731934, -5.227627277374268, 1.1411861181259155, 6.7509355545043945, -4.085379123687744, -7.089451789855957, 12.612335205078125, 5.667693138122559, 4.098660945892334, 6.252622604370117, 1.2631452083587646, -0.8278766870498657, -2.4149019718170166, 10.445843696594238, -19.639312744140625, 6.804823398590088, 10.601716995239258, -2.356022834777832, -3.2581787109375, -7.6129350662231445, 8.647626876831055, 15.358582496643066, -1.642324447631836, -8.364680290222168, 3.3279876708984375, 2.5125412940979004, -9.802167892456055, -4.071216106414795, 9.455055236816406, -6.621174335479736, -14.295207977294922, 14.092613220214844, -10.49026107788086, -4.496179580688477, -25.330188751220703, 4.663424491882324, -10.510730743408203, 1.6216771602630615, -5.9426116943359375, -2.6945996284484863, 15.49170970916748, 19.408754348754883, -14.767046928405762, -8.655202865600586, 10.363113403320312, -7.443418025970459, -5.141699314117432, -3.036362886428833, -1.1255470514297485, 1.129124641418457, 5.099130153656006, 0.5258724689483643, -5.603912353515625, 14.657245635986328, 12.220829010009766, -0.22560617327690125, 1.1849820613861084, 7.712436676025391, -2.66717529296875, 5.33256721496582, -11.565467834472656, 12.068868637084961, 5.290221214294434, 8.665083885192871, 6.805960655212402, 19.536945343017578, -3.9395344257354736, -8.17679214477539, -1.1516327857971191, -9.861916542053223, 6.504548072814941, 7.455678462982178, -4.428212642669678, 23.039958953857422, 16.220670700073242, -16.844375610351562, -6.550097465515137, -4.898026943206787, -7.036247253417969, 0.3971155285835266, 12.975801467895508, 2.4033429622650146, -2.76652455329895, 3.0398333072662354, -3.1099209785461426, 7.664237976074219, -19.30087661743164, 7.824324607849121, -5.297769546508789, -8.330881118774414, 13.07400894165039, 12.024145126342773, -0.662392258644104, 1.329272747039795, -3.8604962825775146, 1.4187791347503662, -6.355897903442383, -5.2561163902282715, 12.32513427734375, 12.47554874420166, -13.156909942626953, 1.0765587091445923, 9.112239837646484, 16.8680362701416, 2.1821377277374268, -8.028145790100098, -7.1007080078125, -3.643563985824585, -10.724658966064453, 1.6007239818572998, 1.4326359033584595, 7.069141864776611, 4.546682357788086, -5.851483345031738, -1.4999613761901855, 1.2883193492889404, -0.6232779026031494, -17.242341995239258, 6.859830856323242, -22.835481643676758, -9.38604736328125, 6.784021377563477, 12.214184761047363, 2.8185067176818848, -9.5370512008667, -2.9552135467529297, 0.015019437298178673, -5.740234375, 16.36164665222168, 2.6037356853485107, -0.10111363232135773, 15.790460586547852, -14.319116592407227, -8.073884010314941, -9.432910919189453, -5.138554573059082, 2.937331438064575, 7.305559158325195, 19.85183334350586, 9.194096565246582, 3.9064626693725586, -19.642393112182617, 3.719513416290283, -21.40256118774414, -1.5488405227661133, -7.895374774932861, 4.484124660491943, 6.468104839324951, -2.3246665000915527, 4.71246337890625, 4.257856369018555, -0.2917206585407257, -10.235431671142578, -3.507842779159546, -9.655900955200195, -5.751070976257324, -8.115913391113281, -8.492230415344238, -1.6330890655517578, 4.631499767303467, -18.130529403686523, -10.594229698181152, 0.6628149747848511, 8.047788619995117, 7.723729610443115, -0.8878415822982788, 3.263292074203491, 6.639991760253906, 2.808511257171631, -5.1412272453308105, 0.31401699781417847, -5.726729393005371, 7.541640281677246, 7.469954013824463, -1.0029364824295044, -8.019674301147461, 8.454707145690918, 1.7370436191558838, -11.603017807006836, 6.607165336608887, 24.79538917541504, 8.652844429016113, 17.948139190673828, 4.388125896453857, 12.638996124267578, -16.00507926940918, 11.981006622314453, 10.088911056518555, 0.7700833082199097, -6.034326553344727, 4.0192389488220215, 2.7992570400238037, -10.464174270629883, 7.2580246925354, 10.419729232788086, -13.39733600616455, 1.3241225481033325, 3.2980427742004395, 12.521757125854492, -8.394801139831543, -6.595111846923828, 18.046781539916992, 3.6866812705993652, 4.0869903564453125, 3.1900784969329834, 3.3649933338165283, -0.5603825449943542, -10.953459739685059, -7.641488075256348, -0.10846962034702301, -12.880599975585938, 4.2841057777404785, -5.776373863220215, 2.096845865249634, 10.067920684814453, 9.382096290588379, -22.045124053955078, -2.5966203212738037, 0.5807394981384277, 3.531491279602051, 1.9385905265808105, 1.2960786819458008, -13.446255683898926, 2.0340750217437744, 1.0784945487976074, 1.6033636331558228, -8.435831069946289, 14.330747604370117, -8.897875785827637, 5.190812110900879, -17.717681884765625, -2.639162540435791, -3.2301297187805176, -1.8030427694320679, 2.5334858894348145, 8.161887168884277, 4.797768592834473, 10.863204956054688, 8.878036499023438, -1.5130589008331299, -8.501047134399414, 3.7962608337402344, -18.733898162841797, 7.426361083984375, 14.581707000732422, 13.903467178344727, 0.4051017165184021, -9.218161582946777, -4.635728359222412, 3.5025575160980225, -14.253236770629883, 4.990065574645996, -7.352442741394043, 3.302820920944214, 1.765099048614502, 5.3590898513793945, -4.154428005218506, -23.676645278930664, -1.785628318786621, -8.06727409362793, -2.613044261932373, -7.527223110198975, 15.28764820098877, 21.477880477905273, -0.2555306553840637, 8.761045455932617, -15.367849349975586, 3.8553035259246826, -5.720708847045898, -4.184534072875977, -5.112236022949219, -4.5962934494018555, 7.003753185272217, -1.5218936204910278, -9.38339614868164, 6.175563812255859, 16.553117752075195, -1.6860657930374146, 5.699217319488525, -10.489158630371094, -3.6130082607269287, 5.33848237991333, 8.166093826293945, 6.177846908569336, 7.21004056930542, 29.516109466552734, 1.2380685806274414, -19.485857009887695, -1.0206177234649658, 4.236130714416504, 1.4496638774871826, 12.678032875061035, -9.250581741333008, -17.59916114807129, -5.754980564117432, -0.4776704013347626, 40.59203338623047, -18.763864517211914, 17.4770450592041, 10.5619478225708, -1.3968528509140015, 18.353899002075195, 12.230937957763672, -0.214118093252182, 2.3015847206115723, -13.262405395507812, 2.560464859008789, -3.753242254257202, -4.180093288421631, -5.9338836669921875, 25.175464630126953, 5.934767246246338, 6.583025932312012, 27.340499877929688, -7.226122856140137, 7.430484771728516, -2.9916043281555176, -10.05314826965332, 3.3975613117218018, 12.978525161743164, -9.03969955444336, 0.22373050451278687, 3.7654531002044678, 5.181474208831787, -1.8563908338546753, -13.197182655334473, -3.1870288848876953, 9.075334548950195, -8.986265182495117, 18.502750396728516, -12.474824905395508, 9.389633178710938, -0.1254923790693283, 5.239344120025635, 9.291438102722168, -1.5436451435089111, 9.150398254394531, 14.78500747680664, 6.465179443359375, -6.734602928161621, 16.470590591430664, 22.160518646240234, 9.117583274841309, -12.31420612335205, 16.5733699798584, -11.481293678283691, -6.3089470863342285, -12.49584674835205, -0.9550957679748535, 3.742774486541748, 2.7741751670837402, 5.6903886795043945, 0.6342904567718506, 2.616067409515381, 5.596959114074707, 1.288933515548706, -4.828625202178955, 1.5563827753067017, 0.12099428474903107, -4.332372665405273, -9.057027816772461, 15.634827613830566, 11.765294075012207, -6.359113693237305, 5.101202964782715, 9.335142135620117, 16.339847564697266, 10.148969650268555, -5.332208156585693, -3.7492856979370117, -23.684329986572266, -15.56279182434082, 10.188865661621094, 9.55447769165039, 18.6121768951416, 3.267853021621704, -5.3187174797058105, -9.448368072509766, 1.8442522287368774, -0.815903902053833, 9.499253273010254, -0.7859880328178406, -0.8054907321929932, -6.019320964813232, -8.627923965454102, 8.828917503356934, -5.174026966094971, 11.888492584228516, -2.544919967651367, -11.794065475463867, 14.986315727233887, 3.5426478385925293, -6.028024673461914, 14.864261627197266, -4.485687255859375, -9.353598594665527, 9.552127838134766, -18.02248191833496, 12.221156120300293, -1.2458875179290771, -10.637961387634277, 13.434659957885742, -17.73834228515625, -7.3328094482421875, -3.0974674224853516, 5.458102226257324, -4.87679386138916, 15.972304344177246, 0.7452194690704346, 11.941551208496094, 1.1229060888290405, 3.752106189727783, 2.0749168395996094, -9.569989204406738, -24.066177368164062, 18.42403793334961, -4.365917682647705, 8.678556442260742, -11.430736541748047, -12.65395736694336, 6.774089813232422, 11.395992279052734, -3.9682693481445312, -14.580846786499023, -1.8666064739227295, 4.440858840942383, -16.533109664916992, -14.812967300415039, 2.8849711418151855, 8.93257999420166, -5.787349224090576, 27.9263973236084, 7.964732646942139, -2.608732223510742, 7.775521755218506, 8.198448181152344, -5.660247325897217, 9.024710655212402, 3.9312515258789062, -7.811189651489258, 2.182234764099121, 7.633723735809326, 4.29582405090332, 20.44992446899414, 17.0760498046875, 4.115963459014893, -18.883346557617188, -17.061885833740234, -8.233953475952148, -17.289539337158203, -3.536604642868042, 16.201549530029297, 10.932975769042969, 23.263999938964844, 4.483826160430908, 5.439590930938721, -0.8650550246238708, 19.790634155273438, -1.3885993957519531, -12.10594367980957, -10.772047996520996, -0.4442179501056671, -9.697758674621582, -3.6533408164978027, -4.07584285736084, 10.743280410766602, -1.4482486248016357, -8.950027465820312, -4.5469970703125, 14.163790702819824, -9.169438362121582, 18.597211837768555, -4.386409759521484, -9.79160213470459, -15.528112411499023, -0.4733337461948395, -2.434847116470337, -15.6495361328125, 3.71736478805542, 10.03609561920166, -25.496856689453125, -15.208593368530273, -0.4418683648109436, -7.461100101470947, -4.430726051330566, -2.3218610286712646, 4.7043280601501465, -3.1592135429382324, -6.2539520263671875, 10.537334442138672, -5.179081439971924, 15.662135124206543, -17.450517654418945, 3.114727735519409, 11.182014465332031, 1.4498140811920166, 8.063755989074707, -17.870128631591797, 3.181955575942993, 9.160655975341797, 4.918674468994141, -8.523906707763672, -7.7048726081848145, 0.6394391059875488, 3.2161831855773926, -3.6437177658081055, 6.2894086837768555, -5.374946594238281, -3.0144309997558594, -0.15383143723011017, 5.1027679443359375, -2.636953592300415, 12.028238296508789, -13.466873168945312, 2.0487141609191895, -5.743742942810059, 9.62022590637207, 9.74599838256836, -8.543012619018555, 1.7047998905181885, 2.0782570838928223, -18.066974639892578, 6.798318862915039, -11.98830509185791, -15.170570373535156, 1.756817102432251, -17.480276107788086, -4.410349369049072, -1.7390607595443726, -20.101886749267578, 3.8396666049957275, 4.3858795166015625, 14.145181655883789, -0.23572203516960144, 0.5694912672042847, 19.94922637939453, 9.401937484741211, 0.5200005173683167, 10.76077938079834, 7.500433444976807, 16.056215286254883, 4.743858337402344, -2.082119941711426, 5.514865398406982, -1.0961405038833618, 1.2212674617767334, -1.949306607246399, -8.598155975341797, 1.376106858253479, -0.4919448494911194, 5.004842281341553, 0.05377674475312233, 7.378567695617676, -3.8821470737457275, 12.145547866821289, -7.794961929321289, 15.325462341308594, -11.432352066040039, -5.2028279304504395, 15.267061233520508, 7.613958835601807, 6.653378486633301, -0.5194234251976013, 7.062492847442627, 21.14544677734375, -9.39945125579834, -12.272785186767578, -5.023725509643555, 7.411452293395996, -2.5848939418792725, 1.6957149505615234, -5.063260078430176, -1.0204318761825562, 2.921299457550049, -11.695727348327637, -1.6705491542816162, -6.483244895935059, -4.673485279083252, -5.861655235290527, 13.78892993927002, -8.055275917053223, 22.436771392822266, -10.63941478729248, 15.953330993652344, 15.627653121948242, 19.88121223449707, 6.405253887176514, -6.901180267333984, 20.282087326049805, 4.0759806632995605, 13.951236724853516, 12.833199501037598, 5.255638122558594, -0.8460357189178467, 17.273900985717773, -2.9232070446014404, 10.696619987487793, 1.7533713579177856, 15.006303787231445, -23.874195098876953, 2.9345340728759766, 20.539968490600586, 2.5096726417541504, 15.101265907287598, -13.47575569152832, 15.461952209472656, -13.36711311340332, -3.7872228622436523, 19.86874008178711, -4.389848232269287, 7.218945026397705, 2.299363136291504, -2.7193775177001953, 10.947484016418457, -9.67830753326416, 18.224502563476562, 0.9640359282493591, 9.577839851379395, -15.466551780700684, -13.45409870147705, 0.644445538520813, 8.698945999145508, -14.97779655456543, 15.49828052520752, -8.600186347961426, 10.268037796020508, 23.556297302246094, -13.231101036071777, 10.969086647033691, -1.8108251094818115, -0.1908641755580902, -10.340418815612793, -6.164562225341797, -3.9959611892700195, -8.345659255981445, -10.138652801513672, -1.5950156450271606, 16.23436737060547, 0.8293908834457397, -8.392557144165039, 4.07981014251709, -1.9862104654312134, -3.5993127822875977, 8.269887924194336, 1.9564313888549805, 11.064544677734375, 0.48764729499816895, 2.815553903579712, -3.462794780731201, 5.320363998413086, 17.92227554321289, -15.515153884887695, -1.1165646314620972, 1.7217447757720947, 5.256882667541504, -3.697122097015381, 10.150958061218262, 12.038318634033203, 0.7946088314056396, -9.652395248413086, -13.030515670776367, -10.218372344970703, -4.2705559730529785, -12.432394027709961, -9.3311185836792, -7.020993232727051, -3.515122175216675, -17.92361831665039, 5.758365631103516, -15.9568510055542, -11.249996185302734, 5.976109027862549, 14.871006965637207, 16.32906150817871, 5.149388790130615, 11.619185447692871, 6.769924640655518, -10.359731674194336, 9.199213981628418, -0.25653043389320374, -6.220951080322266, 6.55963659286499, -6.509286880493164, -14.631797790527344, 9.760071754455566, 4.699095249176025, -3.072347640991211, 4.067128658294678, 17.136266708374023, -8.074644088745117, -6.055922508239746, 22.08298683166504, -22.286245346069336, -1.7337841987609863, -11.197992324829102, -0.5134663581848145, -1.7587147951126099, -4.6742262840271, 5.191673755645752, 10.550973892211914, -0.4786616563796997, -5.945791244506836, 2.4568610191345215, -6.793923377990723, -22.994590759277344, -7.480483531951904, -1.4201246500015259, 8.41401481628418, -1.1484708786010742, -8.424497604370117, -1.2825417518615723, 14.414767265319824, -9.649153709411621, -0.7927799820899963, 1.302297592163086, 0.2762882709503174, 12.098505020141602, -7.0209856033325195, 1.621387243270874, 9.39760971069336, -5.161540508270264, -11.030263900756836, -17.96055793762207, 3.652674913406372, -12.397612571716309, 5.413996696472168, -7.981045246124268, 8.819914817810059, 1.2726049423217773, 7.063067436218262, -1.3568497896194458, -11.41344165802002, 8.4959135055542, 7.412974834442139, -16.632980346679688, -8.033331871032715, 4.5864152908325195, 22.151477813720703, 3.410198926925659, -9.236351013183594, -7.021851539611816, -3.90142560005188, 19.880538940429688, 1.4902350902557373, 11.595210075378418, 9.200689315795898, -0.08301712572574615, -15.953008651733398, 11.975528717041016, -16.64606475830078, -12.096435546875, -10.697480201721191, 11.352086067199707, -8.652048110961914, -9.688200950622559, 5.5648698806762695, -6.00437593460083, 12.519168853759766, 5.069693565368652, 13.873101234436035, -9.600712776184082, 2.528903007507324, 3.0724289417266846, -0.37361395359039307, -31.451618194580078, -9.982978820800781, 1.7444754838943481, -0.9531590342521667, -7.0116167068481445, -2.1318068504333496, 3.7910475730895996, -8.027878761291504, -16.01322364807129, 4.442992687225342, 8.170498847961426, 0.32436466217041016, 3.9681899547576904, -3.5141098499298096, 2.5248026847839355, -0.890181303024292, -12.459016799926758, -5.106382369995117, -13.011880874633789, 13.348588943481445, -12.354171752929688, 1.8795257806777954, 10.90478801727295, 11.29285717010498, -7.6085591316223145, 1.5647627115249634, -9.450087547302246, -7.8025007247924805, -1.2540489435195923, -3.2683799266815186, -2.44462251663208, -5.993359088897705, -6.281533718109131, 7.031883716583252, 6.419923782348633, -17.03285026550293, -9.854390144348145, -2.1650750637054443, 12.013197898864746, 4.51704740524292, 6.098584175109863, 4.911830902099609, 2.691786050796509, 3.540828227996826, -2.3479299545288086, 0.620637059211731, -3.256791591644287, 13.085668563842773, -11.234673500061035, 3.419405221939087, -14.348674774169922, -8.322184562683105, -2.7014529705047607, -17.2869815826416, -27.75304412841797, 12.256097793579102, -10.563709259033203, -10.139904022216797, -2.8967976570129395, -15.15675163269043, 9.901535987854004, 13.51362419128418, -17.684722900390625, -10.861738204956055, 13.024197578430176, -4.572473049163818, 8.709465026855469, -8.809709548950195, 4.012567520141602, -5.973377227783203, -5.872560501098633, -10.565023422241211, -17.571186065673828, 4.055206298828125, 4.553396701812744, 3.3227038383483887, 1.8455193042755127, 4.2853169441223145, 7.742643356323242, 21.427457809448242, -2.2689297199249268, 1.7622568607330322, -8.971698760986328, 9.174592971801758, 3.4214348793029785, -12.721518516540527, -8.002625465393066, -0.706155002117157, -2.755737543106079, -0.6565299034118652, -2.053375720977783, 8.132073402404785, -0.432791143655777, -19.044963836669922, -2.040098190307617, -13.338126182556152, 3.000044584274292, -0.2347823977470398, -14.125490188598633, -2.7697105407714844, -8.532215118408203, -15.508354187011719, -2.482928991317749, -5.288251876831055, -10.890434265136719, -6.859829425811768, -14.310287475585938, -3.841959238052368, -12.339775085449219, -11.35738754272461, -5.8586225509643555, -11.434734344482422, -2.9965174198150635, 17.479045867919922, -11.732105255126953, -19.091991424560547, 5.828173637390137, 17.37194061279297, 11.569509506225586, -2.60162353515625, -3.436450481414795, 7.2445173263549805, 16.110881805419922, 4.876293659210205, -1.8039511442184448, 12.264909744262695, -18.230777740478516, -5.528009414672852, -2.1055362224578857, -11.321133613586426, -21.10540008544922, -1.6985208988189697, -7.797571182250977, 1.1488900184631348, 16.631267547607422, -0.3295954465866089, -15.810975074768066, 5.337374687194824, -0.19650495052337646, -3.2562448978424072, 9.819095611572266, -15.222379684448242, 1.402801275253296, 11.085319519042969, 6.528740882873535, 6.257964134216309, 22.766925811767578, 7.874966621398926, -6.582400798797607, -1.4263702630996704, 4.224390506744385, 10.67814826965332, 17.218734741210938, 1.6178566217422485, 4.604962348937988, 11.928691864013672, 11.289473533630371, 9.840662002563477, -3.8231194019317627, 13.069271087646484, -10.303842544555664, 22.232187271118164, -2.095284938812256, 4.9193525314331055, -7.660958290100098, 1.7095218896865845, 2.0739595890045166, -2.4794921875, -3.9851951599121094, -16.082374572753906, 17.756847381591797, -4.907138347625732, -4.219006538391113, 4.31683349609375, 9.075445175170898, 0.5950031280517578, -0.47298914194107056, -8.336465835571289, -12.181586265563965, -7.07172155380249, -1.2339247465133667, -10.662178039550781, -0.6527340412139893, -6.128403663635254, -9.133600234985352, 11.672880172729492, 10.474419593811035, 5.8331298828125, -5.716183662414551, 3.9394564628601074, 11.651386260986328, 10.214946746826172, -17.60164451599121, 1.2259397506713867, -0.4392552971839905, -3.7939870357513428, -9.616451263427734, -11.666985511779785, -8.197223663330078, -17.53850746154785, 3.702733039855957, 6.04086971282959, -11.727089881896973, -0.4088052809238434, 25.082616806030273, -1.6773921251296997, 18.29113006591797, 0.3560201823711395, -0.9823881983757019, -14.612666130065918, -19.59203338623047, -0.9174091815948486, -1.8880552053451538, 0.7008366584777832, 24.484729766845703, 4.431768417358398, -8.170493125915527, -3.338200092315674, 1.7544597387313843, 9.115148544311523, -8.262847900390625, -14.59218978881836, 0.2888685166835785, -11.531042098999023, 11.019115447998047, -11.492630004882812, -19.109817504882812, -2.566840887069702, -13.05981731414795, -2.0297203063964844, -3.2317357063293457, 1.238455057144165, 5.276479721069336, -6.485476493835449, 8.489631652832031, 10.062063217163086, -2.8509984016418457, -25.036712646484375, -14.63569450378418, -5.120422840118408, -13.213349342346191, 3.912423849105835, 11.288795471191406, 3.4429988861083984, 7.254297733306885, -6.559482574462891, -15.175073623657227, -4.157042980194092, 5.692036151885986, -17.097585678100586, 5.717742443084717, -7.113334655761719, 3.8622055053710938, 2.4791438579559326, -5.508121967315674, -5.115298748016357, 0.38108232617378235, -2.5465619564056396, -11.332120895385742, -9.41786003112793, 12.307206153869629, -18.04344367980957, -12.609899520874023, 1.4476020336151123, -7.111856460571289, 3.7152645587921143, 2.909252166748047, -8.546833992004395, -9.239422798156738, -1.3090044260025024, 5.793728828430176, -4.686126232147217, 7.223670482635498, -3.7282676696777344, -12.666468620300293, 12.559436798095703, 0.5771406888961792, 5.09507942199707, -14.229202270507812, -9.24560546875, 6.452939987182617, 10.190818786621094, -0.16234326362609863, -12.004843711853027, 1.2992439270019531, -6.03537130355835, 4.135202407836914, -3.154303789138794, 13.55666732788086, -1.4213619232177734, 12.624322891235352, 1.9957635402679443, 1.8604154586791992, 3.3075199127197266, 7.856988430023193, -8.79642105102539, -16.268531799316406, -4.833171844482422, 3.676382064819336, -3.439976692199707, -4.99144983291626, -0.691336989402771, -6.53585147857666, -2.141054391860962, 3.6555609703063965, 1.050638198852539, -10.967010498046875, -1.5917712450027466, 14.2998046875, 0.2513652741909027, -4.149936676025391, 1.8220462799072266, 3.870847225189209, 2.6084365844726562, 0.29070475697517395, 2.439232349395752, -2.4260826110839844, -27.184513092041016, 2.369680881500244, -4.236270427703857, 12.64528751373291, -4.808685779571533, 2.5229578018188477, 9.19558334350586, -9.348474502563477, -7.131728649139404, 4.434739589691162, -3.1816043853759766, -0.7441518306732178, 19.6740779876709, 27.018327713012695, -23.11669158935547, -11.689001083374023, 3.2323429584503174, -1.3563560247421265, -14.273214340209961, -3.4515633583068848, -9.258264541625977, -1.8802871704101562, 3.6033735275268555, 7.941259860992432, -11.499466896057129, -7.292997360229492, -4.867122650146484, 21.720401763916016, 4.5876312255859375, 9.48836898803711, 13.856086730957031, -1.5808366537094116, -12.424631118774414, 0.46567168831825256, 12.033824920654297, 4.943133354187012, -20.314128875732422, -2.503356456756592, 18.00674819946289, -7.780142307281494, -0.6043493151664734, 5.846684455871582, -27.60086441040039, 10.241583824157715, -0.16879825294017792, 2.526637077331543, -2.0105502605438232, 3.8832497596740723, 4.892474174499512, 1.6485910415649414, -8.99384880065918, 1.4062159061431885, -3.7131056785583496, 4.383081436157227, -14.61217975616455, -4.933646202087402, 6.955309867858887, -9.097128868103027, 8.928459167480469, -6.72502326965332, -15.051111221313477, -9.12939453125, -7.6699724197387695, 1.2034437656402588, -3.213498830795288, -12.72722053527832, 5.173068523406982, 1.3162364959716797, -14.499174118041992, 5.654170989990234, 1.6013565063476562, -1.4932851791381836, -7.067131042480469, -11.719365119934082, 23.0711727142334, -5.5243635177612305, -1.6578155755996704, 8.373725891113281, -2.8707046508789062, 6.618727684020996, -13.020641326904297, -10.754385948181152, 3.325972080230713, -13.524807929992676, -6.903564453125, -14.893108367919922, -13.409826278686523, 0.2718672752380371, 17.822288513183594, 1.8205440044403076, -15.254202842712402, 4.418754577636719, -2.6799168586730957, 5.690309524536133, -7.586284637451172, -18.983352661132812, 15.843220710754395, -5.645997047424316, 5.358491897583008, -2.392889976501465, -5.576230049133301, -2.8726794719696045, 5.086331367492676, 3.8367128372192383, 1.5702582597732544, -0.7507032752037048, 7.297172546386719, -25.408851623535156, -4.7917799949646, -3.1103878021240234, -7.893219470977783, -16.8648624420166, -3.4787473678588867, -4.984981536865234, 6.123837947845459, -2.008100748062134, 4.113480567932129, 12.909561157226562, -10.253473281860352, -12.41383171081543, 4.426845073699951, -6.645208358764648, 2.4961767196655273, 4.814148426055908, 4.269059181213379, -1.3826611042022705, -14.309602737426758, 7.007054805755615, 2.615511894226074, -2.5411930084228516, -1.0854392051696777, 7.2642669677734375, -0.6799728870391846, 3.5226049423217773, 23.607595443725586, -16.909318923950195, -3.4729442596435547, -8.603322982788086, -10.413297653198242, 0.916843056678772, 11.967066764831543, -1.7556543350219727, 7.5016984939575195, -0.815161943435669, -3.5031487941741943, -2.132673740386963, 19.76689910888672, -3.273207902908325, -14.178133964538574, -5.110976219177246, 3.069432020187378, 8.379851341247559, -7.001257419586182, 2.015294075012207, -13.254653930664062, -6.402425289154053, 8.825841903686523, 18.32583236694336, -7.902738571166992, 4.540467262268066, -20.08646011352539, -6.822443962097168, -15.434752464294434, -8.191963195800781, 6.150445938110352, -16.22787094116211, 11.594744682312012, 7.759071350097656, 16.134702682495117, 3.933802604675293, -5.789679527282715, 1.3862125873565674, -6.967195510864258, 1.6443520784378052, 12.412286758422852, 2.025381565093994, 7.911096572875977, 13.090936660766602, -3.053314447402954, 14.042019844055176, 11.803693771362305, -0.5998302102088928, 5.326733589172363, 5.21368932723999, -1.7590017318725586, 0.9161576628684998, -11.682340621948242, -7.876066207885742, 0.6509873270988464, -12.503033638000488, 1.4103013277053833, 11.94126033782959, -12.83598804473877, 9.705554962158203, -16.747167587280273, 4.941928863525391, -12.215166091918945, 2.0490427017211914, -1.8568958044052124, 10.276512145996094, 7.6022047996521, 3.5981454849243164, -9.098703384399414, -13.10240650177002, 6.310024261474609, 4.134392261505127, 9.921584129333496, 12.494911193847656, -8.763067245483398, 13.789838790893555, 16.547204971313477, -1.7657675743103027, -0.12568286061286926, 13.268909454345703, -6.3716630935668945, -11.048394203186035, 1.2863478660583496, -1.7710928916931152, -8.672098159790039, -4.068685531616211, -9.886816024780273, -2.7540934085845947, 23.5157470703125, -0.21387916803359985, 10.908236503601074, 1.996544361114502, 3.9964678287506104, 19.014822006225586, -8.22883415222168, -17.262723922729492, -4.409431457519531, -21.499500274658203, -2.2543013095855713, -3.7507712841033936, -15.002951622009277, 6.2915143966674805, 10.798151016235352, -0.6990980505943298, 5.875697612762451, 11.648069381713867, 19.513782501220703, 5.381580352783203, -22.235090255737305, 16.294849395751953, -12.955174446105957, -18.211719512939453, -3.6575534343719482, 13.17630386352539, 5.158534526824951, -16.48297119140625, -8.861103057861328, 13.20245361328125, 10.080205917358398, 4.615455627441406, -8.021308898925781, -11.706032752990723, 17.070068359375, 1.8706433773040771, 10.200462341308594, -0.9922035932540894, -1.6021019220352173, -19.516937255859375, 3.0217041969299316, 8.725187301635742, -1.7258954048156738, -7.04038667678833, -5.071384429931641, 1.3026739358901978, 0.9522424936294556, -4.557392120361328, -6.402671813964844, -2.5728371143341064, 6.447629451751709, -0.6745600700378418, -8.271148681640625, 5.857409477233887, -7.120965003967285, 6.824171543121338, 0.5579004883766174, 6.121283054351807, -9.258830070495605, 8.416565895080566, 10.5018949508667, 2.138718366622925, -4.880943775177002, -19.742652893066406, -3.1250829696655273, -6.709638595581055, 7.533625602722168, 2.549924612045288, -3.372260093688965, -0.6330010890960693, -4.152317523956299, 7.485556602478027, -2.701132297515869, -14.764244079589844, 2.0046279430389404, -8.993438720703125, -4.510856628417969, 24.36815643310547, -1.5953184366226196, 5.604128360748291, -1.595729112625122, 6.71356725692749, -4.178853511810303, 16.289276123046875, 6.616224765777588, 15.089855194091797, -1.4096274375915527, 5.541078567504883, 1.4819231033325195, -15.603991508483887, 8.932369232177734, -7.850783348083496, -0.027710067108273506, -0.26860347390174866, 1.5653008222579956, 2.495812177658081, 8.360010147094727, 11.248092651367188, 0.44877851009368896, 0.16870009899139404, -7.074620246887207, -4.517362594604492, 8.641469955444336, 7.491889953613281, -13.567911148071289, -2.0186030864715576, -1.7537906169891357, 10.715330123901367, -7.390760898590088, -8.045882225036621, 7.598137855529785, -1.8345308303833008, -7.702619552612305, -3.2655069828033447, 8.746915817260742, -4.182246208190918, -12.717903137207031, 0.8134369850158691, -16.90595245361328, 6.132455348968506, -3.1723439693450928, -9.983537673950195, -6.9717817306518555, -0.4141388535499573, 13.561968803405762, -15.871392250061035, 8.408612251281738, 6.806216716766357, -3.936318874359131, 13.831705093383789, 13.115052223205566, -0.31758397817611694, -0.7687849402427673, 5.582260608673096, 2.348860263824463, 16.610353469848633, -8.754842758178711, 2.2688446044921875, 11.545270919799805, -8.903435707092285, 5.820176124572754, -7.142958164215088, 5.0588836669921875, 5.9089202880859375, -2.5131044387817383, 12.167162895202637, 4.219500541687012, 10.019444465637207, 2.7940571308135986, -9.833051681518555, -5.801566123962402, 6.205494403839111, 0.3713681697845459, -0.5937762260437012, -1.144380807876587, 25.210573196411133, -17.85481834411621, 1.8116176128387451, -23.363492965698242, -18.42462730407715, 1.759998083114624, 1.4108556509017944, -0.2783556282520294, 0.6964907050132751, 0.8763620257377625, -5.078751564025879, 7.290088653564453, -18.205429077148438, 11.832006454467773, 6.6512017250061035, -8.589639663696289, -3.367940902709961, 13.51836109161377, -13.711353302001953, -13.965725898742676, 3.434553384780884, 3.976696491241455, -3.650456428527832, 9.157035827636719, 2.250455379486084, -11.548879623413086, 10.091489791870117, 7.064937591552734, -21.165786743164062, 0.8641700744628906, 7.543921947479248, -22.975509643554688, -15.698580741882324, 7.032821178436279, -10.243650436401367, -8.36375617980957, -2.274466037750244, -5.786052703857422, -4.222942352294922, 5.9561614990234375, 7.108059883117676, 14.637431144714355, -0.6344547271728516, 15.711894035339355, -26.179161071777344, -7.293305397033691, 1.0014623403549194, -12.397550582885742, -7.751859664916992, -10.274822235107422, 7.119231224060059, 3.8339900970458984, -3.102276086807251, -10.323958396911621, -9.542496681213379, -13.492722511291504, 13.646358489990234, -5.024815559387207, 3.067336082458496, 9.182580947875977, -7.300361633300781, -14.98178482055664, -2.6348824501037598, -7.90817928314209, -13.163116455078125, 22.31698226928711, -10.451118469238281, 9.419588088989258, 5.808801174163818, 16.538639068603516, -3.3554301261901855, -8.420239448547363, 5.89125394821167, -9.142240524291992, -0.038376256823539734, 3.880767583847046, -6.718741416931152, 5.758044719696045, -13.291025161743164, -8.106501579284668, 25.100839614868164, 10.239818572998047, -17.249195098876953, -7.241182327270508, -15.43030834197998, -20.778976440429688, 4.295426845550537, 4.009706497192383, -16.429088592529297, -25.802614212036133, -42.02930450439453, 2.468630313873291, -1.798667311668396, 21.939050674438477, -18.613435745239258, -0.8276227712631226, -5.753209590911865, -4.676233291625977, -0.45399752259254456, -6.4310078620910645, -2.4671642780303955, -3.959813356399536, -2.4476864337921143, -1.786966323852539, 2.808884620666504, -9.396291732788086, -2.208475351333618, -18.687421798706055, -6.925949573516846, -9.014617919921875, -2.2440435886383057, -2.4179835319519043, 6.191302299499512, 9.814170837402344, -13.287507057189941, -1.766885757446289, 11.548762321472168, 5.868163108825684, -16.661746978759766, 3.254950523376465, 2.067523717880249, 3.294572353363037, -15.54316520690918, -21.415300369262695, 14.50745677947998, 2.509981393814087, 20.515520095825195, -5.057404518127441, -6.7826247215271, 3.7061285972595215, -6.7202606201171875, -8.246505737304688, 2.6955273151397705, 1.1158193349838257, 2.0938668251037598, -0.5962038636207581, 3.9156923294067383, 17.855525970458984, 6.5626044273376465, -6.056979179382324, -13.697040557861328, -11.733369827270508, 8.772154808044434, 0.8080368041992188, -0.3748801350593567, 0.5302044153213501, 8.70995044708252, -3.2959375381469727, -2.070749282836914, 9.747511863708496, -1.1727181673049927, 1.1765267848968506, -10.057962417602539, -0.8084544539451599, 6.353420734405518, -6.24655294418335, -0.1485489457845688, -9.015268325805664, -6.924774646759033, -1.7353527545928955, 4.715018272399902, 1.494887113571167, 9.917915344238281, 6.939392566680908, 16.15015411376953, -2.674377918243408, -6.580310821533203, -16.45635414123535, -2.9573912620544434, -20.937898635864258, 15.832260131835938, -4.687900066375732, -12.499509811401367, 10.442533493041992, 10.719990730285645, 1.1020091772079468, 8.634003639221191, 19.323070526123047, 3.4948909282684326, -3.0714402198791504, -13.475276947021484, 11.813493728637695, -11.68871021270752, 2.1319541931152344, 1.0427932739257812, 12.18171215057373, 6.016721248626709, -4.858137130737305, 3.2445173263549805, -3.9259183406829834, -1.7778252363204956, 13.66949462890625, -16.237689971923828, -8.856412887573242, -2.149397850036621, 5.835348129272461, -7.575430870056152, 12.568145751953125, 6.844526290893555, -3.9899425506591797, -13.425668716430664, 7.177483558654785, -18.279626846313477, 0.6930314302444458, 0.8962801098823547, 2.249512195587158, -14.126280784606934, -20.09146499633789, 11.924055099487305, 11.405986785888672, 2.8109118938446045, -5.429858207702637, -1.0830820798873901, -6.390973091125488, 8.170587539672852, 8.997987747192383, -4.262169361114502, -0.4722149670124054, -0.3059580624103546, 1.684810996055603, -10.20673656463623, 7.319941997528076, 6.497172832489014, 12.490654945373535, 4.910691261291504, 16.733610153198242, -4.775303363800049, -4.919050216674805, 1.3472377061843872, 14.817627906799316, 14.939404487609863, 7.0401482582092285, 12.032352447509766, 6.480503559112549, 11.358384132385254, -7.3138957023620605, -11.489091873168945, 3.686190128326416, 6.007315635681152, -1.6600720882415771, -2.06119966506958, 4.8122429847717285, -8.440953254699707, -20.490924835205078, 0.5617929697036743, -3.7603774070739746, 1.1800414323806763, 2.7388663291931152, 18.1621150970459, 6.28091287612915, -7.726235389709473, -1.1389516592025757, 9.778413772583008, -5.5805487632751465, -0.26421064138412476, -0.9947865009307861, -9.915732383728027, 5.597519397735596, 18.936391830444336, 8.136255264282227, -7.822678565979004, -3.9709243774414062, 1.5811588764190674, 3.7242820262908936, -0.5131807327270508, -3.9882895946502686, -12.27992057800293, 9.25735855102539, -5.552072048187256, 12.451104164123535, -1.6275113821029663, 3.6605846881866455, 6.689844608306885, 5.118429183959961, 12.41541862487793, 3.0424964427948, 2.894329071044922, 15.476493835449219, 12.416905403137207, -7.533778190612793, -19.51357078552246, 4.345785617828369, 0.2657356262207031, 4.015079498291016, -0.7279160022735596, -2.929795980453491, -22.67728614807129, -10.123623847961426, -8.696606636047363, 11.984600067138672, 1.040820598602295, 11.077942848205566, 13.705430030822754, -12.45865249633789, -10.72447681427002, -5.037036418914795, 18.099712371826172, 13.703104019165039, -5.295418739318848, 0.09086206555366516, 7.390814304351807, 14.370938301086426, 7.777355194091797, -9.494132041931152, 3.02565860748291, 25.221555709838867, 13.728734016418457, -4.702977657318115, -2.16819167137146, -18.719083786010742, -1.8241028785705566, 18.812686920166016, -1.6205170154571533, 7.204962730407715, 11.910690307617188, 17.16001319885254, 8.697421073913574, 3.5806174278259277, -7.751287937164307, -2.1369247436523438, -4.393229007720947, -13.514876365661621, -10.27025032043457, -8.611601829528809, -6.009525775909424, -4.859244346618652, 7.276923656463623, -3.071964979171753, -16.241966247558594, -0.07960367202758789, 20.260055541992188, 6.215193748474121, -1.8425533771514893, 3.7717769145965576, -2.3552730083465576, -1.8273117542266846, -12.997550010681152, 6.414511203765869, -5.883736610412598, -0.3419634997844696, 4.592335224151611, 1.0232903957366943, -3.0091426372528076, 8.688910484313965, 3.1250224113464355, 6.196871280670166, 11.180991172790527, 5.600457668304443, -3.710392475128174, -0.47431260347366333, 5.575527191162109, 3.7898354530334473, -13.855690956115723, 5.963850498199463, -1.476309895515442, -9.040657997131348, -13.341405868530273, 20.51570701599121, -17.8002986907959, 13.58427619934082, 16.282690048217773, 6.142426490783691, -4.879781246185303, -11.536870002746582, 11.399803161621094, 6.444119930267334, 10.000248908996582, -3.2971370220184326, -10.296746253967285, 4.565488815307617, -1.986060619354248, 7.164768218994141, -2.0021767616271973, -10.427852630615234, -13.98541259765625, 0.14810322225093842, 1.9533199071884155, -8.093067169189453, 3.9445960521698, 3.934025287628174, -3.594398260116577, -7.702455520629883, -14.354528427124023, 11.25445556640625, -13.627642631530762, 3.336437940597534, 12.298483848571777, 14.029029846191406, 3.1668269634246826, 4.127346992492676, -15.877218246459961, 6.468236446380615, -5.809879779815674, -17.87529182434082, -5.247852802276611, -6.011366367340088, -4.09395170211792, 0.6187000274658203, 10.050444602966309, -11.184672355651855, -11.86738395690918, 4.263123989105225, -1.6603684425354004, 22.40238380432129, -20.181751251220703, -13.250164985656738, -8.073493957519531, 5.0765380859375, -8.829702377319336, 17.429359436035156, 8.950082778930664, -3.628065824508667, 10.468859672546387, -5.159128189086914, 11.586606979370117, 0.758778989315033, 11.818181037902832, 20.844463348388672, 5.713109016418457, -11.719533920288086, -13.199636459350586, -10.384303092956543, -28.27437973022461, 14.352977752685547, 4.4442949295043945, -0.976538896560669, -2.823488473892212, -11.114392280578613, -1.9157960414886475, 20.44760513305664, -1.8937883377075195, 1.673775315284729, 4.2971954345703125, 16.783662796020508, -6.79513692855835, 8.27393913269043, 5.80085563659668, -18.549516677856445, -4.175284385681152, 13.302865982055664, -1.0922608375549316, 4.425710201263428, 11.583023071289062, -2.105562448501587, -2.3367598056793213, 4.59987735748291, 7.76153564453125, -1.0188956260681152, -11.8452730178833, 2.4459445476531982, -3.5682084560394287, -14.881685256958008, 2.01839542388916, -7.0885701179504395, -7.23858642578125, 11.41044807434082, -8.414673805236816, 13.694772720336914, -11.037269592285156, 16.276653289794922, -14.693373680114746, -1.3787580728530884, 9.1490478515625, 5.434370994567871, 16.31694984436035, -8.265378952026367, -10.026016235351562, 11.00082778930664, 8.090628623962402, 14.473670959472656, -0.824485182762146, -14.597900390625, -5.496402740478516, -10.221724510192871, -4.317813396453857, 2.8018877506256104, 7.687442302703857, 4.494405269622803, -9.344646453857422, -7.902701377868652, 0.7948949933052063, -9.169361114501953, -14.984363555908203, 7.115571022033691, 1.9473363161087036, -12.898605346679688, 6.079850196838379, 3.733258008956909, -3.8089654445648193, 6.576626300811768, 3.8311614990234375, 19.300922393798828, -0.6973708271980286, -6.58088493347168, -16.062171936035156, -9.150101661682129, -5.876029014587402, -2.842409372329712, -6.950083255767822, 20.094409942626953, 21.852256774902344, -3.7994651794433594, 14.943547248840332, 9.641434669494629, -8.35112476348877, 3.453240156173706, -8.67216682434082, 16.87271499633789, -4.743577003479004, -13.979202270507812, 9.015947341918945, -0.9188036918640137, -18.661237716674805, -7.259814262390137, -8.786341667175293, 10.258736610412598, -16.616382598876953, -8.737358093261719, -10.22070026397705, -5.131319046020508, 25.24292755126953, 1.3491910696029663, 14.766768455505371, 35.89271926879883, 0.5391630530357361, 6.481611251831055, -10.211383819580078, -13.25994873046875, 2.295794725418091, 8.074111938476562, -18.727367401123047, 7.970400810241699, -2.657076835632324, 2.269988536834717, 15.971685409545898, 13.133733749389648, -5.643585205078125, 1.2828631401062012, 4.440620422363281, 14.782306671142578, -9.657073020935059, 10.969640731811523, -9.299325942993164, 5.040899276733398, 12.522587776184082, 0.8221307992935181, 0.7936353087425232, 9.328592300415039, -6.984634876251221, -25.353641510009766, 6.3344926834106445, 17.543941497802734, 7.363278865814209, -0.7581712007522583, 13.834153175354004, -9.171158790588379, -6.956624507904053, -13.597006797790527, 3.3930225372314453, -2.3176493644714355, 3.442626476287842, -5.938138008117676, -13.379484176635742, 11.602924346923828, -23.25672149658203, -0.5149377584457397, 8.157273292541504, -3.5742554664611816, 8.671356201171875, -18.5937442779541, 16.514175415039062, -4.207437992095947, 3.865842819213867, -4.719399929046631, 8.393247604370117, 19.356597900390625, -9.304000854492188, 8.70553207397461, -2.044748544692993, -4.932595729827881, 2.614772319793701, 0.27160927653312683, 6.604578971862793, -6.035327434539795, 12.347025871276855, -15.422621726989746, 7.701839923858643, -3.3165624141693115, -16.967254638671875, -12.186016082763672, -2.8719911575317383, -14.706374168395996, 9.55639362335205, 10.72938060760498, 11.344243049621582, 8.091232299804688, 18.600303649902344, 2.964475154876709, 0.5506461262702942, 18.989328384399414, -4.867829322814941, -9.959193229675293, -12.70048999786377, 2.6796975135803223, 6.715428829193115, 0.25838056206703186, 12.940970420837402, 1.7133727073669434, -6.1032209396362305, 10.990647315979004, -0.8867896199226379, 6.184582710266113, -2.8203704357147217, 6.9767279624938965, 17.096149444580078, -6.061481952667236, 9.691780090332031, 9.751993179321289, -0.19508399069309235, -8.317858695983887, 3.218893051147461, 3.1432414054870605, 13.175066947937012, 12.930858612060547, -0.2956501245498657, -8.530882835388184, 6.074916839599609, -16.65176773071289, 7.568907260894775, -1.2566981315612793, -9.878202438354492, 17.141935348510742, 23.747053146362305, -8.640826225280762, -24.578025817871094, 10.065045356750488, -20.584400177001953, 11.075101852416992, -10.91093635559082, 2.0700840950012207, -5.688017845153809, -6.580886363983154, -3.143718719482422, -0.365192174911499, 14.992334365844727, 8.576658248901367, -4.123027324676514, -4.453560829162598, 11.144423484802246, 5.194980621337891, 11.122989654541016, 1.0785346031188965, -1.526548147201538, 15.151625633239746, 1.9291554689407349, 1.5512259006500244, -1.8711484670639038, 0.24181769788265228, -3.194267988204956, -1.600031852722168, -10.398427963256836, 6.248710632324219, -4.049076557159424, -10.141313552856445, -2.200472116470337, 3.7182908058166504, -10.832709312438965, -4.2802839279174805, 0.7988131046295166, -1.358965516090393, 2.34480881690979, -24.900779724121094, -0.511967658996582, 3.9625823497772217, -3.3305435180664062, 9.569846153259277, -7.255847454071045, -12.729188919067383, 6.699283123016357, -3.2517757415771484, -1.843812346458435, 18.1757755279541, -1.416532278060913, -16.464805603027344, -5.15119743347168, -4.546928882598877, 6.945071220397949, 3.7621495723724365, 8.755353927612305, -17.801071166992188, -5.8394341468811035, -8.82275676727295, 1.68874990940094, 5.403543472290039, -8.257259368896484, 19.083890914916992, -6.386482238769531, 10.077258110046387, 7.717004299163818, 12.590478897094727, -3.006286144256592, -7.601041793823242, 13.609643936157227, 9.032159805297852, 0.27330201864242554, 0.6082311272621155, -14.562601089477539, -10.433463096618652, 9.57354736328125, -14.82097053527832, -16.79620933532715, -3.5108163356781006, -0.971580445766449, 8.214164733886719, -12.894240379333496, -4.233047962188721, 10.248104095458984, -2.370464324951172, -5.128176689147949, 4.080225944519043, -7.583900451660156, -11.038074493408203, -4.752277851104736, 16.826189041137695, -1.1556766033172607, 0.8086078763008118, -3.316892385482788, 1.8851901292800903, 1.2637580633163452, 7.01310920715332, 12.624689102172852, -2.3633251190185547, -7.9906535148620605, 17.39381217956543, -1.2854970693588257, 9.609983444213867, -8.938065528869629, 2.547321081161499, -4.652090072631836, 7.204177379608154, -11.55622386932373, -16.158329010009766, -5.104465484619141, 8.323746681213379, -1.5313507318496704, 8.040996551513672, 3.491807460784912, -8.3121337890625, -15.037951469421387, -0.8699536919593811, -10.946062088012695, 12.450652122497559, -0.7489433884620667, 5.211711406707764, -24.1283016204834, -8.535898208618164, 1.0795996189117432, -10.868934631347656, 0.568504273891449, -0.8357265591621399, -6.317697525024414, 0.3101221024990082, -1.3858996629714966, 24.938419342041016, 23.259279251098633, -9.578343391418457, 11.339864730834961, -11.019145965576172, 3.969087600708008, -3.489513635635376, 13.78090763092041, 7.136837005615234, 5.2297539710998535, -4.114038467407227, 6.151524066925049, -0.9622559547424316, 1.6407420635223389, 9.280022621154785, 4.888638019561768, -8.718446731567383, 8.096019744873047, 4.558582782745361, 9.741393089294434, -2.3422658443450928, -1.2748198509216309, -12.852602005004883, 4.981139659881592, -3.1935997009277344, 15.09707260131836, 2.284360408782959, -2.9137840270996094, -4.001153945922852, -5.8580851554870605, -2.842289924621582, -7.731969833374023, -7.357931137084961, 12.94801139831543, -11.803665161132812, -4.729568004608154, 1.8718867301940918, 0.420958936214447, -6.507998466491699, -6.831679344177246, -13.558429718017578, 7.957513809204102, -1.6508510112762451, -17.203495025634766, -3.056215286254883, 18.865310668945312, -13.26715087890625, -2.3638246059417725, -1.079106092453003, -21.411733627319336, 2.6705942153930664, -7.523317813873291, 8.890846252441406, 5.350490570068359, -20.72162628173828, 8.544057846069336, 5.715852737426758, -0.792258083820343, 8.50728988647461, -19.698293685913086, -1.2204461097717285, -8.936237335205078, 7.983150482177734, 3.6496353149414062, -6.911682605743408, 3.2636115550994873, 14.02997875213623, 13.347684860229492, 11.800131797790527, 4.302647590637207, 4.665635585784912, 8.994375228881836, -3.6842010021209717, -10.667096138000488, 6.304357528686523, 19.63837432861328, -8.382344245910645, 3.4971847534179688, 8.51291561126709, 5.057553768157959, 4.867201328277588, 0.23575085401535034, -3.733319044113159, -12.71811294555664, 1.9012935161590576, -0.9700629711151123, -5.146075248718262, -2.617311477661133, 10.260554313659668, -20.04098892211914, -6.0816192626953125, -5.034148216247559, 1.813714623451233, -7.739541053771973, -10.46479606628418, -0.7453984022140503, 8.227642059326172, -4.742990016937256, 14.633424758911133, 3.5050106048583984, -0.9606030583381653, 3.3151702880859375, -6.732618808746338, 3.116907835006714, 13.206111907958984, -2.555482864379883, -16.50177764892578, 13.55213737487793, -3.1641218662261963, 5.617099761962891, 4.538601875305176, 8.938576698303223, -8.14961051940918, -0.5394583344459534, 10.873762130737305, 1.2496049404144287, -12.207735061645508, 12.89180850982666, -6.301081657409668, 13.007973670959473, 30.774667739868164, 2.8144896030426025, -7.886934280395508, -2.3852133750915527, -11.672714233398438, -13.902992248535156, 3.84089994430542, 5.074267387390137, 4.0384697914123535, -5.1263933181762695, 10.555410385131836, -6.724742889404297, -16.560970306396484, 9.66435432434082, 15.12495231628418, 15.650915145874023, -11.550027847290039, -10.175114631652832, 22.589275360107422, -9.761030197143555, -5.925293922424316, -3.699003219604492, -15.857223510742188, 5.430155277252197, -3.6136748790740967, 12.303762435913086, 7.895686626434326, -10.080224990844727, -1.8814860582351685, 3.3300983905792236, 10.72122573852539, 9.252904891967773, -3.8951303958892822, -3.8148927688598633, 10.60450553894043, -3.3309171199798584, 1.1238847970962524, -8.334835052490234, 0.1194927990436554, -8.00178337097168, -0.11479493230581284, 3.50830078125, -20.286602020263672, 3.7564613819122314, -9.403570175170898, 8.271709442138672, -8.737460136413574, -5.765634536743164, -17.122343063354492, 10.664405822753906, -5.924554347991943, -3.1283178329467773, 7.641153335571289, 6.321101665496826, 5.418548107147217, -4.146618366241455, -13.691417694091797, 11.89152717590332, 1.7428897619247437, -3.4313595294952393, 1.074472427368164, 10.819543838500977, -2.917059898376465, 2.040958881378174, -8.774920463562012, 3.634315013885498, 3.2695415019989014, 0.10651035606861115, -21.93115997314453, 6.923383712768555, -7.075726509094238, 6.799139022827148, 3.0898313522338867, 14.556781768798828, 3.625229597091675, 2.56655216217041, -16.041250228881836, -23.269367218017578, -13.155706405639648, 0.6906264424324036, -13.024945259094238, 8.171770095825195, 5.0061750411987305, -2.7291760444641113, 8.985689163208008, -0.8349695801734924, 8.546663284301758, 0.7337606549263, -2.5162956714630127, 11.559929847717285, -16.976835250854492, 8.52967357635498, 10.288214683532715, -3.6352732181549072, -6.308929920196533, 20.403444290161133, -29.5002384185791, 8.744420051574707, -21.121206283569336, -18.84162712097168, 6.889640808105469, 14.808441162109375, 7.449025630950928, 0.5181609988212585, 4.033137798309326, -10.165555953979492, -3.931028366088867, 4.578177452087402, 11.552520751953125, 16.534381866455078, 10.312177658081055, -6.490749359130859, -4.932639122009277, 2.8344311714172363, -0.8786802291870117, 5.004748344421387, 7.50547456741333, 13.423877716064453, 14.689584732055664, -14.503813743591309, 8.437097549438477, 7.514358043670654, -4.962640285491943, 1.976870059967041, 1.7603271007537842, -5.569101333618164, 2.13177227973938, 6.678142547607422, 1.0362138748168945, 3.560068130493164, 5.450477123260498, 12.094926834106445, -10.241397857666016, 8.586221694946289, -15.946796417236328, -10.492168426513672, -3.5565457344055176, 6.85328483581543, 8.59154224395752, -14.737253189086914, -10.658939361572266, 2.475576162338257, 6.528754234313965, -9.85584831237793, 8.087553977966309, -4.9133429527282715, 16.765918731689453, 5.12840461730957, -4.424905300140381, 31.783416748046875, -9.722079277038574, -15.949763298034668, -8.335413932800293, -25.35394859313965, -3.947066307067871, 9.4279203414917, 4.775699615478516, -5.706609725952148, 6.791719436645508, -1.4073030948638916, -5.033524036407471, -6.8827972412109375, 8.643766403198242, -2.5336337089538574, -11.56613540649414, 2.6048574447631836, 2.3305740356445312, 13.331269264221191, -6.4822821617126465, -6.4371442794799805, -19.93729019165039, -7.301353931427002, -3.634613513946533, -16.84368896484375, 6.9759745597839355, -4.317460060119629, -1.9562263488769531, -6.217800140380859, 5.325661659240723, 15.339493751525879, 3.129213333129883, 1.5761544704437256, 18.053335189819336, -0.0023308517411351204, 23.511075973510742, -18.374919891357422, 15.286927223205566, 16.069276809692383, -12.039909362792969, -8.05356216430664, -5.378868103027344, 1.907058835029602, 25.319459915161133, 8.265477180480957, -4.54310941696167, 6.300309181213379, 1.9642781019210815, 10.46270751953125, -6.407590389251709, -6.501926422119141, -7.5290303230285645, 3.787513017654419, 4.4446516036987305, -17.322965621948242, 8.488973617553711, -8.39390754699707, 7.065728187561035, 3.4771718978881836, -2.3786630630493164, 6.7223711013793945, 11.455358505249023, -9.406447410583496, -1.4233888387680054, -0.8021138906478882, 0.8136756420135498, 6.949944972991943, -20.827083587646484, -6.842895030975342, 2.8299903869628906, 10.117118835449219, -4.122946739196777, 5.457338333129883, -11.657577514648438, 4.714829921722412, 2.7804532051086426, -7.472718238830566, 1.6674096584320068, -7.878894805908203, -7.3169403076171875, 0.291702538728714, -3.9309024810791016, 0.2364884614944458, -11.532201766967773, -13.543050765991211, 10.178384780883789, -12.004195213317871, -11.70067310333252, 4.87493896484375, -10.022089004516602, 1.7057850360870361, 3.069352149963379, -23.833431243896484, -11.18885612487793, -11.067748069763184, 2.6231515407562256, -10.638533592224121, -15.522443771362305, 18.35964584350586, -6.469912052154541, 8.347822189331055, -5.684505462646484, -5.320155143737793, -2.115671396255493, 6.567396640777588, 1.7793564796447754, 16.777917861938477, -15.304999351501465, 26.31534194946289, 14.488860130310059, 17.077333450317383, 7.668442249298096, -3.7405803203582764, -1.9284465312957764, -3.8236947059631348, 5.634732723236084, -9.114044189453125, -1.3739910125732422, -2.39631986618042, -4.420494079589844, 4.382931709289551, 12.584802627563477, 12.840021133422852, 4.050972938537598, -8.843019485473633, 10.106620788574219, -5.448763370513916, -7.697938919067383, -2.1266369819641113, 3.692876100540161, -2.5848472118377686, 8.911846160888672, -0.004104678053408861, -16.123424530029297, 11.685754776000977, 16.853397369384766, -7.736974716186523, 1.9482855796813965, 6.529360294342041, 0.9348280429840088, 10.444907188415527, -0.9683581590652466, 10.377496719360352, -8.430789947509766, 7.1556901931762695, -1.1753820180892944, 11.325542449951172, 7.249435901641846, 10.950286865234375, -18.965251922607422, -9.072633743286133, -17.761802673339844, -3.72314715385437, -8.922550201416016, -15.347748756408691, -9.548297882080078, 9.11280345916748, -9.343976020812988, -0.4193670153617859, -1.804077386856079, -1.1765857934951782, 6.70227575302124, 9.755196571350098, -5.079653739929199, 17.281335830688477, 5.66849422454834, 16.730703353881836, -9.426469802856445, -14.535879135131836, 9.500558853149414, 3.6026158332824707, -0.676453173160553, -3.9098832607269287, -3.7251555919647217, -0.19218826293945312, -3.535008192062378, -7.719144821166992, -17.792705535888672, 2.395124912261963, 5.910013198852539, 1.678830623626709, -14.140741348266602, -0.3586055040359497, 2.381423234939575, -11.014370918273926, -4.891652584075928, 5.982842445373535, 19.575002670288086, -12.716293334960938, 7.951035499572754, -9.045364379882812, -11.185120582580566, 5.7726850509643555, -23.937366485595703, 1.7760660648345947, -2.6966729164123535, -18.209415435791016, 0.37268105149269104, -1.898210048675537, -11.897083282470703, -2.3232929706573486, 4.619273662567139, -4.073969841003418, 2.0267274379730225, -17.45008087158203, 7.673874378204346, -12.396431922912598, -3.3427624702453613, 4.4707932472229, -0.6038575768470764, -16.51593780517578, 0.19720977544784546, 11.795292854309082, 3.645986557006836, -7.3489179611206055, -0.8582611083984375, 11.393716812133789, -11.661407470703125, -6.874194145202637, -11.081801414489746, -1.6174496412277222, 3.7054131031036377, -6.4383087158203125, -1.4033676385879517, -8.903064727783203, 9.2640380859375, -4.987002849578857, 7.5724053382873535, 12.10563850402832, -0.34248149394989014, -18.64688491821289, -8.689355850219727, -11.103205680847168, -3.5782060623168945, -5.432665824890137, -2.246892213821411, 5.060929775238037, 7.267385005950928, 0.048117198050022125, -11.764717102050781, 3.4423506259918213, 16.441509246826172, 4.507150173187256, 2.8986551761627197, -2.6707563400268555, -12.9821195602417, -3.1856093406677246, -9.145233154296875, 14.594315528869629, -0.13185322284698486, -0.3152371644973755, -5.1758809089660645, 0.30308547616004944, 1.776298999786377, -1.3918684720993042, 5.012388229370117, -6.908452987670898, -1.5255475044250488, 3.7061240673065186, 16.81412696838379, 14.07004451751709, 5.392796516418457, -2.8451576232910156, -10.761860847473145, -6.169645309448242, -2.8635170459747314, -8.20310115814209, 2.702249050140381, -25.399044036865234, 11.336668014526367, 0.19971607625484467, -6.845298767089844, 0.2374638020992279, 18.627910614013672, -6.244967460632324, -0.19391706585884094, 3.899949073791504, -4.487768173217773, 2.3258299827575684, -13.769488334655762, -18.773841857910156, 16.189006805419922, -13.789358139038086, 19.190271377563477, -16.36644172668457, 4.136173248291016, -9.02203369140625, 5.145342826843262, -2.6385655403137207, 7.7840576171875, -13.76156234741211, 3.236060619354248, 12.424161911010742, 15.164719581604004, 13.14925765991211, 0.2338118553161621, 0.4278363585472107, -12.384981155395508, -12.861985206604004, -1.2563145160675049, 25.80447769165039, -10.415787696838379, -0.22914372384548187, 2.5149240493774414, -13.314233779907227, 3.558664083480835, 10.641897201538086, 12.87850570678711, 13.637208938598633, -1.481304407119751, 7.701953887939453, -20.685016632080078, -5.112438678741455, -3.468060255050659, -0.6863963603973389, 7.355064392089844, -8.126079559326172, 7.580324172973633, -9.112980842590332, -10.557727813720703, 10.899094581604004, 13.013134002685547, 2.5237388610839844, -1.284018874168396, -2.78436541557312, -2.054121971130371, 3.1516666412353516, -32.78597640991211, -26.55389976501465, 4.380829811096191, -1.489830493927002, -1.5690301656723022, -11.585878372192383, -0.34712928533554077, 9.73122501373291, -13.651662826538086, 10.046194076538086, -7.447850704193115, -14.049958229064941, 9.686067581176758, -13.126432418823242, -7.225596904754639, 0.7444199323654175, -20.00298309326172, 18.651845932006836, -23.81929588317871, 4.882255554199219, 2.9825592041015625, 4.763372421264648, -7.48516321182251, 10.454100608825684, 5.740061283111572, -0.01075697597116232, -2.0319321155548096, 16.69341468811035, -5.7077412605285645, -9.053155899047852, -14.560083389282227, -19.569400787353516, 17.086347579956055, -2.1724259853363037, -9.523921966552734, -5.100240707397461, 3.426145076751709, 9.488310813903809, -10.312931060791016, -8.035879135131836, -23.329248428344727, 16.964244842529297, 5.366148948669434, -5.31632661819458, 2.5606348514556885, 11.70123291015625, 20.814878463745117, 15.67629337310791, -5.171468257904053, -2.573770046234131, 11.349273681640625, 13.435640335083008, 16.860891342163086, -3.717597723007202, 0.20757512748241425, -3.8407721519470215, 5.722125053405762, -1.8771743774414062, -15.7625093460083, -1.6961008310317993, -2.403447389602661, -4.088792324066162, -6.6489410400390625, 8.484939575195312, -4.24544620513916, -19.665237426757812, -4.321493625640869, -13.326827049255371, 6.61642599105835, 1.4268994331359863, -7.577924728393555, 14.905145645141602, 13.573297500610352, -1.0530095100402832, 10.357902526855469, -4.632663249969482, 9.350825309753418, 1.7502821683883667, -6.1251325607299805, -9.69178295135498, -6.767949104309082, -0.2404271811246872, 1.3157408237457275, 10.504298210144043, -4.739538669586182, 5.746201515197754, -7.832240104675293, -9.021650314331055, 3.2949411869049072, 0.1235930547118187, -3.472343921661377, 1.5592533349990845, -6.133512496948242, 6.948780536651611, -3.284507989883423, 1.3464285135269165, -17.78241539001465, 15.295561790466309, -9.200037956237793, 5.122348785400391, -6.49906587600708, 9.933320045471191, -4.18867301940918, 9.767668724060059, -19.744831085205078, 1.9659669399261475, 13.112550735473633, -6.638931751251221, 4.194652080535889, -1.0318405628204346, -0.2023518681526184, -14.772100448608398, 14.1898832321167, 8.408624649047852, 3.441807270050049, -2.14101505279541, 10.390210151672363, -16.481929779052734, -8.6912260055542, 0.3304404318332672, 6.754105091094971, 17.202938079833984, -10.97259521484375, 12.895010948181152, -6.783304691314697, -8.82802677154541, 7.339635372161865, -3.831505298614502, 10.244914054870605, -0.9118009805679321, -19.787931442260742, 9.147283554077148, -13.073736190795898, -0.346617728471756, -2.2453489303588867, 3.5616259574890137, -0.9139772653579712, 2.286700963973999, -15.165632247924805, -0.01882234774529934, 3.150470018386841, -11.166314125061035, 6.083574295043945, -3.0817201137542725, -4.283559799194336, 7.680523872375488, -6.0871500968933105, 3.2405307292938232, 16.621192932128906, -13.894743919372559, 4.337174415588379, -5.7752180099487305, -3.875462532043457, 1.6444259881973267, -31.199378967285156, 4.840592861175537, -5.4940266609191895, -8.820749282836914, 0.4837614595890045, 4.964733123779297, -3.02705979347229, 7.1443891525268555, -0.6805672645568848, -23.236614227294922, 21.815736770629883, -5.198546409606934, 14.579622268676758, -15.093473434448242, -8.240312576293945, -13.897817611694336, 13.462234497070312, -29.243125915527344, 0.39723220467567444, -4.526211738586426, 5.808864116668701, -10.098737716674805, 2.024874210357666, -10.722366333007812, 15.431182861328125, 0.6799517869949341, 16.13753890991211, 16.07280158996582, 9.095684051513672, 9.674798011779785, 6.3442792892456055, -1.8373287916183472, -0.6698748469352722, -3.3377466201782227, -8.703612327575684, 4.117521286010742, -14.018603324890137, -18.83539581298828, 7.721464157104492, 1.8203233480453491, 12.155470848083496, 9.564107894897461, 5.788939476013184, -0.24913230538368225, -6.754263877868652, -1.5211659669876099, 6.006422996520996, 14.318544387817383, 8.374429702758789, 4.845679759979248, -2.621903657913208, 4.763946056365967, 8.110928535461426, -13.216971397399902, 13.169252395629883, 0.32528239488601685, 5.133223056793213, 1.2104393243789673, -12.492912292480469, 0.3599514067173004, -0.2402179092168808, -16.368501663208008, 6.953351020812988, -0.7359138131141663, -7.41566276550293, 7.985478401184082, -1.4927237033843994, 1.3946309089660645, -13.334179878234863, -10.41057014465332, 4.225977897644043, 3.8658950328826904, -1.5640830993652344, -11.532621383666992, -3.78836727142334, -8.808093070983887, -3.646113634109497, 16.26313018798828, -0.8741323947906494, 4.81254243850708, -11.100894927978516, -7.1609907150268555, -2.9881510734558105, 22.06226348876953, -4.582761764526367, 3.9858663082122803, 1.7796623706817627, 3.0050442218780518, 21.459016799926758, -12.303938865661621, 17.51082992553711, -10.482115745544434, -17.336395263671875, 8.620627403259277, -9.161825180053711, -1.8424171209335327, 0.7776592969894409, -1.5467619895935059, -1.6156599521636963, -0.7917568683624268, 15.887018203735352, 10.07508373260498, 3.1359825134277344, -3.4953746795654297, 4.797966957092285, 1.8521089553833008, 27.2893123626709, -9.323138236999512, 3.3730709552764893, 12.586614608764648, -6.265923500061035, 4.986761093139648, -13.463554382324219, 27.744544982910156, 9.915946960449219, -10.646621704101562, 0.8103296160697937, -27.93502426147461, -0.0720037892460823, 2.6940860748291016, 15.458955764770508, 14.57077407836914, 7.281197547912598, -6.8253254890441895, 5.010735988616943, 3.4297890663146973, 3.637498378753662, 6.804567337036133, -5.6574578285217285, -5.302196502685547, -6.3169660568237305, -2.473353862762451, -11.932676315307617, 0.6871143579483032, 4.5069899559021, -4.775335311889648, 10.071813583374023, 1.992901086807251, 17.864604949951172, 5.943680286407471, 0.08855998516082764, -5.795605182647705, -4.9497880935668945, 15.093116760253906, 1.4150160551071167, -18.592449188232422, 2.7555317878723145, 5.667159080505371, 15.886988639831543, 3.7139108180999756, -1.1222563982009888, -7.276923656463623, -15.240304946899414, -17.993261337280273, -3.6814377307891846, 11.179293632507324, -10.497854232788086, -9.982441902160645, -6.76054048538208, -9.355090141296387, 5.711245536804199, 3.332707405090332, 24.333969116210938, 0.7016540765762329, -8.078736305236816, 5.330749034881592, 8.405086517333984, -15.9192476272583, 5.8283562660217285, 9.872152328491211, -11.443889617919922, -0.3272874355316162, 1.4759211540222168, 7.179825782775879, 11.074305534362793, 5.811853885650635, 1.3049461841583252, -9.25075626373291, 3.1029305458068848, -2.9779183864593506, 7.56137752532959, -9.13306999206543, 2.853888511657715, 12.05152702331543, 12.972494125366211, -5.94927978515625, 0.12812745571136475, 27.75661849975586, -11.882219314575195, 0.6856068968772888, 6.555538177490234, -4.824954032897949, -4.802446365356445, -8.236921310424805, -9.893726348876953, -5.764782905578613, -12.620904922485352, 2.7282121181488037, 10.134765625, 4.79966402053833, 9.158502578735352, 4.263575553894043, -2.1228623390197754, -3.324378490447998, 7.863244533538818, -11.596134185791016, 9.016895294189453, -8.333229064941406, 3.346623420715332, -7.136020183563232, -18.710590362548828, -4.815052032470703, 4.170259475708008, -5.849459648132324, -2.836836576461792, 2.681487560272217, -1.8249993324279785, -2.003507137298584, 0.544062614440918, -8.864578247070312, 2.465836524963379, -0.23366039991378784, -7.49735164642334, 5.303377151489258, 1.508569598197937, -4.055882930755615, 5.111513137817383, -16.29676628112793, -5.6894330978393555, 5.524892330169678, -24.271278381347656, 8.472335815429688, -9.954610824584961, -5.676187515258789, 27.555767059326172, -9.753934860229492, 1.3346936702728271, -7.522825717926025, 7.662447929382324, -6.325225353240967, 24.71043586730957, -18.863178253173828, -8.227837562561035, -3.6278109550476074, 6.31746244430542, -8.482839584350586, -1.5559734106063843, 21.90591812133789, 2.8555777072906494, 10.456961631774902, 8.85527229309082, 9.142139434814453, -5.343331336975098, 21.87372589111328, -18.884681701660156, -2.0452675819396973, -13.048056602478027, -13.193004608154297, -16.445331573486328, 16.699491500854492, -4.921442031860352, 10.818948745727539, 6.068228721618652, -11.384910583496094, -19.80475616455078, -5.409200668334961, -9.064277648925781, 5.042319297790527, -9.240060806274414, 2.7786850929260254, 0.9815123081207275, -20.513324737548828, 13.29487419128418, -3.90156888961792, -0.6393567323684692, 2.976102828979492, 1.8888919353485107, -10.798798561096191, 30.340831756591797, 23.898860931396484, -7.832804203033447, 27.921642303466797, 3.235152006149292, -0.9915798902511597, 8.819273948669434, -2.4482715129852295, 12.40207576751709, -12.635553359985352, -6.257765769958496, 27.273780822753906, 3.786261558532715, -2.0476608276367188, -0.5808194875717163, -9.460728645324707, 3.9465115070343018, -4.678750038146973, 4.031056880950928, 33.30412673950195, 3.0931105613708496, 11.15230655670166, 4.954248428344727, -15.81626033782959, 7.124358177185059, 2.1032354831695557, 8.53767204284668, -24.218528747558594, 2.2377686500549316, 4.958838939666748, 7.0331854820251465, 1.9910626411437988, 8.498448371887207, -13.95452880859375, 12.32785701751709, -5.542845726013184, -5.92972993850708, 12.820619583129883, 14.913759231567383, 4.520724296569824, 11.689929962158203, -5.287491798400879, -0.199981689453125, -7.11578369140625, -4.9862565994262695, 16.828035354614258, 9.435511589050293, -13.28173542022705, -8.728747367858887, -2.985196590423584, -4.168299198150635, 0.8148796558380127, 12.10223388671875, 2.0230605602264404, -16.06412124633789, -15.442865371704102, -3.679771900177002, 26.661296844482422, 7.496047019958496, 3.34970760345459, -2.5846550464630127, 1.7352399826049805, -0.7504827976226807, -4.823550701141357, -0.7862710952758789, -15.283102035522461, 17.139129638671875, -1.5871930122375488, -9.575407028198242, 7.346923828125, 3.2600436210632324, 10.833650588989258, 6.6822333335876465, -0.761505663394928, 7.665038585662842, 0.31419384479522705, -1.8609846830368042, 8.730018615722656, -17.17359733581543, -1.8328173160552979, 7.278680801391602, 1.154383897781372, 1.9482979774475098, 7.715845584869385, 4.0691938400268555, 6.850833415985107, 8.181160926818848, 2.3580775260925293, -9.31429672241211, -6.0315775871276855, -13.598442077636719, 10.365285873413086, 6.944379806518555, 12.230369567871094, -1.0007444620132446, 12.473920822143555, 4.649205684661865, 5.223206996917725, -0.9268770217895508, 20.480730056762695, -1.4716888666152954, -0.1594177782535553, -12.175561904907227, -23.151962280273438, 6.525191307067871, -11.320900917053223, -16.28007698059082, 6.581671237945557, 0.8186129331588745, 4.939568519592285, 4.577668190002441, 10.533687591552734, 1.8339060544967651, 1.2437152862548828, -8.26338005065918, -16.70404624938965, 7.662764549255371, 1.19566810131073, 14.386054992675781, 13.06551456451416, -11.040908813476562, 5.865034580230713, -7.260275840759277, -8.45966911315918, 14.274789810180664, 16.92165184020996, -5.600378513336182, -15.811887741088867, -12.705076217651367, 9.761165618896484, -1.9265800714492798, 7.940158367156982, -2.7042102813720703, -8.078911781311035, 7.6305012702941895, -3.0058093070983887, 18.916471481323242, -13.554610252380371, -16.862468719482422, -15.01348876953125, 9.156658172607422, -13.944750785827637, -6.928482532501221, 5.878452777862549, -13.727151870727539, 5.9354329109191895, 15.67291259765625, -7.805965423583984, 5.992123603820801, 3.4042282104492188, -9.038414001464844, -6.491217613220215, 1.6654884815216064, 11.699268341064453, -8.193598747253418, -5.601509094238281, -2.470825433731079, -5.089954853057861, 21.96028709411621, -13.190385818481445, 11.092248916625977, 12.156907081604004, -1.6911852359771729, 9.926177024841309, 10.268241882324219, 5.710837364196777, 2.512216091156006, -7.522792339324951, -11.106911659240723, -9.377785682678223, -11.521005630493164, -6.315898895263672, -7.103375434875488, 9.99718189239502, 1.3761733770370483, 3.4819366931915283, -2.6287176609039307, -15.013429641723633, 8.824268341064453, 1.570849895477295, -1.20750093460083, -6.5816264152526855, 16.345623016357422, 8.409915924072266, -9.866572380065918, 19.057430267333984, 6.444485664367676, -1.03660249710083, -9.852124214172363, 9.11739730834961, -1.6757549047470093, -3.7991514205932617, 3.4166839122772217, -0.17828620970249176, -8.360568046569824, -2.0725724697113037, -12.828699111938477, 1.8980242013931274, -4.1014485359191895, 2.5948545932769775, 20.19418716430664, 2.4424667358398438, -1.6355950832366943, 5.158933639526367, -13.31906795501709, 0.1947278529405594, -7.531569004058838, 6.509471416473389, 13.338013648986816, 0.3960336446762085, 14.716540336608887, 5.473923206329346, 13.507793426513672, 8.268922805786133, 2.1703126430511475, 10.85859489440918, -2.39532732963562, -2.9443554878234863, 1.651432991027832, -2.8965508937835693, -3.004439115524292, -12.248027801513672, -6.841985702514648, 6.348752975463867, 19.874120712280273, -11.244202613830566, 2.907939910888672, -7.431693077087402, 9.217869758605957, 14.482954025268555, -4.03995418548584, 3.4591832160949707, -14.921754837036133, -3.6069579124450684, 2.8369140625, 7.092913627624512, -4.158041954040527, -11.688281059265137, 11.860292434692383, 4.820716381072998, -0.24852706491947174, -5.591269016265869, 3.631187915802002, -13.868779182434082, -12.607518196105957, -9.18762493133545, -6.161220073699951, 4.971234321594238, 6.088233947753906, 17.97613525390625, 12.50352954864502, 8.042533874511719, -0.11697837710380554, 5.459440231323242, 3.6930010318756104, 3.031165838241577, -5.555236339569092, 6.981016159057617, -0.318459153175354, 1.3086165189743042, 6.550294876098633, 7.281394004821777, -7.391196250915527, 10.192760467529297, 7.404540061950684, 7.684196949005127, -7.459064483642578, 3.309631109237671, -10.485062599182129, -12.971267700195312, 12.533565521240234, -1.1455626487731934, -5.602164268493652, 4.564123630523682, -2.5293326377868652, 4.366396903991699, -1.5131959915161133, 8.47718620300293, 15.232583045959473, 4.075497627258301, 12.027790069580078, -9.248308181762695, -6.4132795333862305, 13.497701644897461, 5.445231914520264, -2.5151357650756836, 2.247620105743408, 4.71481466293335, 13.305254936218262, -4.636439323425293, 4.069905757904053, -8.062891960144043, 11.612602233886719, 16.435409545898438, -8.473336219787598, -3.46315860748291, -2.079467535018921, 5.097566604614258, -0.6136665940284729, -7.015867710113525, -5.788308143615723, 3.181330680847168, 0.67429119348526, 5.593395233154297, -17.780834197998047, 9.279431343078613, 7.5135416984558105, -1.6954569816589355, -2.666207790374756, -2.9485366344451904, 8.919007301330566, -2.9541869163513184, -5.720067977905273, 4.385583400726318, 13.43476390838623, 6.923055648803711, 0.2797607183456421, 0.21522091329097748, -13.246604919433594, 11.219972610473633, 1.7340248823165894, -15.079655647277832, 3.3411693572998047, 3.4130349159240723, 17.242341995239258, -8.503804206848145, -3.3554043769836426, 0.046548403799533844, 4.6613450050354, 8.894449234008789, -5.974831581115723, -1.2668886184692383, -4.5333757400512695, 14.652844429016113, 13.915578842163086, -2.891103982925415, -4.2630486488342285, -14.47500991821289, 1.8101203441619873], [1.5792804956436157, -12.33667278289795, 22.472789764404297, -8.987269401550293, -20.873031616210938, 3.2205960750579834, -1.4750995635986328, -0.4880635142326355, 16.874225616455078, -21.18607521057129, -1.1227519512176514, -3.326260566711426, 14.996892929077148, -5.280085563659668, -11.40146255493164, 3.227792739868164, -1.020893931388855, -10.364607810974121, 14.689587593078613, -14.050761222839355, -4.165896415710449, 11.316690444946289, 11.783848762512207, 10.928743362426758, -15.289628982543945, -0.10151609778404236, -10.128279685974121, -25.104820251464844, -10.968719482421875, 10.36020278930664, -0.3945024013519287, 1.423775553703308, -4.551519870758057, -6.488516807556152, 9.616275787353516, 4.372307300567627, -15.316176414489746, 4.215059757232666, -8.399425506591797, 4.624068260192871, -11.012463569641113, -9.995964050292969, -4.016037464141846, 4.505873680114746, -22.340822219848633, 21.01247787475586, 10.419078826904297, -9.38370132446289, 13.605341911315918, -19.938640594482422, -3.471341609954834, -9.491952896118164, 5.524778366088867, -1.3200057744979858, -3.9284210205078125, -11.962937355041504, -3.9537811279296875, 3.123627185821533, 1.1440210342407227, -2.659100294113159, 1.0081201791763306, -8.402190208435059, -15.053339004516602, -7.197685718536377, 8.653738975524902, 14.901122093200684, -0.0719524472951889, -11.500017166137695, -19.332836151123047, 0.21566438674926758, 6.815472602844238, -18.463489532470703, -7.081637859344482, -2.478097915649414, 15.309600830078125, 16.010454177856445, -14.78709602355957, 31.744144439697266, 5.424002647399902, -8.025171279907227, 16.743911743164062, -5.636658191680908, -2.8998074531555176, 23.06240463256836, 11.607823371887207, 20.278303146362305, 2.250781774520874, 2.0757129192352295, -7.085027694702148, 3.956331729888916, 1.0163971185684204, -18.219743728637695, 2.7074341773986816, -3.717090129852295, -21.226398468017578, -4.1077961921691895, -12.380655288696289, -3.3839151859283447, -5.45649528503418, -6.006808757781982, -1.8395819664001465, -6.90299129486084, -8.760697364807129, -14.203323364257812, 4.240239143371582, -7.870061874389648, 1.4696471691131592, -1.6623395681381226, -5.396166801452637, -22.127363204956055, -12.691668510437012, -1.2910256385803223, 2.5035400390625, 10.62482738494873, 3.4406261444091797, 16.266380310058594, 6.4762444496154785, 0.11314547061920166, 12.595295906066895, 2.8000073432922363, -2.887603282928467, -4.729487419128418, -2.816561222076416, 3.8937807083129883, -8.278743743896484, -2.2151944637298584, 0.11212847381830215, -0.34964674711227417, 6.09072208404541, 9.52223014831543, -11.614582061767578, -3.638998508453369, 4.170671463012695, -0.6666099429130554, -11.031394958496094, -24.150686264038086, 14.920035362243652, 6.569278240203857, 9.745274543762207, 4.426090240478516, 3.1066384315490723, -2.71769380569458, 2.8763089179992676, 2.1498935222625732, -18.949487686157227, 10.879121780395508, 4.121381759643555, 1.211726427078247, 9.52945327758789, 5.771018981933594, 15.384138107299805, 2.2753801345825195, 12.584820747375488, 3.967682123184204, 5.871708393096924, -8.157829284667969, -6.5076069831848145, -1.2630623579025269, 4.117885589599609, -14.398850440979004, -13.186094284057617, 13.852124214172363, 4.498827934265137, -7.217137336730957, -4.156317234039307, 19.845823287963867, -0.08268757909536362, 0.5353915095329285, 0.8872764706611633, 8.280588150024414, -9.537700653076172, -22.085044860839844, -0.35901740193367004, 3.9466075897216797, 8.384159088134766, -4.261357307434082, 7.4334282875061035, -3.658064842224121, 2.9160985946655273, -7.01833963394165, 8.598015785217285, -5.146003246307373, -3.2497479915618896, -10.981185913085938, -13.619583129882812, -4.7105207443237305, -12.551750183105469, 4.001381874084473, -5.78216028213501, 13.414673805236816, 1.6124403476715088, -2.6212706565856934, -4.492054462432861, 12.713895797729492, 4.838754177093506, 9.067048072814941, 8.415597915649414, -13.727230072021484, -2.8981521129608154, 0.5155670642852783, 6.912335395812988, -8.805437088012695, 4.232752799987793, 14.108415603637695, 11.898860931396484, -7.1713690757751465, -9.063910484313965, 3.5647785663604736, 8.660296440124512, 8.902827262878418, -4.285805702209473, 15.066784858703613, 10.589166641235352, -1.9931135177612305, 8.3113374710083, -11.500541687011719, 5.016692161560059, 0.3094199001789093, 0.5024676322937012, -5.665666580200195, 10.213977813720703, 7.36086893081665, 9.590641021728516, 11.844881057739258, 2.4275734424591064, -14.174076080322266, 9.883340835571289, 4.7667388916015625, -1.6462388038635254, -5.138545989990234, -4.001825332641602, 12.822578430175781, 12.637457847595215, -7.892531871795654, 9.64980411529541, -7.15984582901001, 9.006360054016113, 5.338762283325195, 8.209077835083008, -6.181318283081055, -2.4852969646453857, 1.3614842891693115, -15.137113571166992, -8.99755859375, -1.0430669784545898, 18.910213470458984, -3.17667818069458, -6.532100677490234, 10.609500885009766, 9.421792030334473, -1.8633365631103516, 1.9614397287368774, 5.102531433105469, -5.62628698348999, 18.423364639282227, 5.216004848480225, 13.176407814025879, -1.0867737531661987, -0.596134603023529, -9.178375244140625, -5.898246765136719, -9.83807373046875, 3.6289069652557373, 14.015768051147461, -2.881974697113037, 12.021614074707031, -3.963953971862793, 6.254915237426758, -2.7651171684265137, -4.87510871887207, -5.252975940704346, -0.3252004384994507, 16.43806266784668, -14.493749618530273, 15.74966049194336, 6.046191215515137, -1.410295844078064, 16.577932357788086, 9.551698684692383, -4.1213698387146, 16.853193283081055, -10.837363243103027, 3.223208427429199, -0.004728999454528093, 10.044363021850586, 5.629246234893799, 0.12517642974853516, -4.4719390869140625, -3.4486422538757324, -5.547876358032227, 6.062312602996826, 4.43961238861084, -1.199501633644104, -13.388614654541016, 30.556720733642578, -4.721497535705566, -4.0694684982299805, 12.98305892944336, -1.6111170053482056, -12.617101669311523, 1.2998290061950684, 4.203227996826172, 8.850074768066406, 6.216754913330078, -10.16392707824707, 14.204487800598145, -8.976184844970703, -10.310301780700684, 8.173728942871094, -4.968501091003418, 4.261877536773682, 5.879010200500488, -15.901345252990723, -3.3474202156066895, 6.110339164733887, 11.771955490112305, 3.1962122917175293, 3.5721383094787598, -1.8227685689926147, -16.445404052734375, 10.66412353515625, -10.147289276123047, 13.045236587524414, -20.818153381347656, 1.810948133468628, -10.342246055603027, 6.842587471008301, -12.578237533569336, 9.792898178100586, 16.244604110717773, -2.3208935260772705, -0.023995842784643173, 3.796499013900757, -10.409696578979492, 7.083401679992676, -13.349810600280762, -2.036139726638794, -16.79119873046875, -1.6621707677841187, -1.170828938484192, -1.1209619045257568, -12.197999000549316, 0.5958478450775146, -12.96901798248291, -7.124855995178223, -4.6577277183532715, -3.8169682025909424, 0.045269809663295746, 14.510489463806152, -5.04854154586792, -0.7528878450393677, 6.644982814788818, -0.07514900714159012, -19.9713134765625, -8.608975410461426, 15.252010345458984, -5.017630577087402, 13.778592109680176, -2.164134979248047, -0.18059980869293213, -4.688205242156982, 6.523945331573486, -15.249186515808105, -22.028430938720703, -2.885500431060791, -7.092203617095947, -0.6856206655502319, 3.7362899780273438, -7.058122634887695, 0.45429185032844543, -11.910102844238281, 10.030969619750977, -10.505290031433105, 27.027864456176758, 0.10746154934167862, -4.7867584228515625, -5.350008010864258, -5.6068830490112305, 6.676762580871582, -4.140382766723633, -5.903603553771973, -3.4425482749938965, 1.0913307666778564, 16.19928741455078, 29.678245544433594, -6.436291694641113, -3.9364826679229736, 0.1989385187625885, 13.480191230773926, -10.977018356323242, 0.06076569855213165, -4.463833808898926, -4.761375904083252, 4.781310081481934, 4.396817684173584, 15.142041206359863, -9.986519813537598, -1.5913249254226685, -6.111621379852295, -15.120950698852539, 6.731962203979492, -9.902746200561523, 14.45970344543457, -3.488809585571289, 2.8287813663482666, 10.688947677612305, 14.13491439819336, -0.30675339698791504, 17.798503875732422, 9.683080673217773, -9.759360313415527, 9.443994522094727, -12.846783638000488, -1.5839662551879883, -9.701910972595215, 1.6590888500213623, -6.6926093101501465, 2.595895528793335, -3.1363258361816406, 7.381021499633789, -22.830846786499023, 10.98590087890625, 11.314669609069824, -10.761246681213379, 2.010240077972412, 4.806910037994385, 2.623347043991089, 6.061586856842041, 2.4918277263641357, 6.448965549468994, -4.7944135665893555, -8.409102439880371, -5.4283623695373535, -10.476089477539062, 22.655723571777344, -15.252294540405273, 17.501365661621094, 3.8394412994384766, -6.527680397033691, -11.702469825744629, -3.414512872695923, -6.925570964813232, -10.691398620605469, -7.0632476806640625, 2.9077742099761963, -11.481311798095703, 20.2645206451416, 9.593109130859375, -1.7530148029327393, 5.1953864097595215, -14.723877906799316, 12.158370971679688, 2.7781736850738525, 10.277597427368164, -5.982205390930176, -4.72115421295166, 1.830007791519165, -15.90376091003418, 10.241727828979492, -11.50455093383789, -4.049689292907715, 13.852603912353516, -0.10505686700344086, 16.051000595092773, -0.18982605636119843, -2.2982544898986816, 11.421233177185059, 11.366329193115234, -11.827068328857422, -9.785703659057617, 5.103579044342041, 4.020572185516357, 15.322916030883789, 3.9005894660949707, -1.5166654586791992, -16.576885223388672, -5.880504608154297, 0.8055611848831177, -7.09069299697876, 22.908588409423828, 8.176198959350586, -3.3517744541168213, 0.23835113644599915, -3.897520065307617, -12.95391845703125, -6.716119766235352, -5.870716094970703, -17.86626625061035, 1.140321969985962, 4.4019598960876465, -13.583541870117188, -3.335082530975342, -14.337465286254883, -8.440959930419922, 2.8145461082458496, -8.360671043395996, -4.580393314361572, 21.324050903320312, -9.426963806152344, -1.7693052291870117, 3.2762041091918945, 5.47858190536499, -17.22017478942871, 4.5115509033203125, 7.420010566711426, 2.902477264404297, 6.26503849029541, 8.93906021118164, -5.397951126098633, 2.775094509124756, 2.513721227645874, 14.391956329345703, 12.923299789428711, 21.196331024169922, -13.414142608642578, -22.083589553833008, 9.922184944152832, 3.117565155029297, -8.081597328186035, 7.847705364227295, 7.906904220581055, 9.057511329650879, 3.289839267730713, 5.0534162521362305, -0.1637713462114334, -16.063507080078125, -2.9459164142608643, 17.788471221923828, -5.413359642028809, -20.71924591064453, -12.380166053771973, 16.30958366394043, 11.575519561767578, -7.0395097732543945, 5.96250057220459, 11.725220680236816, -9.99332046508789, -13.002344131469727, -4.753497123718262, -26.861591339111328, 1.9277269840240479, -8.131342887878418, 7.988315582275391, 34.2589111328125, 1.5176230669021606, 0.0634889006614685, 4.875420570373535, 8.050406455993652, 14.966161727905273, 14.502973556518555, -2.2249467372894287, 6.338906764984131, 1.7102168798446655, -2.8045639991760254, -11.54112720489502, -8.717580795288086, -12.463783264160156, -1.106649398803711, 7.018556118011475, -3.0711703300476074, 5.364553451538086, -13.678136825561523, -6.174260139465332, 1.4501054286956787, -11.247502326965332, -3.7690412998199463, -11.03425407409668, -5.858217239379883, -1.9917490482330322, 12.406122207641602, -9.888406753540039, 3.5225138664245605, -7.941501140594482, 0.27736082673072815, -13.808998107910156, -4.170882701873779, 9.009456634521484, -6.186633586883545, -12.39081859588623, -15.45157527923584, -12.137557983398438, -11.002462387084961, 11.244829177856445, 10.436515808105469, -0.8545404076576233, -0.9101855754852295, -3.1525769233703613, -0.6038784384727478, -20.78958511352539, -2.7642152309417725, 5.803010940551758, 12.582653045654297, -2.565389633178711, -0.02230611816048622, -0.1315353959798813, -25.403520584106445, 6.1060686111450195, 10.903526306152344, -13.270696640014648, 7.580951690673828, -6.593482971191406, 7.267556667327881, 16.168916702270508, -4.126408100128174, 5.840873718261719, -9.048759460449219, -7.230512619018555, -11.48552131652832, 9.506139755249023, -3.843024253845215, -6.356924057006836, 10.508398056030273, -23.403186798095703, -15.165839195251465, 8.692777633666992, -8.37211799621582, -4.562227249145508, -2.3535242080688477, -14.014570236206055, 5.475783824920654, -16.31821060180664, 3.146059036254883, 6.769112586975098, -10.45919418334961, -4.367125034332275, -0.21676957607269287, -0.811803936958313, -15.115697860717773, 5.340801239013672, 1.4449882507324219, -5.03078031539917, -1.5849838256835938, 3.036639451980591, -4.475942611694336, -10.425098419189453, -0.26348400115966797, -17.441410064697266, -0.627244234085083, 4.756416320800781, -5.343476295471191, 1.888730525970459, 7.938662528991699, -1.380736231803894, 2.0572025775909424, 8.697663307189941, 7.416437149047852, -9.51017951965332, 7.344930648803711, 9.551993370056152, 10.307116508483887, 16.323808670043945, -1.4788920879364014, -4.161442756652832, 3.059662342071533, 21.2901611328125, 10.058794021606445, 0.04161301255226135, -8.554636001586914, -0.6161565780639648, 4.4600982666015625, -2.7276382446289062, -8.34287166595459, -2.028496503829956, 9.57786750793457, 14.608744621276855, 2.183159351348877, 0.30234628915786743, -1.5344566106796265, 15.36508560180664, 6.775661468505859, 10.503631591796875, 11.112504005432129, 11.86081314086914, 21.929990768432617, -9.074633598327637, 4.828239917755127, -8.845237731933594, -5.979520797729492, 4.747186660766602, -6.10467004776001, -10.21387004852295, -4.842294216156006, 0.6020838022232056, -10.940864562988281, -4.254475116729736, -10.0632963180542, 1.3183170557022095, 8.818958282470703, 7.070125579833984, 4.897058486938477, 6.218411922454834, 8.78477954864502, -16.30442237854004, -9.029833793640137, -7.149065017700195, 8.655804634094238, 5.200261116027832, 2.9918112754821777, 1.257785439491272, 1.8374186754226685, -9.78821086883545, -8.258750915527344, -11.16421890258789, 10.191522598266602, -17.179893493652344, 8.59422492980957, -1.303955316543579, 5.562877178192139, -16.707359313964844, 13.507976531982422, 4.363112926483154, 0.15865716338157654, -1.156615972518921, -12.235631942749023, 21.921064376831055, -21.734268188476562, 15.302729606628418, -12.8991117477417, 10.212697982788086, 8.361178398132324, -7.0734758377075195, -6.871369361877441, -9.076842308044434, 1.2027912139892578, -12.531665802001953, 12.07796573638916, 16.805709838867188, -1.4646680355072021, 9.929540634155273, -9.603734970092773, -14.231355667114258, 8.474920272827148, -0.05766734480857849, -2.469331979751587, 5.507723808288574, 10.2112398147583, -2.6183366775512695, -1.3011616468429565, -14.975481033325195, -7.754887580871582, 3.2129058837890625, -5.172956466674805, -10.248153686523438, 17.159883499145508, 2.8309860229492188, 4.20803165435791, -17.91276741027832, -8.144007682800293, 1.8883113861083984, 2.51296329498291, -12.335552215576172, 11.575287818908691, -3.4613583087921143, 19.86223602294922, 13.925251960754395, 5.010406970977783, 2.115856885910034, -4.998206615447998, -18.112462997436523, 2.4033279418945312, 2.7179975509643555, -10.227766036987305, -1.8611125946044922, 9.258126258850098, -12.957075119018555, 4.340416431427002, 21.050945281982422, 0.7968508005142212, 1.2466926574707031, 4.433071136474609, -11.872086524963379, -12.019509315490723, 10.292469024658203, -0.8918251991271973, 8.85516357421875, 1.0949287414550781, 14.229667663574219, -11.066829681396484, -0.7535189390182495, -3.8362038135528564, -13.109393119812012, 2.3220765590667725, -10.927413940429688, -0.5164455771446228, -8.563512802124023, -10.858631134033203, -0.24299374222755432, -11.857646942138672, 1.6250957250595093, -3.9533183574676514, -9.146463394165039, 23.742504119873047, 12.459497451782227, -12.362433433532715, -1.297769546508789, -4.9976301193237305, 10.842103958129883, 9.885782241821289, -5.937541961669922, 0.7170228958129883, 13.715167999267578, -1.9585556983947754, 4.906063556671143, 7.415463924407959, 0.9513677358627319, 16.226016998291016, 0.7021452188491821, -7.59704065322876, 7.582435131072998, -8.26917552947998, -2.4260172843933105, -3.050238847732544, 10.556120872497559, 1.6304337978363037, -4.020280361175537, -10.781742095947266, 13.087725639343262, -4.8674163818359375, -9.038981437683105, 8.797136306762695, 0.10583159327507019, -0.3715657889842987, 3.7018651962280273, 10.654617309570312, 1.3227182626724243, -15.401975631713867, -11.117383003234863, -2.0819597244262695, 2.1848948001861572, -9.641725540161133, -2.5925326347351074, -1.8645727634429932, -12.727431297302246, -11.799546241760254, 4.539580345153809, -6.50892972946167, 15.311928749084473, -1.8109427690505981, -18.28527069091797, -6.129487991333008, 10.428613662719727, 1.8527013063430786, -10.800960540771484, -2.1154372692108154, 10.460792541503906, -10.506564140319824, -8.33236312866211, -0.9035356044769287, 3.0490267276763916, -2.6507279872894287, 6.8875627517700195, 3.005587100982666, -8.715773582458496, -9.807703018188477, -22.8657169342041, -2.087794780731201, 0.30662575364112854, 6.562097549438477, 4.394043445587158, -7.018326759338379, 44.78466796875, 5.116761207580566, -2.8402867317199707, 2.9192428588867188, 8.455965042114258, -1.8192138671875, 8.476512908935547, -1.609872817993164, 15.690020561218262, -13.297524452209473, -3.605586290359497, -2.3656516075134277, 12.264263153076172, -10.075940132141113, -20.762292861938477, 6.97514009475708, -8.306194305419922, 3.3402438163757324, 8.312050819396973, 8.358198165893555, -14.208423614501953, 11.59312629699707, 2.864048480987549, 17.967309951782227, -10.308245658874512, 22.088163375854492, -3.1985511779785156, 1.0834758281707764, 13.547151565551758, -9.70624828338623, 6.263859272003174, 14.146432876586914, 6.0449957847595215, -6.986245155334473, -12.065532684326172, -9.873522758483887, -1.6012285947799683, -1.9454352855682373, 0.9928109645843506, -3.6221389770507812, -8.221396446228027, -6.297798156738281, 5.036413669586182, 2.4205708503723145, -18.297088623046875, -14.816873550415039, -8.2564058303833, -0.7057002782821655, -4.977381706237793, -6.079073905944824, -8.26085090637207, -23.26458168029785, 5.179363250732422, 8.955615997314453, -9.99435806274414, -6.3277907371521, 6.13027811050415, 2.0791378021240234, 10.282330513000488, 11.732954978942871, -3.8982298374176025, -1.6910063028335571, -12.892314910888672, 14.361557006835938, 7.395291805267334, -9.288705825805664, -1.5258698463439941, -12.608959197998047, 23.7547607421875, -10.038360595703125, -11.659187316894531, -8.608404159545898, -6.743017196655273, -12.60566520690918, -6.767000198364258, -3.668328285217285, -11.595941543579102, 2.724721670150757, -0.5672265291213989, -3.5967159271240234, -23.00093650817871, 20.418739318847656, 4.610239028930664, -2.6673636436462402, -0.04055090248584747, 9.538148880004883, -8.178337097167969, 16.158254623413086, 0.37923359870910645, 2.3916492462158203, 1.7073936462402344, -14.378826141357422, -10.35184383392334, 7.160229206085205, 9.830516815185547, -15.972440719604492, 15.752653121948242, 11.711379051208496, 9.95745849609375, -7.024263381958008, 4.786231994628906, -9.567033767700195, -0.16461193561553955, -7.075753688812256, -1.5114061832427979, 3.2507548332214355, 3.4681689739227295, 22.34027671813965, 2.1693663597106934, -8.907562255859375, 9.862306594848633, -5.18381404876709, -0.7801819443702698, -0.5195655822753906, -7.834710121154785, -14.137933731079102, 2.496340036392212, 8.78090763092041, -5.933766841888428, 3.7204976081848145, 8.596088409423828, 0.7119304537773132, 2.3052773475646973, -23.086530685424805, -24.80659294128418, -2.2608871459960938, -0.9487723112106323, 7.832247734069824, 2.2385077476501465, 3.0475306510925293, 3.103255271911621, -3.3614492416381836, 5.0109710693359375, -14.105247497558594, 8.087688446044922, -15.9259033203125, -9.75993537902832, 8.30428409576416, -1.6574697494506836, 7.843244552612305, -5.534763813018799, -4.204276084899902, -4.123147010803223, 1.805092215538025, -6.698476791381836, 7.571586608886719, 10.97218132019043, -3.2629621028900146, -14.112348556518555, -6.6327385902404785, -19.530736923217773, -6.389308929443359, 7.037250518798828, -4.026791572570801, 10.086490631103516, 7.322582721710205, -1.4556151628494263, -4.173403263092041, 1.5434119701385498, 1.1070139408111572, -16.149341583251953, -4.087652683258057, -10.255261421203613, -0.7503679990768433, -4.001152038574219, -2.1106603145599365, 5.455752372741699, 3.0605294704437256, 6.387811183929443, 7.133145809173584, -20.58114242553711, -7.391408920288086, 8.885088920593262, 1.334326982498169, 0.9864182472229004, 8.377725601196289, 15.224356651306152, 16.103830337524414, -6.800014972686768, -5.2049560546875, -16.477115631103516, -5.655235767364502, -4.8253326416015625, -8.753463745117188, 10.224939346313477, 5.571274757385254, -7.911524295806885, -19.479629516601562, 7.5584516525268555, -20.13616180419922, 5.3444743156433105, -15.759973526000977, 11.13648796081543, -9.364996910095215, 7.789146900177002, -6.655550003051758, -14.752103805541992, 6.376766681671143, -4.809688091278076, 13.35923957824707, -8.166722297668457, 4.475905418395996, -12.865514755249023, -7.728298187255859, -0.251276433467865, 4.660521984100342, -4.089986801147461, -5.089725971221924, 4.057342529296875, -8.604890823364258, -15.986818313598633, -12.753032684326172, -0.18689769506454468, -20.474489212036133, -4.679344177246094, 27.32822036743164, -13.341208457946777, 17.947021484375, 16.2971134185791, -10.555549621582031, -14.553308486938477, -7.193962097167969, -10.738330841064453, 9.044197082519531, 0.010954133234918118, 7.544059753417969, -11.791923522949219, 0.4076853394508362, -16.84530258178711, 5.501214981079102, 16.53032875061035, 3.868191719055176, 15.708616256713867, 5.258762836456299, 0.38398319482803345, -6.16868257522583, 4.262929916381836, -6.619389533996582, 3.294536590576172, -1.005172848701477, 12.865229606628418, -3.078324794769287, -0.10315479338169098, 9.572396278381348, -4.280169486999512, 4.797878265380859, -4.476047515869141, 8.259343147277832, -10.09439468383789, 9.093488693237305, 4.834279537200928, 1.2153469324111938, 10.441202163696289, -6.220222473144531, 11.03116226196289, 1.8984253406524658, 2.1536202430725098, 2.0166871547698975, -1.9918180704116821, -0.4542388916015625, -7.355359077453613, 6.811807155609131, 6.624721527099609, -6.925698280334473, 3.5988264083862305, 0.18424378335475922, 0.44116976857185364, -2.427529811859131, 3.76792311668396, -8.972100257873535, -5.904261589050293, 8.087416648864746, -3.2339391708374023, -2.27882719039917, -20.78862953186035, 3.5796117782592773, -8.044454574584961, 0.5555762648582458, -1.5191113948822021, 4.489450931549072, -17.370635986328125, 9.157939910888672, -4.060496807098389, 6.758141994476318, -12.699651718139648, 3.3953378200531006, -9.142268180847168, -2.9486827850341797, 8.343816757202148, 1.9290587902069092, 12.322083473205566, 1.4732229709625244, 5.084145545959473, 17.048362731933594, -12.883298873901367, 8.195450782775879, 0.7242023944854736, 5.307924270629883, -5.87790584564209, -6.546322822570801, 0.18553511798381805, 11.187106132507324, 10.459802627563477, -5.628576278686523, 13.378334999084473, -9.537924766540527, -27.018938064575195, -0.5020306706428528, 1.6816201210021973, 4.679795265197754, 8.024656295776367, 0.8984142541885376, -13.792356491088867, -8.658734321594238, 32.85163116455078, 11.756942749023438, -15.145103454589844, -4.9865946769714355, -2.067633628845215, -8.136411666870117, 0.8512041568756104, -1.6156408786773682, 5.627603054046631, 20.054311752319336, -2.6004457473754883, 0.39790797233581543, -13.90510082244873, -4.513550281524658, -13.372896194458008, 11.772659301757812, 1.4490845203399658, -0.6052457094192505, 0.7150651216506958, 14.069319725036621, 10.249028205871582, 1.1012029647827148, 6.266942977905273, 9.007701873779297, 11.165404319763184, -21.254058837890625, 11.613688468933105, 7.100171089172363, 7.867589950561523, -5.400059223175049, 7.246316909790039, -12.853350639343262, -0.7960631847381592, -15.501785278320312, -3.171027660369873, 11.559497833251953, -5.890772819519043, 6.56435489654541, -9.39275074005127, -10.741464614868164, -0.510792076587677, 0.8312637805938721, -3.3343241214752197, 3.572850227355957, -0.2444462776184082, 18.487974166870117, -25.093496322631836, 0.3048636019229889, 2.052001714706421, 13.984041213989258, -5.493595123291016, 14.747440338134766, -0.7540150880813599, -4.427797317504883, 0.2772942781448364, 14.469322204589844, 1.897965669631958, 4.419952392578125, 8.591053009033203, -13.411355018615723, -27.888004302978516, -19.94235610961914, 16.371183395385742, -2.8914899826049805, -9.190903663635254, -12.15967082977295, 10.504423141479492, -16.165544509887695, 2.4792604446411133, -5.52506685256958, -0.5009528398513794, -4.497696876525879, -7.472995758056641, 9.415262222290039, 4.1356048583984375, -4.821670055389404, -0.08162189275026321, -14.170516967773438, -6.746541976928711, 5.556466579437256, 2.020906925201416, -15.068646430969238, -11.00779914855957, -2.9927163124084473, -0.6502672433853149, -1.40776526927948, -11.367592811584473, -13.201266288757324, 1.7219816446304321, 3.187166452407837, -9.576074600219727, 22.309249877929688, 10.083137512207031, -11.040934562683105, -14.5694580078125, -9.691176414489746, 12.789764404296875, 23.02202033996582, -4.692543029785156, 12.301674842834473, 8.25944709777832, 8.750656127929688, 10.307951927185059, -4.543661594390869, -12.231563568115234, 8.500086784362793, -17.489139556884766, -4.789046764373779, -15.040539741516113, -5.46151065826416, -9.23331356048584, 10.587448120117188, -4.103717803955078, 20.67071533203125, 24.70076560974121, 2.9442074298858643, 7.143876552581787, 7.751491069793701, -13.754583358764648, 7.619225978851318, -12.399148941040039, -0.2147345095872879, 8.872961044311523, -7.42874813079834, -3.4405529499053955, 6.456861972808838, -15.150775909423828, 1.650429368019104, -4.502232551574707, -6.158210754394531, 10.642348289489746, -11.450088500976562, -2.8316330909729004, 3.2655441761016846, -5.972426414489746, -13.143636703491211, 13.082193374633789, 17.455827713012695, 15.414782524108887, 12.389121055603027, 5.081541538238525, -18.12623405456543, 6.153848171234131, -10.398998260498047, -3.402170181274414, 11.334243774414062, -20.650283813476562, 4.170074939727783, 18.07369041442871, -10.11626148223877, 7.482222557067871, 6.469449520111084, -2.506985902786255, 5.2286224365234375, -0.3853335380554199, 16.9163875579834, -18.87843894958496, -14.095138549804688, 6.455219268798828, 1.830747365951538, -9.020294189453125, -18.983518600463867, -9.154669761657715, 1.4029929637908936, 6.725303649902344, 21.257619857788086, 1.2252064943313599, 7.102040767669678, 1.5117862224578857, 2.3073723316192627, -8.728601455688477, 19.342790603637695, 23.233184814453125, 4.614365577697754, 1.9569669961929321, -6.078932285308838, -6.9957356452941895, 0.41847988963127136, -7.264404296875, 1.3267912864685059, 7.47035026550293, 5.363840103149414, 3.304863452911377, -7.4216413497924805, 8.136042594909668, 3.9290080070495605, -6.58349609375, 2.884981632232666, -10.471244812011719, 6.1224775314331055, 6.692745208740234, 19.0941162109375, -29.298534393310547, 11.59384536743164, -14.294828414916992, -15.354084014892578, -13.521821975708008, 5.279409885406494, 12.233880996704102, -0.013177746906876564, 11.891958236694336, 12.829375267028809, 7.692378044128418, 14.392942428588867, 5.273948669433594, -1.957932710647583, -1.9510908126831055, -6.539891242980957, 9.025069236755371, 8.458290100097656, -11.22958755493164, 1.6099231243133545, -6.584104537963867, -10.150050163269043, -0.7705071568489075, 21.870603561401367, -12.552546501159668, -6.327790260314941, -13.59445571899414, 8.3187255859375, 3.690342426300049, 1.528416633605957, -18.627517700195312, 2.060699224472046, 13.810272216796875, -14.78131103515625, -16.24726104736328, 5.316311836242676, 17.518775939941406, -2.994898796081543, 10.584272384643555, -1.3973528146743774, 10.065788269042969, 10.244874000549316, 34.05233383178711, -3.536257743835449, 20.644332885742188, 6.248716831207275, 13.902366638183594, -1.0742809772491455, 8.37175178527832, 2.0717508792877197, 5.720691680908203, 0.6364789605140686, -5.7497687339782715, 13.417770385742188, -4.365110874176025, 8.874845504760742, 0.8422282338142395, -6.897079944610596, -19.277088165283203, 2.8518028259277344, -12.527985572814941, -11.779427528381348, 14.683627128601074, -0.9927239418029785, -6.943726062774658, 23.84076690673828, -3.1316044330596924, 28.3615779876709, 0.3119087815284729, -8.238409042358398, -1.2701655626296997, 7.435091018676758, 0.34812894463539124, -5.847921848297119, 0.7964562773704529, 12.951705932617188, 8.013867378234863, 10.093990325927734, -3.9401803016662598, 22.082019805908203, 1.5974135398864746, 2.1193840503692627, 3.7544074058532715, 10.119000434875488, -5.8273820877075195, 16.12718963623047, -7.705258369445801, -5.21981954574585, -16.790245056152344, -14.765546798706055, 5.223578929901123, -22.432729721069336, -9.828961372375488, -11.006032943725586, 3.0329954624176025, -11.907244682312012, 3.8279318809509277, 3.6115355491638184, -5.781838893890381, 13.835111618041992, 11.137871742248535, 9.708599090576172, 6.472987174987793, 2.7134246826171875, 2.54022216796875, -7.227113246917725, -14.383064270019531, -2.6286306381225586, -7.574029922485352, 11.429435729980469, 3.4602551460266113, 7.100164413452148, 3.0600264072418213, 8.953568458557129, 0.6309232115745544, -2.287492513656616, 13.280012130737305, -19.02320098876953, -10.176958084106445, -10.465328216552734, 10.276958465576172, -22.65606117248535, 2.5923311710357666, -9.841123580932617, -19.51666259765625, 6.760744094848633, -3.0239367485046387, -0.4877784252166748, 7.0383219718933105, 12.769886016845703, -12.019120216369629, -1.6355050802230835, -10.299419403076172, -11.751394271850586, 1.9944612979888916, -4.718662261962891, -1.4793001413345337, 5.087116241455078, -4.526979923248291, 2.503713369369507, -5.626437664031982, 17.742080688476562, 9.552257537841797, -0.40498220920562744, 6.86190128326416, -2.225698947906494, -5.43765926361084, 0.995573878288269, -14.140010833740234, -13.587264060974121, -0.6196787357330322, -3.070030450820923, -7.583211898803711, -20.09824562072754, 12.333671569824219, -8.529081344604492, -8.642072677612305, 2.010035514831543, -12.396965980529785, -7.712846755981445, -21.21051025390625, -9.090977668762207, 17.39637565612793, -2.1474599838256836, -14.338207244873047, 9.446083068847656, -6.605469703674316, -10.95792007446289, 4.43411111831665, 0.16305306553840637, -7.781369209289551, 2.695305824279785, -3.869352102279663, -17.611053466796875, -2.003108263015747, 0.08283102512359619, 2.786876916885376, 1.1508907079696655, -10.821603775024414, 0.8571937084197998, 17.20915412902832, -27.813968658447266, -4.105556964874268, -9.777356147766113, 7.3142290115356445, -17.50891876220703, 0.8884754776954651, 7.328324317932129, 1.2368354797363281, 3.345118522644043, 22.855331420898438, 10.975051879882812, 5.875067710876465, 1.380712866783142, -9.670706748962402, -9.007917404174805, -9.263195037841797, 6.6661906242370605, 10.103606224060059, -9.070737838745117, 1.4975887537002563, -2.2421793937683105, 18.174318313598633, 15.115880966186523, 13.107190132141113, -10.710539817810059, 1.1186861991882324, -26.001413345336914, 18.52798843383789, -17.942285537719727, 4.9192914962768555, 7.402042865753174, -6.544563293457031, -3.49261736869812, 14.362349510192871, -4.962843418121338, 6.681001663208008, -3.949340343475342, -8.803561210632324, 2.2913215160369873, 3.5632448196411133, 12.871524810791016, 0.6463521718978882, 4.661428451538086, 13.750067710876465, -9.528578758239746, -15.314530372619629, -4.135672092437744, -10.426355361938477, -1.3822377920150757, -0.025447877123951912, 15.862683296203613, -4.096835613250732, -12.442728042602539, -10.386713027954102, 8.895848274230957, 6.498252868652344, 10.787692070007324, -0.7896243333816528, -10.00805377960205, 19.558263778686523, 5.908473968505859, 4.734633445739746, -11.03040599822998, -19.37802505493164, 17.63610076904297, 3.279019594192505, -2.715393543243408, -23.875961303710938, -15.190930366516113, -7.266798496246338, 10.697063446044922, -4.373730659484863, 14.945919036865234, -5.742359638214111, 7.327723503112793, -4.756442070007324, -20.96172523498535, 11.118979454040527, 21.744068145751953, -13.937780380249023, 7.623845100402832, 1.3258904218673706, -6.874846935272217, 10.296345710754395, -4.956431865692139, -1.2565133571624756, -5.028138160705566, 5.004487037658691, 10.506509780883789, 8.824210166931152, -6.575432300567627, -7.369333267211914, -15.364355087280273, 5.404026508331299, -7.195837020874023, -0.5333724617958069, 4.779808044433594, -15.977636337280273, -10.969160079956055, -3.774609327316284, 10.290620803833008, 0.6049002408981323, 5.960147857666016, -15.52729606628418, 8.93353271484375, -4.540820121765137, -2.4979348182678223, -10.663749694824219, 6.148275375366211, 3.6312055587768555, -11.491766929626465, -3.7005081176757812, 17.632225036621094, 3.437046527862549, -1.42035973072052, -2.090575695037842, 6.483675956726074, 0.9969505071640015, -2.2865517139434814, -2.5499801635742188, -4.452163219451904, 3.5029616355895996, 12.173103332519531, -2.1300926208496094, -4.644099235534668, -0.3329068422317505, -10.304213523864746, -14.28661823272705, 1.0324451923370361, 13.968013763427734, -1.6218128204345703, 4.655524253845215, -3.7207603454589844, 10.108030319213867, -1.4559768438339233, -1.0172094106674194, -12.094278335571289, -1.1483629941940308, 9.897651672363281, 7.868995666503906, 1.946585774421692, 8.254332542419434, 3.761216163635254, 4.343414783477783, 11.264628410339355, -13.865300178527832, 20.724369049072266, 6.937309265136719, 3.1834604740142822, -5.362112045288086, 3.227184772491455, -9.585631370544434, -8.105226516723633, -14.505006790161133, -3.622769355773926, 0.6883019208908081, -11.57503604888916, -13.603836059570312, 4.962660312652588, -11.05439567565918, -7.477720260620117, -1.330841064453125, 3.09079909324646, 3.130769729614258, 0.6644003987312317, -7.447854995727539, -8.716249465942383, 6.821331024169922, 7.8314595222473145, -0.9549630880355835, -4.472871780395508, -0.11958352476358414, -0.49467745423316956, 8.345527648925781, -0.37468868494033813, 11.850785255432129, 6.3245158195495605, -3.2483041286468506, 1.9552185535430908, 5.648300647735596, 5.611889362335205, 10.298556327819824, 3.961806058883667, -1.7087727785110474, 14.357854843139648, -11.572942733764648, 7.522429943084717, 24.41179656982422, 14.697531700134277, -5.802918434143066, -28.934165954589844, -11.970499992370605, 9.29132080078125, 2.4743518829345703, 6.964674472808838, 2.7267849445343018, -5.086567401885986, 5.701622009277344, 1.198499083518982, 0.8819279074668884, 6.3962225914001465, -10.757972717285156, -4.727327346801758, -18.393285751342773, 1.364534616470337, -8.611181259155273, 3.5001935958862305, -1.4655483961105347, 2.354398012161255, -3.0159730911254883, 6.0269365310668945, -4.522378921508789, -3.1619601249694824, 13.228105545043945, -4.134467601776123, 14.224595069885254, -4.373567581176758, 3.0916619300842285, 1.5921781063079834, 13.921637535095215, -10.224848747253418, 4.483209133148193, -3.9694249629974365, -2.8561105728149414, -7.960756301879883, 4.934548854827881, 2.513936758041382, 14.512369155883789, -14.17296028137207, -0.2973627746105194, 15.42447566986084, -0.4528850317001343, 3.8272886276245117, -0.2583692967891693, 7.037883281707764, 14.484830856323242, -3.406128406524658, -17.40852165222168, -1.0701898336410522, -17.989883422851562, 9.291977882385254, -13.904306411743164, 3.4728426933288574, -5.207475662231445, 0.12008346617221832, -5.387517929077148, 6.401033401489258, -6.926578998565674, -6.652718544006348, 5.054908275604248, -11.226823806762695, 2.0041699409484863, -22.483020782470703, 10.37082576751709, -1.5334950685501099, 2.038557529449463, -7.713339805603027, 8.182891845703125, 21.373218536376953, -3.383319616317749, 6.142014980316162, -4.908589839935303, -12.056941032409668, -5.97374153137207, 6.230024814605713, -5.128427028656006, 17.326454162597656, 2.644653797149658, -2.0415217876434326, 3.550023078918457, 3.8607094287872314, 13.96609115600586, -1.706686019897461, -1.0843251943588257, -1.2179473638534546, -23.8819522857666, -12.419014930725098, 6.987057209014893, 3.6896018981933594, 9.85297679901123, -8.464249610900879, 17.282405853271484, 8.77369213104248, 16.567689895629883, 4.976245880126953, 7.049544334411621, -6.351125240325928, 5.912404537200928, -6.051121234893799, -0.5278830528259277, 7.822145462036133, 12.013239860534668, 15.557418823242188, 1.297716498374939, -14.564839363098145, 17.467493057250977, 21.972118377685547, 5.600691795349121, 13.012540817260742, -23.09957504272461, 8.900548934936523, -3.1459879875183105, -3.011963367462158, -17.296390533447266, 7.014568328857422, -11.326814651489258, -18.962081909179688, 1.8885207176208496, 0.939251184463501, 3.8319449424743652, -5.223891258239746, -1.2838521003723145, -6.260964870452881, -13.167911529541016, 11.863849639892578, 12.721024513244629, -15.544239044189453, 12.544477462768555, 5.380289077758789, 12.188302040100098, -2.1174404621124268, 7.623773097991943, -8.28541374206543, 0.1716078817844391, 17.259750366210938, 9.450000762939453, 1.2573528289794922, 7.420277118682861, 10.602720260620117, -15.305200576782227, 4.131041049957275, 0.2922741770744324, -1.1033934354782104, 9.583467483520508, -4.366721153259277, 3.7393789291381836, -10.386701583862305, -0.7532370090484619, 5.672516345977783, 2.281848430633545, -5.5390543937683105, -2.7133665084838867, 5.89463472366333, 4.0245490074157715, -15.430925369262695, -1.2464935779571533, 6.639771938323975, 0.8927499651908875, -1.0134317874908447, -6.416287899017334, -9.895724296569824, 7.626184463500977, -3.726529836654663, 0.900709867477417, 4.867993354797363, 16.790908813476562, -0.3792695701122284, 13.320919036865234, -4.787123680114746, -4.56400203704834, 0.059542007744312286, -2.0457310676574707, 7.965128421783447, -19.837987899780273, -5.337766647338867, -1.6705732345581055, -11.601510047912598, -0.5001544952392578, 9.79754638671875, -0.1835061013698578, 20.373382568359375, -10.989112854003906, 4.721335411071777, 3.483538866043091, -4.253185272216797, -1.1911677122116089, 1.246852159500122, -1.770272135734558, 5.766810417175293, -5.87740421295166, 20.63582992553711, 9.50169849395752, 14.986897468566895, -0.9455006718635559, -5.7868757247924805, 17.300399780273438, -6.027209281921387, -13.117073059082031, 11.238313674926758, 9.760366439819336, 1.457838535308838, 15.109864234924316, -2.1151506900787354, 17.532546997070312, 16.133535385131836, 3.417595624923706, -7.1939778327941895, 22.506328582763672, 9.830831527709961, -12.294055938720703, -24.364892959594727, -9.792869567871094, -2.601512908935547, -5.948636054992676, 6.400373458862305, -5.647125244140625, -5.344182014465332, 1.7149062156677246, -1.5282148122787476, -11.210649490356445, 1.7868150472640991, -5.920031547546387, -26.073204040527344, 7.937655448913574, -18.9012508392334, -6.151300430297852, -18.13627052307129, -1.117361068725586, -19.486284255981445, -7.26064395904541, -14.2142972946167, -3.7489075660705566, 17.65064811706543, -4.81232213973999, 6.244156837463379, 1.4807379245758057, 5.143305778503418, -9.900394439697266, 0.6174356937408447, -5.636969566345215, -0.09631537646055222, -4.200459003448486, -5.98973274230957, -3.3498902320861816, -1.5936869382858276, -10.675960540771484, 7.851583480834961, -14.65018367767334, 1.9296507835388184, -2.086143732070923, -3.4468867778778076, 4.422177791595459, -0.9315452575683594, -21.836938858032227, -1.531386137008667, 9.580753326416016, -3.52329683303833, -8.190019607543945, -28.824932098388672, 7.621126174926758, -1.5649878978729248, 5.964235782623291, -11.420884132385254, -4.232032775878906, -19.390228271484375, -5.863857269287109, 6.702973365783691, -8.169057846069336, 9.486259460449219, 10.931289672851562, -0.4847412705421448, -12.649824142456055, -2.734755277633667, -0.7868916392326355, 5.804852485656738, 5.1014275550842285, -1.6656886339187622, -16.546873092651367, 3.5730152130126953, -9.801246643066406, -10.282184600830078, -4.293379783630371, 5.958195686340332, 8.045059204101562, 20.548460006713867, -2.486191749572754, 12.135551452636719, -21.273223876953125, 14.95383358001709, 13.614553451538086, -3.6364998817443848, -4.2746477127075195, 3.6462135314941406, 7.768528461456299, -4.701624393463135, -2.5452420711517334, 16.484392166137695, -6.212163925170898, 6.077966213226318, 2.5781943798065186, -3.289717197418213, 7.313160419464111, 0.5071485042572021, -0.0027347158174961805, -12.605037689208984, -1.1251403093338013, 15.344796180725098, -3.1015751361846924, -3.605070114135742, -4.283414363861084, 6.607847690582275, 9.098770141601562, -1.2201969623565674, -19.40290641784668, 14.272607803344727, 0.948634684085846, 7.991479873657227, -2.0375521183013916, 10.481563568115234, 4.468472480773926, 4.689244270324707, -10.722121238708496, -2.2953085899353027, 4.804679870605469, 14.331971168518066, 8.226428985595703, 17.172643661499023, -10.592490196228027, -5.641518592834473, 0.27982228994369507, 9.366338729858398, -0.13244673609733582, -3.6696906089782715, -5.421123027801514, -4.003004550933838, 16.145030975341797, 4.533280372619629, -5.623696327209473, 1.8432670831680298, 9.756111145019531, -1.5794026851654053, -15.504914283752441, -7.173760414123535, 1.0020581483840942, -13.762451171875, -7.866997241973877, 3.9101269245147705, 12.18776798248291, -0.6914899349212646, -18.619260787963867, 20.516517639160156, -9.050835609436035, 4.744886875152588, -18.111682891845703, -5.493093490600586, 8.388256072998047, -5.245376110076904, 14.531694412231445, -4.9091997146606445, -6.477087020874023, 4.787552833557129, 2.8596651554107666, -2.946803569793701, -10.124960899353027, 2.1983859539031982, -10.314377784729004, -10.164961814880371, 7.590303897857666, -2.9429409503936768, 20.890419006347656, 14.344966888427734, 8.443807601928711, 2.2203683853149414, 1.2921223640441895, -1.0135592222213745, 8.081436157226562, 15.961648941040039, -9.847630500793457, -1.5611357688903809, -10.510427474975586, 3.010918140411377, 4.729621887207031, 11.36335277557373, 2.077332019805908, 4.059347629547119, -23.137901306152344, -0.2571307122707367, 16.535598754882812, 3.460941791534424, -33.56787109375, -1.5093611478805542, -24.65865135192871, 0.17080366611480713, 6.498901844024658, -10.722799301147461, 28.115692138671875, 16.191457748413086, -18.269439697265625, 5.1626973152160645, -16.79049301147461, 1.168136477470398, -3.8967604637145996, 6.405167579650879, 3.6256918907165527, 3.005099058151245, -15.437617301940918, -15.629681587219238, 13.726612091064453, -9.55451774597168, 4.720090389251709, -9.023386001586914, 10.761214256286621, 7.470273971557617, -4.940809726715088, -0.8156181573867798, -16.30510902404785, 12.15638256072998, -5.38043212890625, 7.917817115783691, 13.074661254882812, 8.843735694885254, -4.923198699951172, 4.7698163986206055, -0.6807196140289307, -23.435192108154297, -2.514110803604126, 4.268213272094727, 8.494585990905762, 4.7856550216674805, 0.8428698182106018, -0.49048274755477905, -6.585885047912598, -15.50756549835205, -5.408957481384277, 2.3961708545684814, 1.5487031936645508, -0.09901009500026703, 4.5156121253967285, -0.885364294052124, -10.097953796386719, 10.367780685424805, 4.181957721710205, -5.4342217445373535, -8.644848823547363, 6.918410301208496, 1.4556751251220703, 13.116965293884277, 1.5753518342971802, -13.000179290771484, -7.74882698059082, 7.908459186553955, -14.238033294677734, -4.205294132232666, -2.303321123123169, 1.308968186378479, 7.180405139923096, 5.528826713562012, -2.6749629974365234, 11.423473358154297, -3.660079002380371, 4.292962074279785, -31.29146385192871, -4.211521148681641, 4.13924503326416, -13.152048110961914, -2.7986552715301514, -5.651607036590576, 15.445131301879883, -7.371020793914795, 16.832244873046875, 2.406885862350464, -2.6732258796691895, -0.3150981068611145, -10.304473876953125, 12.744217872619629, 9.473539352416992, 0.7902446985244751, -2.416214942932129, -19.733428955078125, -15.223823547363281, 3.7723941802978516, 4.5840229988098145, 7.5348734855651855, -7.4819793701171875, -9.523513793945312, -11.45637321472168, -5.099364280700684, 1.4572120904922485, -5.729948043823242, 13.126214981079102, -16.745378494262695, -4.3282575607299805, -0.20936211943626404, 11.995733261108398, -1.123048186302185, 6.398746967315674, 6.645129203796387, 11.069001197814941, 13.419662475585938, -0.5802949070930481, -12.801018714904785, 8.98354721069336, -14.465683937072754, -4.951077938079834, -9.145137786865234, -5.298237323760986, -4.109700679779053, -15.39897346496582, -8.818415641784668, 21.403257369995117, -6.011920928955078, 17.476383209228516, 0.027948040515184402, -0.4453272223472595, -27.64508056640625, 13.240665435791016, -12.023202896118164, 5.918078422546387, 8.701752662658691, -1.5248003005981445, -5.315995693206787, -10.681408882141113, -13.02769660949707, 2.14990496635437, -0.2387290596961975, -5.584775924682617, 11.540071487426758, -1.5889060497283936, -20.992870330810547, 0.4270821511745453, 0.36003977060317993, -9.497625350952148, -1.4115281105041504, 6.016152381896973, -1.474998116493225, -2.873084545135498, -3.2723209857940674, -4.930749416351318, -8.840375900268555, -2.3594067096710205, -15.460114479064941, -16.678531646728516, -8.646220207214355, 13.23909854888916, -2.743288040161133, -1.9087448120117188, 5.511743545532227, -4.673209190368652, -4.864405155181885, 17.25288963317871, -2.6824772357940674, 12.80803108215332, -5.975694179534912, 5.837120532989502, 12.980875968933105, -5.556992530822754, -1.8334842920303345, 15.164179801940918, -2.8837056159973145, 1.2328757047653198, 2.964693069458008, -8.637828826904297, -19.417802810668945, -14.778169631958008, -3.3920817375183105, 2.506997585296631, -6.923598766326904, -9.824127197265625, -1.0780798196792603, 0.9333437085151672, -0.21666395664215088, -1.767266869544983, 8.482916831970215, -3.847487211227417, 2.1914150714874268, -0.8415451645851135, -2.9398844242095947, -4.608841419219971, -28.963836669921875, -12.592772483825684, 5.458309173583984, 4.4325642585754395, 5.924410343170166, -6.1256937980651855, 5.795133590698242, -0.4971937835216522, 1.747873306274414, 2.2606966495513916, -2.424055814743042, -11.385099411010742, 6.093866348266602, -17.624921798706055, 4.14694881439209, 15.492472648620605, 5.070137023925781, -0.8813191056251526, -3.523110866546631, 18.449735641479492, 0.7641082406044006, -0.8991312980651855, 12.738194465637207, 2.7319674491882324, 5.362457752227783, 20.627412796020508, -19.099300384521484, 3.7734451293945312, -7.095188617706299, 4.337018966674805, 5.296792030334473, 0.7254502773284912, 4.231593132019043, 8.442669868469238, 5.162171840667725, -6.561364650726318, 1.2199616432189941, 0.06099905073642731, -4.16906213760376, -7.36428165435791, 4.942748069763184, -5.434135437011719, 1.9351446628570557, 5.764217853546143, 15.479511260986328, 10.186738014221191, 8.621570587158203, 2.8949732780456543, 5.472039699554443, -13.62070083618164, 5.724903106689453, 5.4802985191345215, 6.01560640335083, -9.280200004577637, -9.770661354064941, -11.99447250366211, 7.464576244354248, 1.7103874683380127, -1.9252039194107056, 2.379850149154663, -0.6566672325134277, -12.383278846740723, 3.5118179321289062, 1.9073021411895752, 5.2753424644470215, 7.249934673309326, 4.364647388458252, 6.804710388183594, -14.693175315856934, -0.32233062386512756, -7.795190811157227, -3.2104859352111816, -2.7392072677612305, -18.197919845581055, 0.6244308948516846, 5.047048568725586, -4.215395927429199, 2.12306809425354, 4.670596599578857, -1.9855748414993286, -11.841135025024414, 1.4155797958374023, -10.321220397949219, -5.9778971672058105, -8.424606323242188, -9.30689525604248, 3.0297250747680664, -1.0287537574768066, 7.577194690704346, -8.790135383605957, 1.2737689018249512, -8.902386665344238, -12.12496566772461, -5.1407599449157715, 7.2403764724731445, 6.42617654800415, -3.4068288803100586, -10.053000450134277, -1.7885105609893799, -1.0519542694091797, -12.13990592956543, -3.75557804107666, 11.705885887145996, -14.291983604431152, -2.6470980644226074, -26.947826385498047, -11.273326873779297, 18.494951248168945, 7.087008953094482, -8.601065635681152, 2.028790235519409, -2.150439739227295, -17.62677574157715, 1.2710106372833252, 10.699462890625, 22.367469787597656, 2.504545211791992, 8.947037696838379, -4.186219692230225, 12.27230167388916, -0.40665361285209656, -6.62968111038208, 3.245250701904297, 5.1225996017456055, 8.758216857910156, 9.999002456665039, -5.499163627624512, 3.617175817489624, 9.73386001586914, 10.67127799987793, -8.640047073364258, 7.593031883239746, -8.900450706481934, 7.180187702178955, 6.370301723480225, -3.0071866512298584, 17.407094955444336, 7.273900032043457, -17.631916046142578, 5.269162178039551, -0.9822540283203125, 15.173011779785156, 11.606761932373047, -2.6256213188171387, 2.9790279865264893, 1.6484295129776, -9.89974594116211, 2.478781223297119, 1.6115492582321167, -3.4132578372955322, 2.2311668395996094, 1.3184893131256104, 0.29872238636016846, 4.87661075592041, -5.201701641082764, -3.2521581649780273, -3.2357163429260254, -0.2389708161354065, -4.281216144561768, -5.886238098144531, 2.2046027183532715, 9.827592849731445, 22.081470489501953, -10.717028617858887, 16.24883460998535, -4.594471454620361, 12.348685264587402, 2.2091755867004395, 17.369539260864258, -7.216035842895508, 4.189001083374023, 29.440820693969727, -4.921200275421143, -5.655777931213379, -7.929632663726807, -7.224030017852783, 11.615837097167969, 6.450396537780762, -9.86030101776123, 2.6447200775146484, 8.009635925292969, 9.195719718933105, -3.5111818313598633, -12.533248901367188, -4.471697807312012, -21.124996185302734, 1.6474525928497314, 4.143661975860596, -9.2012300491333, -2.8677616119384766, 10.510951042175293, 5.679311275482178, 3.2087037563323975, 6.687179088592529, -12.028804779052734, 7.43727970123291, 3.0803489685058594, -2.585752248764038, -0.34983545541763306, 0.14278338849544525, 8.652523040771484, -6.902859687805176, 1.9229416847229004, 2.09057354927063, 3.3280162811279297, -0.8484771847724915, -3.415069341659546, -2.5407135486602783, -1.3340402841567993, -6.96357536315918, 2.5216965675354004, 13.156303405761719, 23.706045150756836, -4.561275959014893, -2.10137939453125, 10.280498504638672, -12.968990325927734, 5.372095108032227, -24.081501007080078, 16.949993133544922, 19.814884185791016, -5.274995803833008, -3.501303195953369, 7.906306266784668, 12.849569320678711, -1.181523084640503, -20.320398330688477, -1.0432742834091187, 11.406961441040039, -4.508912086486816, 4.208729267120361, -6.66703462600708, -6.0470871925354, 4.120922565460205, -8.370195388793945, 5.255887031555176, 6.078548908233643, 21.04001808166504, -19.948013305664062, 4.080204486846924, -17.360654830932617, -17.735761642456055, -16.12911605834961, 7.191762924194336, 7.201406478881836, -5.807694911956787, 19.908111572265625, 5.088448524475098, -15.259498596191406, 9.666133880615234, 10.377334594726562, 9.684354782104492, 0.3105607330799103, -10.923334121704102, 3.783364772796631, -7.335836887359619, -18.129959106445312, 19.970966339111328, -20.69176483154297, 1.4607740640640259, 11.126856803894043, 0.8002240061759949, -13.980504035949707, 2.825018882751465, -6.389302730560303, 14.218982696533203, 9.928326606750488, -20.383167266845703, -23.986265182495117, -10.37097454071045, 12.305244445800781, 7.905556678771973, 1.9348440170288086, -8.165661811828613, -12.743378639221191, 9.051465034484863, -10.200504302978516, -3.7771594524383545, -3.2504818439483643, 1.2379517555236816, -5.865311145782471, 12.961471557617188, 11.304950714111328, 12.499603271484375, -8.26070499420166, -0.4422505497932434, 7.457947254180908, 1.6248092651367188, -3.9782657623291016, -15.09013557434082, -7.544154167175293, -11.582876205444336, -10.058084487915039, -8.968454360961914, -3.1774520874023438, 7.826925277709961, 15.955406188964844, -0.308422327041626, -14.302186012268066, 2.9627161026000977, 9.588473320007324, 5.351372718811035, -3.1243228912353516, -6.799798011779785, -6.18925666809082, 6.318121910095215, -6.9701409339904785, -8.768573760986328, 5.07903528213501, -3.9968106746673584, 1.5454787015914917, -18.777786254882812, -1.3497250080108643, -2.254364013671875, -2.4894752502441406, -14.897687911987305, 13.155980110168457, -17.790916442871094, -5.028448581695557, -0.014671283774077892, 12.661896705627441, 15.922954559326172, 5.5392022132873535, 2.5755703449249268, 8.569262504577637, -2.0246081352233887, -30.923137664794922, -2.9876039028167725, -2.0821306705474854, 9.05068588256836, 2.1653833389282227, -4.954684257507324, -7.13847541809082, 12.152584075927734, -6.9661946296691895, -12.272729873657227, 8.285664558410645, -2.2732250690460205, -12.43311882019043, 12.57063102722168, 4.606363296508789, -16.214540481567383, -5.522064685821533, -1.756220817565918, -0.20437952876091003, -12.321281433105469, 20.39535140991211, 5.360646724700928, 21.164520263671875, 0.6736053228378296, -3.3272693157196045, -5.510790824890137, 11.041776657104492, -1.2905539274215698, 0.676558256149292, 0.966930091381073, -18.10139274597168, 1.2304800748825073, -5.872808456420898, -15.530986785888672, -1.5750012397766113, 9.981307029724121, 17.00293731689453, -19.453632354736328, -8.156890869140625, -7.323706150054932, -13.23638916015625, -1.232674241065979, -6.4586076736450195, -3.0741353034973145, 7.388772964477539, -1.6392841339111328, -17.662357330322266, 5.288219928741455, -3.379621982574463, -4.789112567901611, -3.0930142402648926, -6.420417308807373, 32.1521110534668, 5.240396976470947, 0.3697817325592041, -11.596604347229004, -0.1189727932214737, -2.4793877601623535, -17.620664596557617, -0.6077135801315308, 14.58743953704834, 5.667079925537109, 3.3498806953430176, 3.9367058277130127, 9.143436431884766, 8.634521484375, 11.022042274475098, -3.3536806106567383, -4.649104595184326, 4.092108726501465, -5.163318634033203, -13.021077156066895, 5.961671829223633, -0.2054869830608368, 2.6937971115112305, 4.039251804351807, 4.91605806350708, -8.415571212768555, -7.041830539703369, 0.5424798130989075, 9.355724334716797, -20.17228889465332, 5.506921291351318, 1.8488190174102783, -0.7737588882446289, 7.214691638946533, -3.6756787300109863, 24.63941764831543, -2.1146271228790283, -5.195891380310059, -2.95215106010437, -11.738178253173828, -0.727085530757904, -14.801522254943848, 15.040651321411133, 3.5944197177886963, 9.691530227661133, 0.7857522964477539, 12.379961013793945, 4.554933547973633, 7.5731425285339355, 13.360939025878906, 4.946315765380859, -2.7557783126831055, 22.03451919555664, 13.438940048217773, -4.424501895904541, -1.2202928066253662, 6.945159435272217, 4.7943925857543945, 2.7921409606933594, -8.045635223388672, 3.489694833755493, -15.92709732055664, 0.7640472054481506, -5.130390167236328, 1.721412181854248, -14.762253761291504, -7.241524696350098, 1.0364288091659546, -7.005167484283447, -0.04267329350113869, 0.5175913572311401, -11.5278959274292, 16.328054428100586, -5.6168389320373535, -11.149264335632324, 6.317580223083496, 14.924718856811523, 8.398482322692871, -1.677828073501587, -0.490988552570343, 3.0969901084899902, -2.4084091186523438, 3.4552695751190186, 8.096235275268555, -4.969015121459961, 3.5645172595977783, 21.566665649414062, -13.720781326293945, 9.344949722290039, 17.518417358398438, -0.45392072200775146, 7.793710708618164, -1.6952924728393555, 9.439483642578125, 6.487473964691162, 11.976194381713867, 3.4891018867492676, 11.669170379638672, -3.4490180015563965, -3.701820135116577, -8.857157707214355, -19.356571197509766, 8.704460144042969, -11.33947467803955, -7.519800662994385, 7.204807281494141, 7.037230014801025, -8.091140747070312, 3.6755218505859375, -14.792821884155273, -4.81735897064209, 3.9316294193267822, 14.200152397155762, 12.928323745727539, -14.311169624328613, 8.42153549194336, 4.401510238647461, 14.334989547729492, 5.10672664642334, -19.871261596679688, -5.817978382110596, -2.024142265319824, 5.563421249389648, 2.259101629257202, -5.959830284118652, -29.861228942871094, -1.7332642078399658, 16.902362823486328, 4.119811534881592, 17.991117477416992, -5.103734016418457, -15.333924293518066, -10.014348983764648, 6.539756774902344, -21.86244773864746, -4.799147605895996, 19.383676528930664, 14.598691940307617, -4.152742385864258, 3.1574223041534424, 8.346556663513184, -6.664609432220459, -16.602943420410156, 9.799010276794434, 8.486451148986816, 1.810271143913269, -16.292821884155273, 0.31486010551452637, -6.098201751708984, 2.9067273139953613, -10.539669036865234, -8.265616416931152, -3.5126304626464844, -7.206717491149902, 3.2394485473632812, 14.864128112792969, -5.379065990447998, 2.9559266567230225, 10.049033164978027, -1.579988718032837, 14.62802505493164, -11.673582077026367, -2.8395938873291016, -4.390981674194336, 10.918241500854492, 1.5726683139801025, -7.790199279785156, 17.909923553466797, 6.234871864318848, -17.014720916748047, 2.7085204124450684, -8.231088638305664, -16.20169448852539, -0.4959288537502289, -16.132349014282227, 4.7077460289001465, 2.4986865520477295, 6.48866081237793, -6.365538597106934, -8.638171195983887, 16.19194221496582, -9.37298583984375, 15.602112770080566, 4.658144950866699, -3.103879928588867, -14.201265335083008, -1.2084076404571533, -1.6263645887374878, -2.7907142639160156, 8.363202095031738, -3.2574236392974854, 8.98900032043457, 0.5024423599243164, -15.988515853881836, 3.796055793762207, -1.8735002279281616, 15.694562911987305, 4.412143707275391, -14.415218353271484, 13.114551544189453, 11.318883895874023, -8.346231460571289, 15.438910484313965, -4.053398132324219, -4.592866897583008, 7.868459701538086, -8.862258911132812, 20.974397659301758, -2.5676023960113525, 3.7184581756591797, -7.065662384033203, 0.9604830145835876, -14.057321548461914, 10.645864486694336, -2.4818074703216553, 2.3054494857788086, 3.6447548866271973, -4.627468585968018, 6.333674430847168, 0.8695766925811768, -2.53690505027771, -4.279294490814209, 19.33249282836914, 15.515889167785645, 0.24794858694076538, -2.611661911010742, 20.24368667602539, -4.796353340148926, 16.00492286682129, 18.337400436401367, 9.451152801513672, 4.999529838562012, -10.538032531738281, -11.074468612670898, -4.200250148773193, 10.23284912109375, -15.829488754272461, -12.956916809082031, 6.958518028259277, 1.1271629333496094, -12.17949104309082, 19.653282165527344, -2.6135478019714355, -16.24735450744629, -13.664365768432617, -4.371095180511475, -3.9474403858184814, 10.452823638916016, -9.766045570373535, 1.6740198135375977, 2.1654984951019287, -3.7375400066375732, 0.7512808442115784, -2.565793752670288, 14.012414932250977, -13.423982620239258, -17.65509033203125, -6.852360725402832, -4.746576309204102, 7.106019020080566, 10.087379455566406, -11.45663070678711, -10.994730949401855, -11.625710487365723, -6.794294834136963, 11.788496017456055, -18.60609245300293, -5.805156707763672, 4.636127471923828, 5.29563045501709, -13.306078910827637, -7.779306888580322, -2.035835027694702, 7.1868391036987305, -6.605942249298096, -11.371623992919922, 10.875839233398438, 18.6483154296875, 16.47380256652832, -13.846993446350098, -17.70889663696289, -15.868058204650879, 8.368332862854004, -2.94985032081604, 6.887150287628174, -3.5092880725860596, -13.137116432189941, 7.290470600128174, -16.333187103271484, -6.131697654724121, -1.42433762550354, 5.500431537628174, -2.422966718673706, -5.244472026824951, 9.960933685302734, 3.68923282623291, -15.394319534301758, -3.542177200317383, 23.746341705322266, -8.861576080322266, 8.914215087890625, -4.705604076385498, -7.118721961975098, -3.4630088806152344, 1.2026822566986084, -21.456144332885742, -9.4827299118042, 12.481675148010254, -3.1586461067199707, 5.002845764160156, -12.448102951049805, 12.382515907287598, -0.9457488059997559, 0.6723923683166504, -0.12916377186775208, 15.050048828125, -10.867435455322266, 8.978302955627441, 7.076775550842285, 2.138129711151123, 24.366168975830078, 8.499030113220215, 4.22717809677124, 1.2119216918945312, -7.344229698181152, -1.1948741674423218, -0.8874357342720032, -7.946037292480469, -9.322380065917969, 7.298603057861328, 3.6567578315734863, 5.9524736404418945, 0.3017287254333496, 6.888998031616211, -15.100378036499023, -7.798460006713867, 14.114828109741211, -1.984522819519043, 9.452990531921387, 11.77995491027832, 10.613136291503906, 7.1107635498046875, 17.47394371032715, 11.340644836425781, -0.571607768535614, 9.499463081359863, -15.72871208190918, 8.359916687011719, -0.5009756088256836, -5.327674865722656, 0.4709276258945465, 2.3081283569335938, 0.5043255090713501, -9.809393882751465, 7.309017181396484, 22.636184692382812, -5.37614107131958, 6.220174789428711, -9.38966178894043, -0.36718058586120605, -9.038558959960938, -10.04894733428955, -13.136353492736816, -7.187631130218506, -4.984821319580078, -3.1591267585754395, -24.17203712463379, -8.317233085632324, 8.08471393585205, -18.01650619506836, 5.113818645477295, -22.671802520751953, -15.44344711303711, 8.470733642578125, -1.556459665298462, 9.239761352539062, 7.685162544250488, 4.430803298950195, 3.867445230484009, -3.8452746868133545, 4.535583972930908, -6.002620697021484, -3.115086317062378, -14.60435676574707, 1.506508708000183, -5.134957313537598, -3.4505093097686768, -9.466011047363281, -2.727043390274048, 7.896446228027344, 5.392585754394531, -17.39691734313965, 9.363585472106934, 3.046349287033081, -7.636584281921387, 3.0339155197143555, 1.083762526512146, 6.122898101806641, -9.163946151733398, 3.6875197887420654, 5.0745368003845215, 17.70246696472168, -23.129215240478516, 12.204862594604492, 10.512534141540527, -8.641825675964355, -0.029713185504078865, 9.14680290222168, -6.096921443939209, 8.038030624389648, -4.048900127410889, 4.054764747619629, -7.313823223114014, 1.1566715240478516, -8.896820068359375, 1.843935489654541, -8.408947944641113, -0.42288631200790405, 7.182576656341553, -3.6762044429779053, 6.4182281494140625, -11.545480728149414, -25.142776489257812, 5.667901039123535, -2.091096878051758, 0.5605328679084778, 0.35100698471069336, -3.5202348232269287, -23.675128936767578, -7.916522026062012, -11.469267845153809, 14.830705642700195, -17.91571044921875, 0.4755234122276306, -18.822467803955078, -20.540254592895508, 4.617791652679443, 11.39138412475586, -3.931621551513672, -9.343716621398926, 19.549890518188477, 5.084839820861816, 1.756548285484314, -11.670489311218262, 4.827429294586182, 5.540704727172852, -24.179868698120117, 6.817445278167725, -18.493465423583984, -5.392124652862549, -15.732251167297363, 0.8847994804382324, 14.797590255737305, 10.219840049743652, 1.1732045412063599, -3.886171340942383, 4.938416004180908, -16.074951171875, 10.624947547912598, -0.9614927768707275, 2.874840497970581, 1.4327858686447144, -20.41328239440918, -6.759814739227295, -6.692960262298584, 2.5797176361083984, -19.602624893188477, 22.270877838134766, 15.286542892456055, 10.965328216552734, 15.1563081741333, 19.757253646850586, 0.5504470467567444, -0.4615718424320221, 4.365106105804443, 7.536831855773926, 14.331818580627441, -26.092958450317383, 0.6465084552764893, -7.221770763397217, 7.637627601623535, -0.8391994833946228, -14.396615028381348, 3.3182530403137207, -7.068614482879639, -3.098196506500244, -2.0131983757019043, -9.126611709594727, -1.2974882125854492, -3.0372886657714844, 11.85576343536377, -10.136622428894043, -8.539764404296875, -4.712523460388184, 0.2521538734436035, 11.31517505645752, -2.264251947402954, 4.757727146148682, 8.340437889099121, 2.6659393310546875, -2.9323792457580566, -5.388139724731445, 13.60145092010498, 8.517501831054688, 5.457538604736328, 16.14626121520996, -8.611934661865234, 11.646854400634766, -1.279914140701294, -16.354772567749023, -9.211649894714355, 1.9841927289962769, 12.970904350280762, -20.930496215820312, 11.118511199951172, -2.412566661834717, 0.8932993412017822, 5.256104469299316, -15.2974271774292, 2.471562147140503, -2.8400115966796875, -0.10020826756954193, 22.357101440429688, 2.949885129928589, -1.506320834159851, 1.1212422847747803, 22.537755966186523, 15.526041984558105, 9.511955261230469, 1.289243221282959, 0.7430512309074402, -11.649253845214844, -13.963713645935059, 13.104911804199219, 7.9400434494018555, 1.474955677986145, -0.23488101363182068, -10.660469055175781, 10.268410682678223, 3.5887458324432373, 3.4230637550354004, -12.206707000732422, 18.99046516418457, -2.989997386932373, -8.157379150390625, -1.2617002725601196, -3.1372153759002686, 10.037979125976562, 1.868294358253479, -9.782214164733887, -15.459627151489258, -18.930959701538086, -8.964123725891113, 2.7561581134796143, -12.211584091186523, 0.3656345307826996, -15.961305618286133, -0.5649851560592651, -3.5671210289001465, -11.389240264892578, 5.145684242248535, 11.95375919342041, -2.520242214202881, -2.0728251934051514, -5.05618953704834, 1.0530462265014648, 5.439119338989258, 9.471816062927246, -0.9551517963409424, 21.75598907470703, 30.713134765625, 25.52515983581543, -20.38497543334961, -13.554380416870117, -6.657111167907715, 1.1715449094772339, 10.388108253479004, -4.368647575378418, -7.933949947357178, -4.636687278747559, -10.817338943481445, -9.416582107543945, 0.3288368582725525, -31.935325622558594, 20.371618270874023, 4.787035942077637, 5.295958518981934, 14.08321475982666, -6.044523239135742, -2.351858377456665, 2.017477512359619, -8.398538589477539, 0.43333524465560913, 1.3069202899932861, 4.68228816986084, 11.350870132446289, 6.311611175537109, -5.174227237701416, -10.550004959106445, -6.054868221282959, 6.780416488647461, -22.132797241210938, 2.794790267944336, -11.483918190002441, -7.41959285736084, -1.5290254354476929, -15.302962303161621, 1.1797749996185303, -0.33488091826438904, -0.08644784986972809, -9.460042953491211, 22.872089385986328, 0.7316949963569641, 11.563173294067383, 6.881661891937256, -13.048930168151855, 3.9722609519958496, 11.051053047180176, -9.28763484954834, 8.256059646606445, 15.751083374023438, -5.701056003570557, -2.167851448059082, -6.654350757598877, -5.820259094238281, 12.174558639526367, -9.662049293518066, -16.86521339416504, -6.585445404052734, 19.339563369750977, -1.600101113319397, 2.762053966522217, 7.4470930099487305, 5.421066761016846, 9.786425590515137, 18.049339294433594, -0.04325641691684723, 4.803661346435547, 11.173151969909668, -11.702669143676758, 16.929908752441406, -9.019782066345215, 19.790334701538086, -7.249095439910889, 7.3166303634643555, -9.89197063446045, -1.8337812423706055, -3.6163341999053955, -2.593799591064453, 2.050506353378296, -5.730795860290527, -7.232156753540039, 13.831148147583008, -9.906925201416016, -16.51565933227539, 17.41930389404297, 22.072601318359375, -4.961126804351807, 8.369311332702637, -0.6407202482223511, 8.943251609802246, 1.9913972616195679, -7.736804962158203, 7.065339088439941, 24.851953506469727, -7.712922096252441, 14.985910415649414, -20.38262939453125, -3.6011459827423096, 11.11715316772461, 13.524225234985352, -5.502312660217285, 10.379382133483887, 2.425156593322754, 8.193601608276367, 12.734251022338867, -10.736154556274414, -2.412097692489624, -16.470664978027344, 16.894376754760742, -3.304918050765991, -9.471887588500977, -20.40420150756836, -0.10921765863895416, -12.545088768005371, -5.098536491394043, 13.456746101379395, 3.094080686569214, 0.36409202218055725, 13.68220329284668, 17.745168685913086, 0.3652428984642029, -7.982964515686035, -7.833996295928955, -1.4660488367080688, 8.646913528442383, 12.066335678100586, -17.346513748168945, 1.1521581411361694, 4.827688694000244, 9.0670747756958, 20.87779998779297, 6.619718551635742, -1.3729689121246338, -9.805316925048828, -15.542692184448242, 12.421516418457031, -7.933659076690674, -5.074204921722412, 8.71388053894043, 6.585356712341309, 2.9137396812438965, 10.493887901306152, -12.067795753479004, 0.5028945207595825, 6.585653305053711, 0.627143383026123, -11.535593032836914, -17.32589340209961, 6.255446434020996, 4.724708557128906, -8.341017723083496, -9.427056312561035, 5.713079452514648, 5.082316875457764, -8.08393669128418, -7.366767406463623, -0.06144434213638306, -10.506357192993164, 4.897744178771973, -13.68462085723877, -17.057769775390625, 13.96449089050293, -17.78809928894043, -6.271828651428223, 4.716397762298584, -11.579692840576172, 21.884239196777344, -3.5024266242980957, 4.1389617919921875, -13.627152442932129, -4.5707855224609375, -13.478015899658203, 14.28580093383789, -8.071023941040039, 3.0588955879211426, -6.111278057098389, 5.508496284484863, 1.6774194240570068, 3.4017703533172607, 23.36988639831543, -7.632865905761719, 2.242520570755005, 1.8327302932739258, 17.97687339782715, -7.720585823059082, 12.27743911743164, 6.182701587677002, -1.509291410446167, 15.008914947509766, -0.12980690598487854, 1.5799331665039062, 3.5840771198272705, -9.268636703491211, -2.2251431941986084, -1.9863536357879639, -3.391666889190674, -0.09057092666625977, 6.644464015960693, 6.250660419464111, -1.9425685405731201, 1.8726904392242432, -2.5188281536102295, -10.754467010498047, -1.1644867658615112, 0.9854138493537903, -4.840460777282715, -23.962526321411133, 17.241744995117188, -5.067026615142822, -7.655184745788574, -10.841808319091797, 4.57857608795166, -5.646002292633057, -9.529006958007812, -8.514442443847656, 4.907188892364502, -11.334810256958008, -6.165811538696289, -0.48855191469192505, 1.6552186012268066, -4.601574420928955, -5.618671417236328, -4.773486137390137, 5.343541145324707, 12.283851623535156, -5.958823204040527, 3.625953197479248, 6.306507110595703, -0.020208626985549927, 3.825711965560913, -13.482507705688477, 9.966765403747559, -11.15284538269043, -13.661762237548828, 1.2071956396102905, -3.3069193363189697, 19.527544021606445, 0.7949590086936951, 19.022174835205078, -9.894620895385742, 32.31453323364258, -3.271965742111206, 10.735115051269531, -18.22145652770996, 2.2972123622894287, -10.617985725402832, -0.0503683015704155, -14.94326114654541, 18.380115509033203, 8.35058307647705, 14.235651016235352, 12.313627243041992, -7.050368785858154, -2.7014846801757812, 4.699531078338623, 26.0240535736084, 3.5348546504974365, 5.234584331512451, -9.609768867492676, 1.5196014642715454, -8.592422485351562, 8.649276733398438, 10.409395217895508, 7.418749809265137, -4.259284019470215, 4.00343656539917, 2.8963656425476074, -12.535001754760742, 15.986562728881836, -8.356782913208008, -4.885652542114258, -19.919992446899414, 2.8354129791259766, -7.705327033996582, 5.620992660522461, 14.675437927246094, 19.153621673583984, -0.15928655862808228, 2.7194321155548096, 18.039209365844727, -8.065470695495605, -13.528117179870605, -20.872724533081055, -11.521430015563965, 0.45335862040519714, 7.744889259338379, -4.275478839874268, -1.4761226177215576, 9.235406875610352, 11.529401779174805, -4.163328647613525, -14.545951843261719, 2.4136123657226562, 19.91388702392578, 0.9393298625946045, -7.843076705932617, -12.87417984008789, -10.33266830444336, -0.2292483150959015, 2.65431547164917, -7.937747955322266, -17.255691528320312, 6.380912780761719, 2.6715469360351562, 6.108120918273926, -18.95708465576172, -6.421223163604736, 4.343271255493164, -4.404911041259766, -10.30725383758545, 19.221555709838867, -1.346477746963501, 2.7243967056274414, -5.5444746017456055, -0.8449735641479492, -12.625813484191895, 3.3013110160827637, -2.8601465225219727, 14.630989074707031, -4.444332122802734, -4.763161659240723, -6.8653974533081055, 11.133987426757812, 8.887890815734863, 0.012703617103397846, -5.814438819885254, -19.597984313964844, 3.389193058013916, 2.4226298332214355, 3.8890483379364014, -27.53787612915039, 1.3684813976287842, -2.6557013988494873, -10.933073043823242, 4.737213134765625, 15.371928215026855, 15.543007850646973, -20.219802856445312, 2.7785286903381348, 14.089731216430664, 4.729673385620117, -6.342790603637695, -2.8392882347106934, -1.7110939025878906, 1.7264093160629272, -12.124419212341309, -17.598796844482422, 8.269315719604492, 3.992119789123535, -1.9835299253463745, 10.56049919128418, -7.525740623474121, -6.446817398071289, 1.2108076810836792, -6.154391288757324, 7.459268093109131, 14.475362777709961, -7.828860759735107, 1.8221080303192139, 1.3169777393341064, 3.864370584487915, -12.12347412109375, -12.619054794311523, -11.70749282836914, 3.1542861461639404, -4.460712432861328, -0.7622603178024292, -5.6981916427612305, -4.547990322113037, -1.2102457284927368, 14.562065124511719, 1.6400094032287598, -1.8389348983764648, 0.23887845873832703, 9.761131286621094, -12.788347244262695, -10.138426780700684, -13.924790382385254, 13.904213905334473, 2.264075517654419, 2.4638054370880127, 13.546299934387207, -1.1212776899337769, 6.998142719268799, 2.5921273231506348, -6.3722734451293945, 16.462703704833984, 19.329307556152344, -9.989648818969727, 0.5249788165092468, -14.126220703125, 19.24530792236328, -1.2218647003173828, -12.259870529174805, 0.15478457510471344, 9.6109037399292, 2.408616542816162, 6.2551116943359375, 0.3295299708843231, 6.285382270812988, -2.669051170349121, -3.5543065071105957, -11.397603988647461, -14.405664443969727, 5.881457805633545, -3.4458231925964355, 5.8802690505981445, 1.240830898284912, 0.23702022433280945, 2.2027950286865234, -14.788686752319336, 2.0371716022491455, -0.8973636627197266, -23.80925178527832, -10.406079292297363, 1.0287842750549316, 8.991579055786133, -6.80687952041626, 7.080431938171387, 10.499300003051758, 9.086216926574707, 5.242965221405029, -4.864297389984131, -11.583087921142578, -7.119502544403076, 3.1319327354431152, -28.90341567993164, -1.43580961227417, -1.8719468116760254, 8.441598892211914, -7.272442817687988, 12.240675926208496, -4.178735256195068, 0.8080095648765564, -8.204704284667969, -7.116256237030029, -9.71217155456543, -7.975046634674072, -0.9776006937026978, -14.883970260620117, 8.260920524597168, 7.89430570602417, -6.62983512878418, -2.4294352531433105, -16.37396812438965, 0.587096631526947, 17.356985092163086, -3.7380239963531494, 6.274486541748047, -10.480262756347656, 33.1743278503418, 0.9831697940826416, 16.49203872680664, -6.27060604095459, -3.394896984100342, 15.64896011352539, 0.31512850522994995, -2.462050199508667, 2.6950831413269043, -16.992774963378906, -0.05122211202979088, -5.009750843048096, -12.80663013458252, 2.5723483562469482, -2.0373001098632812, 0.3191301226615906, -8.873153686523438, 5.88314151763916, 6.003611087799072, 1.3562451601028442, -4.031487464904785, -24.22289276123047, -6.618056297302246, -4.170351505279541, -15.803106307983398, 18.82613754272461, -8.534822463989258, -6.279332160949707, -13.85006046295166, -19.389328002929688, -2.3528435230255127, -1.732946515083313, 4.589946746826172, 6.563752174377441, 3.6486082077026367, -7.569660186767578, 6.925915718078613, -12.772422790527344, 18.488035202026367, 2.2369346618652344, 15.27534008026123, 21.604961395263672, -29.367467880249023, -4.760998725891113, 5.013030052185059, -4.107738971710205, -2.0653467178344727, 7.2966790199279785, 7.327020645141602, 10.056443214416504, -3.9071238040924072, 4.089863300323486, 0.11323920637369156, 0.643683671951294, 1.5717087984085083, -8.32450008392334, 3.431532621383667, -7.11602258682251, 10.016985893249512, -1.005378007888794, 4.169793605804443, -11.290929794311523, -3.532886266708374, -18.148229598999023, -18.0636043548584, 3.7616047859191895, -7.552433967590332, -8.116053581237793, 14.118593215942383, -11.698148727416992, -1.2832581996917725, -6.80283260345459, 2.127121686935425, -8.545724868774414, -19.568984985351562, -14.610172271728516, -0.03958088159561157, 10.619982719421387, -10.1505765914917, -0.5857892036437988, 0.32663121819496155, -2.653846263885498, 10.982213973999023, -2.033261299133301, 11.613792419433594, 13.349332809448242, -7.1755781173706055, -9.198352813720703, -10.846277236938477, 5.358211994171143, -6.035168170928955, -8.202716827392578, 7.933952331542969, -2.608529567718506, 28.497207641601562, 5.892755508422852, 12.177473068237305, 1.504903793334961, 8.012401580810547, 0.9448773860931396, 4.876536846160889, -1.0788081884384155, 14.05306339263916, 3.083922863006592, 0.2759524881839752, -7.294654846191406, 11.826109886169434, 5.2857465744018555, -12.156118392944336, 1.8486716747283936, 24.949586868286133, -1.9880479574203491, 3.234889507293701, 11.027303695678711, -3.151620388031006, 9.03197193145752, 8.656109809875488, 1.371870517730713, -12.31421947479248, 13.961710929870605, -6.914939880371094, -13.248152732849121, -7.446994304656982, 18.587060928344727, -7.12005615234375, -8.698212623596191, 0.007208478637039661, -3.8704214096069336, -8.70155143737793, -10.730143547058105, 0.29863330721855164, -5.355170249938965, -12.43760871887207, 20.5808162689209, 9.016732215881348, -7.687536239624023, 4.698921203613281, 13.624478340148926, 3.514305353164673, -3.73872709274292, 13.12725830078125, 1.119455337524414, 6.784278869628906, -3.4040021896362305, 0.6149371266365051, 20.34734344482422, 9.811206817626953, -1.4567936658859253, 8.370179176330566, 3.052304983139038, -2.2148277759552, -0.07577282935380936, -1.339386224746704, 0.6442397236824036, -1.9513657093048096, 6.67130708694458, -0.40584877133369446, -6.906795978546143, 6.324797630310059, 1.9786466360092163, -17.678186416625977, -9.606618881225586, 4.977660655975342, 8.221473693847656, 11.889777183532715, -10.750993728637695, 12.502965927124023, 3.754103422164917, 2.8339834213256836, -5.453648567199707, 6.419643402099609, -2.09702730178833, -0.11091209948062897, 22.99703598022461, -4.308763027191162, -1.1776434183120728, -2.084317445755005, -13.212193489074707, -4.44423770904541, -5.48679256439209, -8.967020034790039, 8.868537902832031, 17.881568908691406, 0.4690605700016022, -2.5911200046539307, 5.1067609786987305, 0.4676569998264313, 10.516753196716309, -9.237074851989746, -2.5144078731536865, 20.9681339263916, 1.9199018478393555, 13.502805709838867, 16.90538215637207, 3.124002456665039, -3.9731311798095703, -5.804324150085449, -29.552824020385742, 6.8608293533325195, -10.969110488891602, 20.649778366088867, -0.33253949880599976, -35.507835388183594, -0.7289218902587891, -10.85254192352295, 5.324014663696289, -2.039557456970215, 1.6943814754486084, -5.261507034301758, -6.465569496154785, -17.44305419921875, -14.413651466369629, -18.027143478393555, 4.547467231750488, 9.725868225097656, 2.55005145072937, -3.260526418685913, -5.689239025115967, -11.019163131713867, 10.656980514526367, -28.37813377380371, -1.4849231243133545, -16.574771881103516, 14.844197273254395, -13.69404411315918, -12.781349182128906, 0.03453109785914421, 9.385330200195312, 0.42438384890556335, -6.277937889099121, -10.009239196777344, -11.013650894165039, -10.644632339477539, -0.6540347337722778, 6.247082233428955, -5.954067230224609, -7.329692363739014, -1.0033650398254395, -11.252297401428223, -11.586832046508789, 14.212650299072266, -5.309823989868164, 11.278314590454102, 7.045404434204102, -12.763769149780273, 14.984893798828125, 2.1708126068115234, 5.853664875030518, 11.433469772338867, -6.183976173400879, -6.472430229187012, -12.132404327392578, 5.02632999420166, -27.021408081054688, 0.2737900912761688, -0.7565256357192993, -3.4759058952331543, -6.7143707275390625, -12.976577758789062, 0.7524499297142029, 6.537513732910156, -2.827549457550049, 20.321969985961914, 4.973270416259766, 0.2535232603549957, 3.3723835945129395, -4.1234612464904785, -7.398667335510254, -5.976390838623047, 4.393491268157959, 10.158836364746094, 2.501291036605835, 11.136940956115723, -4.931423187255859, -4.994912624359131, -8.107234954833984, 4.655735015869141, -13.848152160644531, 2.30845308303833, 1.0132333040237427, -3.9767239093780518, 1.7364872694015503, 5.829530239105225, 9.625200271606445, -14.373872756958008, 3.9756574630737305, -2.881927251815796, -13.454286575317383, -6.397531509399414, -3.119443655014038, -5.06410026550293, -21.153472900390625, -12.865342140197754, -3.700321912765503, 1.8882869482040405, 9.70427131652832, -12.948070526123047, 12.391435623168945, -2.551072120666504, 4.602325439453125, 4.164767742156982, 4.456110954284668, -16.137636184692383, -2.9806995391845703, 3.21600079536438, -0.08893140405416489, 12.629793167114258, 12.640132904052734, -2.2836697101593018, 9.528434753417969, -8.63941764831543, -20.04735565185547, 7.006809234619141, 10.275030136108398, 13.085709571838379, 6.202338218688965, -8.37486457824707, 6.098172187805176, -8.486409187316895, 11.175049781799316, 5.716825485229492, -4.015097141265869, 14.85338020324707, 2.5417308807373047, 8.2449951171875, -15.224227905273438, -0.42171937227249146, 4.264362335205078, 12.972742080688477, 6.791693687438965, -0.5708537697792053, -3.821598529815674, -9.485701560974121, 3.9107255935668945, 20.02519416809082, -4.765570640563965, 0.37683141231536865, -17.124961853027344, 2.596435070037842, -13.160324096679688, -10.07796859741211, -3.475919723510742, -17.9985294342041, 11.5594482421875, 17.220550537109375, 1.9313337802886963, 4.7332916259765625, -6.155856132507324, 15.814748764038086, 9.128405570983887, -15.472173690795898, -9.781926155090332, 1.399120807647705, 21.129417419433594, 4.352902889251709, -13.203763008117676, -8.536044120788574, 15.450753211975098, 14.625502586364746, 9.620369911193848, -1.9160759449005127, 2.440669536590576, -0.3927609324455261, -4.604331970214844, -17.142127990722656, -23.02682876586914, 8.51469612121582, 3.723227024078369, 11.364049911499023, 0.5140094757080078, 12.412109375, -10.386567115783691, 10.585391998291016, -7.088846683502197, 2.774116039276123, 6.955183029174805, -0.024320457130670547, 8.613364219665527, -2.8818063735961914, 17.063093185424805, 6.085413932800293, -1.9176757335662842, 23.53921890258789, -5.709145545959473, 3.136329174041748, 11.670714378356934, 4.612666606903076, -7.387013912200928, 9.255388259887695, -20.848798751831055, -4.893334865570068, 5.49963903427124, 1.7487431764602661, -5.368928909301758, -1.9552397727966309, 1.3660882711410522, 2.159475326538086, 5.473600387573242, -9.068988800048828, -10.96767807006836, 14.962512969970703, -7.720248699188232, -10.532135009765625, 3.7207210063934326, 13.032367706298828, -3.4069273471832275, 12.016159057617188, 5.829763889312744, 10.093022346496582, 7.168609619140625, -9.888320922851562, -2.1756598949432373, -1.1179563999176025, 16.684492111206055, 13.029878616333008, 9.941445350646973, -0.7357431054115295, -1.3535504341125488, -17.03285026550293, 13.369924545288086, -2.7746667861938477, -0.6558359861373901, -12.6152982711792, -2.6446568965911865, 28.814775466918945, 11.006680488586426, 13.910083770751953, 4.285401821136475, -1.4548622369766235, -22.15300750732422, -6.023955345153809, -9.443520545959473, -0.9684545397758484, 4.823577404022217, 10.187858581542969, 5.772369384765625, 2.9090099334716797, 7.159576892852783, 5.264739036560059, 0.29542869329452515, 10.240751266479492, -15.7590970993042, 31.782146453857422, -12.443767547607422, 5.169990539550781, 14.161629676818848, -0.7588083148002625, 9.115226745605469, -4.856333255767822, 0.38427025079727173, 21.712635040283203, -10.674067497253418, -4.6309919357299805, 8.587757110595703, 20.045211791992188, 4.679046630859375, -10.037642478942871, -8.60331916809082, 11.827912330627441, -11.045944213867188, -1.3072364330291748, 2.1582088470458984, 11.925439834594727, 14.453387260437012, -5.716909408569336, -7.306948184967041, 26.650171279907227, 5.533910274505615, -5.111248970031738, 15.92959213256836, 10.236891746520996, 2.028754711151123, -8.813700675964355, -23.426631927490234, 7.852628231048584, 6.854639530181885, 7.335061550140381, -7.219440460205078, 3.031895637512207, 6.099151611328125, 2.6612443923950195, -17.031394958496094, -11.700267791748047, -0.68218994140625, 2.023744821548462, 14.374711990356445, -2.5858802795410156, -12.258071899414062, -11.687016487121582, 10.309280395507812, -11.01399040222168, -1.7222111225128174, -12.662538528442383, 4.954251766204834, 9.930774688720703, -15.153404235839844, -11.644563674926758, 22.214691162109375, 8.218326568603516, -11.633193969726562, -3.687020778656006, 13.013715744018555, 3.7821407318115234, 13.068245887756348, 4.762946128845215, 13.298389434814453, 4.183925151824951, -2.946840524673462, 3.1992971897125244, 0.3796634376049042, -1.96531343460083, -0.0021704924292862415, -3.400447130203247, 11.066312789916992, -4.340212821960449, -1.6997864246368408, -1.3671321868896484, 0.7818370461463928, -6.915024757385254, 8.786003112792969, 1.7689181566238403, 4.21929407119751, -18.386117935180664, -0.20197130739688873, -0.710659921169281, -7.02902889251709, 7.343325138092041, -2.8478095531463623, 1.9466735124588013, -3.422853469848633, 4.748394012451172, 6.069398880004883, 13.00699234008789, 12.32206916809082, 10.26102066040039, -5.2063446044921875, -4.30115270614624, 8.678132057189941, 0.4091033339500427, -6.915078639984131, 4.229248046875, 5.18752384185791, 15.630721092224121, -2.538404941558838, -4.929117202758789, -12.718554496765137, -0.9098889827728271, -8.653176307678223, -7.664227485656738, 7.343762397766113, 11.359493255615234, 4.029365539550781, 4.33212947845459, -9.335328102111816, 10.20089340209961, -11.899222373962402, 5.923769950866699, -1.706939458847046, -8.102336883544922, -16.088497161865234, 11.015348434448242, 19.499897003173828, 12.670622825622559, -14.760650634765625, 2.3389933109283447, -9.394346237182617, 21.468164443969727, -10.782427787780762, -6.001911163330078, -15.605741500854492, -5.975481986999512, -7.749913215637207, 10.995405197143555, 1.1112644672393799, 4.3467698097229, -4.887138843536377, 2.9028537273406982, 2.3938653469085693, 8.940479278564453, 10.715034484863281, 4.911258220672607, 10.984638214111328, 8.467780113220215, 18.4561767578125, 6.959132194519043, -18.98953628540039, -1.968308925628662, -8.113353729248047, 2.1056528091430664, 7.6320319175720215, -4.9949188232421875, 19.20562171936035, -7.345307350158691, 10.552022933959961, -5.270373344421387, -6.285023212432861, -0.9536947011947632, 10.005491256713867, -2.0347580909729004, -9.476204872131348, -22.84247589111328, 1.6171534061431885, 19.919208526611328, -4.362433910369873, -0.8440084457397461, -0.15254804491996765, -7.3319597244262695, 14.075871467590332, 8.8489990234375, 9.275185585021973, -20.676651000976562, 7.622063636779785, 7.189730644226074, 11.40717601776123, 21.870361328125, -11.750137329101562, -0.3311289846897125, -5.425960540771484, 4.667414665222168, -6.598494529724121, 16.142789840698242, -3.7872986793518066, -8.674017906188965, -14.391631126403809, 2.4734504222869873, 2.3608312606811523, 9.880880355834961, -8.170368194580078, 4.1310505867004395, 14.930170059204102, -11.68443775177002, 0.8506583571434021, 6.05980110168457, -2.166106939315796, 9.97527027130127, 6.892419815063477, -2.3402459621429443, 8.211509704589844, -6.585351943969727, -0.41634392738342285, 12.004997253417969, 0.7932264804840088, 4.429548263549805, 9.937908172607422, 1.8857803344726562, -2.8275668621063232, 3.5045652389526367, 6.541825294494629, -25.464323043823242, -0.0723191425204277, 4.31364107131958, 11.056629180908203, -5.509688377380371, -5.191542625427246, 7.399238586425781, -11.367273330688477, 1.5215915441513062, -8.10606575012207, -10.624673843383789, -15.552634239196777, 0.46191152930259705, 3.9502501487731934, -3.518949508666992, -6.1050896644592285, -6.203375816345215, -0.11068689823150635, -11.686622619628906, -0.8673227429389954, 9.035918235778809, 10.666910171508789, -8.90624713897705, 3.995393753051758, 1.8446342945098877, 14.422980308532715, 7.91070556640625, -1.383638620376587, 0.6792705655097961, 13.267763137817383, -9.539464950561523, -3.6178178787231445, 11.935041427612305, 10.1763916015625, 5.751885890960693, -7.703718185424805, -1.1210570335388184, -18.067729949951172, -9.661160469055176, -2.746633529663086, 16.213539123535156, 18.6745662689209, 5.870533466339111, -1.4203076362609863, 0.0344684012234211, -12.191957473754883, 5.031911849975586, -9.183961868286133, -7.352938652038574, -6.618118762969971, -2.613593339920044, -6.501280784606934, -9.87030029296875, -13.511417388916016, 4.54776668548584, -8.90605354309082, 10.18241024017334, 16.58547592163086, -5.466195106506348, -0.90439373254776, -16.196611404418945, 13.442272186279297, -13.992777824401855, 1.3151370286941528, 4.704321384429932, -4.1162614822387695, -7.741403102874756, -8.519767761230469, -8.193524360656738, -14.585531234741211, -8.89511489868164, -0.6256299018859863, 2.5635790824890137, 7.474708080291748, 4.4800705909729, -16.10378646850586, -6.943974494934082, 17.33618927001953, -6.57435417175293, 1.4307866096496582, 21.723031997680664, 4.543229103088379, -6.774752140045166, -1.797959804534912, -16.799196243286133, -13.168746948242188, -6.732332706451416, -19.827442169189453, 22.424318313598633, 8.111607551574707, 0.03611653670668602, 8.70668888092041, -1.306868553161621, -14.83134651184082, -7.509105682373047, -12.25367546081543, 6.945889472961426, 1.2255022525787354, -8.243294715881348, -13.151740074157715, -0.5779778957366943, -1.5747926235198975, -4.567331790924072, -6.36932897567749, -12.126059532165527, 6.055510520935059, -15.865616798400879, 5.453557968139648, 0.594182014465332, -0.7759349346160889, -3.473222494125366, 2.635843515396118, 16.896623611450195, 13.3269681930542, 5.731654167175293, 3.712223768234253, 1.0639525651931763, 4.038434982299805, -19.499652862548828, 6.720736503601074, -13.542253494262695, 4.417690753936768, -18.659399032592773, 3.3072259426116943, -5.141720294952393, -3.9435510635375977, -1.460361361503601, 10.554130554199219, -12.019292831420898, -9.143725395202637, 7.621511936187744, 17.907238006591797, -13.534993171691895, 14.277167320251465, -6.942502021789551, -0.006786380894482136, -11.861053466796875, -9.826780319213867, 0.7801514267921448, 1.2928032875061035, -9.437993049621582, 8.841277122497559, 22.519227981567383, 10.449390411376953, -17.958999633789062, -10.134679794311523, -1.7395007610321045, -1.7624919414520264, -6.575929641723633, -8.41919994354248, 6.868854522705078, -16.34494972229004, -1.2255775928497314, 5.1092634201049805, -15.88547134399414, 0.5547928810119629, 8.619102478027344, -8.315533638000488, 10.425576210021973, 3.3889386653900146, -1.2286109924316406, -0.8045225739479065, -2.3893158435821533, -1.8570290803909302, 7.282086372375488, 15.631763458251953, -16.47138214111328, 0.5616098046302795, 13.100799560546875, -12.021628379821777, -1.1185455322265625, 2.3420488834381104, -8.343843460083008, 0.6283703446388245, -13.419301986694336, 11.541004180908203, 4.260947227478027, -17.94498062133789, -17.888269424438477, 0.6560932397842407, 18.17129898071289, -6.862187385559082, 21.453001022338867, 7.8589653968811035, 10.186759948730469, 17.394454956054688, 20.102985382080078, -4.05863618850708, 7.272627830505371, 10.842557907104492, -3.173635244369507, -14.787471771240234, 0.8375985026359558, -14.515373229980469, -4.315628528594971, -11.048717498779297, 10.819454193115234, -8.038270950317383, 0.9589495658874512, -6.432788848876953, -9.11840534210205, -15.009788513183594, 14.28274154663086, 9.246018409729004, 18.290584564208984, -2.177581310272217, 5.2603631019592285, -16.457374572753906, 15.65111255645752, 12.107738494873047, -15.910100936889648, -7.874011039733887, 4.026221752166748, 2.891691207885742, -24.808029174804688, -1.1369545459747314, -2.2310078144073486, 4.314924716949463, 5.325297832489014, -21.089496612548828, 9.610505104064941, 18.465255737304688, -4.750624656677246, -3.5778095722198486, -4.01277494430542, -17.19549560546875, 5.883350849151611, -6.130318641662598, -7.0020432472229, -12.868095397949219, 1.620687484741211, -4.632419586181641, -0.14112916588783264, 3.7867355346679688, -4.198541164398193, -3.1423842906951904, 5.68688440322876, -7.041229724884033, -11.175623893737793, -4.680553436279297, 1.8969907760620117, -3.827266216278076, 10.970273971557617, 3.565664529800415, -1.3720840215682983, 2.372535228729248, -4.5073018074035645, 2.4937808513641357, -7.848972797393799, 11.682610511779785, 7.3277740478515625, -23.544052124023438, 3.8780486583709717, -15.046128273010254, -5.13411808013916, -5.662664413452148, 2.9832472801208496, 2.645353317260742, -12.140941619873047, -3.030336856842041, 7.458987236022949, -4.02504301071167, -3.088066577911377, -2.043701410293579, 8.686935424804688, -2.3037679195404053, 2.9471263885498047, 5.137810230255127, 19.841861724853516, 4.471911907196045, 2.1905295848846436, 5.589818000793457, -0.2447955161333084, -9.97895336151123, -5.223011016845703, -2.988581895828247, 7.988635063171387, -5.3615593910217285, -4.5857768058776855, 4.699051856994629, 4.943187236785889, 2.12113881111145, 7.2724432945251465, 1.1840523481369019, -4.850865364074707, 17.65415382385254, -10.078996658325195, 2.726229190826416, 5.182492733001709, 5.872659206390381, -2.2606024742126465, -1.6559584140777588, -3.439832925796509, -23.975200653076172, 11.607452392578125, -1.9309301376342773, 15.363129615783691, 13.211740493774414, 3.055945873260498, -9.161201477050781, -2.6436543464660645, 16.280105590820312, 11.45034408569336, 13.724838256835938, 12.590079307556152, -15.36393928527832, 14.116291046142578, -15.754117012023926, 11.936368942260742, -1.5982017517089844, -13.81866455078125, -10.072114944458008, -13.807857513427734, 22.876914978027344, -0.3089410662651062, -1.9710015058517456, 1.0051476955413818, 12.207666397094727, -1.3016157150268555, 10.184247970581055, 10.103659629821777, 10.972710609436035, -3.270138740539551, -0.8968117833137512, -10.283875465393066, -1.0049681663513184, 5.626614093780518, -6.516600608825684, -0.249551460146904, 9.616401672363281, 1.9983850717544556, -9.743907928466797, -8.279610633850098, 12.453764915466309, 2.2490627765655518, 14.050530433654785, 12.497674942016602, 8.55795669555664, -3.430144786834717, -8.32691764831543, 7.125211238861084, -0.15443947911262512, 6.18752384185791, 14.475261688232422, -3.2206268310546875, -22.862918853759766, 5.374006271362305, -1.917172908782959, 5.36069917678833, -0.9915313124656677, -7.720952033996582, 6.733765602111816, -0.2979772686958313, -1.5832818746566772, -3.096437454223633, 8.91493034362793, 12.845243453979492, -3.075957775115967, -5.155306339263916, 1.0082452297210693, -12.368278503417969, -12.002580642700195, -9.067124366760254, 1.587702751159668, -1.2459628582000732, -5.17000675201416, 6.913967132568359, -6.260690212249756, -7.060844898223877, -12.844087600708008, 0.6744605898857117, 6.396815776824951, -17.818096160888672, -3.2200167179107666, 1.9986921548843384, 8.574740409851074, 2.0663907527923584, -8.592954635620117, 8.82505989074707, -2.003694772720337, 18.804832458496094, -3.532158851623535, 13.359700202941895, -5.031908988952637, 11.628390312194824, -11.097098350524902, 3.9477968215942383, 1.981835126876831, 13.179282188415527, -1.476006031036377, 7.9875640869140625, -3.157416582107544, 7.190376281738281, -1.596206545829773, -3.5326921939849854, 3.570913791656494, -0.3819977641105652, -10.168715476989746, -0.03767740726470947, -8.858635902404785, -1.9225656986236572, -4.136613845825195, 17.155006408691406, 5.256341934204102, -16.058639526367188, 10.44978141784668, 9.255560874938965, 12.389822959899902, 10.71245288848877, 0.9203683733940125, 7.455346584320068, -9.855711936950684, 15.128254890441895, 11.26166820526123, 11.431304931640625, -1.208513855934143, -1.3847696781158447, -3.3729019165039062, 8.677129745483398, -20.97431182861328, -11.855167388916016, 6.293180465698242, 13.345161437988281, 1.233750820159912, 7.3259782791137695, -5.579276084899902, -2.211843490600586, -12.38603401184082, 1.9027255773544312, 0.7891830801963806, -30.35159683227539, -9.998284339904785, -5.274392127990723, -1.504079818725586, 10.238040924072266, 6.138200283050537, -3.0218019485473633, -0.04477561265230179, 3.4544708728790283, 8.210402488708496, -6.645910739898682, 9.61046314239502, -10.099810600280762, 0.2795717716217041, 3.079131603240967, 1.618793249130249, -4.695901870727539, -12.86999797821045, -14.676789283752441, -6.955468654632568, -23.716053009033203, -5.946404457092285, 7.352744102478027, 4.597130298614502, 2.087984085083008, -5.226892948150635, 21.812990188598633, 9.608723640441895, 0.9077963829040527, -0.7852306962013245, 10.898506164550781, -6.591121673583984, -7.579626083374023, 0.9973463416099548, -4.9610795974731445, -13.159612655639648, -2.157721757888794, -18.414514541625977, -16.32239532470703, 0.06602458655834198, 1.3927371501922607, -8.531072616577148, 0.5608646869659424, -5.802192211151123, -2.0829427242279053, 6.0064849853515625, -0.3574555516242981, -2.8051435947418213, 2.35711669921875, 12.125016212463379, 6.68466329574585, -14.082698822021484, 4.099453449249268, 16.383569717407227, -1.3187986612319946, -0.7266127467155457, -0.6893078088760376, -4.344509124755859, 6.517331600189209, 0.5740649104118347, -13.301656723022461, 6.0500640869140625, 3.966883659362793, 1.1077297925949097, -13.483736038208008, 0.20556899905204773, -2.732245922088623, -3.3869240283966064, 9.478092193603516, -7.4936699867248535, -21.349014282226562, 2.2962753772735596, 8.091350555419922, -4.242554187774658, -19.537790298461914, -16.241525650024414, -7.769872665405273, 2.9451515674591064, -5.568123817443848, -17.033740997314453, -27.00437355041504, 16.265348434448242, -0.3519115149974823, 10.57030200958252, -6.960559368133545, -9.312830924987793, 3.186371326446533, 12.879524230957031, -2.1341333389282227, 1.3812181949615479, -8.908784866333008, 6.616125106811523, 14.408921241760254, 4.123369216918945, 3.0616543292999268, 23.01332664489746, 12.6179838180542, -17.544296264648438, -1.3874365091323853, -8.302322387695312, 5.458116054534912, -5.857272148132324, -10.86948013305664, -12.279881477355957, -10.919461250305176, -1.7203961610794067, 3.2454757690429688, 3.134284019470215, -20.003555297851562, 7.598396301269531, -5.1691107749938965, 0.9857460260391235, -17.567773818969727, 5.980352878570557, 0.5872429609298706, 10.167539596557617, 6.0428595542907715, -14.270186424255371, 9.676732063293457, 19.908803939819336, -9.570552825927734, -19.832487106323242, -15.975410461425781, 1.5871638059616089, -5.482908248901367, 14.85933780670166, -18.505985260009766, -9.170767784118652, -4.466057777404785, -6.6219987869262695, -11.631074905395508, 11.901970863342285, 2.8213768005371094, 7.802792549133301, -6.453406810760498, -3.3664331436157227, 9.676244735717773, 4.509559631347656, 9.113393783569336, 1.1687945127487183, -8.214933395385742, 4.657226085662842, 8.496949195861816, 4.654509544372559, 10.22559642791748, 2.120396852493286, 9.072842597961426, -3.8309378623962402, -2.947420597076416, -4.991118431091309, 1.9865082502365112, -5.2514801025390625, 8.982547760009766, -5.814244270324707, -16.819950103759766, -7.810000419616699, 6.6556525230407715, 11.169523239135742, 10.612889289855957, 5.261178970336914, 8.678845405578613, -17.754776000976562, -22.574682235717773, -16.53792381286621, -2.19061541557312, 8.977210998535156, -4.360416889190674, 8.91553783416748, -1.152195692062378, 3.6619389057159424, -9.03735065460205, -18.994956970214844, 5.083919048309326, 3.87259840965271, 5.862366676330566, -4.666327953338623, -2.8768510818481445, 8.648231506347656, 12.901434898376465, 0.7604128122329712, -14.156203269958496, -5.4920573234558105, 4.182158470153809, 6.8067121505737305, 6.037031173706055, -8.104752540588379, -1.7122488021850586, -3.698481321334839, -6.010270118713379, 14.829439163208008, -3.497307062149048, -2.563115119934082, 1.5268152952194214, 2.540227174758911, -3.6549160480499268, 1.456268072128296, -4.596844673156738, 1.5590232610702515, -11.02754020690918, 14.073863983154297, -11.268890380859375, -12.820643424987793, 10.307708740234375, -8.590685844421387, -20.998031616210938, 13.50853157043457, -7.9281110763549805, -3.4726409912109375], [13.82327651977539, -10.017062187194824, -11.517134666442871, -14.60951042175293, -3.7098560333251953, -6.193845748901367, -2.6133930683135986, -13.560399055480957, -0.6707144379615784, 20.126035690307617, -8.61313247680664, 5.239631652832031, -3.6057136058807373, -3.841806650161743, -14.679536819458008, 11.44532299041748, 11.774084091186523, 1.160006046295166, 5.94752311706543, -6.406887054443359, 6.31646728515625, -11.333045959472656, 9.51458740234375, 0.5924581289291382, -6.626067638397217, -4.539907455444336, -0.7407381534576416, 9.440842628479004, -5.284669399261475, -11.753662109375, 8.4037446975708, -6.184547424316406, 18.068601608276367, -8.243033409118652, 4.235678195953369, 4.5538411140441895, 2.102924346923828, -10.587839126586914, -12.269590377807617, -9.375643730163574, -15.145930290222168, 16.96106719970703, -18.426572799682617, -8.294442176818848, -10.25728702545166, 5.398041725158691, 2.100285053253174, -8.531170845031738, -14.419881820678711, -0.5251020789146423, -10.2612886428833, -12.89670467376709, 4.61391544342041, -1.8918741941452026, 6.411562919616699, 4.045799732208252, 7.353349685668945, 13.242754936218262, -3.418586015701294, -15.015913009643555, 5.052547931671143, -10.0972900390625, -13.330163955688477, 3.179384231567383, -4.545177459716797, 0.16099707782268524, -3.532928228378296, 4.389973163604736, -11.140528678894043, 10.372687339782715, 24.437801361083984, -3.853424072265625, -6.695169925689697, 6.878157615661621, 25.635204315185547, -6.448997974395752, 3.253175735473633, 0.8140137791633606, -7.363690376281738, -17.74940299987793, -14.449785232543945, -3.093456745147705, 14.432406425476074, -7.203338146209717, -6.241302490234375, -1.4335815906524658, 19.726449966430664, 2.880927324295044, 7.834050178527832, 5.590388298034668, 2.5184412002563477, -6.6699981689453125, 2.107963800430298, 9.973615646362305, 1.7171787023544312, 9.574540138244629, -6.977608680725098, 6.829837799072266, 11.5985107421875, -3.005910873413086, -2.8505709171295166, -5.634090900421143, -23.745628356933594, -2.9777348041534424, 15.529626846313477, -3.7402172088623047, 0.29552173614501953, -3.335557460784912, -15.288898468017578, 3.0718026161193848, -1.2992192506790161, 3.8882250785827637, -10.717534065246582, 8.630620956420898, 26.923389434814453, -2.70296049118042, 29.23304557800293, 2.741854190826416, 2.302804470062256, -0.47913360595703125, 4.6937456130981445, -0.5131711959838867, -5.145906925201416, 2.943080425262451, -12.647418975830078, -0.9595474600791931, 7.0885701179504395, 8.689613342285156, 17.920108795166016, 5.718349456787109, 1.6688055992126465, 2.2044131755828857, -7.866161346435547, -9.222344398498535, -9.956116676330566, -6.894235134124756, -5.789914131164551, 2.0120887756347656, 3.147045373916626, -1.0465586185455322, 20.507497787475586, -1.0286927223205566, 3.0135769844055176, 0.2142939567565918, 14.126373291015625, -8.139887809753418, -3.4573254585266113, 27.63840103149414, 0.01772303506731987, 8.230846405029297, 13.9403715133667, -1.3546383380889893, -15.575981140136719, -8.611924171447754, 9.87303352355957, -8.677165031433105, 13.476042747497559, -3.53423810005188, 13.790679931640625, -5.236080169677734, -3.4277968406677246, 17.07975196838379, -10.821198463439941, -7.5182085037231445, 11.602460861206055, 5.44819450378418, 17.225833892822266, 3.4946932792663574, 10.195371627807617, -1.1054863929748535, -2.3846988677978516, 11.668455123901367, -0.04730840027332306, -17.362659454345703, 13.599528312683105, -5.620018005371094, -7.883100509643555, 14.174516677856445, -18.3785400390625, 11.224279403686523, -4.774805068969727, 14.981161117553711, 6.42430305480957, -10.132841110229492, 32.75914764404297, 2.168184280395508, -1.1414554119110107, 24.398754119873047, -12.072944641113281, -2.9799880981445312, -1.3557677268981934, -13.557456970214844, 3.5560572147369385, 17.33856964111328, 13.023480415344238, -13.696006774902344, -11.248883247375488, -4.814207077026367, 8.037711143493652, 12.913046836853027, 9.044416427612305, -11.119855880737305, -1.0307912826538086, -24.73299217224121, -3.4067883491516113, -3.2258403301239014, -10.537175178527832, -12.276374816894531, -2.0170650482177734, 0.1080358698964119, 2.880624532699585, 1.0004656314849854, -5.313460826873779, -2.781158447265625, 5.996703624725342, 3.919912815093994, 12.447359085083008, 5.995689392089844, -3.502519369125366, -4.210468292236328, -3.023641586303711, -3.0675411224365234, 14.696789741516113, 2.806713342666626, 9.81834602355957, 4.795254230499268, -2.2967159748077393, 0.655305027961731, 1.9890077114105225, 5.64628791809082, -11.011085510253906, 1.625112771987915, -7.15000581741333, 3.5324645042419434, 8.469141960144043, 1.6850322484970093, 0.7087181806564331, 6.074492454528809, -5.894762992858887, -5.852346897125244, 0.937801718711853, -3.832002639770508, -9.066526412963867, -10.78258991241455, 6.756768226623535, 6.1780476570129395, 2.4186577796936035, 20.080615997314453, -7.882546901702881, -4.334028244018555, -18.08713722229004, -8.606087684631348, -21.212997436523438, 11.842890739440918, 8.419198036193848, 10.202341079711914, -0.27560728788375854, 9.277664184570312, 8.524420738220215, 11.356954574584961, -21.539859771728516, 3.2770583629608154, 8.728731155395508, 3.1402981281280518, -13.247389793395996, 18.78465461730957, 1.576712727546692, 21.157772064208984, -6.680745601654053, 1.0945225954055786, -5.613537311553955, -3.307950973510742, -6.060933589935303, -6.661639213562012, 12.606851577758789, -5.756405830383301, -5.434993743896484, -2.492325782775879, -12.246162414550781, -6.549723148345947, 7.836150169372559, -7.076560974121094, 11.944564819335938, 8.337441444396973, 8.856759071350098, -3.0352425575256348, 17.7996768951416, -13.717727661132812, -4.592389106750488, -6.486536979675293, -15.631213188171387, 2.9802896976470947, 12.718782424926758, 0.5742782354354858, 2.6102094650268555, -5.281193256378174, 2.3878579139709473, -10.749799728393555, -0.19825676083564758, 9.680927276611328, -10.178682327270508, -4.499507904052734, 16.356975555419922, -5.9166483879089355, -10.129125595092773, 5.876317501068115, 4.218690872192383, -4.520336627960205, -1.581133484840393, -6.395911693572998, -3.128350019454956, 5.976111888885498, 7.8662919998168945, 12.899447441101074, -19.250783920288086, -2.1619184017181396, -8.04817008972168, 5.377231597900391, -7.424004077911377, 12.549816131591797, -5.955219268798828, 8.755487442016602, 19.850566864013672, -5.139980316162109, 11.10286808013916, 16.03964614868164, 13.051854133605957, 14.20306396484375, 16.52823829650879, 3.8038973808288574, 16.160396575927734, 15.090336799621582, -4.125815391540527, 4.518032550811768, 19.942834854125977, -10.494040489196777, 0.9469619393348694, 4.45512056350708, 9.419793128967285, -9.127142906188965, -12.947893142700195, 0.7104347944259644, 2.2104332447052, 8.934056282043457, -0.23986434936523438, 1.985133171081543, -3.8036699295043945, 16.453697204589844, -4.330652713775635, -2.346743583679199, -6.091021537780762, -14.044236183166504, -3.30719256401062, -9.086786270141602, -1.6295708417892456, 1.595992922782898, -3.906862497329712, -4.59312629699707, -2.23551082611084, 2.5529396533966064, 4.479795455932617, -4.17605447769165, -0.7130845785140991, -2.3191587924957275, 18.583995819091797, -6.754759788513184, -6.86913537979126, -1.4796841144561768, -20.071184158325195, 6.134419918060303, 3.860644817352295, -0.43967753648757935, 1.5488253831863403, 11.67394733428955, 4.308895587921143, -8.551286697387695, -0.688051700592041, 5.163798809051514, 25.459068298339844, -6.9837541580200195, -17.917282104492188, 2.939368486404419, 4.823690414428711, 10.468083381652832, 5.510332107543945, -5.723476886749268, 4.175565719604492, -4.418760299682617, 9.80029296875, 5.769553184509277, -5.900748252868652, -2.1224071979522705, -0.005077388137578964, 8.56406307220459, 4.661713600158691, 2.154853105545044, -16.113643646240234, 13.653985977172852, -3.8776326179504395, -7.070922374725342, -7.556333065032959, -14.610730171203613, -19.74486541748047, -16.74169158935547, 9.821545600891113, -4.329573631286621, -15.746740341186523, 3.1746764183044434, -13.919744491577148, -3.8146848678588867, -7.455880165100098, 14.543258666992188, 12.238741874694824, -14.945396423339844, 7.672019958496094, -10.150598526000977, 7.9680376052856445, 6.082526206970215, -19.487016677856445, -1.3173402547836304, 4.705408096313477, -17.31009292602539, -15.216723442077637, 0.26736196875572205, 18.432174682617188, -0.15411162376403809, -1.3399709463119507, 9.437517166137695, 9.368000030517578, 0.3858185410499573, 8.31401538848877, -9.105575561523438, 5.783993721008301, -0.27627798914909363, 13.599464416503906, 11.478660583496094, -3.58864688873291, 2.9140000343322754, 3.463641405105591, 2.251298666000366, -1.3433375358581543, -8.594669342041016, 3.5636215209960938, -2.8976426124572754, -7.652382850646973, 20.037784576416016, 6.508238315582275, -5.021266937255859, -16.20142936706543, -0.7579007148742676, -9.21774673461914, -5.248920917510986, 0.9643752574920654, 20.14946174621582, 7.792355537414551, 7.277501106262207, -4.132643699645996, 10.634642601013184, 10.105392456054688, 7.650571346282959, 15.800541877746582, 1.4915580749511719, 9.848204612731934, -22.741748809814453, -8.058584213256836, -2.8882384300231934, 9.842950820922852, -6.6280927658081055, -4.699304103851318, 6.432526588439941, -6.779760837554932, 2.9379167556762695, -11.077186584472656, 10.710793495178223, -2.489108085632324, 7.564632892608643, 11.557984352111816, 2.682525634765625, -3.5434935092926025, 17.968231201171875, -2.845268487930298, -16.28546142578125, 7.7023210525512695, -11.126704216003418, 2.611629009246826, -10.83918571472168, 7.526335716247559, 2.759038209915161, -8.290603637695312, 8.253861427307129, -21.291893005371094, -9.333828926086426, -5.367760181427002, -3.7356576919555664, -9.467446327209473, 16.26423454284668, 7.101633071899414, 2.5679988861083984, -11.127842903137207, 4.7407732009887695, -6.243002414703369, 4.171897888183594, -16.60137939453125, 3.7348365783691406, -0.17021527886390686, 9.757404327392578, -16.531295776367188, -4.755191326141357, 1.5587078332901, -6.901822090148926, 2.880336284637451, 20.331411361694336, -1.6367663145065308, 5.016766548156738, 2.8423187732696533, -11.768796920776367, -7.287892818450928, -2.3897523880004883, 1.5901212692260742, -4.26621675491333, 13.017526626586914, -8.648079872131348, -3.393507242202759, -2.9852700233459473, 10.803204536437988, -4.1292314529418945, 9.877981185913086, 4.539307594299316, -5.665749549865723, 5.312979221343994, -2.3104851245880127, -11.33492374420166, 8.258868217468262, 2.4463374614715576, -18.634868621826172, -0.7398095726966858, -16.699308395385742, -0.2934676706790924, -14.033082962036133, 8.918547630310059, -24.825332641601562, -2.9922099113464355, 9.332257270812988, -2.623220682144165, 10.150594711303711, -1.7828811407089233, -12.549284934997559, 11.645486831665039, -6.8348283767700195, -4.916842460632324, 14.347541809082031, 2.4076685905456543, -18.1292667388916, -3.6008808612823486, -3.2451510429382324, 1.891553282737732, 2.9784584045410156, 1.5922677516937256, -4.725982189178467, -5.609147071838379, 11.606284141540527, 12.616572380065918, 7.240647315979004, -9.677234649658203, -5.516843795776367, 7.652744293212891, 2.194523572921753, 7.07535457611084, -11.687013626098633, -3.271545171737671, -9.570384979248047, 28.83286476135254, 6.7473883628845215, 6.5491132736206055, -14.005722999572754, -9.986169815063477, 2.216068983078003, -0.2085750699043274, -1.391653299331665, -0.9366127252578735, 3.9440557956695557, 7.4371442794799805, -3.8521780967712402, -5.749115943908691, 11.668496131896973, 0.2466294765472412, 1.038199782371521, -13.15762996673584, 17.141002655029297, 6.713259220123291, -16.109596252441406, 0.384176105260849, -3.4982099533081055, 20.266319274902344, 1.8408536911010742, 5.6844258308410645, -12.332277297973633, -9.54593276977539, 10.655046463012695, 5.966297149658203, 7.267009735107422, 15.491382598876953, 1.390087604522705, 13.822881698608398, 3.7709853649139404, 6.39939022064209, 1.2914116382598877, -13.66379165649414, 6.627786159515381, -10.600485801696777, 7.551268577575684, 5.742556571960449, -8.634057998657227, 3.460552453994751, -7.723540306091309, -0.6670482158660889, -8.465865135192871, 13.832744598388672, -6.117550849914551, -10.971797943115234, -18.10805892944336, 12.588658332824707, -7.924032688140869, -13.017662048339844, -7.108076095581055, -4.891185760498047, 2.4971704483032227, 10.897701263427734, 10.106355667114258, -6.1501336097717285, 9.566341400146484, 8.94345474243164, -1.61667001247406, 4.424754619598389, -3.0673158168792725, -3.8789334297180176, -9.569175720214844, -1.4848082065582275, 4.901539325714111, 4.150904655456543, -6.634699821472168, 13.266033172607422, -11.599726676940918, -13.094156265258789, 7.53272008895874, -9.598831176757812, -18.899723052978516, -16.756023406982422, 1.0357569456100464, 14.16347885131836, -4.239950180053711, -1.5549479722976685, 16.47476577758789, -3.820892572402954, -3.010540008544922, -12.496596336364746, -9.386251449584961, 12.539007186889648, 5.46533203125, 3.727968692779541, -11.299729347229004, -18.876605987548828, 2.551234245300293, -5.185503005981445, -7.703412055969238, 3.3446736335754395, 1.6874922513961792, 15.809229850769043, 0.8592984676361084, -6.745391845703125, -0.37344789505004883, 20.086095809936523, -4.452561378479004, 12.232236862182617, -5.936966896057129, 9.821020126342773, -23.05472183227539, -18.423494338989258, -4.999751091003418, -10.009857177734375, 6.107223033905029, -0.6152951121330261, 0.5616384148597717, -8.066286087036133, 5.10886287689209, 10.258112907409668, -14.48579216003418, 20.425537109375, 5.03190279006958, -4.338295936584473, -0.5743783116340637, -7.016953945159912, -10.890769958496094, 9.104036331176758, 19.622102737426758, -4.154199600219727, -1.026611089706421, 2.725656032562256, 8.427835464477539, 5.556082248687744, -16.82132911682129, 3.1503591537475586, 6.567499160766602, 14.652329444885254, 7.461224555969238, 5.550487041473389, 3.2358720302581787, -10.441347122192383, -12.729608535766602, -0.8300272226333618, 8.209044456481934, 12.11311149597168, 3.6208128929138184, 3.0248210430145264, -9.067272186279297, 7.9173970222473145, 4.642518520355225, 3.277376890182495, 14.984054565429688, -16.31929588317871, -8.632429122924805, -0.7761348485946655, 1.6116938591003418, -1.6181418895721436, -21.450294494628906, 10.88127326965332, 6.752254962921143, -15.721799850463867, 9.566352844238281, 15.744203567504883, -2.2450902462005615, -10.122245788574219, -5.931015968322754, 0.7225807905197144, -4.5671305656433105, 14.45705509185791, -12.291242599487305, -8.33129596710205, 3.2477126121520996, -7.709712028503418, -0.8404174447059631, -25.677507400512695, 11.237283706665039, -9.686240196228027, 2.7640180587768555, -2.0525875091552734, 21.162073135375977, -15.446715354919434, 1.7239023447036743, -11.177362442016602, 0.17287811636924744, 6.137967586517334, 0.5785394310951233, 7.299806594848633, 12.080598831176758, -5.185912132263184, -6.3361005783081055, 5.3360795974731445, -7.509847164154053, 21.235675811767578, 10.981205940246582, 1.782012939453125, -0.5623107552528381, -0.3473482131958008, -3.346405506134033, 1.274727463722229, 11.501516342163086, -0.30682915449142456, 3.906893730163574, -6.742978572845459, -4.992379665374756, -3.9236583709716797, 6.396356582641602, -6.454619407653809, 5.660440921783447, 11.517783164978027, -15.13745403289795, -9.432820320129395, 5.017350196838379, 12.960283279418945, 2.329547166824341, -2.4512929916381836, -2.806342124938965, -6.371404647827148, 7.36027193069458, -3.8082523345947266, -4.939252853393555, -10.912813186645508, -7.79679012298584, -7.022915840148926, 1.3544071912765503, -6.968484878540039, 14.659154891967773, 0.8080916404724121, 5.963925838470459, -23.50912094116211, 16.102603912353516, -21.471607208251953, 4.354707717895508, -5.143747806549072, 16.560684204101562, -8.084656715393066, -10.924020767211914, 4.118570327758789, 3.521360397338867, 11.123964309692383, 5.1665754318237305, 4.5627312660217285, 25.603029251098633, 7.073960781097412, 0.34391912817955017, 8.22625732421875, 12.250737190246582, -2.5224804878234863, 15.506631851196289, 14.891101837158203, -0.7745306491851807, -24.00230598449707, 8.469615936279297, 15.416143417358398, -7.0055036544799805, -1.2231355905532837, -3.7053024768829346, 9.993545532226562, -15.0215425491333, -0.6093818545341492, 10.783487319946289, -0.01110621728003025, 1.2771577835083008, -6.323129653930664, -7.546346187591553, 0.4937894642353058, 17.29393196105957, -2.0548758506774902, -8.92611026763916, -6.174917697906494, -3.00939679145813, -17.39213752746582, -13.316365242004395, 4.301284313201904, -10.731637954711914, -2.6223464012145996, 3.65090274810791, -4.529262542724609, 1.5831749439239502, -3.8752050399780273, -1.475861668586731, -4.192237377166748, 7.524834632873535, 9.268486976623535, -27.613990783691406, 2.5345537662506104, 9.969610214233398, 0.24471990764141083, -3.8706774711608887, 15.439244270324707, 4.067775249481201, -2.760775327682495, -1.7635111808776855, 7.590280055999756, -3.7507989406585693, 18.72491455078125, -0.9815441370010376, -1.3315317630767822, 12.425504684448242, -5.735321998596191, -4.717173099517822, 4.740565299987793, -11.651824951171875, 5.742347240447998, 10.563173294067383, 7.131650924682617, 12.082648277282715, 12.179346084594727, -0.8725333213806152, -8.332473754882812, 4.714369297027588, 3.0689125061035156, -1.5451066493988037, 2.9056687355041504, 2.8912224769592285, 0.674107015132904, -8.264660835266113, -6.611083984375, -14.243383407592773, -14.635480880737305, -2.803426742553711, 15.022734642028809, -6.863391399383545, 1.1463289260864258, 3.3754281997680664, -6.717746257781982, 3.570225238800049, -0.9025001525878906, -2.7902376651763916, -15.400787353515625, -0.8745520710945129, -9.875100135803223, 0.3314485549926758, -8.997182846069336, 16.780885696411133, 20.850467681884766, 19.748863220214844, 8.248498916625977, -3.3002092838287354, 2.369967222213745, 7.904936790466309, 6.674147605895996, 22.077083587646484, 4.646873950958252, -6.964290142059326, 0.3621584177017212, -17.273828506469727, -22.150352478027344, -19.214826583862305, 0.0028755704406648874, 10.71053695678711, 6.520814418792725, -5.138836860656738, -4.658486366271973, -0.41992396116256714, 0.2367393523454666, 16.714059829711914, 14.421541213989258, -11.145746231079102, 2.199974775314331, 6.345280647277832, -10.213289260864258, 1.4255428314208984, -4.754871845245361, -3.0229287147521973, 5.671358108520508, 1.2772934436798096, -7.44757080078125, -5.310299396514893, 5.349933624267578, 11.248074531555176, -0.8961716890335083, -6.916257858276367, -7.607319355010986, -12.468324661254883, -1.2794551849365234, 14.608403205871582, 14.540209770202637, 5.37992525100708, -11.832728385925293, -26.840007781982422, -15.401602745056152, 5.988394737243652, -7.052402496337891, -20.587936401367188, 12.121895790100098, -12.492743492126465, -7.031739234924316, 17.76424789428711, -1.494025707244873, -5.013018608093262, -1.6836919784545898, -27.038881301879883, 1.9087272882461548, 9.31186294555664, -3.142488956451416, 10.357122421264648, 0.06541454792022705, -14.966337203979492, 5.280527591705322, 6.151561260223389, -3.1850483417510986, 4.518989086151123, -2.622788190841675, 8.578452110290527, 1.43314528465271, 0.7039237022399902, -14.24526309967041, -10.032941818237305, 3.58833909034729, 11.382549285888672, 5.610854148864746, 2.444836378097534, -11.390643119812012, 1.095291256904602, -0.8699845671653748, 1.9105708599090576, 2.3967788219451904, 1.9400619268417358, 24.009071350097656, 22.544132232666016, -22.73777198791504, -15.175325393676758, 18.277385711669922, -9.32754898071289, 4.347493648529053, -5.213907241821289, 9.103242874145508, 13.416311264038086, -9.613807678222656, 6.750100135803223, -1.2588515281677246, -1.505400538444519, 4.221100330352783, 7.472600936889648, 7.2790069580078125, -12.836202621459961, 1.6630098819732666, -6.40143346786499, -11.49828815460205, -7.414207458496094, -0.3274552524089813, -9.341676712036133, 3.334908962249756, -2.678622007369995, 12.895331382751465, -10.309338569641113, -0.816642165184021, -1.8182036876678467, 26.694995880126953, 5.254125595092773, 2.4104807376861572, -22.208393096923828, 5.903757095336914, -6.977051734924316, 11.507102966308594, 12.439369201660156, -8.840035438537598, -17.92483901977539, 12.01959228515625, -5.924069881439209, -4.724475383758545, 9.29640007019043, 4.076049327850342, 2.0736052989959717, 9.972888946533203, 5.782565593719482, -12.211795806884766, 4.773831844329834, 0.17980116605758667, -12.045948028564453, 2.1427159309387207, -14.522923469543457, 9.85058307647705, 0.8632951378822327, 3.747836112976074, -7.586658000946045, 4.2012481689453125, 19.517906188964844, -0.884576141834259, -0.5247887372970581, -3.0502138137817383, 4.527065277099609, 17.802461624145508, 12.187581062316895, 9.093238830566406, -10.841336250305176, 18.97878646850586, 1.4734095335006714, 1.9805408716201782, -3.2303643226623535, 6.5380330085754395, 8.276771545410156, -9.367374420166016, -3.4098756313323975, 11.249222755432129, 9.46745777130127, -7.4934539794921875, -7.160665512084961, 12.764163970947266, 12.983642578125, 4.946674823760986, -16.667325973510742, 14.085271835327148, -23.419784545898438, -7.451600074768066, -15.772989273071289, 7.71357536315918, -13.530155181884766, -7.929389476776123, 11.190224647521973, 3.3814120292663574, -9.975835800170898, -1.644648790359497, 2.7442238330841064, 2.412569999694824, 2.9350948333740234, 5.596776008605957, 8.043242454528809, -0.02490914613008499, -11.275934219360352, 13.371240615844727, -0.8011007905006409, -2.2728259563446045, -6.965682029724121, 2.05951189994812, 7.295881271362305, 3.820380210876465, -4.007373809814453, -8.075998306274414, 0.9288425445556641, 1.1167268753051758, 6.617985725402832, -8.515270233154297, -18.185871124267578, -0.8593116998672485, -14.210121154785156, 13.262605667114258, -14.654058456420898, -17.480934143066406, 8.861654281616211, 3.020937442779541, 7.308815956115723, 6.77150821685791, 10.485124588012695, -3.316849708557129, 8.892023086547852, 4.4493303298950195, -5.698381423950195, 21.237071990966797, -4.557584285736084, -13.683477401733398, 15.724244117736816, -1.169032096862793, 6.164780616760254, 1.1418977975845337, 4.68727445602417, 4.887916088104248, 15.006921768188477, 7.067809104919434, -1.876218318939209, 3.990492820739746, 7.435532093048096, -1.8216246366500854, -0.07241928577423096, 9.38035774230957, 12.890233993530273, 2.4292633533477783, -6.5147013664245605, 6.578988075256348, -2.8821969032287598, 17.549339294433594, 1.6338218450546265, 7.476201057434082, -4.993114948272705, -0.5537620782852173, -5.820910930633545, -9.932222366333008, 11.241816520690918, -9.847780227661133, -0.19199049472808838, 6.660227298736572, 7.307944297790527, -4.346287727355957, 0.3582574427127838, -1.311481237411499, -4.77134895324707, -3.1256332397460938, 5.08659553527832, -6.93587064743042, 6.653349876403809, -11.958403587341309, -1.1444132328033447, 2.3140392303466797, -11.935308456420898, -6.183189392089844, -10.185516357421875, -2.0659232139587402, -12.576576232910156, -19.163816452026367, -3.317171096801758, 5.328125953674316, 3.5055384635925293, -0.7202596068382263, -3.093747854232788, 7.495282173156738, -4.037074089050293, 3.343552350997925, 9.269186019897461, -1.7602111101150513, -7.8147077560424805, 13.767861366271973, 1.4066073894500732, -8.926085472106934, 13.698121070861816, 3.7709105014801025, 8.04684066772461, -2.942021369934082, 0.5319925546646118, 6.152202606201172, 5.49213981628418, 13.968706130981445, 0.5827630162239075, 14.616937637329102, -36.89600372314453, 13.322916984558105, -8.440197944641113, 10.469635009765625, 3.2178664207458496, -21.79790496826172, -11.596982955932617, 3.1006834506988525, -7.424007415771484, -2.423677921295166, -0.404913991689682, -1.2614657878875732, 9.492820739746094, -11.75756549835205, 9.324581146240234, -1.9341076612472534, 12.581552505493164, -12.155275344848633, -0.6239961385726929, 1.6007107496261597, -2.8666677474975586, -5.545989036560059, -2.8651790618896484, 3.7476186752319336, -0.3033836781978607, 7.741289138793945, -3.173292636871338, -3.9254088401794434, -13.729304313659668, -6.3135271072387695, -4.277698516845703, 9.703849792480469, -1.412427306175232, -2.559508800506592, 15.2162504196167, 8.045491218566895, 14.02905559539795, -1.7601287364959717, -20.374881744384766, 5.293704986572266, -11.99532413482666, -17.083831787109375, 0.7746595144271851, 12.339805603027344, 9.046300888061523, -13.210860252380371, -12.823637008666992, 23.56924057006836, -1.7716684341430664, -1.130314588546753, 10.839838027954102, 1.4749730825424194, -5.162020683288574, -1.7798118591308594, -8.770143508911133, 17.21055793762207, -15.596105575561523, 2.547398805618286, 2.337763547897339, 2.61791729927063, -1.3269262313842773, 18.370397567749023, -7.201399326324463, -0.9834930300712585, -9.350272178649902, 4.272970676422119, 8.290556907653809, -8.427156448364258, 10.720809936523438, 17.788005828857422, -9.668548583984375, 7.972527503967285, 1.0337798595428467, 4.385580062866211, 5.818081378936768, 15.194424629211426, 12.998838424682617, -14.707226753234863, -13.741575241088867, -4.302380561828613, -10.81398868560791, 5.210397243499756, 18.004304885864258, 2.3957040309906006, -0.06307592988014221, -6.675715923309326, -9.029380798339844, 11.89614486694336, 8.88385009765625, 12.670080184936523, 10.737567901611328, 11.461679458618164, 6.375672340393066, -15.700628280639648, 8.129111289978027, 6.963412284851074, 5.124603271484375, 0.5948036313056946, -15.324480056762695, -13.632221221923828, 2.83620548248291, 5.725769519805908, 9.826611518859863, 4.807044982910156, 4.432058811187744, -13.685663223266602, 3.254361152648926, 4.618044853210449, -9.49299430847168, 7.465317249298096, -13.927000999450684, 17.019681930541992, 3.2230818271636963, 5.932306289672852, 3.264444351196289, -20.880773544311523, 6.723798751831055, -3.5141327381134033, 11.885442733764648, 21.122047424316406, -15.49238109588623, -9.036103248596191, 3.797670841217041, -10.263461112976074, 3.1309266090393066, 6.229499816894531, 4.180639743804932, 1.0696157217025757, 8.179551124572754, -14.876474380493164, -2.991978168487549, -2.1051135063171387, 5.924679756164551, 1.0205415487289429, 7.025928974151611, -0.14386168122291565, 4.3667683601379395, 8.194416046142578, -1.7478351593017578, 1.9737857580184937, -4.284749984741211, 5.746340751647949, -5.443036079406738, 1.4721118211746216, 11.374252319335938, 7.923833847045898, 3.5560898780822754, 1.9955213069915771, -5.878829479217529, 11.063355445861816, -3.5513625144958496, 19.365840911865234, -1.058854341506958, -6.979377269744873, -15.28529167175293, -8.627429962158203, 17.102294921875, 18.395172119140625, 10.908336639404297, -1.4605014324188232, -12.180702209472656, -5.631700038909912, 11.408218383789062, 3.989276885986328, 0.9731841683387756, 17.578140258789062, -7.090634346008301, 3.5766096115112305, -0.7883669137954712, 15.406211853027344, 22.340591430664062, -2.621192455291748, -1.8077343702316284, 5.064542293548584, -14.625638961791992, -17.991268157958984, 1.562382698059082, -2.9692845344543457, -2.0097806453704834, -10.734687805175781, -13.784225463867188, 4.417796611785889, -3.576429843902588, 12.149503707885742, 17.335201263427734, 6.82073974609375, -5.691944122314453, -2.896897792816162, -3.811061382293701, -1.8369321823120117, 16.16147804260254, -4.865129470825195, -0.1706010401248932, -12.680543899536133, -4.781041622161865, 0.27052971720695496, -9.629439353942871, 10.977784156799316, 18.624927520751953, -7.759219169616699, 6.72322940826416, -4.883169651031494, -2.7059314250946045, 16.72954750061035, 5.0227556228637695, -1.7891329526901245, -15.770662307739258, 3.01267147064209, 20.25253677368164, 8.93518352508545, -10.690449714660645, 6.615301132202148, 19.602176666259766, -8.351007461547852, 12.739943504333496, -3.3100593090057373, -5.2458624839782715, 20.495384216308594, -5.2910566329956055, -2.3824398517608643, 4.723043918609619, 8.045395851135254, -0.42152684926986694, 9.902966499328613, 3.8027091026306152, -12.750200271606445, 9.638108253479004, 9.62939453125, -2.7520787715911865, -2.5020155906677246, 15.815107345581055, 13.25635814666748, 6.064929485321045, -21.567049026489258, 9.529226303100586, -3.5929059982299805, -26.046850204467773, -3.474220037460327, 2.6776466369628906, 13.488395690917969, 0.551472544670105, -10.53770923614502, -3.513448476791382, 2.386449098587036, -28.832475662231445, 11.993346214294434, 3.4992527961730957, -4.391689777374268, 12.216200828552246, 10.209047317504883, 9.122698783874512, -21.34282875061035, 20.284198760986328, -2.708207607269287, -6.742677211761475, -11.026314735412598, -0.6608365774154663, 0.2604118883609772, 2.1179630756378174, -0.9288583993911743, 9.003949165344238, 5.6609015464782715, 1.7209984064102173, -1.2606871128082275, -2.71268367767334, 10.56106185913086, 2.789769411087036, 0.9389854669570923, -8.041484832763672, -16.380916595458984, -6.707503795623779, -7.086644172668457, 1.8090559244155884, -13.981945037841797, 18.60436248779297, -6.590092182159424, 6.474562168121338, -7.911003112792969, -18.016521453857422, -7.937384128570557, -1.8742185831069946, 3.991854429244995, -12.423306465148926, 1.1016578674316406, 6.013698577880859, 4.205262660980225, 5.75235652923584, -9.97762680053711, -12.269715309143066, 11.254037857055664, 17.83120346069336, -2.144832134246826, -7.790944576263428, 2.291712999343872, 11.699264526367188, 2.3537583351135254, 17.535493850708008, 11.618510246276855, 6.210808277130127, -11.46310043334961, -10.49235725402832, 1.9570673704147339, 6.090712070465088, 6.993672847747803, 0.18660269677639008, -28.532352447509766, 2.9444007873535156, 14.16963005065918, 18.42795181274414, -8.160175323486328, 11.378299713134766, -0.09138689935207367, -8.986909866333008, -7.95404052734375, 6.073117733001709, -4.465087890625, 1.2005667686462402, 15.133880615234375, 0.9573451280593872, 17.296213150024414, -5.040528297424316, 1.4235172271728516, -3.7221453189849854, -14.152946472167969, 4.717172622680664, -12.458980560302734, -5.539553642272949, 2.9123125076293945, 6.937445640563965, 0.9708858132362366, 0.18225058913230896, -14.103306770324707, 13.471526145935059, 23.234529495239258, 8.469614028930664, -3.3474884033203125, -2.065932512283325, -20.562423706054688, 19.995527267456055, -20.299877166748047, 9.93730354309082, -8.317184448242188, 3.1180193424224854, 13.701183319091797, 6.666536331176758, 4.806155204772949, -8.581449508666992, 5.183446407318115, 0.6852660179138184, -10.023536682128906, -1.108835220336914, 6.970084190368652, 4.1575727462768555, -12.670668601989746, -19.442588806152344, -0.1350187212228775, -1.3781810998916626, 15.033393859863281, -16.789506912231445, 0.31096768379211426, -1.7991507053375244, 4.056722640991211, -17.314773559570312, 4.920133590698242, 5.283256530761719, 6.1356000900268555, -9.119382858276367, 0.23497463762760162, 13.166658401489258, -2.0845744609832764, -0.01146668754518032, 15.09313678741455, 15.42833137512207, -13.359736442565918, -21.67367935180664, -1.281355381011963, -11.467127799987793, -5.33989143371582, -9.55626106262207, 0.22318249940872192, -10.761306762695312, -12.135086059570312, 5.843735694885254, -11.787660598754883, 6.323225021362305, -1.5695350170135498, -12.21601676940918, -4.407859802246094, -28.43463706970215, -16.677474975585938, 0.5374758243560791, 8.371498107910156, -11.770180702209473, -3.766385316848755, 5.58244514465332, 11.80984115600586, -4.838930130004883, -8.875847816467285, 0.8917384147644043, -2.699409008026123, 5.7840704917907715, 5.126310348510742, -19.50796127319336, 24.42920684814453, 3.319195508956909, -7.814144611358643, -2.287522315979004, 10.095963478088379, -8.035865783691406, 12.631538391113281, 10.47045612335205, 8.793502807617188, 14.3162202835083, 9.665903091430664, -3.237807512283325, 2.1291332244873047, 7.311300277709961, -6.576437950134277, -5.8140177726745605, 3.346329927444458, 1.3401145935058594, -2.238337516784668, -11.7972993850708, 7.772176742553711, -14.71574878692627, 4.30870246887207, -6.253864765167236, 7.584417343139648, -4.713170051574707, 10.432256698608398, 9.758583068847656, 5.666765213012695, 6.748603820800781, -3.899188995361328, 8.40652847290039, -18.361114501953125, 4.575417518615723, 4.320371627807617, -0.334378182888031, -13.3966064453125, -3.946704626083374, 10.829734802246094, -4.476737022399902, -13.26032543182373, 5.572938442230225, -9.67504596710205, 8.764700889587402, -26.052326202392578, 2.7622060775756836, -11.390176773071289, 13.28835391998291, -12.678784370422363, 16.14122200012207, 2.904606819152832, 1.020198106765747, -2.7745909690856934, 3.12003493309021, -4.94932746887207, -0.990014910697937, 8.58761215209961, 14.860294342041016, -4.77551794052124, 4.1741790771484375, -12.540603637695312, -8.792550086975098, -3.2103803157806396, -17.907405853271484, 10.393377304077148, 5.901261329650879, -6.934144973754883, 7.952928066253662, 21.178089141845703, -5.423913478851318, -1.4848542213439941, -8.944221496582031, 10.549243927001953, -11.357048034667969, -3.6106972694396973, -4.197171211242676, -8.618658065795898, -6.01970911026001, -3.3938279151916504, -0.48203563690185547, -9.196115493774414, 8.428791999816895, -12.816234588623047, 1.549903154373169, -10.787059783935547, 13.703036308288574, 1.6798605918884277, 20.40540885925293, -9.483002662658691, -18.814781188964844, -4.1789231300354, 2.5544543266296387, 3.902693271636963, -6.778831481933594, 4.24895715713501, -3.663832902908325, -15.498115539550781, 6.558873176574707, 21.981552124023438, 7.482948303222656, -15.467548370361328, 17.409507751464844, 2.633603096008301, -10.640487670898438, 8.83938217163086, 7.022055625915527, 0.5473184585571289, -2.2035326957702637, 2.7329752445220947, 4.432555198669434, -9.882465362548828, 4.156133651733398, -28.017112731933594, 0.023819874972105026, 7.8601813316345215, -1.4896924495697021, 3.6795222759246826, -10.369301795959473, 2.4841527938842773, 1.0477291345596313, 0.5447183847427368, 6.560193061828613, 0.9555118083953857, 4.640560626983643, -4.009586334228516, -11.362382888793945, -10.5137939453125, 11.057724952697754, 11.73021411895752, 0.05196722224354744, 4.496387004852295, -6.722992897033691, -7.208244800567627, -3.1989169120788574, -3.027438163757324, -15.785015106201172, -4.312453746795654, 0.9139983654022217, -3.28436279296875, -11.87700366973877, -3.579123020172119, 17.9952449798584, -15.389364242553711, 7.293612957000732, -10.792241096496582, -15.98593521118164, 1.1488007307052612, -1.1931347846984863, -0.43467140197753906, -0.9234538674354553, -11.58107852935791, 12.718870162963867, -6.536540508270264, -9.826814651489258, 9.883365631103516, 5.317148685455322, -14.521093368530273, -9.942646980285645, -4.762536525726318, 4.5331902503967285, -7.488157272338867, 3.2732863426208496, 16.831228256225586, 12.568538665771484, 0.3955051600933075, -0.32651323080062866, 2.8252930641174316, -3.0304644107818604, -6.2711262702941895, -22.275421142578125, 4.559898853302002, -10.966753005981445, 21.4985294342041, 17.485736846923828, 1.8197531700134277, 19.88546371459961, -7.740027904510498, 4.889242649078369, -24.427688598632812, -7.420938491821289, 7.7962775230407715, 10.214690208435059, -11.092111587524414, -30.868270874023438, -22.852678298950195, 7.657787322998047, 5.571679592132568, 16.279943466186523, -3.174423933029175, 3.8491275310516357, -2.8597631454467773, -14.072423934936523, -2.304823398590088, 18.953903198242188, -7.630948543548584, 12.394858360290527, 14.580263137817383, -4.183754920959473, 2.337627649307251, -12.687091827392578, 7.787940502166748, -3.0848522186279297, 3.327813148498535, -2.772341728210449, -3.5175511837005615, -13.712163925170898, -6.264439582824707, 16.414472579956055, -7.151317596435547, 18.344324111938477, -10.0422945022583, -5.4811296463012695, -2.8595924377441406, -4.302309989929199, -5.500300407409668, 16.855472564697266, -12.092037200927734, 12.005020141601562, 9.92261791229248, -20.99469757080078, -11.025136947631836, -3.594506025314331, 13.757061004638672, -3.349102020263672, -6.678354263305664, -10.503996849060059, -1.2267042398452759, 23.51602554321289, 0.8012714385986328, -6.122314453125, -6.740942478179932, 0.8725127577781677, -8.458488464355469, -23.589914321899414, 2.858598232269287, -3.628535270690918, -3.81467604637146, 3.0363190174102783, -9.48471450805664, -1.0352327823638916, -0.07537388056516647, 29.831445693969727, 4.762629985809326, 4.377847671508789, -0.7789340019226074, 27.145919799804688, 6.775489807128906, -17.984655380249023, 11.021422386169434, 3.3131072521209717, 11.038362503051758, 4.6240434646606445, -3.1699588298797607, -9.48495864868164, 8.877771377563477, -13.687721252441406, -5.54489803314209, -11.12561321258545, 4.618799686431885, -0.24020293354988098, 5.067133903503418, -11.756193161010742, -7.384700775146484, 11.486928939819336, 5.023212909698486, -4.420981407165527, 1.228127121925354, -11.231070518493652, 0.7853978872299194, -16.805789947509766, 9.817181587219238, 4.665400505065918, 1.0564501285552979, 0.5913423895835876, -0.7575743794441223, 2.4474918842315674, -16.17691421508789, 16.20487403869629, -4.641204833984375, 15.697967529296875, -7.026620864868164, 8.675503730773926, -3.693143129348755, -9.889518737792969, -8.659191131591797, 9.056309700012207, -11.689348220825195, 4.356116771697998, -20.927959442138672, 0.273170530796051, -2.401899814605713, 11.892263412475586, 14.348175048828125, 15.181190490722656, 0.7355222105979919, -10.773820877075195, 7.124460220336914, 5.053903579711914, -13.185285568237305, -16.813257217407227, -6.259422779083252, 0.07623222470283508, 0.96359783411026, 10.702096939086914, 8.631770133972168, -5.278443336486816, -11.678655624389648, 14.8858642578125, -0.18155820667743683, -11.309717178344727, 19.09393310546875, -2.8542122840881348, -23.89103126525879, -16.379745483398438, 8.80700969696045, -0.7807235717773438, -13.058241844177246, -4.295228958129883, 19.000551223754883, -2.8694794178009033, 17.75100326538086, -9.365084648132324, -15.398801803588867, -19.917570114135742, 4.164886951446533, 9.56772232055664, -14.73217487335205, 5.649948596954346, -3.68548846244812, -8.637589454650879, -16.261798858642578, -15.617616653442383, 0.3105595111846924, 2.465843915939331, -9.993363380432129, 11.305950164794922, -10.40519905090332, -30.5260066986084, 10.520964622497559, 5.48484992980957, 17.955583572387695, -11.101250648498535, -12.914132118225098, -15.355974197387695, -3.9126780033111572, -11.580902099609375, 15.88248348236084, -5.020186901092529, 9.542187690734863, 18.851234436035156, -6.7886881828308105, -11.825522422790527, -4.617466449737549, -2.268049955368042, 11.895814895629883, -3.713996648788452, 13.50007152557373, 31.174930572509766, 5.579874038696289, -5.442939281463623, 11.172913551330566, -0.9998497366905212, 10.948456764221191, 4.409638404846191, -5.502506256103516, 9.635293006896973, -5.7007036209106445, 10.738673210144043, 9.87553882598877, -2.733203172683716, 5.090802192687988, 8.799405097961426, 11.912734985351562, 15.874015808105469, 11.851282119750977, 12.52692699432373, 12.388042449951172, -8.428873062133789, 1.8116284608840942, 4.654604911804199, -1.168785572052002, -0.11883054673671722, 6.267075538635254, 13.86290454864502, 6.534451961517334, -5.757058143615723, -8.194865226745605, -6.389937877655029, 3.7903051376342773, 13.457931518554688, 8.529047012329102, -3.930375099182129, 7.5276899337768555, 6.030824661254883, -2.6744933128356934, 10.533267974853516, -8.788331985473633, 11.815229415893555, 15.97163200378418, -12.087727546691895, -3.931840658187866, 4.801374912261963, 4.504793167114258, 8.883625030517578, -4.955373287200928, 6.5222601890563965, -8.075296401977539, 9.077800750732422, 17.55615997314453, 6.283040523529053, -9.60962200164795, -1.0427556037902832, -12.975807189941406, -9.519420623779297, 4.065919399261475, -17.959171295166016, -25.527557373046875, 11.71716594696045, -5.293439865112305, 6.436038494110107, -6.217577934265137, 13.234737396240234, 0.9414509534835815, 0.061105694621801376, -11.390968322753906, -1.0039631128311157, -0.972132682800293, -6.501402854919434, -3.201594829559326, 14.85441780090332, 4.630724906921387, -1.8552556037902832, -11.972566604614258, -2.782504081726074, 0.25924599170684814, 8.886898040771484, 14.944878578186035, -0.4699924886226654, 15.750442504882812, -4.132771968841553, -7.203894138336182, -6.108707427978516, -6.45233678817749, -12.133179664611816, -17.322546005249023, 2.326186180114746, 4.676635265350342, 9.323716163635254, 1.401929259300232, -5.864021301269531, -6.675641059875488, 8.77901840209961, 4.114354133605957, -8.491775512695312, 6.290664196014404, -0.6481847763061523, -11.502533912658691, 8.086470603942871, -13.237103462219238, 0.767707347869873, -11.5997896194458, -7.356614589691162, 1.4874556064605713, 3.1775057315826416, -19.29258155822754, 20.19054412841797, 5.426813125610352, -5.875821113586426, -2.312523126602173, 14.712223052978516, -12.280479431152344, 9.202400207519531, 5.630586624145508, -0.3107343316078186, -2.2624659538269043, -6.033917427062988, 6.120248317718506, 7.115068435668945, -7.806550979614258, -6.012950897216797, 0.2701517641544342, 0.3447462320327759, -3.6030876636505127, 1.9794927835464478, 27.859098434448242, -5.30157470703125, -6.404052257537842, 8.105362892150879, 14.935304641723633, -5.447005748748779, -2.113924264907837, 3.410707950592041, 22.32456398010254, 8.285294532775879, -1.7396695613861084, 5.223828315734863, -5.251564025878906, 5.265015602111816, -4.6907525062561035, 12.574125289916992, 12.718484878540039, -1.049587607383728, -5.804773330688477, 7.799605369567871, 1.9595283269882202, 9.617790222167969, -4.917655944824219, 5.650864601135254, 6.229747772216797, -6.949862957000732, 12.93128776550293, 1.9543085098266602, -5.253932476043701, -6.1631269454956055, -9.921830177307129, 0.835075318813324, -15.257936477661133, -3.7024264335632324, -5.231432914733887, -14.92501163482666, 4.686131000518799, -1.6331290006637573, -6.517549514770508, -9.641119003295898, 4.086866855621338, 4.377624034881592, -11.61599349975586, 6.452373504638672, -4.597834587097168, 9.655482292175293, -5.154799938201904, 6.192397594451904, 4.1614460945129395, -2.5801520347595215, 3.353010654449463, -1.3978793621063232, 11.705879211425781, 1.5782384872436523, 0.4287634789943695, 9.936614990234375, -12.345846176147461, -7.306605339050293, 4.5637969970703125, 3.870713710784912, 1.8719873428344727, -0.692563533782959, 2.7153992652893066, -1.7829113006591797, -11.624303817749023, -8.61548900604248, -5.04311990737915, -17.487646102905273, 9.666041374206543, 0.37822026014328003, -11.102811813354492, -8.231501579284668, 9.85861587524414, 21.09137535095215, 14.05483341217041, 12.800589561462402, 0.6264374256134033, 11.956887245178223, -4.873211860656738, 0.35399115085601807, 16.454946517944336, -3.4795637130737305, 0.14532770216464996, -0.19642339646816254, 13.143604278564453, 18.215599060058594, -19.732872009277344, 4.50999641418457, 11.129188537597656, 9.815627098083496, 2.832235813140869, 7.084375381469727, -18.2955322265625, -9.562811851501465, 4.896350383758545, -22.472848892211914, 5.276117324829102, 0.11186981201171875, -10.336799621582031, -5.933802604675293, -1.4296512603759766, -7.032153129577637, 18.277660369873047, 0.11411573737859726, -2.5493760108947754, -6.7313551902771, -12.92543888092041, -7.631529808044434, 18.86992073059082, -6.036509990692139, 2.7558045387268066, -11.810720443725586, -7.572512149810791, -23.248764038085938, 8.510185241699219, 2.3593616485595703, -5.451927661895752, 9.291726112365723, 4.427889823913574, -15.560461044311523, 0.10371895134449005, -10.636219024658203, -11.652947425842285, -2.417217969894409, 6.539893627166748, 4.64465856552124, -0.6303136944770813, -9.812638282775879, -6.0132317543029785, -14.507584571838379, 9.02066421508789, -4.153865814208984, -10.2162504196167, 6.749210357666016, -7.744354724884033, -6.713040351867676, -8.38720417022705, -0.49843770265579224, 5.014492034912109, -1.4113538265228271, 8.746488571166992, 5.828680515289307, -22.796113967895508, 2.5512585639953613, -5.347036838531494, 10.30186653137207, 11.936477661132812, 5.769443511962891, 4.086177825927734, 12.079858779907227, -0.7362375259399414, -3.348393201828003, -1.3171957731246948, 3.344452381134033, 6.500437259674072, 1.0730867385864258, 4.822657585144043, 20.984067916870117, 8.661182403564453, 7.841763019561768, 5.871364593505859, -23.859153747558594, -9.222576141357422, 12.25120735168457, -11.644214630126953, -0.3758881986141205, -5.9956254959106445, 3.6722664833068848, 0.5224137306213379, -12.900657653808594, -0.4285871982574463, 4.425010681152344, -3.6877191066741943, 5.981134414672852, 3.0279994010925293, -17.82760238647461, -1.5891947746276855, -2.198739528656006, 10.491617202758789, 3.50821852684021, 1.038079857826233, -5.488973617553711, 13.771255493164062, -15.659467697143555, 11.375551223754883, 6.283597946166992, 11.982219696044922, 13.976119995117188, 6.602387428283691, 8.045106887817383, -13.053322792053223, -17.88826560974121, 8.594428062438965, -7.675606727600098, -3.660754680633545, -7.973914623260498, -13.512802124023438, -20.55266571044922, -0.34625017642974854, 5.671604156494141, 23.155620574951172, -10.31559944152832, -21.432411193847656, -11.371469497680664, 0.8636671900749207, -16.858905792236328, 9.125414848327637, 8.96994686126709, -10.094644546508789, -0.052963946014642715, 4.128371238708496, -6.489253044128418, 0.4377897381782532, -8.898561477661133, -4.276058197021484, -11.541379928588867, 16.40769386291504, 3.4546518325805664, -7.160206317901611, 7.467393398284912, 1.9925158023834229, 2.492542266845703, -19.699750900268555, 7.485661506652832, 7.2408013343811035, -12.873662948608398, -10.227035522460938, -14.838573455810547, 2.1506638526916504, -12.696480751037598, -10.7582368850708, 6.640255451202393, -3.9948983192443848, -9.175216674804688, 4.036352634429932, 2.318014144897461, 5.171780586242676, -4.782083511352539, 4.590390205383301, -10.152185440063477, -17.668380737304688, 9.405817985534668, 8.663200378417969, 2.796128273010254, 0.17657695710659027, -6.563723564147949, -4.703786849975586, 7.814960479736328, -29.142274856567383, 6.619610786437988, 0.17288373410701752, -23.373634338378906, -11.580962181091309, 0.5675380229949951, -14.49541187286377, -4.637206077575684, -15.246660232543945, -4.317263603210449, -0.666447639465332, -2.577291965484619, 23.36719512939453, -0.263143926858902, -25.645410537719727, -19.729764938354492, 7.054771423339844, 3.7898755073547363, -8.187095642089844, 9.48067855834961, -13.997089385986328, -5.779602527618408, -5.755128860473633, -2.192204475402832, -1.3230078220367432, 2.970869302749634, 4.547192096710205, -7.93080997467041, -1.2666704654693604, -1.0026130676269531, 0.5083740949630737, -7.561959743499756, 0.579501211643219, 0.44636741280555725, -10.22940444946289, 6.994542598724365, 6.581281661987305, -10.370482444763184, -18.610464096069336, -5.283253192901611, 18.724233627319336, -11.036853790283203, -2.1600594520568848, 2.37675142288208, 10.386133193969727, -14.14659309387207, -10.250436782836914, -11.510541915893555, 5.490779876708984, -3.60676908493042, -9.841569900512695, -15.536977767944336, 8.667890548706055, 10.225357055664062, -6.961015701293945, -8.691651344299316, -31.61278533935547, 14.145264625549316, 8.889188766479492, -5.939409255981445, -8.830331802368164, 0.23085114359855652, -10.256052017211914, -1.5830856561660767, -0.7292729020118713, -0.32559725642204285, 0.44068992137908936, -4.271134376525879, 6.493098258972168, -1.5560264587402344, 22.452030181884766, -4.627144813537598, 2.074974536895752, 3.9356324672698975, 5.271111011505127, -6.631575107574463, -5.4210638999938965, 7.177698612213135, 4.593815326690674, 15.463323593139648, -2.8023457527160645, -5.802338600158691, 11.473567962646484, 2.5553107261657715, 1.4142868518829346, 8.794581413269043, -9.399523735046387, -2.629491090774536, 11.539243698120117, -14.156082153320312, 7.2260613441467285, -0.889096200466156, -9.850577354431152, 1.5509158372879028, 9.561432838439941, -1.4126218557357788, -9.660553932189941, 0.17007030546665192, -7.280910491943359, 6.608155250549316, -3.285555601119995, -6.566438674926758, 26.83717918395996, 6.105748176574707, 0.007138713728636503, 2.4797821044921875, -5.1379923820495605, -23.627004623413086, 11.90025806427002, -18.824378967285156, 7.392211437225342, -6.654269218444824, 0.8028353452682495, 9.15488338470459, 2.795828342437744, 8.041413307189941, 7.8291778564453125, 13.868646621704102, 18.893386840820312, 3.590761184692383, 6.256190299987793, 7.40934944152832, 14.80489730834961, 8.606085777282715, 2.1952872276306152, 5.406208038330078, -11.661575317382812, -5.27649450302124, -19.319732666015625, -8.819485664367676, 8.52877426147461, 6.906915187835693, -7.13416862487793, 3.5863513946533203, -17.157791137695312, 1.6918869018554688, -12.922962188720703, -1.9222437143325806, 0.46865516901016235, -9.863992691040039, -2.5469260215759277, -9.10545825958252, -4.343615531921387, 4.86891508102417, -12.536992073059082, -1.5496559143066406, -13.910177230834961, -13.34572982788086, -8.871607780456543, -17.366411209106445, 0.8069700598716736, 0.5877147316932678, -2.4573495388031006, -4.0755157470703125, -13.413782119750977, -10.415362358093262, -5.9557695388793945, -3.151453971862793, 0.5942252278327942, -0.6425943970680237, -10.25415325164795, 3.5664596557617188, 11.150093078613281, 2.687011957168579, 6.4716477394104, 2.822421073913574, -8.769939422607422, -7.355680465698242, 3.0018258094787598, -19.164033889770508, 0.1700381189584732, -8.369571685791016, -3.6592230796813965, -18.953678131103516, -0.6033729910850525, 16.97264289855957, -17.794851303100586, 16.709835052490234, 3.406864643096924, 0.6284370422363281, -14.055777549743652, -16.865007400512695, -9.86237907409668, -16.986330032348633, -2.740234375, -2.9230761528015137, 11.019867897033691, -1.4618855714797974, 10.06247615814209, 0.5747602581977844, 15.044913291931152, 4.686600208282471, -25.387836456298828, 2.7214016914367676, 11.840988159179688, -5.875916004180908, -0.3078005909919739, -3.132751226425171, -1.2976795434951782, 14.934274673461914, 4.527176856994629, -6.068728446960449, -0.0629354938864708, -4.49686861038208, -2.2769665718078613, 1.645473837852478, 6.72919225692749, -0.5337602496147156, -3.6806833744049072, -0.5432957410812378, 2.2216155529022217, -0.26867035031318665, -14.974245071411133, -18.52669334411621, -11.285382270812988, 2.625607490539551, 9.677467346191406, -5.714126110076904, 4.559427261352539, -7.006901741027832, -8.125601768493652, 12.775492668151855, 14.667120933532715, -4.641121864318848, 4.4990739822387695, -4.7692389488220215, 3.276895523071289, 0.479276180267334, -13.489550590515137, 3.0703725814819336, -5.954835414886475, -6.729979515075684, 14.024113655090332, 1.050813913345337, 1.5342607498168945, 9.074169158935547, 16.51456642150879, 3.7689528465270996, -6.564060211181641, -6.258063316345215, 5.2448625564575195, -3.1167144775390625, 0.25569766759872437, -6.380428791046143, 0.03125309199094772, 8.844202041625977, 9.201887130737305, -3.386162519454956, -16.3875732421875, -1.2494463920593262, -13.848546981811523, -11.236045837402344, -10.181058883666992, -7.928955554962158, 2.5764501094818115, 6.836369037628174, -8.93869686126709, -0.2371896654367447, 7.022407054901123, -5.168822288513184, 8.956632614135742, -5.836119651794434, 8.535731315612793, 7.172467231750488, -1.9747016429901123, 8.551895141601562, -2.5349621772766113, -2.4456868171691895, 24.57718276977539, -1.1096370220184326, -5.406620025634766, 1.0163829326629639, -24.363094329833984, 3.42531156539917, -14.521503448486328, 2.2930262088775635, 4.117569446563721, 5.9073615074157715, 5.634835243225098, 13.770866394042969, -16.10411834716797, -10.414491653442383, -8.623737335205078, -0.617988646030426, -5.427431583404541, 14.681951522827148, 6.21213436126709, 8.117952346801758, 6.060369491577148, -10.596090316772461, -12.185760498046875, -1.201023817062378, -0.5022061467170715, 11.263717651367188, -14.48426342010498, 19.10106086730957, -2.76134991645813, -0.4341201186180115, 1.552166223526001, 12.311259269714355, 2.1927123069763184, -14.257104873657227, -9.963251113891602, 8.57581615447998, 16.620996475219727, -5.671487808227539, -9.323226928710938, -5.818582534790039, 1.8554736375808716, 10.35784912109375, -4.1708879470825195, 6.952689170837402, -5.069380283355713, -14.54824447631836, 10.091887474060059, 1.1459544897079468, 13.374082565307617, -1.2263314723968506, 6.841726779937744, -10.97777271270752, -1.4494638442993164, 7.561526775360107, 1.9706401824951172, 3.507305383682251, 5.500654697418213, -10.032073020935059, -6.044771194458008, 8.394357681274414, 8.99770736694336, -2.897479295730591, 2.947956085205078, 5.316106796264648, 18.833995819091797, -7.3398823738098145, 3.017843723297119, -13.201967239379883, 8.997961044311523, 4.89420223236084, -16.02863311767578, -13.458236694335938, 6.927767276763916, 3.675766944885254, 0.3482001721858978, 1.1106386184692383, -12.662040710449219, 7.009124755859375, 5.118703842163086, -2.82830810546875, 12.498522758483887, -6.7764129638671875, 14.609054565429688, 1.2701482772827148, -1.1110490560531616, -8.96850872039795, 0.924531102180481, 3.5454366207122803, -16.97576904296875, 1.298051118850708, -10.991930961608887, -3.424349069595337, 19.35403060913086, 11.232329368591309, -2.624046564102173, 9.440948486328125, -11.790000915527344, 0.7740062475204468, 12.538877487182617, 4.363605499267578, 22.11095428466797, -13.6065673828125, 4.406740188598633, -11.51883602142334, -0.387894868850708, 0.4754483699798584, -7.5648980140686035, 4.211817741394043, -20.933574676513672, -5.587213039398193, 2.7816853523254395, -5.349478244781494, 5.2954487800598145, 9.54131031036377, -0.8891170620918274, 16.851688385009766, 2.5712778568267822, -3.207630157470703, -2.3540213108062744, -4.7885613441467285, 12.17947769165039, -2.524661064147949, -2.293649673461914, -3.0457091331481934, -9.470067977905273, -9.540919303894043, -12.049224853515625, -4.79143762588501, 4.143091201782227, -3.494831085205078, 4.116511821746826, 3.5694360733032227, -0.07335781306028366, -8.308058738708496, 2.063325881958008, 5.114479064941406, 10.120782852172852, 14.11410903930664, 2.621013879776001, -9.013822555541992, 3.206045627593994, -10.234457015991211, -11.436290740966797, 9.509499549865723, 6.30573844909668, 27.0602970123291, 9.884546279907227, 4.831352710723877, -6.207050800323486, -6.848803520202637, -7.930598258972168, 12.82693099975586, -9.514222145080566, -3.3321595191955566, -2.0310020446777344, 10.493568420410156, -5.540373802185059, -1.0164291858673096, 8.473776817321777, 5.306090831756592, 9.689002990722656, -24.163066864013672, -16.403348922729492, 4.266800403594971, 7.723069190979004, 1.745270013809204, 10.58598518371582, -0.5362696647644043, 6.403262615203857, -1.7240930795669556, 7.00432825088501, 7.339632987976074, 14.897135734558105, 5.3680949211120605, -5.195921897888184, 5.304265975952148, 14.468817710876465, 1.6296948194503784, 4.677657127380371, 8.540647506713867, -6.2778167724609375, -8.548394203186035, -8.560182571411133, 6.943073749542236, -0.8834741115570068, -7.173181533813477, 6.8893818855285645, -5.8086700439453125, 0.3169727325439453, 14.864099502563477, -0.9877544045448303, -4.244802951812744, 0.15256619453430176, -4.507116794586182, 13.016558647155762, -9.468323707580566, 16.839412689208984, 6.126979827880859, 13.99013900756836, 1.1484558582305908, 8.978897094726562, -3.2384185791015625, 15.813024520874023, -2.438723087310791, -14.202937126159668, 13.245553970336914, 0.09525664895772934, -6.389111518859863, 5.341455459594727, -8.747695922851562, -2.300069570541382, -6.633919715881348, -4.034444332122803, 1.7708959579467773, -3.8396453857421875, 3.048656940460205, -19.586021423339844, 4.51607608795166, -12.022192001342773, 7.150282382965088, -2.800790309906006, 5.22777795791626, 6.050422668457031, 1.4855927228927612, -11.060297012329102, 10.213958740234375, -4.514243125915527, -8.024171829223633, -3.179459571838379, -13.951249122619629, -7.29188871383667, 1.1481653451919556, -6.885122299194336, 0.40480339527130127, -2.1620945930480957, 4.649525165557861, 2.2197632789611816, -9.366338729858398, 0.9266314506530762, 12.558061599731445, 11.914042472839355, 26.494651794433594, -3.786189079284668, 0.871565043926239, -2.039658308029175, 4.205243110656738, 23.053647994995117, 13.044047355651855, 8.24064826965332, -22.666168212890625, 2.556159019470215, -11.542654991149902, 11.749761581420898, 11.387648582458496, 12.857136726379395, 8.87695026397705, -5.278777122497559, 8.229743957519531, 2.5526645183563232, 10.896915435791016, -4.54172420501709, -17.97532844543457, 6.632980823516846, 16.833227157592773, -9.025076866149902, -4.968844413757324, 1.5466021299362183, -1.3537293672561646, -12.78021240234375, -28.990615844726562, 6.258859634399414, 10.840331077575684, -13.07646656036377, 6.933175086975098, 2.9438834190368652, -4.8533935546875, -6.2859601974487305, -1.11440908908844, -14.18402099609375, -10.753816604614258, -1.429768681526184, -7.360630512237549, 13.446688652038574, 22.96727180480957, -0.8514812588691711, -2.2317490577697754, 0.9683883190155029, -17.806724548339844, -2.615333080291748, 1.5565322637557983, 10.382627487182617, 5.05488395690918, 3.370736837387085, -4.529361724853516, -4.989659309387207, -12.956737518310547, 8.97685432434082, 11.742517471313477, -17.918277740478516, 1.468475580215454, 2.039616584777832, -6.661524772644043, 4.811069488525391, -9.990080833435059, -18.512950897216797, -11.899567604064941, 6.324892997741699, -18.325136184692383, -13.44770336151123, 18.530445098876953, -5.942183017730713, 4.584569454193115, 0.31391412019729614, -12.790443420410156, -10.17486572265625, -7.464476108551025, -11.672931671142578, -1.7757428884506226, 11.78060531616211, 4.646695137023926, -7.2490668296813965, -6.316699504852295, 11.040759086608887, 6.507282733917236, 4.776311874389648, -18.947118759155273, -13.89431381225586, 8.76694107055664, 9.625824928283691, -2.704756259918213, -8.925169944763184, -10.075722694396973, -6.8896284103393555, -8.491867065429688, 4.587207794189453, -14.413227081298828, 9.878974914550781, -1.0334972143173218, 3.6079554557800293, 4.897298336029053, 6.8060150146484375, -6.589815139770508, 19.440767288208008, -0.4642561674118042, -3.445920705795288, -10.556979179382324, -5.013266563415527, -0.8447664976119995, -0.6753852963447571, 10.18642520904541, 18.310192108154297, -7.961593151092529, 2.658230781555176, -0.5026708245277405, 3.9212441444396973, 1.8836054801940918, -5.629868984222412, -4.880902290344238, -0.05247940868139267, 15.193851470947266, 12.762264251708984, 3.9759626388549805, 0.3513076901435852, -1.7224628925323486, -4.572807788848877, 7.3460917472839355, 12.181234359741211, 13.726091384887695, -22.666866302490234, -6.964607238769531, 25.469972610473633, 24.560766220092773, -3.0444655418395996, 16.52881622314453, -5.3471808433532715, -18.014616012573242, 6.89914608001709, 0.9790563583374023, 3.9771652221679688, -13.008052825927734, -5.875410556793213, -13.811205863952637, -9.934464454650879, -9.712936401367188, -10.650808334350586, 2.0515129566192627, 16.73108673095703, -1.929358959197998, -13.521985054016113, -8.132460594177246, -6.485452175140381, 1.8568639755249023, -18.76356315612793, -15.674234390258789, -7.512935638427734, -1.5257428884506226, 25.950592041015625, 3.6919212341308594, -0.8708617687225342, 6.221251964569092, 19.289512634277344, -2.41703724861145, 6.259817600250244, 13.61127758026123, 15.925564765930176, 1.8323482275009155, 13.235054016113281, -19.758182525634766, -3.3765945434570312, 22.512657165527344, -4.984563827514648, 6.263365745544434, 3.6672492027282715, 21.436241149902344, 12.60734748840332, -0.909660816192627, 15.450153350830078, -6.666735649108887, -12.425955772399902, -15.649886131286621, -11.370387077331543, -12.251422882080078, -0.011435468681156635, -1.426513433456421, -0.6356751918792725, 2.7633204460144043, 3.913480281829834, -28.27571678161621, -11.515207290649414, 5.511866092681885, 0.1437748372554779, -1.5182125568389893, 6.5871734619140625, 13.308228492736816, 0.3249475359916687, 14.950078964233398, 5.957090854644775, -1.6044223308563232, 8.426549911499023, -17.256484985351562, 1.9121277332305908, 2.946279525756836, -7.353639125823975, -3.870121955871582, -1.51185142993927, 10.616477966308594, -3.607851505279541, 14.622417449951172, -17.208650588989258, -4.946896553039551, 1.8577944040298462, -13.320623397827148, 1.307246446609497, -2.7592506408691406, -1.601775884628296, 6.0116095542907715, 12.37699031829834, -1.8893296718597412, 9.385492324829102, -7.405223369598389, -7.491302013397217, -0.8456243276596069, 3.044910430908203, 13.475407600402832, -7.616374969482422, -6.105441570281982, 5.891053676605225, 9.504708290100098, -1.5570383071899414, -6.922545433044434, 9.937614440917969, 4.888071537017822, 0.17790129780769348, 1.4068503379821777, 9.156364440917969, 9.444862365722656, 1.9010422229766846, 9.838821411132812, -5.202402114868164, -20.382347106933594, -4.039526462554932, 4.6675310134887695, -7.953665256500244, 12.350935935974121, -1.5201293230056763, -8.901571273803711, 7.4663214683532715, -2.861544132232666, 15.543807983398438, 14.684646606445312, 3.513087749481201, -18.706756591796875, 0.5038276314735413, 11.854886054992676, -17.596830368041992, 10.905437469482422, 7.655494213104248, 10.494482040405273, 13.057341575622559, -6.406002998352051, -6.134028911590576, 8.479789733886719, 8.513629913330078, 4.5885725021362305, 5.215279579162598, -3.194375514984131, -6.132885932922363, 8.32648754119873, 2.8571524620056152, -2.448528289794922, -15.975992202758789, -10.582635879516602, 1.1559014320373535, -3.534295082092285, 25.099262237548828, 17.827465057373047, 0.6472446918487549, -2.3288092613220215, 0.09852516651153564, 4.684348106384277, -10.760499000549316, 1.6703076362609863, 5.84092378616333, -10.743340492248535, 13.946130752563477, 1.5784512758255005, -13.780006408691406, 10.004252433776855, -0.042445823550224304, -12.940951347351074, 4.577597141265869, -11.318962097167969, 9.214571952819824, 2.5130648612976074, -15.657527923583984, 2.011909246444702, 14.59475040435791, 6.856747150421143, 7.0798234939575195, 12.141465187072754, 5.190271854400635, 0.1988275945186615, -7.147614479064941, 22.764644622802734, -6.442464828491211, -5.105655670166016, -1.7898601293563843, 4.300384044647217, -12.257298469543457, 7.8445048332214355, -20.517333984375, 4.589980125427246, 0.9177864789962769, -9.715980529785156, 14.332071304321289, -16.886484146118164, 7.492409706115723, 1.1333777904510498, 7.351150989532471, -6.577579498291016, 3.625154972076416, 1.7186431884765625, -1.3817224502563477, 2.4744627475738525, 2.783569097518921, -12.539721488952637, -0.4892485737800598, 13.481389999389648, -1.2509658336639404, 8.327540397644043, 1.9994968175888062, 20.740657806396484, -2.6735787391662598, -7.208018779754639, -13.954442977905273, 10.779193878173828, 3.3728597164154053, -6.1974358558654785, 12.802142143249512, -2.6731553077697754, -5.356199264526367, -2.8777847290039062, 2.319514513015747, -9.707297325134277, -5.522129058837891, -2.245197296142578, -4.827767848968506, 5.454279899597168, -18.38999366760254, -14.616202354431152, 4.632959365844727, -26.344459533691406, 18.522972106933594, -3.7546839714050293, 1.6936534643173218, 8.485591888427734, 1.0293203592300415, 1.2455787658691406, 14.775286674499512, 8.924054145812988, -11.763843536376953, 9.636744499206543, 8.966713905334473, 9.203941345214844, 0.20668359100818634, -0.5208635926246643, 21.863414764404297, 3.6185505390167236, 20.942222595214844, -21.855714797973633, 8.449390411376953, 8.360273361206055, 18.73090362548828, 18.317113876342773, 2.2878973484039307, 7.922870635986328, 6.839419364929199, 9.07591724395752, 18.582412719726562, -11.720723152160645, -2.1086320877075195, -5.9996843338012695, 0.6860693693161011, -9.690652847290039, -7.608664035797119, 2.684213161468506, 18.994386672973633, 7.421080589294434, -12.767976760864258, 7.004880428314209, 1.9757534265518188, 10.548587799072266, 18.78510284423828, -13.394880294799805, -0.003150706645101309, 12.960663795471191, -20.2919921875, 3.0452613830566406, -0.9619572162628174, -6.369010925292969, -15.454261779785156, -8.117263793945312, 5.014474391937256, -2.3589823246002197, -0.8887196779251099, 6.375076770782471, 5.379540920257568, -12.448314666748047, -0.7393897175788879, -4.108849048614502, -10.125007629394531, 5.094711780548096, -10.755178451538086, -14.489364624023438, -28.048866271972656, -12.589856147766113, 17.693574905395508, -16.501081466674805, 7.552359580993652, 2.3346235752105713, 0.07689504325389862, -3.782642364501953, 5.6387810707092285, -10.612630844116211, 2.761066198348999, 3.7529754638671875, -3.0557596683502197, -4.549340724945068, 5.0431013107299805, -5.657707214355469, 3.642327308654785, 0.7412442564964294, 17.70478057861328, -11.986408233642578, -6.045770168304443, 0.3905482888221741, 8.567313194274902, 5.005295753479004, -5.118411064147949, 8.105936050415039, -2.48441743850708, 0.34941935539245605, -2.015139579772949, -7.196883201599121, 1.3156808614730835, -19.041858673095703, 12.489645004272461, 4.555115222930908, -8.318330764770508, 1.3977282047271729, -6.175899982452393, -10.956921577453613, 16.19342803955078, -10.880767822265625, -1.9616844654083252, 10.080842018127441, -0.3601810336112976, -0.6986424326896667, -0.37972939014434814, 10.516340255737305, 13.80037784576416, -5.0271124839782715, 2.864994525909424, 8.01896858215332, -2.7406115531921387, 23.076810836791992, -17.745214462280273, -8.628849029541016, 6.886604309082031, -1.5562797784805298, -2.7387661933898926, 5.057624816894531, 1.6791285276412964, 11.89089584350586, -8.768747329711914, 7.392125129699707, -8.160619735717773, -16.202617645263672, 3.5860190391540527, -17.538543701171875, 2.4596304893493652, 5.3225297927856445, 8.689992904663086, 6.06838321685791, -6.62629508972168, 15.732436180114746, -16.106969833374023, 12.01429557800293, 3.7792012691497803, -6.423830509185791, -6.462955951690674, 2.226363182067871, -2.686363458633423, 5.000560760498047, 6.572467803955078, 6.792354106903076, 6.556532859802246, -8.151885986328125, -5.51318883895874, 13.314491271972656, 1.0386770963668823, 6.903233051300049, 5.857378959655762, 9.815085411071777, -0.17431914806365967, -6.761208534240723, 7.207489967346191, 8.930704116821289, -2.5410828590393066, -7.996834754943848, 5.725839614868164, -18.775148391723633, -2.4967474937438965, -3.808504104614258, 12.046834945678711, -18.114513397216797, -4.117276668548584, 1.1594812870025635, -20.00933837890625, -6.84530782699585, 18.96021270751953, -4.14483642578125, -0.9022119641304016, -19.902481079101562, -7.5476274490356445, 4.196120738983154, -2.7104344367980957, -13.321100234985352, 7.698576927185059, 2.8606810569763184, 3.2110042572021484, 8.194143295288086, -4.281742572784424, 7.332022666931152, -23.112079620361328, 16.445022583007812, -16.304424285888672, -16.89923858642578, -8.204432487487793, 6.892983913421631, -0.7510883808135986, 12.753087997436523, 5.718979835510254, -0.225010484457016, -15.22464656829834, 7.847821235656738, -0.7289887070655823, 11.103662490844727, -2.3295228481292725, 5.499227523803711, 1.7421653270721436, 15.347504615783691, 5.6067633628845215, 12.01960563659668, -4.176985740661621, 5.983911037445068, 19.690702438354492, -2.58851957321167, 1.325160026550293, -2.577690601348877, 14.50959300994873, -1.5814554691314697, -7.74554967880249, 7.724443435668945, 2.6073157787323, -6.103786468505859, 0.7828945517539978, -3.5248804092407227, 4.680001258850098, -2.195974349975586, -8.91920280456543, -21.780437469482422, -1.6657767295837402, -8.52667236328125, 5.010319232940674, -8.212279319763184, -10.004209518432617, -13.548840522766113, 4.160190582275391, -16.645849227905273, -2.951690912246704, 16.26621437072754, 0.6384411454200745, -10.330644607543945, -1.1037592887878418, 6.329290390014648, 1.3117055892944336, 13.258434295654297, -19.26325225830078, -4.31544828414917, -8.750520706176758, 11.548376083374023, 9.062613487243652, -1.279463291168213, -4.115776062011719, -4.234972953796387, 4.978649616241455, -11.116175651550293, 6.06782865524292, 0.4852491319179535, 0.020026566460728645, -8.044466018676758, 8.901655197143555, 19.660856246948242, 17.525833129882812, 2.0489726066589355, 2.975606918334961, -6.223971366882324, -27.393550872802734, 9.399062156677246, -4.382932662963867, 8.361562728881836, -0.7770103812217712, 19.817590713500977, 2.118671417236328, -3.197732925415039, -2.2750253677368164, 7.137048244476318, 10.585564613342285, 6.262229919433594, -0.9070488810539246, -3.8877604007720947, -0.07354261726140976, -32.53310775756836, -7.672701358795166, -4.69828462600708, -13.068418502807617, -6.961978912353516, 12.076175689697266, -9.176243782043457, -6.677617073059082, 13.808589935302734, -7.632386684417725, 3.3012752532958984, -2.1291496753692627, -15.065754890441895, 10.99393081665039, -2.6701500415802, -10.515810012817383, -6.342755317687988, -4.090978622436523, -6.095861911773682, -7.655274391174316, 0.3356028199195862, 18.61972999572754, -1.5288136005401611, -9.45958137512207, -9.577043533325195, 18.806251525878906, 0.9776949286460876, 4.202610969543457, -3.0161311626434326, 7.288384437561035, 8.140510559082031, -10.429173469543457, -20.984561920166016, -7.605219841003418, 4.284106254577637, 14.586548805236816, -17.07708740234375, 16.376039505004883, 10.861001014709473, 14.207538604736328, -21.434566497802734, -16.499364852905273, -6.503209590911865, -5.767516613006592, 2.1396572589874268, -11.647356986999512, 0.17814049124717712, 1.7584699392318726, -13.431570053100586, -1.6366782188415527, -3.087718963623047, -2.9527554512023926, 4.366667747497559, -8.989494323730469, 0.29362809658050537, -9.53692626953125, -0.6678715944290161, -11.861766815185547, 8.63257884979248, 5.255902290344238, -3.2705445289611816, -0.7609665989875793, -5.170417785644531, -3.5611538887023926, 6.2256975173950195, 1.464524507522583, -7.22571325302124, 10.930780410766602, 3.815551996231079, 13.124019622802734, 5.585516929626465, -17.280881881713867, -6.374597072601318, 10.79409408569336, -7.888524055480957, 9.874245643615723, -3.2920515537261963, 6.206608772277832, -0.547477662563324, -12.837635040283203, -18.70575523376465, -5.291350364685059, 0.8084136843681335, -16.09758186340332, -13.370594024658203, -12.134270668029785, 4.572349548339844, -7.214440822601318, 13.031683921813965, 20.89807891845703, 4.585083961486816, -10.326966285705566, 2.490269660949707, 10.275555610656738, 9.439836502075195, -2.629704236984253, 6.55944299697876, 10.529245376586914, 3.0929341316223145, -3.521251916885376, -14.344362258911133, 12.12663745880127, 6.265981674194336, -16.481651306152344, 4.857510566711426, 5.5660481452941895, -22.82557487487793, -12.468657493591309, 7.829039096832275, 6.703072547912598, -3.0179390907287598, -10.199529647827148, -13.705568313598633, 3.68068265914917, -3.9206137657165527, 0.23836882412433624, -4.069793224334717, 9.497712135314941, 3.8382437229156494, -8.112274169921875, 1.1874563694000244, -17.40363883972168, 7.341314315795898, -10.056666374206543, -9.288925170898438, 6.569300174713135, 6.268343448638916, 10.56378173828125, -10.069061279296875, -19.292070388793945, 6.302676200866699, 4.633218288421631, 12.673114776611328, -8.683499336242676, -0.4778217077255249, 5.173978328704834, -11.84557056427002, -11.30778694152832, 20.166349411010742, 1.0326571464538574, 6.417840957641602, 0.37287500500679016, -13.287164688110352, 12.820137977600098, 13.003951072692871, -14.130593299865723, 2.9010021686553955, -0.8371427655220032, 1.1861090660095215, 9.852839469909668, -10.653196334838867, 2.8137292861938477, 5.456415176391602, 21.03569221496582, 19.105289459228516, -3.981355667114258, -10.491989135742188, -4.2939043045043945, -5.935726165771484, 9.668108940124512, 2.2664973735809326, 15.009566307067871, -2.8958632946014404, -14.265061378479004, 5.515380382537842, -4.578342437744141, 1.7774627208709717, -2.3765807151794434, -11.819987297058105, -5.683982849121094, -12.44067096710205, -4.628050327301025, -2.008523941040039, -11.944559097290039, 19.029014587402344, 3.895218849182129, -1.2116326093673706, -7.369838714599609, -3.783425807952881, -19.471969604492188, -14.950723648071289, -5.617315769195557, -10.404058456420898, -2.439535140991211, -12.991582870483398, 11.621849060058594, -9.297138214111328, -10.867042541503906, -1.263232946395874, -12.885530471801758, 28.957019805908203, -3.284059762954712, -7.791952610015869, 13.23084831237793, -6.997243881225586, 6.1524810791015625, -8.058939933776855, -6.879406929016113, 7.445254325866699, -15.21440601348877, -7.431166172027588, -13.435673713684082, 16.197275161743164, -13.255395889282227, 6.9531707763671875, -2.469480514526367, 7.382339954376221, 6.699392318725586, -1.7051416635513306, -13.227554321289062, 5.235187530517578, 5.665895938873291, -2.070605993270874, -14.098633766174316, 8.4708890914917, 15.76253890991211, 11.647028923034668, -13.162385940551758, 14.485135078430176, -7.082805156707764, 3.0382871627807617, -18.00531578063965, -2.2513911724090576, 11.52286148071289, -4.714211940765381, 12.096809387207031, 1.1847695112228394, 5.794003486633301, 1.0858781337738037, 4.91935396194458, 18.737258911132812, 6.0143609046936035, -26.147775650024414, -5.860440731048584, 14.848730087280273, -1.0953736305236816, -15.176040649414062, -9.130610466003418, 2.9459593296051025, -17.471683502197266, 15.572725296020508, 10.063118934631348, 3.854321241378784, -8.131246566772461, 9.248222351074219, -15.228355407714844, 11.38348388671875, -2.5918636322021484, -6.818697929382324, -0.6241147518157959, 12.724260330200195, 9.792886734008789, 0.6784186363220215, 0.82869553565979, 6.534160614013672, -0.6667795181274414, 6.21321439743042, -10.253499984741211, -5.432565689086914, -6.439741134643555, 4.476500511169434, -6.158198356628418, -7.974462509155273, 4.490280628204346, -5.6733903884887695, 8.8870849609375, 4.424893856048584, 1.0904264450073242, 1.9226051568984985, -7.745990753173828, -18.877595901489258, 1.1346923112869263, -2.6407594680786133, 12.71144962310791, 5.175037384033203, -10.652570724487305, 3.213632583618164, 2.4297032356262207, -14.66147232055664, 9.89086627960205, -1.5747390985488892, 5.830230236053467, 6.051129341125488, 13.210886001586914, 8.946967124938965, 13.347366333007812, 4.838645935058594, 4.888255596160889, -9.620864868164062, 4.27295446395874, -17.52278709411621, -3.059718608856201, -4.57245397567749, 1.0932875871658325, 16.757070541381836, 6.247598648071289, -2.6502761840820312, -0.0037008896470069885, 14.12122917175293, 18.499942779541016, 3.703296661376953, -18.675622940063477, 7.8460187911987305, 14.447626113891602, -8.730788230895996, -11.345161437988281, -7.133874416351318, -9.507718086242676, -7.668140411376953, -9.05294132232666, 6.660221099853516, -16.396501541137695, 10.395964622497559, 9.027122497558594, 8.953901290893555, -1.8809490203857422, -0.6993167400360107, -11.247149467468262, 20.958356857299805, -5.283956050872803, 20.3282470703125, 1.3689607381820679, -14.454550743103027, -11.00876235961914, -0.09710829704999924, -12.97033405303955, -9.157892227172852, -10.655435562133789, 6.6899495124816895, -15.042869567871094, 7.492935657501221, 2.696383476257324, -6.6714887619018555, 3.3649630546569824, -3.63662052154541, 12.257343292236328, 14.292964935302734, 5.052499771118164, 3.4566566944122314, 8.901101112365723, -2.914278745651245, 1.0147545337677002, 3.1038589477539062, 3.5660972595214844, 14.72928237915039, -6.692338466644287, 5.731319427490234, 13.719016075134277, -5.605034351348877, -10.510531425476074, -20.689441680908203, 11.802703857421875, -10.428609848022461, 8.088939666748047, -6.4999680519104, 2.1906769275665283, -16.243722915649414, 5.3911590576171875, -6.161321640014648, -6.861867904663086, -7.280823707580566, -3.4815480709075928, 0.24855336546897888, -10.108848571777344, -1.5762903690338135, 5.11931848526001, 4.053426742553711, -2.0826501846313477, -5.892395973205566, -7.42828893661499, 3.433087110519409, 2.9703686237335205, 6.697387218475342, 0.32009536027908325, 3.4309353828430176, 0.8326014876365662, 2.364116668701172, -11.562736511230469, -0.6779107451438904, 0.9388430118560791, 3.766752243041992, 10.294332504272461, -3.432054042816162, 0.19983352720737457, 4.1667070388793945, -11.541980743408203, 13.188100814819336, -10.302205085754395, 2.908892869949341, 2.6140265464782715, -3.9872539043426514, -5.629047870635986, -9.5423583984375, -7.866105556488037, -15.443066596984863, 6.788440704345703, -13.102705001831055, -3.265019655227661, 3.330775499343872, -5.9621195793151855, 6.614194869995117, 2.7864792346954346, -1.9874464273452759, -7.00593900680542, 10.076298713684082, -18.12851333618164, 10.488798141479492, -0.7308573722839355, 18.57950782775879, -3.912759304046631, 9.0481538772583, 2.2147841453552246, -14.953582763671875, 2.761697292327881, -6.435037612915039, 2.7409145832061768, 6.888983726501465, 10.724488258361816, 3.200124502182007, -13.254341125488281, -10.27931022644043, 5.189248085021973, 1.347722053527832, 9.219205856323242, -8.452770233154297, 0.05116508901119232, -5.857241630554199, 14.917911529541016, -4.3844709396362305, -5.828771591186523, -1.65871262550354, 7.5828704833984375, -13.29546070098877, -11.85051441192627, -1.4123022556304932, 6.1663665771484375, -6.043483734130859, 8.762378692626953, -6.011375904083252, 1.492590308189392, -4.655205726623535, 3.3322925567626953, 5.783210754394531, -0.07825911790132523, 5.387068748474121, -0.008348464034497738, 10.201208114624023, 4.367757797241211, 10.262974739074707, -8.104349136352539, 6.076740264892578, 5.25106143951416, 1.0522024631500244, -6.3823628425598145, 5.106373310089111, 13.41556167602539, 1.1366095542907715, -17.110626220703125, 10.755667686462402, -3.7880146503448486, -11.829028129577637, 6.227256774902344, 4.6657328605651855, 8.356291770935059, -11.759008407592773, 8.12441635131836, 12.50694751739502, -8.964239120483398, -5.297614097595215, -7.451227188110352, 4.4118266105651855, -13.294925689697266, -6.232982158660889, -21.7991943359375, -10.920732498168945, -2.6626858711242676, 5.4235639572143555, 9.949151992797852, -5.943727016448975, 3.227098226547241, -10.772709846496582, 16.37406349182129, 4.624240875244141, 2.9448413848876953, -5.405572891235352, -11.125938415527344, 15.595297813415527, -4.618412494659424, 7.946930408477783, -4.789767742156982, 4.521906852722168, -7.261629104614258, 10.050050735473633, 12.074054718017578, -2.6107401847839355, 12.586955070495605, 1.5070747137069702, 2.2662177085876465, -2.0557873249053955, 18.184206008911133, 6.376239776611328, -2.234055995941162, 1.8501471281051636, 12.074049949645996, -4.381079196929932, 1.2910230159759521, -27.173809051513672, -5.439448833465576, 10.757896423339844, -12.839335441589355, -4.255802154541016, 5.703672885894775, -5.314201354980469, 16.409090042114258, -8.393491744995117, -2.9376139640808105, 1.2623119354248047, -0.2420164793729782, 2.8360698223114014, 12.596900939941406, 2.8180785179138184, -12.557319641113281, -9.76910400390625, 1.5015487670898438, 9.64954948425293, 3.7074408531188965, 4.512907028198242, -7.752623081207275, 15.53782844543457, -3.433617115020752, -11.829644203186035, -2.8692052364349365, 2.0517690181732178, 7.878421783447266, -10.976709365844727, -2.524514675140381, -8.212750434875488, -4.7149858474731445, -0.20421987771987915, -1.2202223539352417, -8.159741401672363, -7.336244583129883, 7.862635612487793, 12.462512016296387, -5.521820068359375, 4.404195785522461, 6.675498008728027, -2.8335607051849365, 5.092628479003906, -2.710963249206543, 9.209212303161621, 2.5668869018554688, -3.6472415924072266, 5.81059455871582, -5.407166481018066, 13.398985862731934, 5.585972785949707, -7.52086877822876, -23.364158630371094, 2.866004467010498, -3.380666971206665, -11.974244117736816, 12.792612075805664, -0.12463133782148361, -9.340361595153809, 2.014042377471924, -6.384166717529297, -11.002605438232422, -25.45370864868164, 1.7740669250488281, 2.1731467247009277, 13.120306015014648, -12.29367446899414, -6.433441638946533, 8.334800720214844, -7.676539421081543, -2.501692295074463, -9.688233375549316, 14.315668106079102, 12.633322715759277, 13.90826416015625, 7.196135997772217, -6.873846530914307, 13.784643173217773, -11.282591819763184, 8.058349609375, 7.0583953857421875, -6.388671398162842, 6.956792831420898, 9.912212371826172, -12.881757736206055, -28.37897491455078, -7.811488628387451, 7.181125164031982, 1.5626643896102905, 13.518877983093262, 4.074535846710205, 12.974262237548828, -5.632011413574219, -8.834456443786621, 6.785252571105957, 10.043231010437012, 0.0033039352856576443, -10.640925407409668, 14.253377914428711, -18.160444259643555, 16.360193252563477, -12.842626571655273, -4.115942001342773, -0.22694188356399536, 10.796494483947754, 16.396276473999023, -18.526247024536133, 9.54664421081543, -5.552239418029785, -2.3219549655914307, 8.466407775878906, -7.770830154418945, -2.381093740463257, 6.2482380867004395, -10.854848861694336, 7.039864540100098, 8.144163131713867, -12.455150604248047, -4.029191017150879, -7.500998497009277, 19.1141300201416, 2.564129590988159, 16.5303955078125, -6.105649471282959, 0.3656824231147766, -6.676331520080566, 5.147610187530518, -11.192717552185059, 10.29680061340332, 6.355655670166016, 9.216400146484375, 5.759181499481201, 4.81990385055542, -14.160414695739746, -5.0502777099609375, -2.2902908325195312, 6.457930088043213, 0.7041633129119873, -7.270036220550537, -14.302650451660156, 7.398636817932129, -5.27440071105957, 6.051587104797363, 12.936803817749023, 1.4942429065704346, -1.5741400718688965, -4.487523555755615, -2.5922279357910156, -8.598125457763672, 10.844770431518555, -9.257831573486328, -6.497715950012207, -3.3042595386505127, -5.219285011291504, -2.5587263107299805, -2.4869046211242676, -10.78583812713623, 2.396697521209717, -5.885205268859863, -6.272377014160156, -5.92172908782959, 6.055477142333984, 8.117198944091797, 9.407144546508789, -23.547163009643555, -9.13901138305664, 13.988269805908203, 4.400203227996826, 3.965522050857544, -4.112865924835205, 8.861056327819824, 7.651218891143799, 1.0079716444015503, 7.773380756378174, -3.14778995513916, 1.814605951309204, 3.0142459869384766, 9.540760040283203, -8.357365608215332, -7.144412994384766, -8.159486770629883, -6.426416397094727, -1.494507908821106, 25.215190887451172, -4.43218994140625, -7.177216529846191, 0.11693567037582397, -13.190372467041016, 0.5181975364685059, 8.060227394104004, -0.6947109699249268, 8.73096752166748, 13.339479446411133, 2.431283950805664, 2.0977327823638916, -1.4063899517059326, 2.9972167015075684, -11.565763473510742, 8.261098861694336, -14.804115295410156, 2.769256830215454, -5.349246978759766, 6.030675888061523, -5.684647560119629, -7.360187530517578, -13.681631088256836, -10.402827262878418, 0.2341850847005844, -15.333403587341309, 30.602848052978516, 1.3950934410095215, -23.637916564941406, -5.027695178985596, -4.1885666847229, -6.701522350311279, -10.588027000427246, 17.944290161132812, 21.01390838623047, 12.095929145812988, -0.26382091641426086, -1.2973214387893677, -0.4446965456008911, -0.3830465078353882, 10.730060577392578, 3.8913753032684326, 21.188610076904297, 0.17749673128128052, 11.275996208190918, -5.895057201385498, -0.9868699908256531, -0.0017727334052324295, -1.380803108215332, 1.2243684530258179, 15.816536903381348, 11.941813468933105, 9.952178955078125, -0.09515783935785294, -9.199287414550781, -0.15725992619991302, 9.187346458435059, -8.836272239685059, -2.8364663124084473, 2.6153364181518555, -14.346464157104492, -1.9536759853363037, -0.27269044518470764, -0.13386473059654236, 23.729326248168945, -2.899029493331909, 14.753255844116211, -12.726987838745117, -7.339310646057129, 23.73253059387207, -5.588964939117432, 25.111064910888672, 8.496505737304688, -3.5005710124969482, 19.15298080444336, -20.606876373291016, 25.611074447631836, -3.668220043182373, -1.808921217918396, 27.569049835205078, -7.51308012008667, -0.13337144255638123, 4.551346302032471, -3.1599063873291016, 10.481891632080078, -9.419085502624512, -8.447464942932129, -13.324638366699219, 7.171289920806885, 8.68912124633789, -7.524474143981934, 14.877041816711426, -5.547845363616943, -8.130892753601074, 2.873464822769165, -1.666593074798584, -20.009952545166016, 7.969302177429199, -1.461971402168274, -1.9805636405944824, 0.8863038420677185, -8.534177780151367, 9.400491714477539, -2.9316015243530273, -13.506041526794434, -10.745645523071289, -16.231700897216797, -11.659351348876953, -10.580954551696777, -5.700279235839844, 13.610100746154785, -3.816361427307129, 7.370155334472656, -15.316658973693848, -3.058229684829712, -3.180816650390625, -4.927059650421143, 1.043287754058838, 12.693873405456543, -1.692996621131897, -9.14232349395752, -3.8652148246765137, 0.809619665145874, -11.005435943603516, -2.9948065280914307, -3.9259934425354004, -8.331369400024414, 5.732501983642578, -12.302084922790527, 12.292886734008789, 2.239757537841797, 6.188460350036621, 10.652413368225098, 12.65876579284668, -7.945094585418701, -8.206217765808105, 3.469310998916626, -7.259803295135498, -3.640641689300537, 0.8180332183837891, 11.48141098022461, -10.898361206054688, 2.5962905883789062, -0.02106398344039917, 17.828580856323242, 16.04446029663086, -16.607288360595703, 10.195486068725586, -3.2716898918151855, -2.357058048248291, 11.438039779663086, -0.49829038977622986, -0.9645332098007202, 15.301241874694824, -12.972554206848145, -14.70400333404541, 9.439440727233887, 5.963988780975342, -16.166614532470703, 9.44138240814209, -3.0892839431762695, 7.606987953186035, -9.17818546295166, 10.464670181274414, 16.468935012817383, 9.053498268127441, -15.520118713378906, 9.066658020019531, 13.72152328491211, 21.99814224243164, -3.052231788635254, 5.783968448638916, -4.522139072418213, 1.7711197137832642, -1.4139479398727417, 5.635440349578857, 3.2429094314575195, 3.0989465713500977, -8.410614013671875, 9.364557266235352, -10.397407531738281, -5.940802574157715, -12.404722213745117, -2.2083160877227783, 22.83037567138672, -3.324516773223877, -1.994921088218689, -4.521082401275635, -3.5913143157958984, -10.878826141357422, -12.068115234375, 4.5105085372924805, -8.84598445892334, 1.8846653699874878, 13.295182228088379, -10.268810272216797, -9.621708869934082, -4.59286642074585, 11.367568016052246, 27.527141571044922, 3.9990150928497314, 16.562971115112305, -1.8010092973709106, 9.148479461669922, 17.1864013671875, 7.207632541656494, -6.435479164123535, -1.2451832294464111, 2.3998937606811523, 5.956692695617676, 10.658548355102539, -19.55050277709961, 0.9089773893356323, 11.221746444702148, 11.733427047729492, 0.3555460274219513, 12.640039443969727, -13.858867645263672, -3.8317809104919434, 2.840945243835449, 3.8338539600372314, -1.2995208501815796, 11.53996467590332, -12.544307708740234, -7.326662540435791, 15.841699600219727, -3.8426806926727295, 4.867345809936523, -1.8200092315673828, 5.940101623535156, -0.6636382341384888, -9.610794067382812, -0.557709276676178, 0.09365297853946686, 2.347700357437134, -2.4854018688201904, -0.9342910051345825, -8.005773544311523, -21.303956985473633, 5.872650146484375, 6.9433088302612305, 7.01051664352417, -18.865036010742188, -11.177871704101562, 2.7727911472320557, -6.622707843780518, 4.143422603607178, 1.9083223342895508, -2.658994436264038, -9.621942520141602, 0.18759430944919586, -13.214021682739258, -3.4883599281311035, -4.451596736907959, -8.520809173583984, 4.2059712409973145, -4.160845756530762, 3.5881080627441406, -23.225128173828125, -3.841517448425293, -15.598154067993164, 10.565699577331543, -23.7001953125, 27.54793357849121, 13.046436309814453, 16.352535247802734, 4.14851713180542, -0.6627842783927917, -12.659773826599121, -7.84694766998291, 8.956202507019043, 11.750215530395508, -8.675663948059082, -12.87830924987793, 2.3095359802246094, -4.0355682373046875, -4.912714004516602, 12.603178977966309, -6.354041576385498, -3.8719983100891113, -0.4146028161048889, -11.936367988586426, -3.255497932434082, -13.26740837097168, 1.1496903896331787, -5.982269287109375, -1.4658535718917847, 6.502589702606201, 4.86667537689209, 10.482070922851562, -17.510683059692383, 16.491865158081055, -1.4318962097167969, -8.417213439941406, 4.320996284484863, 10.8953275680542, 10.022259712219238, -3.640554189682007, -10.171449661254883, 9.150015830993652, 4.275847911834717, 10.618987083435059, 1.3053596019744873, -7.462647438049316, -25.861404418945312, -3.6391100883483887, 2.392630100250244, -5.775684356689453, 7.725353240966797, -10.131287574768066, -6.248257637023926, -2.0232954025268555, -9.182228088378906, -4.566927433013916, -1.1471889019012451, -6.266429901123047, -0.5354687571525574, -4.746481418609619, 8.331442832946777, -9.79343318939209, 13.561697959899902, -6.562288284301758, 18.641361236572266, -11.498100280761719, 14.83263111114502, -11.70902156829834, -16.41164207458496, -6.323460578918457, 5.590924263000488, 0.2518109679222107, -0.7721091508865356, -1.0771400928497314, -15.663174629211426, 13.92222785949707, 5.17991828918457, 3.4306235313415527, 18.705238342285156, 2.9922423362731934, -9.190250396728516, 3.8569393157958984, -1.0113921165466309, -14.90487289428711, -9.307550430297852, 11.021480560302734, -11.007955551147461, 3.431074619293213, -6.5743584632873535, -7.270334243774414, 1.8442184925079346, -13.343981742858887, 7.218015193939209, -12.973737716674805, 7.0582098960876465, 2.5342845916748047, 16.563255310058594, 19.551279067993164, -16.917123794555664, 12.846760749816895, 15.840581893920898, 17.84305191040039, 3.488614559173584, 5.628608703613281, -1.128175139427185, 9.035377502441406, -3.4782896041870117, 8.561500549316406, 8.685783386230469, 18.97132110595703, 5.057687759399414, -3.5375962257385254, 13.171187400817871, -5.488329887390137, 2.848202705383301, 7.684545040130615, -15.880688667297363, -20.671157836914062, 10.42664909362793, 2.1475229263305664, 5.829967498779297, 2.676100730895996, 11.28636360168457, 2.7034640312194824, -7.317790985107422, 5.976769924163818, -0.5515552163124084, -2.0642290115356445, -3.1290316581726074, -8.630524635314941, 3.547642707824707, 10.664554595947266, 14.224672317504883, -3.0049681663513184, -16.262592315673828, 10.567055702209473, 5.431367874145508, 8.964272499084473, 15.77823257446289, -27.074859619140625, 10.23702621459961, -13.665226936340332, 15.453324317932129, 13.449651718139648, -11.45455265045166, -2.903796434402466, 14.729368209838867, 16.62659454345703, -1.8548462390899658, 1.4353328943252563, -10.37803840637207, 7.624520301818848, 1.8157963752746582, -0.4132020175457001, 3.6297903060913086, -15.99657154083252, 7.8153300285339355, -7.308117866516113, -7.409032821655273, -3.0068752765655518, 2.6258134841918945, -10.476072311401367, 9.428444862365723, 9.617103576660156, -5.232646465301514, -18.68837547302246, 13.294830322265625, 12.14232063293457, 11.562566757202148, -1.4685404300689697, 10.874431610107422, -0.507082462310791, 4.2302680015563965, 0.04045465961098671, 3.0134382247924805, 22.68356704711914, 15.684462547302246, -10.917176246643066, 7.707018852233887, -6.017586708068848, -11.198588371276855, 8.367668151855469, 3.8372421264648438, -0.18461892008781433, 14.30855941772461, -2.4938361644744873, -5.175338268280029, -14.30609130859375, 0.8380270600318909, -7.419008255004883, 4.120824813842773, 0.39470914006233215, 7.686592102050781, 4.852602481842041, -3.8960835933685303, 3.5453808307647705, 2.3842811584472656, 15.92208480834961, 2.0362462997436523, -13.524441719055176, 1.533738613128662, 9.20970344543457, 12.254281044006348, -11.82696533203125, -7.912999153137207, -1.134782314300537, 14.501055717468262, -3.7915573120117188, -6.179825782775879, -4.647496223449707, -10.486727714538574, 0.897026777267456, 0.7764548063278198, -8.913505554199219, -5.813508033752441, -1.3788182735443115, -8.486516952514648, 17.34438133239746, -2.0454583168029785, -8.561569213867188, -7.9048309326171875, 6.562877655029297, 3.777010202407837, 16.162921905517578, -4.201280117034912, -18.46259117126465, -8.302806854248047, 2.714581251144409, -11.706485748291016, 10.929288864135742, -9.404855728149414, -7.491673946380615, -7.216595649719238, 3.382183074951172, 4.40201997756958, 0.21252226829528809, -5.892404556274414, -11.702204704284668, -1.4208617210388184, -7.602672576904297, 8.355486869812012, -10.756217002868652, 4.809374809265137, 8.3826322555542, 7.212695121765137, 9.719301223754883, -15.850214958190918, 6.957374095916748, -12.3084716796875, -17.422758102416992, -11.668648719787598, -4.9056315422058105, 10.994441986083984, 11.875707626342773, -6.139909744262695, -6.839209079742432, 0.8771461844444275, 7.692431449890137, 2.922893524169922, 12.27684497833252, -10.097187042236328, -6.810534477233887, 0.4161913990974426, -4.271268367767334, -5.482668876647949, -7.819633960723877, -15.053035736083984, -5.255098819732666, 10.100563049316406, 10.04369831085205, -2.332775115966797, 17.572153091430664, -6.02311897277832, -1.836269497871399, -21.756168365478516, 6.451735496520996, -15.070724487304688, -15.471964836120605, -4.664620399475098, -6.547154903411865, 6.53541374206543, 0.9996711611747742, -2.6454427242279053, -10.413122177124023, 11.331270217895508, -12.07890510559082, -21.378173828125, 17.13127899169922, -5.016241550445557, 16.53133201599121, 5.083913326263428, -12.435697555541992, -5.9862213134765625, 9.319597244262695, -0.6252362728118896, -2.6873552799224854, -19.495811462402344, -3.58357310295105, 19.324235916137695, -8.936177253723145, 0.2805079519748688, 12.913959503173828, 2.9819579124450684, -11.641324996948242, 2.1562087535858154, 12.546562194824219, -14.60686206817627, 22.092517852783203, 2.4713704586029053, 20.11286735534668, 3.21280574798584, 1.7523164749145508, 2.0107266902923584, 5.109445571899414, -8.403755187988281, -0.7641980051994324, 14.677946090698242, -11.78778076171875, -13.778026580810547, 7.13674783706665, 16.308265686035156, -8.828688621520996, -2.8225769996643066, -12.634393692016602, -0.45809271931648254, 2.677591323852539, 0.15853700041770935, -13.12608528137207, -3.2289037704467773, -1.937684416770935, -6.97277307510376, 2.8419981002807617, -2.0715389251708984, -11.181435585021973, 6.971318244934082, 3.947321891784668, 3.4153785705566406, 4.802088260650635, -19.293712615966797, 1.0906758308410645, 0.14886857569217682, -15.886541366577148, -13.716089248657227, 13.256914138793945, 4.383979320526123, 3.6090078353881836, -4.602721214294434, 8.152355194091797, -1.6545603275299072, -8.259160041809082, 2.0964529514312744, -0.1685906946659088, -4.689765453338623, 8.31479263305664, 9.805686950683594, 5.040677070617676, 0.9781474471092224, -7.0878071784973145, 2.3125877380371094, 4.079182147979736, 9.490036010742188, 21.720849990844727, -12.496075630187988, -4.0096259117126465, 1.0680453777313232, 2.0362467765808105, -14.953452110290527, 16.481534957885742, 0.9252963066101074, 11.322654724121094, 1.2530277967453003, -1.7950342893600464, 5.522134780883789, -4.522280693054199, -0.6715483665466309, 8.977848052978516, 7.236791133880615, -9.195640563964844, -0.1540042757987976, -12.850398063659668, 11.06360912322998, 8.89842700958252, 8.435556411743164, 3.578735589981079, -1.203677773475647, 0.9680169224739075, 10.110528945922852, -3.482988119125366, -19.134891510009766, 2.6398303508758545, 4.571681976318359, 20.172866821289062, -14.85329532623291, 10.48786735534668, 3.210573434829712, 5.320713043212891, 8.953161239624023, -9.035760879516602, -5.507106781005859, 3.3883702754974365, -23.344812393188477, 11.381624221801758, -5.953119277954102, -4.070653438568115, 3.5513086318969727, 8.436676025390625, 15.393134117126465, -2.4491543769836426, 3.458047389984131, -5.530407905578613, 17.9399356842041, -9.043499946594238, -16.92378807067871, -2.3277435302734375, -9.348085403442383, 5.865544319152832, 11.033050537109375, 3.960028648376465, -1.8388381004333496, -5.894017219543457, -0.48862430453300476, 1.644439935684204, -14.99333381652832, -12.886760711669922, 9.336272239685059, -2.0160531997680664, -5.766026496887207, 8.305167198181152, 6.269614219665527, 4.928412914276123, -3.504908561706543, -12.15872859954834, -15.504212379455566, -2.1197798252105713, -8.527556419372559, 0.23420335352420807, 2.3447751998901367, -2.857170581817627, 4.947762966156006, -3.110961437225342, 4.97514009475708, -18.058563232421875, -13.100973129272461, 4.627511501312256, 10.427518844604492, 14.285499572753906, 0.35364773869514465, -1.7208168506622314, -12.432723045349121, -14.314613342285156, -1.7878515720367432, -4.317419052124023, -6.469576835632324, -0.4239126443862915, -6.511511325836182, -22.68630027770996, 1.1117876768112183, 0.6479016542434692, -7.106148719787598, 0.6967548131942749, -4.780640602111816, -4.532137393951416, 11.993188858032227, 11.126440048217773, 13.801103591918945, -8.822548866271973, 18.125797271728516, -12.55211353302002, -2.9389162063598633, 2.5201807022094727, 5.323200225830078, 18.00833511352539, -0.8890828490257263, -4.365702152252197, -11.194089889526367, -5.1078691482543945, -10.856878280639648, 11.39010238647461, -0.7725321650505066, 2.214920997619629, -22.77197265625, -6.432189464569092, -1.2511262893676758, 20.767269134521484, 9.055526733398438, 12.284293174743652, -16.879119873046875, 2.8023977279663086, -8.163553237915039, 4.723028182983398, 2.056180715560913, 4.500457286834717, -17.15236473083496, -2.397627353668213, -19.286659240722656, 1.1811546087265015, -4.889009475708008, -4.8524489402771, -16.658870697021484, 5.072803020477295, 3.4120540618896484, -5.560189247131348, -8.628816604614258, -11.93687629699707, -12.001238822937012, -8.731826782226562, -6.308229446411133, -4.475210666656494, -3.5764827728271484, 9.834428787231445, 9.12978744506836, 0.18737444281578064, 0.9102120399475098, 3.4955105781555176, -4.351986885070801, 0.3190455138683319, 5.987642288208008, -22.798215866088867, -13.745135307312012, 12.185033798217773, 0.6972288489341736, -2.1442325115203857, -2.5388219356536865, 1.528963327407837, 1.9687639474868774, -2.0879645347595215, 9.674545288085938, -13.879453659057617, -4.898250579833984, 6.582065582275391, -6.000685214996338, -0.3794443905353546, 4.847273349761963, 0.20132926106452942, -5.721475601196289, 18.16411590576172, 0.8776562809944153, -11.738947868347168, 6.847903251647949, 17.446277618408203, -11.35928726196289, -8.96202278137207, -28.357629776000977, 6.597336769104004, 26.020137786865234, 3.804766893386841, -5.618119239807129, 10.996655464172363, 7.392180442810059, -2.9613709449768066, 5.922022819519043, 10.146232604980469, -15.839797973632812, 2.911393880844116, 3.587770938873291, -3.0061416625976562, -1.4400722980499268, -19.023317337036133, 6.806421279907227, 6.222552299499512, 9.556755065917969, 13.296679496765137, 9.42325496673584, 0.3564581871032715, -19.69027328491211, 7.686400413513184, 5.501127243041992, -9.172218322753906, 13.30142879486084, -6.20988130569458, 9.960081100463867, 17.795013427734375, 16.366758346557617, 4.75213098526001, 11.67849063873291, 7.1167402267456055, -16.328155517578125, -6.062961578369141, 10.093925476074219, -9.841272354125977, 0.3756447434425354, 26.863872528076172, -2.6782426834106445, 3.179018974304199, 9.579219818115234, -4.268491744995117, -0.49970582127571106, -4.654869556427002, -4.431968688964844, 8.227158546447754, 2.6651504039764404, -11.195406913757324, 2.828465461730957, 2.177302837371826, 1.9573426246643066, -7.910747528076172, 5.689929008483887], [-1.5570515394210815, 10.247611045837402, 6.234637260437012, 13.32761287689209, -8.334504127502441, 8.006502151489258, 0.9127441048622131, 1.1039609909057617, 0.16007515788078308, -11.409192085266113, -6.294216632843018, -19.799306869506836, 13.591755867004395, 4.104052543640137, -3.6292409896850586, -5.467777252197266, -8.412247657775879, -3.560706377029419, -9.961484909057617, 22.596332550048828, 2.2257068157196045, 25.519432067871094, 2.248281955718994, -5.486349105834961, 2.2948696613311768, 8.796969413757324, 10.007661819458008, -7.582275390625, 1.412090539932251, 5.917619705200195, -4.659411907196045, 2.1568617820739746, 4.242614269256592, -0.934669017791748, 10.744197845458984, -18.47385597229004, -3.4720423221588135, 12.98499584197998, -2.356905937194824, -2.830871105194092, -13.859893798828125, 3.298410177230835, -10.724715232849121, -0.2968103289604187, 7.814374923706055, 6.690922260284424, -0.9705352187156677, -5.6018548011779785, -4.014357566833496, 9.741827011108398, 3.889652729034424, -12.813426971435547, -18.295486450195312, 2.8400139808654785, 1.1793783903121948, -13.527664184570312, 1.0670613050460815, 10.863537788391113, -2.196463108062744, -4.740561008453369, 3.732301950454712, 2.105055093765259, -5.793440341949463, -6.389945983886719, -6.088849067687988, -0.7635604739189148, 18.312110900878906, 14.313902854919434, -4.766270637512207, -12.982550621032715, -3.2646665573120117, 2.236811876296997, -4.653090476989746, -0.4109678864479065, 1.1241846084594727, -6.694433212280273, -13.002257347106934, 8.295480728149414, 13.434518814086914, -6.579213619232178, 6.487399101257324, 3.642127752304077, 13.661266326904297, 10.251616477966309, 5.675617218017578, -1.0857475996017456, 22.71824073791504, 12.455967903137207, 8.202756881713867, 2.154665946960449, -0.5472184419631958, -8.44866943359375, -1.4692912101745605, 10.485176086425781, -19.070846557617188, 2.6043999195098877, -19.07456398010254, -12.574417114257812, 11.360492706298828, 27.174072265625, -4.576750755310059, 4.758901596069336, 5.656223297119141, -13.487709045410156, -7.3958353996276855, -6.688651084899902, 4.520654678344727, 4.748223304748535, -0.37289273738861084, 0.21332424879074097, 7.9972357749938965, -6.807405948638916, 2.9134104251861572, 9.566826820373535, 3.6289830207824707, -6.908319473266602, -0.9085553288459778, -7.2156829833984375, -1.838072419166565, 10.993193626403809, 9.462959289550781, -6.531242847442627, 17.169002532958984, 9.032279014587402, 9.101783752441406, -0.054535236209630966, 7.480863571166992, -13.561921119689941, -7.199378967285156, -4.039402961730957, -4.5357136726379395, -15.771589279174805, -4.051754951477051, -5.767797470092773, 3.3924224376678467, 13.841877937316895, 20.659366607666016, 8.332508087158203, -1.0840481519699097, 12.80915641784668, -21.90227699279785, 19.536434173583984, 4.429619312286377, -16.05243682861328, -11.409789085388184, -7.484249114990234, -1.3927814960479736, 5.5007429122924805, -3.6278297901153564, -6.094750881195068, 9.771364212036133, -3.492501735687256, -10.328511238098145, -19.236387252807617, 6.543845176696777, 15.388984680175781, 2.439082145690918, -11.762479782104492, -19.86294174194336, 10.944877624511719, -0.6523929834365845, 9.616044998168945, 7.775025844573975, -14.704695701599121, -12.214012145996094, -3.923201560974121, 3.513043165206909, -11.803155899047852, 2.8315541744232178, 4.692515850067139, 12.481916427612305, -7.785082817077637, -2.261119842529297, -5.757899761199951, -10.63451862335205, 18.434560775756836, 0.29878053069114685, 9.808026313781738, -21.659517288208008, -17.112102508544922, 4.087463855743408, -2.7247226238250732, 5.7285542488098145, 1.6015136241912842, 8.900745391845703, 19.292001724243164, 23.959819793701172, 1.4263159036636353, -13.218607902526855, -8.03853702545166, -2.756054639816284, 0.8657788634300232, 6.315117835998535, -14.27328872680664, -8.937955856323242, -4.012940883636475, 0.8851664066314697, 10.769372940063477, 7.183191776275635, 7.429222106933594, 24.691532135009766, 4.686121940612793, 0.3580271005630493, 4.236176490783691, 13.581550598144531, -1.1152575016021729, 5.103590965270996, 2.1164615154266357, -0.3535141348838806, 10.21086597442627, -0.2528417706489563, 16.053903579711914, -17.61107635498047, 5.321485996246338, 15.212315559387207, 1.920422911643982, 16.662446975708008, -9.863598823547363, 16.209148406982422, -11.610283851623535, -1.4863214492797852, 0.01930922642350197, 12.710772514343262, 10.932550430297852, -0.9334975481033325, -6.490838527679443, 4.036008358001709, 4.622228145599365, -6.923966884613037, 9.380637168884277, 2.019639492034912, 11.96240234375, -7.213647842407227, 6.316905498504639, 19.339529037475586, 6.726141452789307, 12.313651084899902, -6.533376693725586, 3.3063266277313232, -7.371849060058594, 0.5757733583450317, 6.502015590667725, 1.7436325550079346, -12.121907234191895, 0.932205319404602, -18.740760803222656, -0.5425388813018799, -4.257585048675537, 4.0408806800842285, 1.9592314958572388, 3.6500155925750732, -6.894430160522461, -9.532609939575195, 5.136747360229492, 2.3701698780059814, 7.231845855712891, 1.1067285537719727, -12.744893074035645, 9.653708457946777, 16.567180633544922, 7.889715194702148, 1.1537212133407593, -13.606600761413574, -6.525821208953857, 10.694976806640625, 8.113636016845703, -0.06781639903783798, 2.754239082336426, -10.53829288482666, -1.3344593048095703, -5.777509689331055, -0.01792815700173378, -7.451327323913574, 3.9804320335388184, 12.33254337310791, 12.964128494262695, 1.6093908548355103, -5.019045829772949, -3.034700393676758, -11.927529335021973, 10.414652824401855, 6.948945045471191, -10.902904510498047, -2.3881635665893555, 3.3501412868499756, -0.5324124693870544, -11.379371643066406, 6.748185634613037, 9.485970497131348, -2.269969940185547, 4.844837665557861, 1.5700702667236328, -8.873201370239258, -4.125400066375732, 7.4377946853637695, 7.939119815826416, 1.0129530429840088, 9.209598541259766, 16.775590896606445, 3.4950485229492188, -6.667115688323975, -5.729479789733887, 3.1297574043273926, 5.189551830291748, 11.011594772338867, -13.212546348571777, -11.855390548706055, -4.874554634094238, -13.478636741638184, -4.4644389152526855, -5.726984024047852, 3.221045732498169, -5.2739362716674805, 12.520180702209473, -7.803399085998535, -3.8843493461608887, -3.09706449508667, 6.818477630615234, 2.2243635654449463, -0.4424351155757904, 0.4921966791152954, -6.66765022277832, -14.871264457702637, 1.0250041484832764, -2.9151437282562256, -5.0345988273620605, 5.026002883911133, 8.193970680236816, -4.87269401550293, -12.54747200012207, 3.769369125366211, -7.3374505043029785, -2.263333320617676, 9.025155067443848, 11.66690731048584, 11.22859001159668, 2.1383216381073, 0.8206658959388733, 2.099348783493042, -6.358902454376221, -13.481431007385254, -1.5574365854263306, 24.451251983642578, 3.2031173706054688, -0.43308302760124207, 0.8977428078651428, 11.996379852294922, -8.161391258239746, 16.629108428955078, -9.599187850952148, 12.42491626739502, 22.29680633544922, 9.63849925994873, 14.671049118041992, -7.10385274887085, 1.001474142074585, -10.331930160522461, 4.7974534034729, 23.071781158447266, 3.7861528396606445, -9.857477188110352, -12.166851997375488, 11.752289772033691, -2.5375499725341797, -0.7627036571502686, -15.129728317260742, 15.618000030517578, 18.445283889770508, 14.237107276916504, 6.188841819763184, -13.453914642333984, 7.543251037597656, 11.632208824157715, 3.6259536743164062, -1.450235366821289, 20.436803817749023, -5.774856090545654, 0.07322277128696442, 2.8211560249328613, -7.156054496765137, -2.8193726539611816, 5.988289833068848, 3.92349910736084, 9.035545349121094, 4.250766754150391, 5.814304351806641, 3.7898755073547363, -3.65203857421875, -2.342273473739624, 14.539898872375488, -7.307090759277344, 3.5790109634399414, -8.065701484680176, -4.666288375854492, 2.6791346073150635, 2.511805534362793, 5.791587829589844, 9.779346466064453, -8.572009086608887, -0.08385966718196869, -13.81117057800293, -8.184414863586426, 15.410832405090332, 1.9249496459960938, -3.6090409755706787, -4.453309059143066, -6.8923797607421875, -8.135924339294434, 17.796092987060547, 16.779151916503906, 9.289261817932129, -1.8974500894546509, 15.40291976928711, -17.885658264160156, 1.9151852130889893, -8.28795337677002, -0.7336353659629822, -7.1515727043151855, 11.560803413391113, -0.26685988903045654, 17.919233322143555, -0.8564145565032959, 1.4581873416900635, 0.10216599702835083, 6.485304355621338, -6.6183762550354, -0.49853456020355225, 5.140064239501953, -0.09556856751441956, 3.5173795223236084, 5.608736991882324, -12.772425651550293, -4.724244117736816, 17.264211654663086, -16.37834358215332, -12.477039337158203, -4.025820255279541, 1.2782806158065796, -0.8936571478843689, -14.645546913146973, 6.428909778594971, 1.9991637468338013, -5.837677001953125, -10.925864219665527, -16.293272018432617, -10.78323745727539, -6.5720601081848145, 10.726637840270996, 2.3069114685058594, 2.214026689529419, -3.245800018310547, 7.395374774932861, -4.134344577789307, 9.167471885681152, 20.457408905029297, 1.8009967803955078, 5.2124834060668945, 0.4172740578651428, 6.496525764465332, 10.702468872070312, 8.191221237182617, 3.277071714401245, 4.060335159301758, 3.432096481323242, -0.8922300338745117, -1.5167264938354492, -17.503719329833984, -1.310887098312378, -9.964286804199219, -5.327361106872559, -0.4183787405490875, 8.86395263671875, 5.203848361968994, -13.231012344360352, -5.953932285308838, 3.7022628784179688, 10.859461784362793, 8.102493286132812, -0.12835478782653809, -5.7219343185424805, 2.603396415710449, -0.7323337197303772, -4.439807415008545, 11.771310806274414, 5.181750774383545, -20.107643127441406, 9.227811813354492, 6.205904960632324, -4.540929317474365, -18.577733993530273, -15.864692687988281, -18.414722442626953, 0.7156385183334351, 12.603843688964844, 2.5333869457244873, 5.217159271240234, 4.0251359939575195, 10.799362182617188, -3.2479355335235596, -1.2451984882354736, 5.195789813995361, 4.350480079650879, -6.073065757751465, 1.516100287437439, -5.0399394035339355, 14.05856990814209, -1.3311197757720947, 6.826682090759277, 1.3472272157669067, 3.4922969341278076, -17.047574996948242, 18.8532772064209, 0.3873955011367798, -16.9042911529541, -6.3265581130981445, -11.470203399658203, -6.1188812255859375, 3.9128050804138184, -2.165506362915039, 3.5821950435638428, 3.5689516067504883, -12.379569053649902, 2.9460086822509766, -3.553016424179077, 0.3173219859600067, 10.749303817749023, 8.535713195800781, 4.547972679138184, 13.124484062194824, 9.6776762008667, 18.37519645690918, 8.743173599243164, -5.466546058654785, 3.587675094604492, 4.418910980224609, -4.768857955932617, 10.956534385681152, -4.085705757141113, 4.776144504547119, 1.098486065864563, 8.884562492370605, 19.329647064208984, -19.052820205688477, -19.316877365112305, 17.634679794311523, -15.86854076385498, -18.38258934020996, -6.492053985595703, 9.622693061828613, -1.3330168724060059, 14.158530235290527, 10.549192428588867, -2.066438674926758, 0.2528432607650757, 0.8293420076370239, -14.909831047058105, -15.848997116088867, -8.035104751586914, 1.1706253290176392, 9.303342819213867, 6.762763023376465, 6.304868698120117, -5.334875106811523, -0.9512433409690857, 3.0123279094696045, -4.730406761169434, 12.341106414794922, 10.658676147460938, -0.48438844084739685, 12.505186080932617, -1.8735194206237793, 6.674376487731934, -4.849168300628662, -0.4321519136428833, 18.36842918395996, 8.752009391784668, -2.578533887863159, -20.106924057006836, -12.199462890625, 14.448758125305176, 14.061979293823242, 18.353775024414062, 3.993781805038452, -7.765805721282959, -4.148950099945068, -6.6933698654174805, -0.5806843042373657, -3.108773708343506, 11.64521598815918, 10.663749694824219, -13.973794937133789, -1.49583101272583, 2.7846267223358154, 2.8465616703033447, 2.829155683517456, -7.142258167266846, -10.974249839782715, -5.561891555786133, -3.740830659866333, 16.02202033996582, 11.535839080810547, -7.2488203048706055, 10.171070098876953, 1.2140917778015137, -6.485311508178711, 1.4785079956054688, 7.93400239944458, 21.00243377685547, -9.575321197509766, -7.2695817947387695, 4.25881290435791, 12.293224334716797, -1.665939211845398, 6.980995178222656, -1.4673326015472412, 4.714751720428467, -2.631500244140625, -7.9175286293029785, -3.701070785522461, 1.9717897176742554, 4.466952323913574, -7.043442726135254, 5.209737777709961, 2.048809051513672, -9.52459716796875, 6.571106433868408, -1.91221284866333, -18.00705909729004, 2.1426663398742676, -13.731709480285645, -7.5193071365356445, 9.865489959716797, 2.5875213146209717, -15.348030090332031, -3.3402962684631348, -4.0500946044921875, 9.571464538574219, 6.562524795532227, -6.6734161376953125, -12.836915016174316, 20.598739624023438, -8.644988059997559, -1.1945159435272217, -8.760823249816895, 2.718174457550049, -12.663562774658203, 10.322613716125488, -5.119381904602051, -1.314560890197754, 0.6216232776641846, 3.409517288208008, -3.1169443130493164, 2.571878433227539, -3.018900156021118, -0.4485352039337158, -6.352767467498779, 11.6029052734375, 19.61688804626465, 17.57331657409668, 16.227142333984375, -1.2804248332977295, -4.569453716278076, -23.717344284057617, 2.7750234603881836, -4.299256324768066, -8.238121032714844, 12.089599609375, 7.16287899017334, 9.860047340393066, 2.1309428215026855, 1.760838508605957, -7.492068290710449, -5.948809623718262, 6.094236850738525, 1.6697230339050293, 7.446930408477783, -0.20986077189445496, -1.6544528007507324, 3.1363883018493652, -8.441448211669922, 17.715417861938477, 12.328857421875, -8.949939727783203, 2.4609375, 7.688133239746094, 6.7661848068237305, 6.702408313751221, -1.1942847967147827, 8.707159996032715, 7.0857133865356445, 4.386358261108398, 16.419418334960938, 8.932016372680664, 7.861902236938477, 0.4545394778251648, -3.4166417121887207, -10.301183700561523, 0.8096280694007874, -7.517486572265625, -8.674633979797363, 3.935105323791504, 6.985185146331787, -6.899460792541504, -5.476073741912842, 11.500572204589844, 8.543163299560547, 6.740917682647705, -15.053838729858398, 3.9109573364257812, -11.514564514160156, 22.502361297607422, -0.928659200668335, -9.262066841125488, -0.9041277170181274, 13.199575424194336, -5.98452091217041, -2.7394235134124756, -10.95669174194336, -25.961530685424805, 4.4205708503723145, -9.63798713684082, 1.4296677112579346, -11.155721664428711, 1.0521949529647827, 9.44163703918457, 1.6057243347167969, -3.4531455039978027, 7.948846340179443, -21.935211181640625, 9.860188484191895, -7.423752784729004, 6.800940036773682, 9.916889190673828, 2.515108823776245, 5.561459541320801, -17.113096237182617, 16.180713653564453, -9.99167251586914, 15.372075080871582, 0.7203818559646606, -7.735609531402588, -6.9246978759765625, -15.401898384094238, -6.222339153289795, 4.444301605224609, 3.438384771347046, -2.994290590286255, 6.377737045288086, -2.8336286544799805, -7.764368534088135, 26.64697265625, -4.412545680999756, -6.88778829574585, -3.5471739768981934, -3.7201132774353027, -0.2855340540409088, 8.701687812805176, 28.394880294799805, -3.9724385738372803, -7.036979675292969, -8.327300071716309, 9.558137893676758, 1.760998249053955, -2.6577205657958984, 0.4021311104297638, -1.5908997058868408, 4.447131633758545, 9.168069839477539, -10.176025390625, 1.9969048500061035, -11.949752807617188, -25.526081085205078, -2.3658838272094727, -7.0445146560668945, -1.9710760116577148, 6.636914253234863, 0.523703932762146, -4.308114528656006, 5.635831832885742, 7.811967849731445, 17.848621368408203, 0.6063971519470215, 4.52910852432251, -3.4672536849975586, 9.968138694763184, -3.938432216644287, 11.890350341796875, 11.934112548828125, -6.778613090515137, -5.053971290588379, 20.978538513183594, -8.287678718566895, 6.866405487060547, -14.455489158630371, 11.148431777954102, -11.384910583496094, -17.97834587097168, -16.399131774902344, -29.889509201049805, -9.044185638427734, 6.891707897186279, 8.4900541305542, 4.567270278930664, -17.54828453063965, 4.333941459655762, 9.979500770568848, 0.18804503977298737, -9.55400276184082, 23.451171875, -3.9425692558288574, 2.6146128177642822, 19.140804290771484, -15.601211547851562, 1.9812066555023193, -7.685820579528809, 0.9227229356765747, 11.201504707336426, 12.049731254577637, -5.363144874572754, 0.808885395526886, 12.227235794067383, -6.6224365234375, -3.0693304538726807, -21.87567710876465, -3.3075342178344727, 3.6915602684020996, -13.773337364196777, -4.270579814910889, 14.487560272216797, -6.792121887207031, -0.9291752576828003, 8.344704627990723, 6.084312915802002, -10.20306396484375, -5.9168901443481445, -0.20711927115917206, 5.742763996124268, 13.616006851196289, 18.26113510131836, -4.6464409828186035, 8.337912559509277, -2.3509809970855713, 7.5791473388671875, -17.233261108398438, -10.845697402954102, -9.352888107299805, 8.281522750854492, -5.015639305114746, -4.146351337432861, -11.020584106445312, 14.511116981506348, -9.29894733428955, -1.2121367454528809, 13.24506950378418, 11.919479370117188, -7.0552873611450195, -2.0550425052642822, 8.302648544311523, -5.705574035644531, -11.304271697998047, -7.9995622634887695, -17.62613868713379, 3.174262046813965, 9.575126647949219, -2.3436341285705566, -9.807470321655273, -4.224640369415283, -7.6355671882629395, 1.6872661113739014, -1.4163503646850586, 2.315369129180908, -7.049104690551758, -20.65317153930664, -6.5938849449157715, 12.110540390014648, -8.292793273925781, 21.02216911315918, -0.2763599157333374, 4.41758394241333, -0.46214890480041504, 17.86444854736328, 20.634010314941406, -10.943742752075195, -9.231975555419922, 14.992145538330078, 8.093756675720215, -5.434122085571289, 5.338519096374512, 3.608041286468506, -8.205265045166016, 6.431187629699707, -4.993317604064941, -20.534465789794922, -17.72709083557129, -13.68248176574707, -7.4072675704956055, 8.510246276855469, 3.4569895267486572, -15.364089965820312, 10.404451370239258, -9.536216735839844, -1.0741612911224365, -3.138334274291992, 11.48062801361084, 1.0337578058242798, -3.655834197998047, -18.992826461791992, -5.943515777587891, 1.0677745342254639, -16.635292053222656, 1.1063971519470215, -4.792111396789551, -18.584999084472656, 18.171024322509766, 6.956689357757568, 9.214674949645996, 4.954030990600586, -4.041839599609375, 14.019564628601074, 10.019869804382324, -0.17269405722618103, -25.8511905670166, 1.1030759811401367, 18.559167861938477, -10.522529602050781, -12.79526138305664, 30.33404541015625, -1.9151790142059326, 9.407865524291992, 12.156563758850098, -11.558152198791504, 17.809795379638672, -3.0774641036987305, -1.8475618362426758, -3.3347578048706055, 8.756535530090332, 15.660577774047852, -5.010071277618408, -1.0248119831085205, 19.016260147094727, -10.388181686401367, -14.182184219360352, -1.9690015316009521, -5.964105606079102, 2.0835251808166504, 5.36163854598999, 9.154470443725586, -6.561666011810303, -11.750164985656738, -20.991641998291016, -15.711792945861816, -0.634416937828064, -4.922974109649658, 2.437436103820801, 19.080883026123047, 6.942432403564453, 0.3926120400428772, 17.787147521972656, 12.266698837280273, -13.685673713684082, -2.0071799755096436, -2.759941577911377, -12.500970840454102, 20.586238861083984, -0.5679028630256653, 5.155318260192871, 5.944942474365234, -4.640550136566162, -4.333547115325928, -8.19568920135498, -3.9921956062316895, 1.98763108253479, 7.519094944000244, 3.8684308528900146, -4.685698509216309, 21.315780639648438, 3.1983602046966553, -9.152872085571289, -15.443023681640625, 14.068022727966309, 1.2087570428848267, -6.967838287353516, -6.578632354736328, -14.660346984863281, -13.140279769897461, -3.092622756958008, -13.223442077636719, 3.0455100536346436, -0.5391894578933716, 9.989925384521484, 9.863945007324219, -12.025236129760742, -6.849321365356445, 10.248717308044434, 1.5988037586212158, -11.278409957885742, 8.602266311645508, 11.31950569152832, -6.035599708557129, -3.8185434341430664, 2.5200204849243164, -3.5729386806488037, 5.170258522033691, 3.6570045948028564, -11.349662780761719, -4.665802955627441, -0.7308334708213806, 10.204242706298828, -12.073457717895508, 20.953197479248047, 2.5728578567504883, -1.2538598775863647, -3.589783191680908, 6.600778102874756, -3.1590285301208496, 16.978960037231445, -9.97192096710205, 25.210512161254883, 9.584380149841309, -13.17475414276123, -7.836079120635986, -0.7809481024742126, -10.504558563232422, 7.275296211242676, 1.001778483390808, 0.3722742795944214, -4.07960844039917, 15.848724365234375, -7.312317371368408, 14.212082862854004, -6.761513710021973, -13.04013729095459, 9.353343963623047, -8.170853614807129, 6.259792327880859, 20.894250869750977, -2.3533811569213867, -2.882556438446045, -6.341187953948975, -9.269926071166992, 1.8163979053497314, 6.9828691482543945, -5.200912952423096, 6.742836952209473, 17.91153335571289, -3.584245204925537, 9.831107139587402, 16.357433319091797, 6.21087646484375, -4.742796897888184, -1.448104977607727, 10.44893741607666, 3.735078811645508, -6.48026704788208, 1.5435653924942017, 3.0206127166748047, -12.844606399536133, -1.0837937593460083, 4.0044264793396, -2.1541082859039307, -7.179477691650391, -3.7202346324920654, -5.926120281219482, -3.544851779937744, 7.20853328704834, 13.240089416503906, 5.426324844360352, -10.418539047241211, -5.672665119171143, 8.40389347076416, 0.41917455196380615, -1.2078179121017456, 0.6882808208465576, -8.823711395263672, 18.419553756713867, 7.667428016662598, 0.40438565611839294, -3.0321707725524902, -11.911063194274902, -12.829503059387207, -9.031466484069824, -3.6204380989074707, -0.11523266136646271, 14.566737174987793, 13.488910675048828, -14.209761619567871, 0.5023844838142395, -5.792674541473389, -2.538262367248535, -24.291866302490234, -8.153936386108398, -4.525959014892578, -10.279542922973633, -3.328735828399658, 1.1819039583206177, 0.665647029876709, -4.756085395812988, 5.180308818817139, 4.4830121994018555, -5.79549503326416, 13.23240852355957, -12.155960083007812, 2.603726863861084, 16.79707145690918, 5.807454586029053, -2.7956743240356445, 8.509941101074219, 7.253137588500977, -6.248112678527832, -0.6558600068092346, 0.44296085834503174, -14.728341102600098, -15.006647109985352, -11.468104362487793, -20.46160888671875, -1.9562751054763794, -8.990943908691406, 16.625680923461914, 21.2569637298584, -16.219974517822266, 8.699222564697266, -7.416118621826172, -14.367061614990234, -2.5667903423309326, 27.44797706604004, 8.230257987976074, -1.5707188844680786, 5.223736763000488, -8.050904273986816, 16.100448608398438, 3.3264219760894775, -8.392641067504883, 3.3614017963409424, 0.07568027079105377, -9.04835319519043, -6.383869647979736, 11.704263687133789, 14.976787567138672, -1.2192366123199463, -11.924437522888184, 17.554115295410156, -10.960140228271484, 8.638209342956543, -5.631171226501465, 13.178277969360352, 15.17791748046875, -13.224848747253418, 0.9007262587547302, -2.6170132160186768, 6.406152248382568, -4.429709434509277, -8.57957649230957, -7.016325950622559, -4.046708106994629, -3.007615804672241, -11.941070556640625, -0.8376246690750122, 5.178377628326416, -3.0736634731292725, -4.032728672027588, 1.4638699293136597, 4.450864315032959, 11.889739036560059, -9.378756523132324, -19.572053909301758, 3.432384967803955, -10.956212997436523, -11.678771018981934, -5.08990478515625, -1.943690538406372, -4.584836959838867, -8.060958862304688, 0.28799664974212646, -4.715052127838135, 16.241844177246094, -16.815364837646484, -7.421855449676514, -8.329093933105469, 1.853922724723816, 2.7432429790496826, -7.2234320640563965, -7.453135967254639, -1.076282024383545, -7.621700763702393, -6.856745719909668, -9.506014823913574, -0.7555220127105713, 1.2375411987304688, -8.923404693603516, 3.3838727474212646, 10.34105396270752, -6.954596042633057, 0.18957191705703735, -13.588937759399414, -8.432282447814941, 9.801265716552734, 10.4203519821167, -15.962389945983887, -10.88754653930664, 7.809375762939453, 7.126617908477783, 9.822647094726562, -5.207985877990723, 1.196556806564331, -1.544622778892517, 1.0630662441253662, 6.335210800170898, -3.8514456748962402, 1.117982268333435, 1.5062692165374756, 6.264873504638672, -3.6413776874542236, -16.538179397583008, 8.471808433532715, 15.263933181762695, -5.335570335388184, -9.518900871276855, -7.87558650970459, 0.20551973581314087, -10.290188789367676, -17.18984031677246, -6.4864678382873535, -14.946615219116211, -9.895748138427734, 4.893916130065918, -8.210529327392578, -18.8530330657959, -5.026514530181885, 12.06063461303711, -9.044784545898438, 1.7720611095428467, 5.475152969360352, 3.831798791885376, 2.011021614074707, -17.412111282348633, 4.457283973693848, -20.602745056152344, -3.5942893028259277, -1.1722047328948975, 5.545193672180176, -19.293479919433594, 0.3080877363681793, -6.431764602661133, -5.921959400177002, 25.83491325378418, -19.35211181640625, -22.15847396850586, 21.06755256652832, 16.930896759033203, 2.11407732963562, -13.06103515625, 22.302799224853516, -5.8066606521606445, -5.223477363586426, -5.533873558044434, -9.953582763671875, 1.5270692110061646, 0.6705088019371033, 14.869146347045898, 8.755400657653809, -13.1751070022583, 2.6654491424560547, -7.337801456451416, 2.9881227016448975, 8.241331100463867, -14.081729888916016, -3.6959595680236816, 7.047457695007324, 3.455181360244751, 3.365583896636963, -0.38456523418426514, 19.904787063598633, -9.607521057128906, 0.02152913808822632, -18.468585968017578, -15.623233795166016, -1.9203518629074097, 10.906023025512695, 0.5133715867996216, 4.476044654846191, -0.34447911381721497, -8.969527244567871, 4.691201686859131, 4.175603866577148, 4.603409290313721, 5.213706970214844, 8.438919067382812, -9.650090217590332, 1.8640410900115967, -5.458246231079102, 1.3043804168701172, 15.579963684082031, -2.870642900466919, -6.2267866134643555, -9.628211975097656, 25.675556182861328, 16.54598617553711, 11.431777954101562, 7.791860103607178, 18.521814346313477, -6.420872211456299, -16.615402221679688, -5.574726104736328, -4.049880027770996, 4.145516395568848, 5.411879539489746, 11.87990665435791, -2.88209867477417, -8.789344787597656, -6.3907976150512695, 1.8774349689483643, 21.0170841217041, 9.563613891601562, -4.246947765350342, -18.689716339111328, 11.017385482788086, 10.204075813293457, -12.972649574279785, -1.0801708698272705, -0.11156292259693146, -16.2568359375, 8.888650894165039, -10.624371528625488, -9.631797790527344, -11.526074409484863, 0.044663265347480774, 2.3566455841064453, -3.483438491821289, -8.412345886230469, 2.970731735229492, -1.5549991130828857, 9.842233657836914, -1.7676386833190918, 16.911781311035156, -11.993989944458008, -6.215513229370117, 12.719137191772461, 9.937559127807617, 9.020588874816895, 13.405200004577637, -6.787906646728516, -8.128850936889648, -7.759225845336914, 12.615218162536621, -6.2328901290893555, 3.318260669708252, -10.305606842041016, -14.010400772094727, -6.044677734375, 11.907098770141602, -4.882970809936523, 7.408750057220459, -0.21712280809879303, 8.714142799377441, -3.368788003921509, -4.6229248046875, -4.172300815582275, 8.783787727355957, -9.917040824890137, 1.173711895942688, -4.853289604187012, -4.503890037536621, -3.3422746658325195, -9.314976692199707, 10.735920906066895, -18.494455337524414, -15.927714347839355, 18.390499114990234, 5.279820442199707, 1.870086908340454, -7.281698703765869, 14.093851089477539, 4.082065105438232, -10.796512603759766, 5.4175567626953125, -9.792013168334961, 10.88960075378418, -5.948910236358643, 18.34189796447754, 5.409823417663574, 5.457680702209473, -4.797878742218018, -1.6799157857894897, -1.7657504081726074, -12.239171028137207, -1.0557265281677246, -6.7360968589782715, 20.426151275634766, -8.911727905273438, -16.71646499633789, -4.5109076499938965, 0.4023851156234741, 16.48377799987793, -7.974236488342285, 1.191507339477539, -2.8165230751037598, -8.264126777648926, -7.231029510498047, 18.394208908081055, 3.226926803588867, 7.121541500091553, 0.8779107928276062, 1.6009814739227295, -8.88601303100586, -0.25357839465141296, 19.356760025024414, -7.477263927459717, 0.7019690275192261, 2.7828402519226074, 7.0147223472595215, -10.237445831298828, 7.023487567901611, 6.554901123046875, 6.050439357757568, -5.204185962677002, -8.95045280456543, 0.2461363673210144, 7.525206089019775, 3.500870704650879, -11.839190483093262, 6.810911178588867, 6.962501525878906, 0.10132219642400742, -0.8196483850479126, -4.469468593597412, 12.93920612335205, -10.784427642822266, 25.241100311279297, 9.276660919189453, 8.29881477355957, -16.69182014465332, 2.0157010555267334, 9.652777671813965, 17.494525909423828, 10.530611038208008, -9.99365234375, -12.927358627319336, -11.41543960571289, -17.420251846313477, 8.399008750915527, 12.074260711669922, -0.012158213183283806, 16.99127197265625, -7.420726299285889, -4.3197479248046875, 0.8556673526763916, -10.051201820373535, 0.9663810729980469, 1.312949776649475, -20.318870544433594, -12.702186584472656, -2.9340195655822754, 2.2355053424835205, -2.1824188232421875, 13.360491752624512, -4.203356742858887, -4.996737957000732, -21.988065719604492, -1.655082106590271, 1.7174369096755981, -3.998296022415161, -13.457504272460938, 2.652693271636963, -8.288107872009277, -16.719589233398438, 2.68540620803833, 7.473601341247559, 9.554136276245117, 10.560755729675293, 3.794013023376465, -9.433999061584473, -6.2064080238342285, 10.105424880981445, 10.3449068069458, 6.226164817810059, -17.777429580688477, -14.649886131286621, 1.5076684951782227, -0.9762094020843506, 11.254066467285156, -17.888084411621094, 18.693906784057617, 21.952896118164062, -1.1838160753250122, 22.101306915283203, -1.6610159873962402, -13.405203819274902, 17.13054084777832, 6.22252082824707, 2.4156198501586914, -5.222223281860352, 0.10797717422246933, 11.917093276977539, 15.360760688781738, 0.7915708422660828, 11.601715087890625, 5.317648887634277, 11.681215286254883, -12.415223121643066, 1.4623749256134033, 6.581414222717285, 9.760221481323242, 3.487867832183838, 7.413256645202637, 3.8406052589416504, -1.8076727390289307, 13.69356918334961, 36.468875885009766, 4.019944190979004, -14.960161209106445, -15.008325576782227, -2.3665566444396973, -4.074625492095947, -5.780495643615723, 5.947779655456543, -14.417747497558594, 8.390974998474121, -21.101043701171875, 10.90212631225586, -0.08613385260105133, 7.357270240783691, 1.2176743745803833, 2.4593989849090576, -6.041247844696045, 10.397591590881348, 4.426887512207031, 14.704898834228516, -7.32002067565918, -12.185158729553223, 9.875856399536133, -15.179362297058105, -2.275282382965088, 25.3262996673584, -11.86194896697998, 23.552791595458984, 2.279891014099121, 11.29847526550293, -12.536412239074707, -8.817628860473633, 22.367046356201172, -6.380529403686523, 6.142239093780518, -4.909487247467041, 8.535045623779297, -6.706013202667236, -3.1255648136138916, -2.0684452056884766, -11.601165771484375, -0.37614908814430237, -13.419769287109375, 19.452377319335938, -8.35731029510498, -19.69293975830078, 5.572696685791016, -0.38689538836479187, -4.179090976715088, -15.585861206054688, 7.692567825317383, -4.498654365539551, -7.439052581787109, 17.03046989440918, -2.1677768230438232, 10.075395584106445, -6.748859882354736, 2.965320587158203, 5.583446502685547, -8.481485366821289, 2.99290132522583, -13.246931076049805, -3.4260129928588867, -0.1797901839017868, 1.9505889415740967, 11.402140617370605, 1.9548975229263306, 5.60771369934082, 3.3231637477874756, -12.516891479492188, -8.661314010620117, 14.5440673828125, -0.25287482142448425, -11.682782173156738, -5.627505779266357, 4.397939205169678, 1.292054533958435, 1.4268064498901367, -16.490503311157227, 5.035247802734375, -9.751561164855957, -9.13937759399414, 1.2137569189071655, 5.15313196182251, -9.22195053100586, 7.936938285827637, -9.960652351379395, 2.9217326641082764, -3.7000179290771484, -9.357264518737793, -5.048763751983643, 1.2199625968933105, -0.3728971481323242, 4.576335430145264, 7.433582305908203, -20.339460372924805, 2.9964468479156494, 3.2757017612457275, 1.1379300355911255, -5.027407646179199, 7.661261081695557, 2.9236812591552734, -1.0733590126037598, -15.445743560791016, -5.497763156890869, 3.6904730796813965, 11.495361328125, 13.262900352478027, -1.3205431699752808, -2.089808464050293, 9.769304275512695, 11.899243354797363, -5.534739017486572, 5.326864719390869, -11.083375930786133, -3.9146461486816406, -0.5808796882629395, -8.795369148254395, -3.3585381507873535, -8.375121116638184, -19.262142181396484, 21.80774688720703, -10.493865966796875, 6.973258018493652, -7.4916181564331055, -1.226611852645874, 8.088220596313477, -14.96330738067627, -17.443164825439453, 2.7728774547576904, -1.4303452968597412, 5.002569198608398, -6.078908443450928, -0.5321522951126099, -0.29365959763526917, -14.23062515258789, 1.9952824115753174, 5.09013557434082, -3.8019700050354004, -5.6667680740356445, 21.44570541381836, -1.3748586177825928, 7.854205131530762, 11.016301155090332, 4.218242168426514, -10.792472839355469, -13.946771621704102, -13.896530151367188, 0.7082341909408569, -5.9909892082214355, -9.233711242675781, -4.4866461753845215, 4.469651222229004, -10.81575870513916, -5.911845684051514, -12.284745216369629, 13.128800392150879, 5.932587623596191, -10.06385326385498, 12.813949584960938, -3.4277775287628174, -12.691811561584473, -7.161173343658447, -5.70831298828125, -9.352409362792969, 5.255387306213379, 6.003109931945801, 1.4961804151535034, -8.03769588470459, -6.714381217956543, 13.495275497436523, -0.973482072353363, 19.208606719970703, 3.1892573833465576, -5.5980024337768555, -4.743285655975342, -5.928568363189697, -7.925665378570557, 14.832291603088379, -4.6474809646606445, -9.237391471862793, -4.585443496704102, 2.0537726879119873, 14.950752258300781, 0.7309195399284363, 0.37277308106422424, 6.494298458099365, 13.130346298217773, -0.5575293898582458, -7.581930160522461, -6.443567752838135, -14.451069831848145, 12.398698806762695, 10.79852294921875, -1.3970820903778076, -5.582632541656494, 10.269198417663574, -3.878413200378418, -13.392338752746582, 1.0526950359344482, 20.958118438720703, 2.0903334617614746, -12.134947776794434, 9.958351135253906, -6.052513122558594, -2.9868361949920654, 7.723146438598633, -14.801616668701172, 27.809846878051758, 4.022073745727539, 13.850090026855469, -6.906275272369385, 5.483813285827637, 10.346065521240234, -3.4116768836975098, 5.484968185424805, 18.08009910583496, 6.4003586769104, 1.268316626548767, 4.028414726257324, 5.615405559539795, -8.924440383911133, 3.0477280616760254, 11.147733688354492, -15.117607116699219, 7.335320472717285, -4.071064472198486, 4.517301082611084, -16.08859634399414, -2.9496450424194336, -2.711857318878174, 0.8640148639678955, 10.944992065429688, 0.8698246479034424, -18.58209228515625, -9.23822021484375, -0.025859307497739792, -9.838641166687012, 1.2255558967590332, 1.0542099475860596, 5.83984899520874, 2.512345790863037, 8.220056533813477, -4.016135215759277, -8.397439002990723, -20.338897705078125, -11.698026657104492, -8.031336784362793, -13.36839485168457, -3.433936834335327, 0.7915326356887817, -0.573326587677002, 8.184205055236816, -16.74919891357422, -4.654767990112305, 4.4351301193237305, -3.695345163345337, -16.145132064819336, -6.800471305847168, 0.6995835304260254, 1.6367524862289429, 2.459587335586548, -7.977290630340576, -1.3199461698532104, -10.528366088867188, 5.512912273406982, 11.79012393951416, -2.7924375534057617, -14.230793952941895, 8.917686462402344, -8.315897941589355, -11.868837356567383, -16.40315055847168, 3.349240779876709, 4.6494550704956055, 6.618610382080078, -2.894397258758545, -12.96312141418457, 12.420207977294922, 0.10064041614532471, -2.1858487129211426, 15.604525566101074, 2.131021499633789, -6.235515594482422, 15.111261367797852, -6.079736709594727, 11.913269996643066, 12.992183685302734, -3.1111836433410645, 0.7284849286079407, 10.434561729431152, 4.149157524108887, -17.46639060974121, 1.1891547441482544, 2.2570512294769287, 2.2385482788085938, 8.126935005187988, 5.829220294952393, -10.736804962158203, 4.148918628692627, 10.45881175994873, 3.480886936187744, 7.924643039703369, -4.276134490966797, 3.344216823577881, 19.65872573852539, -7.250857353210449, -15.396707534790039, 11.063447952270508, 7.733014106750488, 7.633829593658447, 0.6063569784164429, 10.403824806213379, 5.199051856994629, -3.0907154083251953, -9.937141418457031, -2.8588340282440186, -5.300451278686523, -2.14989972114563, -2.082479953765869, -12.730009078979492, 4.092750549316406, 4.565368175506592, 20.892391204833984, -15.192875862121582, 3.951557159423828, -2.6916542053222656, -22.577543258666992, 9.231426239013672, -0.260955810546875, -15.479331016540527, -7.756954193115234, 2.2782604694366455, 11.800891876220703, 0.5690813660621643, -11.889822006225586, 2.0948598384857178, 12.9766263961792, 7.775712490081787, 18.965606689453125, -14.4835205078125, 0.9386866688728333, 6.025287628173828, -17.227352142333984, 5.194572925567627, -13.822052001953125, 1.6230530738830566, -13.262035369873047, -12.96962833404541, 18.273244857788086, -14.460033416748047, -8.942110061645508, 0.802997350692749, -20.78596305847168, 16.041467666625977, -1.008087158203125, 7.043444633483887, 6.566315174102783, 4.682115077972412, 4.5995683670043945, -13.326807022094727, -0.674148440361023, -11.219949722290039, 14.236808776855469, 9.717382431030273, -12.898398399353027, -3.8313472270965576, 13.139196395874023, -10.039453506469727, 9.123828887939453, 5.350091934204102, 3.271310567855835, -3.271848201751709, -2.597966432571411, -8.546438217163086, 11.881301879882812, 17.642730712890625, -5.759466171264648, -6.767197132110596, -20.162761688232422, 1.124705195426941, 4.525278568267822, -12.216264724731445, -15.953888893127441, -1.5231847763061523, 12.820842742919922, -1.277587890625, -0.9248255491256714, -4.356553077697754, 0.03125838562846184, 2.103647232055664, -1.5288079977035522, -3.6792218685150146, 11.354118347167969, -2.6244142055511475, 1.4168767929077148, 24.409626007080078, 4.509510517120361, -4.003791332244873, 18.768646240234375, -7.203995704650879, 0.9568182229995728, 4.925324440002441, 13.183821678161621, -3.917435646057129, 3.8963160514831543, 13.083829879760742, -0.7154451608657837, -6.641647815704346, 14.420208930969238, -4.859248161315918, 1.9628523588180542, -2.241386651992798, -24.015653610229492, 1.3208807706832886, -8.253951072692871, -6.820330619812012, -7.749242782592773, -5.137825012207031, -1.2774457931518555, -4.1232590675354, 18.26959228515625, -2.924431800842285, -1.5238280296325684, -10.600700378417969, 1.9250457286834717, -17.881452560424805, 11.655387878417969, -4.932590484619141, -5.3643035888671875, 3.78572678565979, 20.33160400390625, 4.159850597381592, -7.8240461349487305, 10.127581596374512, -7.390449047088623, 12.913137435913086, 1.1621536016464233, 9.150251388549805, -7.18158483505249, 0.26519808173179626, 2.1400530338287354, -1.4307342767715454, -1.7462517023086548, 15.65006160736084, 17.304468154907227, 6.602473735809326, 3.3316409587860107, -10.943758010864258, 11.92855453491211, -4.281620025634766, -10.979766845703125, -5.475892543792725, 5.898603439331055, -7.344268798828125, 14.859493255615234, 11.604022979736328, 2.2806003093719482, 1.9347470998764038, -3.9241256713867188, 5.056124210357666, -5.6775288581848145, 9.325963020324707, -9.903556823730469, 4.130197048187256, -15.059858322143555, 7.0270280838012695, 8.392152786254883, -5.479278564453125, 4.1735100746154785, -14.65048885345459, -6.064597129821777, 1.7882968187332153, 7.0480451583862305, -6.482527256011963, -25.32227325439453, -0.3121841549873352, -8.052562713623047, -20.68089485168457, -18.062973022460938, 9.342592239379883, -6.8185625076293945, 18.203174591064453, 3.1759896278381348, -8.507837295532227, 11.569398880004883, -3.636046886444092, -8.065715789794922, 12.152872085571289, 1.1297693252563477, 12.83979606628418, 4.2198591232299805, -9.860942840576172, 1.6575968265533447, 8.769796371459961, -13.374185562133789, -6.923803329467773, 12.086320877075195, -0.9756070971488953, 10.174877166748047, 9.235494613647461, -0.5695109963417053, 5.299267768859863, 0.5489572882652283, 3.8447751998901367, -3.4503183364868164, 12.451991081237793, -9.771821975708008, 26.461008071899414, -1.7681999206542969, 8.497282028198242, -12.31551742553711, -2.4796712398529053, 3.481604814529419, -1.7288386821746826, -10.676654815673828, 7.376838684082031, 7.499951362609863, -5.575595378875732, -0.7646622657775879, -19.988525390625, -10.593406677246094, -2.93084979057312, -21.87921142578125, 2.1165647506713867, -8.041547775268555, -1.9137288331985474, 11.628388404846191, -2.1741225719451904, -0.21338284015655518, -8.719250679016113, -0.6224627494812012, 1.0133628845214844, 3.252488136291504, 4.750272750854492, -4.466212749481201, -14.875465393066406, -5.915381908416748, 13.132568359375, 9.58910846710205, 7.483109474182129, -5.037004470825195, 2.550816059112549, 1.2606282234191895, -17.21068572998047, -8.325809478759766, -14.497032165527344, 3.4141838550567627, -2.281599998474121, 7.795760631561279, 15.49020767211914, -14.618239402770996, -4.7488884925842285, 16.87214469909668, -0.11229963600635529, 1.4416872262954712, -3.2245774269104004, 1.0526278018951416, 17.593446731567383, -4.1620283126831055, 0.8194169402122498, 2.2011451721191406, -2.790255308151245, 9.158819198608398, -8.827716827392578, 3.773280382156372, -22.26802635192871, 9.512614250183105, 7.877227783203125, 3.969966173171997, -4.433866500854492, -0.31247615814208984, -8.903221130371094, -6.0501556396484375, 7.600467205047607, -4.932511329650879, 3.00270414352417, -5.727987289428711, -15.02224063873291, 1.0813992023468018, -29.567148208618164, 8.318713188171387, -13.330968856811523, 15.13747787475586, 3.287111759185791, 4.3770036697387695, 3.4273269176483154, -5.430047035217285, -1.1288647651672363, 4.015972137451172, 1.0127053260803223, -4.982204914093018, 2.0446527004241943, -4.851140022277832, -1.4346723556518555, -11.761678695678711, -1.2400038242340088, -17.97803497314453, -6.893587112426758, 4.402900695800781, 12.943546295166016, -8.83084774017334, 31.550506591796875, 3.6673531532287598, -4.366949081420898, -0.09263362735509872, -13.215246200561523, 23.860233306884766, -10.61463451385498, -0.027134548872709274, -8.343574523925781, 13.814030647277832, -5.574259281158447, -8.735503196716309, -1.0331919193267822, -9.383596420288086, 0.06277590245008469, -5.842437744140625, -1.6362709999084473, -5.237857818603516, -4.627127647399902, -10.267291069030762, 1.2301485538482666, 0.6031436920166016, -11.112264633178711, -11.074165344238281, 9.333760261535645, -19.895402908325195, 8.126224517822266, 17.02545738220215, 2.927921772003174, 23.226058959960938, -7.069758415222168, -4.493307590484619, -9.462535858154297, -0.09024302661418915, -0.18507489562034607, 5.148920059204102, -1.301881194114685, 5.915174961090088, 5.270540714263916, 15.265146255493164, 0.21997423470020294, -7.740514278411865, 0.6306837797164917, 6.385298728942871, -15.413095474243164, 10.06568717956543, -2.044386386871338, 3.650287628173828, -10.902377128601074, 5.096434593200684, -14.72744083404541, 13.132843971252441, 8.159263610839844, -10.651788711547852, 7.784943103790283, 1.2627625465393066, -5.124971389770508, 2.279305934906006, -18.844205856323242, 10.217808723449707, 23.107818603515625, 4.330312728881836, -10.06209945678711, -4.174992561340332, -0.7629736661911011, -10.037583351135254, 15.29034423828125, -1.2017847299575806, -2.7533493041992188, -7.299917221069336, -23.49005889892578, -2.9914629459381104, -4.281279563903809, -8.209465026855469, 11.08467960357666, -6.764394760131836, 13.203065872192383, -11.082755088806152, 9.77009105682373, -9.146295547485352, -1.398584246635437, 3.5866167545318604, -3.604607343673706, 9.095858573913574, 8.821152687072754, -10.897308349609375, -1.0496647357940674, 1.4841036796569824, 2.4274542331695557, -1.9486205577850342, 0.06163189187645912, -6.81951904296875, -15.199098587036133, 16.826953887939453, -8.60997200012207, 10.881457328796387, 2.537600517272949, 11.004561424255371, 5.959484577178955, 10.984477043151855, 3.526892900466919, -2.650960922241211, -21.60174560546875, 15.867277145385742, 13.177241325378418, -9.678337097167969, 19.41806411743164, 7.0973615646362305, -10.782142639160156, -10.634650230407715, 11.611560821533203, -1.1380906105041504, -1.3195066452026367, 21.68828582763672, 1.1190948486328125, 19.723941802978516, 0.7034211754798889, -3.357267141342163, -16.87207794189453, 12.021492004394531, -19.026138305664062, 11.511100769042969, 5.559253692626953, -1.2882460355758667, -6.282461166381836, -11.676006317138672, 4.670389175415039, -4.881969928741455, 0.17292959988117218, 9.863933563232422, -4.729259490966797, 14.771480560302734, 1.963335394859314, 7.526986598968506, 5.407666206359863, 14.206926345825195, -20.468769073486328, 18.085712432861328, -1.5741827487945557, 14.261489868164062, -5.450210094451904, -6.219992160797119, 1.6845965385437012, -3.165316581726074, -10.729222297668457, -6.76806640625, -31.393505096435547, -0.9709508419036865, -5.679801940917969, -5.303079605102539, 6.653910160064697, -13.269294738769531, 4.864898681640625, 11.160837173461914, -15.61284065246582, -7.122986793518066, 2.6748313903808594, 4.344315052032471, -19.899450302124023, -1.4922529458999634, 4.167316436767578, -5.533550262451172, -1.2621076107025146, -5.482221603393555, -0.493435800075531, -10.663199424743652, -1.5215564966201782, 5.861751556396484, 8.341662406921387, 6.746377468109131, 12.121194839477539, 12.509737014770508, 3.6194591522216797, -9.997516632080078, 13.789633750915527, 16.002277374267578, 5.2732062339782715, 3.1029140949249268, 6.136100769042969, 22.21765899658203, -0.4858538508415222, -8.552729606628418, 21.959068298339844, 2.9299540519714355, 6.788115501403809, -10.61132526397705, 0.43528491258621216, 0.1127401664853096, -1.5747601985931396, -5.444614887237549, -4.637245178222656, -1.2519501447677612, -12.03518295288086, 0.5450115203857422, 7.528440475463867, -28.096202850341797, -0.8020203113555908, -2.1249468326568604, 14.544710159301758, -12.201348304748535, 5.23586368560791, -0.2560408115386963, -3.4241600036621094, 9.978464126586914, 1.2757153511047363, 2.118710994720459, 4.674097537994385, 0.00958778616040945, 2.5140812397003174, 4.25256872177124, 4.1508283615112305, 1.4593946933746338, 1.6941561698913574, 15.376335144042969, -7.0342512130737305, 21.371318817138672, 9.706625938415527, 1.80873703956604, -2.679457187652588, -0.536350667476654, 4.509281635284424, 10.70660400390625, -5.93223762512207, -13.746021270751953, -2.232032537460327, 3.736870050430298, -2.882578134536743, 0.5138784050941467, -11.372400283813477, -9.036127090454102, 18.137985229492188, -5.235290050506592, 12.784420013427734, -5.888314247131348, -3.075232982635498, 2.533519744873047, -34.01587677001953, 7.892398834228516, -0.4480060338973999, -5.196774482727051, 1.1865376234054565, -0.7517461776733398, 5.925877094268799, 6.188926696777344, 10.513320922851562, 16.55132293701172, -17.255136489868164, -3.876119613647461, -3.664531946182251, 1.9728167057037354, 2.331185817718506, 11.215497970581055, -16.048019409179688, -16.219226837158203, -8.939352989196777, 5.004055976867676, 3.4140584468841553, -6.698618412017822, 16.455766677856445, 1.9471949338912964, 23.249094009399414, 5.508055686950684, -28.434755325317383, 1.9439213275909424, -1.2368870973587036, -2.8493335247039795, -3.7834367752075195, -7.2689409255981445, -26.27823829650879, -9.056343078613281, -24.021759033203125, 6.310149192810059, 2.508871078491211, 7.642271995544434, 4.998794078826904, -13.931750297546387, -6.552059173583984, 7.574126243591309, 0.3509339392185211, -2.9252848625183105, -0.4566184878349304, 9.805160522460938, -14.301708221435547, 7.200364112854004, 1.683673620223999, -16.143007278442383, -7.336801528930664, -4.948080062866211, -8.42639446258545, -11.504362106323242, 2.405364990234375, 10.36335563659668, -6.366715431213379, -4.514518737792969, -7.635674476623535, 6.8378448486328125, -4.6700592041015625, -2.304910659790039, 3.2208142280578613, -13.11220645904541, -6.374492645263672, -2.0583267211914062, 17.410003662109375, 16.86511993408203, -1.6780378818511963, 7.506704330444336, 5.842284679412842, 8.474669456481934, -8.196919441223145, -16.39409828186035, -6.739114761352539, -4.365201473236084, 7.7697954177856445, -4.102510452270508, -1.6521456241607666, -19.649269104003906, -13.486961364746094, 1.8207443952560425, -3.504061460494995, 5.114114284515381, -3.3960342407226562, -14.42497444152832, 9.012202262878418, 4.289249897003174, 8.625713348388672, -4.571547985076904, -6.848065376281738, 0.6718138456344604, -11.813474655151367, 1.4913779497146606, 5.091395378112793, -10.027305603027344, 2.3753039836883545, 8.195307731628418, -4.127997398376465, 4.604002952575684, 6.256900310516357, 11.588988304138184, -7.62696647644043, -6.968790054321289, 5.276515483856201, -5.420431137084961, -1.157591462135315, 9.826582908630371, -9.50891399383545, 3.2757413387298584, 7.606087684631348, 9.382301330566406, -7.480495452880859, 21.637821197509766, -3.78971791267395, -0.2683555483818054, 13.000455856323242, -1.572035312652588, 4.278109073638916, -8.757580757141113, 0.6742318868637085, 8.543717384338379, -4.359501838684082, -9.985669136047363, 0.5913408994674683, 4.543816566467285, -5.05211067199707, -14.458550453186035, 13.719196319580078, 0.6121726036071777, -4.961043357849121, 19.737287521362305, -16.50583839416504, 1.9822919368743896, 7.609286785125732, -0.924267053604126, 3.7149393558502197, -3.646744728088379, 19.68395233154297, 4.917731285095215, 2.9396142959594727, -3.3781447410583496, 2.4393646717071533, 14.39040756225586, 18.007347106933594, -13.068014144897461, 0.8002560138702393, -4.6922221183776855, 7.683477878570557, 4.982419490814209, -5.531019687652588, 15.287285804748535, 1.0080265998840332, -7.799553394317627, 13.752470016479492, -6.419798851013184, -8.887714385986328, -2.7173104286193848, 7.800373554229736, -15.530523300170898, 5.882117748260498, -7.827365398406982, -4.688361644744873, 24.960548400878906, 15.419401168823242, 4.296542167663574, -17.162479400634766, -9.38331413269043, -5.2093706130981445, -7.231765270233154, -6.288771629333496, -14.110727310180664, -2.355363130569458, -10.258355140686035, -4.470212459564209, 1.7388068437576294, -3.536851644515991, 11.229411125183105, -14.536651611328125, -5.308263301849365, -1.7188787460327148, -15.757326126098633, 1.8189948797225952, 4.851879119873047, -6.879129886627197, 1.172707200050354, -3.1478652954101562, -22.800579071044922, -4.452429294586182, -12.819061279296875, -16.49224090576172, -3.4232540130615234, 10.072040557861328, 9.958296775817871, -19.08998680114746, 21.222665786743164, 5.337398529052734, -6.34309720993042, 6.819908142089844, -0.12251802533864975, 11.937917709350586, -21.103742599487305, 1.1876226663589478, 8.147908210754395, 12.773055076599121, 11.198759078979492, 3.944824695587158, 25.913293838500977, -1.960999846458435, 0.6592119336128235, -6.5465497970581055, 7.275310039520264, 5.234580039978027, 3.3720600605010986, 11.593221664428711, -4.066699981689453, 12.733394622802734, 10.506097793579102, -0.1894722282886505, -4.571990966796875, 17.802448272705078, -3.418642997741699, -0.18177247047424316, -5.995758056640625, 8.689560890197754, -8.53150749206543, 1.1157710552215576, 3.5594263076782227, -5.722301483154297, -5.771145343780518, -2.4989173412323, 19.01645851135254, 1.7176939249038696, 3.8913888931274414, -13.294135093688965, 13.83360767364502, -1.4460443258285522, -14.951034545898438, -10.17884349822998, 17.067167282104492, 0.713270366191864, 10.827127456665039, 24.24231719970703, 8.804642677307129, -15.308578491210938, 0.21385642886161804, 2.488215684890747, -8.97048568725586, 5.962202072143555, -23.733455657958984, -6.8018903732299805, 7.86948299407959, 0.6039405465126038, -11.302658081054688, -6.068721294403076, 28.371246337890625, -3.0789594650268555, 11.320108413696289, -0.7645385265350342, 12.801841735839844, -5.7918195724487305, -2.937368869781494, -16.4935359954834, 14.074478149414062, 0.06446496397256851, 18.382600784301758, 2.374191999435425, -1.4816230535507202, -2.6639602184295654, 11.434222221374512, -11.800796508789062, 5.222498893737793, 3.078502655029297, -13.352714538574219, 7.522749423980713, 10.432701110839844, -13.666879653930664, 8.386756896972656, 4.03758430480957, -16.749101638793945, 9.994314193725586, -1.1023145914077759, 6.757469654083252, 5.0466389656066895, 6.7203755378723145, -6.578052520751953, 16.208099365234375, -9.81594181060791, -8.502923011779785, 3.9788010120391846, -13.9170560836792, 10.911247253417969, -14.49963665008545, -11.585899353027344, -0.508670449256897, 2.4748830795288086, -14.872129440307617, 14.741010665893555, 3.6969735622406006, 3.027820587158203, 4.22314453125, 21.27608299255371, -6.155273914337158, 0.04811728373169899, 2.5796666145324707, -10.178841590881348, -10.532283782958984, 5.8488898277282715, -2.870570421218872, 2.568575143814087, 11.808069229125977, 8.915626525878906, -2.8484268188476562, 8.75712776184082, -12.866040229797363, -22.337955474853516, 5.476128578186035, -10.475957870483398, 14.476970672607422, -3.2432589530944824, 28.10997772216797, 8.65247917175293, -13.414546012878418, -20.660646438598633, 1.6832497119903564, -5.068516254425049, -5.730612754821777, -5.177402019500732, -9.579154014587402, -2.6311209201812744, -6.158481597900391, 2.288105010986328, -17.8653507232666, -3.077457904815674, -9.251492500305176, 6.007155418395996, 3.68182373046875, 0.14126409590244293, -8.340192794799805, 7.311015605926514, 1.0859025716781616, -0.1861756294965744, -6.6874847412109375, 1.6117430925369263, 11.629948616027832, -15.773658752441406, 7.092272758483887, -15.07776165008545, 9.391134262084961, 6.873763084411621, -3.567481279373169, 1.1395936012268066, -11.788054466247559, 2.9438555240631104, 3.731748104095459, -6.635377883911133, -5.130828857421875, 11.52137279510498, 5.416798114776611, 1.3025615215301514, 0.007083459757268429, -3.8289988040924072, -15.249889373779297, -4.935859680175781, 9.687370300292969, -6.256173133850098, 2.245360851287842, 8.370299339294434, -13.525915145874023, -0.18568897247314453, -18.07283592224121, -4.584331035614014, 1.3449363708496094, 4.769187927246094, 8.935293197631836, 6.586027145385742, 20.834348678588867, -4.370473861694336, -9.487174987792969, 0.8552274107933044, 21.36389923095703, -17.19429588317871, -3.876999616622925, 7.542795658111572, -3.5521838665008545, -21.638078689575195, -2.745331287384033, 0.08987584710121155, 1.475738286972046, 4.804052352905273, 10.593725204467773, 22.82382583618164, 1.5017484426498413, -17.032135009765625, -17.331851959228516, -5.830236434936523, -3.332122564315796, 0.5924776792526245, -5.068525791168213, -17.691253662109375, -9.159099578857422, -4.237851142883301, -5.505466461181641, -13.048208236694336, -14.924515724182129, 7.845411777496338, -2.736576557159424, -12.258411407470703, 4.133551120758057, -5.413601398468018, -0.33704060316085815, 8.13575553894043, 7.601595401763916, -2.279930591583252, 16.25278663635254, 26.70708656311035, 4.468230724334717, 9.372814178466797, -5.119933605194092, 11.699146270751953, -7.615061283111572, -0.7428085207939148, -3.6747193336486816, 11.812389373779297, -9.79871940612793, -19.363136291503906, 3.6279067993164062, 7.256610870361328, 6.31618595123291, 2.8730952739715576, -1.412885069847107, -6.9675421714782715, -1.3618403673171997, -0.19141145050525665, -17.543485641479492, -0.5537628531455994, 21.437150955200195, -24.453510284423828, 9.665096282958984, -0.2221524566411972, 1.1543751955032349, 6.435707092285156, 7.192699909210205, -14.915779113769531, -3.426762819290161, 6.016473770141602, 1.6369588375091553, -19.026935577392578, 12.87396240234375, 6.275532245635986, 22.060619354248047, 0.5757110118865967, -13.249101638793945, 13.195955276489258, -13.283430099487305, -11.250757217407227, -1.4016176462173462, 0.9575880765914917, 3.1766602993011475, -9.56954288482666, -1.3747276067733765, -6.659002780914307, -11.735533714294434, 2.5457239151000977, 4.323608875274658, -7.935197830200195, 3.710338830947876, 12.037189483642578, 5.393702983856201, -4.898612976074219, 11.805523872375488, -1.7412972450256348, 10.626165390014648, -3.8242526054382324, 7.087646007537842, -0.10770133137702942, 10.068463325500488, -17.917343139648438, -11.49712085723877, 16.342891693115234, -6.107593059539795, 6.202791213989258, 9.064421653747559, -13.03067398071289, -12.63883113861084, -0.7598283290863037, -7.225714206695557, -43.699745178222656, -10.277097702026367, -5.057681083679199, 11.613616943359375, -7.342450141906738, 1.3734803199768066, 6.03168249130249, 12.76989459991455, -7.706511497497559, 2.7362985610961914, 3.306708574295044, 3.920842409133911, -4.898122787475586, 16.73973274230957, 8.38936996459961, -18.551055908203125, -2.772365093231201, -6.512770175933838, -4.493614196777344, 9.404692649841309, 12.0814208984375, 4.35377311706543, 1.3009597063064575, 7.082871913909912, 6.61564302444458, -4.9995574951171875, 12.523210525512695, -0.5921556949615479, 8.115522384643555, 14.272146224975586, -1.4559569358825684, -13.48575496673584, 5.2488298416137695, 8.365029335021973, -5.012164115905762, -17.732505798339844, -8.280881881713867, -16.5396671295166, 6.095876216888428, 6.9246907234191895, 15.7252779006958, -0.9581717252731323, 13.860978126525879, 6.004525184631348, -20.88976287841797, 0.39817678928375244, -0.6359642148017883, -10.22663688659668, 1.7269288301467896, 2.538010835647583, 8.755406379699707, -8.615619659423828, 8.74012565612793, 8.949933052062988, -2.328479051589966, -4.9809184074401855, -5.602138042449951, -6.551316261291504, -7.390222549438477, -7.344467639923096, 6.083266258239746, -8.322258949279785, 1.1157691478729248, -5.264741897583008, 13.223669052124023, -1.5411620140075684, -12.751903533935547, 0.12470333278179169, 0.6547799706459045, -8.977395057678223, 2.974252223968506, 12.077190399169922, 2.2112560272216797, -2.2332592010498047, -21.107952117919922, -3.3687515258789062, -2.9805543422698975, 5.563144207000732, -13.724733352661133, 14.782012939453125, -2.589184284210205, 20.34058380126953, 0.1811194270849228, -11.358539581298828, 1.684592843055725, 10.943021774291992, -18.131385803222656, 5.258668899536133, 0.674741804599762, 11.03188705444336, 2.764376640319824, -0.42974844574928284, 16.262420654296875, -4.701099872589111, -3.7718963623046875, 7.893943786621094, 7.900755882263184, -9.134109497070312, 4.120113849639893, -1.0984481573104858, 3.3143014907836914, -5.91530704498291, -25.104724884033203, -8.638935089111328, -6.970623970031738, -4.841465473175049, -10.937752723693848, -3.3059515953063965, 12.624641418457031, 19.84427261352539, -5.456027507781982, -3.5147976875305176, -3.2093725204467773, -18.756515502929688, -3.2979605197906494, -5.871545791625977, -4.783630847930908, 1.5937976837158203, -5.0135040283203125, 26.899646759033203, -5.412816047668457, -11.218856811523438, -6.340920925140381, -7.00892448425293, 1.902738094329834, -10.136188507080078, 15.267881393432617, -22.5296630859375, -1.9129307270050049, -15.492047309875488, -2.7755239009857178, 0.08500120788812637, -4.512150287628174, -19.923912048339844, 9.688937187194824, -16.1940860748291, -19.299076080322266, 15.424180030822754, 4.292572975158691, -13.19921588897705, -8.383834838867188, -16.566097259521484, -24.61188316345215, 20.484840393066406, 9.566064834594727, 2.160651922225952, -6.6098408699035645, -10.274919509887695, -5.112696647644043, -9.849614143371582, -4.390645503997803, -5.438015460968018, -16.071102142333984, 2.2608659267425537, 19.716890335083008, 10.762166976928711, 3.4070205688476562, 17.651493072509766, -6.035240173339844, -12.016816139221191, -6.597353935241699, 0.16603240370750427, 8.822315216064453, 8.557196617126465, -4.510995864868164, 12.333053588867188, -12.43916130065918, 16.334516525268555, 11.032958984375, 5.200631618499756, -1.0437862873077393, -5.540177822113037, 1.0371332168579102, 5.921960353851318, -22.299165725708008, 17.197492599487305, -1.7224527597427368, 10.207454681396484, -16.189516067504883, 13.23921012878418, -2.104526996612549, 6.314725399017334, -11.69357681274414, -7.104937553405762, 8.760049819946289, 3.3191347122192383, 3.779118537902832, 4.54371976852417, -0.44936126470565796, -1.7373487949371338, 8.612533569335938, -7.163979530334473, -1.977594017982483, 14.249405860900879, 9.047833442687988, -1.846566915512085, 11.768238067626953, 6.293359756469727, 5.412711143493652, -18.534717559814453, -11.396265029907227, -5.405089855194092, -6.551311016082764, 14.500870704650879, -17.10106086730957, 10.747896194458008, -14.845782279968262, 2.4704484939575195, -5.795010566711426, -8.71468734741211, 5.087782859802246, 15.79413890838623, 0.9905915856361389, -0.1595761626958847, 3.0138301849365234, -5.01188325881958, 4.528671741485596, 2.0811660289764404, 6.463749408721924, -4.78438663482666, -8.919109344482422, -0.20630407333374023, 12.470149040222168, -0.2839587330818176, -8.18416690826416, -8.193116188049316, -3.5797455310821533, 13.124874114990234, 15.134502410888672, -7.769374847412109, -3.1691174507141113, -15.778464317321777, 18.718698501586914, 11.4613037109375, -4.9873223304748535, 6.3983540534973145, -18.960607528686523, -2.1967129707336426, -1.5325310230255127, -1.7675323486328125, -20.8829288482666, 0.28637129068374634, -2.1164517402648926, -2.746922016143799, -11.116961479187012, 0.7405784130096436, -0.18738387525081635, -16.11902618408203, -4.128726959228516, 6.075422286987305, 4.938661575317383, -8.942037582397461, 27.897253036499023, 14.186745643615723, -0.2594531774520874, 4.054146766662598, 16.134445190429688, -3.625364303588867, -11.958808898925781, -5.261078357696533, 12.647980690002441, 7.398186683654785, 10.475007057189941, 18.942710876464844, 14.173966407775879, -5.08872127532959, -2.990074396133423, -5.661289215087891, 7.608345985412598, 11.929369926452637, 1.2464005947113037, 11.534049987792969, -10.604225158691406, 5.304041862487793, -6.827746391296387, -14.657533645629883, 8.084142684936523, 4.288991928100586, 6.629562854766846, -19.657442092895508, -3.458585262298584, -8.592695236206055, -1.748194694519043, 10.619000434875488, -6.224834442138672, 17.798721313476562, 10.39193058013916, -2.813429594039917, -1.1624066829681396, 2.6503043174743652, 4.133345603942871, 29.925825119018555, -2.8729894161224365, -8.293455123901367, 0.041063472628593445, -11.134605407714844, 3.4640958309173584, 8.786429405212402, -4.0348076820373535, 7.248002052307129, -3.213179349899292, -0.6673008799552917, 15.45292854309082, 12.422138214111328, 3.119264841079712, 8.644365310668945, -1.7825441360473633, -20.447484970092773, 5.492292404174805, 1.378061294555664, -5.026111602783203, 1.6115158796310425, -13.592615127563477, 6.688272476196289, 5.936402320861816, -19.369525909423828, -11.299652099609375, 2.7886903285980225, 6.7628326416015625, -23.113365173339844, 0.5300388336181641, 5.239675045013428, -15.090044021606445, -6.991903305053711, -12.550683975219727, 3.84574294090271, -5.759673595428467, -21.6505126953125, -1.882337212562561, 1.0014281272888184, 14.990187644958496, 4.429095268249512, -5.002642631530762, 14.738174438476562, -4.190962314605713, 1.7643107175827026, -0.6539043188095093, -16.590534210205078, 2.106309413909912, 24.660198211669922, 10.071420669555664, -0.5648341774940491, 4.447563171386719, 5.878178596496582, -7.894063472747803, -14.891374588012695, 2.6750712394714355, 12.595394134521484, 7.8093719482421875, -13.755379676818848, 4.839628219604492, 4.622895240783691, -26.352155685424805, 6.333855628967285, -1.5793615579605103, -1.477427363395691, 3.315128803253174, -1.6460521221160889, 4.307131290435791, 16.244909286499023, -0.09449513256549835, -8.962709426879883, -4.280508041381836, 21.812313079833984, -17.58721160888672, -6.23605489730835, -7.75475549697876, -14.883682250976562, -5.085539817810059, -2.777296543121338, 10.340707778930664, 4.830469608306885, -11.96646499633789, -6.443399906158447, 5.51004695892334, 18.0775146484375, 4.8017377853393555, -5.411818027496338, -10.0452241897583, 8.398959159851074, -15.750236511230469, -1.8711977005004883, 21.916881561279297, -17.018756866455078, 3.761486053466797, -16.892316818237305, -11.736244201660156, 11.539348602294922, -2.576199531555176, 3.595412015914917, -4.0372633934021, -7.639791965484619, -6.163639068603516, 4.128472328186035, 13.958281517028809, -0.45271217823028564, 22.239696502685547, 14.474981307983398, -16.749160766601562, -12.777107238769531, 3.6829211711883545, 4.206785202026367, -11.207305908203125, -5.31978178024292, -14.135570526123047, -4.717992305755615, 11.603022575378418, 4.521785259246826, -6.386013984680176, 17.438432693481445, 26.242938995361328, -7.847993850708008, 0.15217240154743195, 13.958242416381836, -5.954511642456055, 10.811012268066406, -11.281005859375, -16.019681930541992, 10.001283645629883, 8.515935897827148, 7.397260665893555, 8.822598457336426, 7.123175144195557, 6.917989253997803, 3.043954849243164, 12.25674819946289, 12.959035873413086, 8.848125457763672, 15.031469345092773, 13.098187446594238, -4.99671745300293, -2.8267481327056885, -1.4376939535140991, -6.177825450897217, 11.031594276428223, -9.837457656860352, -14.351625442504883, -4.6551971435546875, 7.225720405578613, 6.498260974884033, -6.367590427398682, 8.998369216918945, 5.642230033874512, 1.2522399425506592, -2.9108991622924805, -7.937414169311523, 13.920694351196289, 5.834004878997803, 11.816807746887207, 15.049339294433594, 7.554730415344238, 1.6363780498504639, -3.333604574203491, 5.085932731628418, -4.652105331420898, -14.766748428344727, -0.9027746915817261, 1.8471647500991821, -9.425302505493164, 10.376832962036133, -9.449752807617188, -3.72444486618042, 16.41845703125, -5.068075656890869, 3.8309214115142822, -14.842568397521973, -3.6877079010009766, 6.5313310623168945, -2.408324956893921, 2.5342960357666016, -7.396763801574707, 9.334307670593262, -6.512345790863037, 0.5433053374290466, 5.688557147979736, 11.391469955444336, 5.900259017944336, -3.648037910461426, 10.97706413269043, -2.3427865505218506, -1.2261918783187866, -1.6059701442718506, 4.375943183898926, 4.984699249267578, 7.469066619873047, 6.78232479095459, -10.203103065490723, -6.285122871398926, 14.629225730895996, -13.154754638671875, 12.985011100769043, 5.672112464904785, -12.70590877532959, 1.1860395669937134, 6.344635009765625, 8.868770599365234, -1.3304166793823242, -14.826830863952637, -1.0875974893569946, -1.2585996389389038, -4.250564098358154, -4.673274993896484, -18.744903564453125, 9.169942855834961, -13.523202896118164, -6.594547271728516, -4.799056053161621, -8.912276268005371, -2.913469076156616, 2.027174949645996, -6.283444404602051, -2.3750362396240234, -6.373366832733154, 17.615846633911133, -0.8936980366706848, 30.344959259033203, 9.918824195861816, 14.382658004760742, 13.069676399230957, 12.780303955078125, 2.370271921157837, 6.414709568023682, 1.1387519836425781, 2.8567557334899902, -1.5569247007369995, -9.215548515319824, -10.3663969039917, -12.542015075683594, 0.8985786437988281, 9.44291877746582, -1.5117360353469849, -1.7685232162475586, 3.8017163276672363, -7.090844631195068, 16.708709716796875, -5.37454891204834, 5.942447185516357, -10.360995292663574, -8.467826843261719, 1.1938766241073608, 2.385713815689087, 9.500237464904785, -8.424970626831055, 10.370418548583984, 15.529197692871094, -33.72758483886719, -11.038228988647461, -17.390880584716797, 3.626551628112793, -0.7129260897636414, -8.968266487121582, 6.1172051429748535, 6.140753746032715, -15.053067207336426, -1.1991586685180664, 6.151352405548096, 13.351344108581543, 7.922486782073975, -9.639993667602539, 18.956771850585938, 6.977295875549316, -6.136493682861328, 4.338178634643555, 0.33195760846138, -12.838838577270508, 8.13550853729248, 10.48917293548584, 2.966150999069214, -18.281036376953125, 11.938680648803711, 4.318548202514648, 11.681907653808594, 25.05293083190918, -7.248014450073242, 8.75040054321289, -5.667119026184082, -2.711702823638916, 3.1066582202911377, 9.061914443969727, -4.122683525085449, -15.744572639465332, -6.403887748718262, 6.869403839111328, -0.19326189160346985, 5.788822650909424, 7.558717250823975, 20.870174407958984, -1.9771217107772827, 29.54216766357422, -8.300786018371582, 3.318983316421509, 9.634918212890625, 5.776815891265869, 2.1497950553894043, 10.44124984741211, 18.174671173095703, 1.407523274421692, -18.970195770263672, -5.390937328338623, -4.971772193908691, 8.633211135864258, 7.450641632080078, -25.03371238708496, 17.540189743041992, -3.948820114135742, -14.451390266418457, -15.619888305664062, 13.429086685180664, -6.876422882080078, 2.597917079925537, -6.565680980682373, 3.520839214324951, -13.443163871765137, 11.525991439819336, -3.510897636413574, -6.000792980194092, 16.454631805419922, -9.99052906036377, 3.9470648765563965, 2.8448033332824707, 12.024832725524902, -7.957081317901611, -0.18329250812530518, 3.4591329097747803, 4.182036876678467, 6.83501672744751, 1.0479655265808105, 4.334115505218506, -9.758720397949219, -10.447278022766113, 9.103019714355469, 0.36138269305229187, 3.042653799057007, -1.2799440622329712, -11.912946701049805, -11.623248100280762, -2.1553401947021484, -8.056463241577148, -28.96919059753418, 8.35481071472168, -9.370025634765625, 1.8301827907562256, -20.142030715942383, 0.8202916979789734, -11.530156135559082, -4.6276655197143555, 4.729694366455078, 1.001246690750122, 12.439563751220703, -4.409837245941162, -16.352022171020508, -0.173982173204422, 8.25790023803711, -13.7601900100708, -0.5026950240135193, 4.388789176940918, 3.045799493789673, 12.834763526916504, -1.3018590211868286, 5.507489204406738, -8.690950393676758, -0.4417419731616974, -14.42956829071045, -13.774089813232422, -0.4632951617240906, 7.265978813171387, -2.2927472591400146, 0.7685306668281555, -9.037454605102539, 11.195164680480957, -6.954962730407715, 7.446906566619873, 5.120994567871094, -3.498861789703369, -17.089082717895508, 7.592011451721191, -19.446935653686523, -5.041182518005371, 5.817647933959961, 0.15834560990333557, 18.812772750854492, 21.17591094970703, 12.016276359558105, 23.7352352142334, 11.93243408203125, -20.50893783569336, -3.4077274799346924, 23.321474075317383, 25.45883560180664, 7.8731865882873535, 7.426407337188721, -14.637752532958984, 5.155339241027832, 6.453036308288574, -9.145630836486816, 11.08593463897705, 8.315205574035645, -1.487181305885315, 20.914812088012695, 0.6402480006217957, -3.6199960708618164, 4.842916488647461, -3.780221939086914, 12.151408195495605, 14.044251441955566, 0.17068523168563843, 20.761455535888672, -8.26236629486084, 4.106875419616699, -5.66720724105835, 21.67413902282715, 10.466094970703125, 10.908900260925293, 19.2430477142334, 9.86286735534668, 1.653781533241272, -1.7635332345962524, 0.5111613273620605, -3.2220637798309326, -5.236302375793457, 3.757814407348633, 6.990708827972412, -1.5967943668365479, -5.953059196472168, -6.123157024383545, -2.364060640335083, 3.6228463649749756, 11.05527114868164, 0.9853276610374451, 29.35711669921875, 1.5669448375701904, 5.823385238647461, 18.120941162109375, -3.771282196044922, -9.837889671325684, -0.8980652689933777, -2.1938490867614746, 3.6911094188690186, -10.423709869384766, 3.6455821990966797, -13.166108131408691, 3.5313539505004883, -2.6177549362182617, -18.12816619873047, 21.692569732666016, -13.850489616394043, 1.8612028360366821, -14.313783645629883, 10.356128692626953, -4.284414768218994, 10.526357650756836, 5.321640968322754, 9.095253944396973, 5.746036529541016, -13.72066879272461, 3.362293243408203, -9.534090995788574, 8.693770408630371, -10.01192569732666, 12.451939582824707, 9.166831016540527, 5.569488525390625, 7.2736639976501465, -7.536260604858398, -17.580482482910156, -0.7569097876548767, -5.102405071258545, 3.95001220703125, 2.7651546001434326, 5.874903202056885, 4.7195210456848145, -0.2187291383743286, -11.742557525634766, 8.536541938781738, -6.960806369781494, 8.70007610321045, 17.079856872558594, 9.069038391113281, -11.146992683410645, -7.886448860168457, 10.853158950805664, -9.04079532623291, -6.051888465881348, -4.995483875274658, -11.355347633361816, 7.662904262542725, 15.336603164672852, -10.962467193603516, 12.791526794433594, -10.007522583007812, 1.1344019174575806, 10.798548698425293, -3.471822500228882, -1.4991713762283325, -3.353304862976074, 0.04075290262699127, -3.739823818206787, -4.265523433685303, -10.966657638549805, -11.628355026245117, 2.9543418884277344, -2.317917585372925, -8.296490669250488, -9.711883544921875, 1.921911358833313, 11.332134246826172, 0.24971061944961548, 10.734272003173828, 3.305345058441162, -7.965825080871582, -4.023420810699463, -26.9130802154541, 9.32404613494873, -0.36987635493278503, -4.921173572540283, 15.529073715209961, -7.59135627746582, 12.078750610351562, -10.003117561340332, 2.636277198791504, 2.817551612854004, 8.72428035736084, 13.298629760742188, -9.231472969055176, 9.012811660766602, 6.398509979248047, 4.327376365661621, -1.1137750148773193, -21.1875, -0.2647980749607086, -8.92514419555664, 3.175809383392334, -6.57487678527832, -10.198114395141602, 2.4150991439819336, -26.056194305419922, 2.8151955604553223, 9.141446113586426, -12.100292205810547, -9.19352912902832, -6.390180587768555, 9.104109764099121, 9.12950325012207, -9.488818168640137, 11.38170337677002, -3.634364366531372, 5.632644176483154, 0.8265279531478882, 10.219527244567871, 0.10365929454565048, 8.184492111206055, 3.103668212890625, 13.556513786315918, 15.12560749053955, 14.688207626342773, -11.27365493774414, -2.615471839904785, -0.7262040376663208, -6.256514072418213, -8.693008422851562, -0.5091065764427185, -10.119009017944336, -5.569746971130371, -6.223166465759277, 6.504640102386475, 4.357343673706055, -6.548430919647217, -5.6148905754089355, 3.7631754875183105, 22.00418472290039, -1.8271639347076416, 14.494677543640137, -11.281498908996582, 26.857107162475586, 12.409963607788086, 7.0411295890808105, 1.7936079502105713, 11.65742301940918, -0.4687322974205017, 3.7741031646728516, -12.153616905212402, -1.3660223484039307, 7.214000225067139, 3.2965052127838135, 4.419754505157471, 26.15831756591797, -23.81702423095703, -12.666219711303711, 5.439688682556152, 2.349247932434082, 3.4880292415618896, -3.2421014308929443, 9.814620018005371, 13.935599327087402, 4.615777015686035, 1.6619822978973389, -14.759376525878906, 2.898402214050293, 10.16313362121582, 0.6694365739822388, 12.454044342041016, -0.659536600112915, 3.8458821773529053, 22.88057518005371, -10.574313163757324, 1.3646489381790161, 21.300907135009766, 11.381900787353516, 5.875759124755859, 5.089457035064697, -29.25952911376953, 8.478840827941895, -2.282398223876953, 4.1418023109436035, -3.519603967666626, -6.603893756866455, -25.967987060546875, 1.2695083618164062, -4.431410789489746, -1.353757381439209, -3.989215850830078, 2.405550003051758, -4.149209976196289, 1.8806946277618408, -9.850340843200684, 7.841531753540039, -2.958954334259033, 1.9037303924560547, -3.4851431846618652, 10.361560821533203, 12.309149742126465, 1.6475393772125244, -7.360278129577637, 12.666625022888184, 6.287159442901611, -0.5357567667961121, -5.522925853729248, 8.628609657287598, 7.452136039733887, -22.044248580932617, 0.6223169565200806, -21.723451614379883, 7.313857078552246, -3.120258331298828, -2.735750675201416, -0.4109175205230713, -0.32602959871292114, 14.627985000610352, 3.3430776596069336, -0.7322264313697815, 1.4318687915802002, 0.43485140800476074, 16.84818458557129, 12.746662139892578, 7.559975624084473, -0.6713372468948364, -10.667396545410156, -12.910713195800781, -17.074554443359375, -6.52095890045166, -4.9586710929870605, 7.547041893005371, 9.380837440490723, 2.2651174068450928, -5.777242183685303, -2.6829309463500977, 2.0990021228790283, -2.608184576034546, 5.119142055511475, 1.976130723953247, 15.98963737487793, 2.956040382385254, 3.6976823806762695, 3.0688867568969727, 0.8776708841323853, 1.7119476795196533, -9.633450508117676, -6.640668869018555, -14.404244422912598, -5.544434070587158, -18.792688369750977, 12.556971549987793, 24.19919204711914, 6.075038433074951, -15.35661792755127, 4.484589099884033, 16.449337005615234, -16.979175567626953, 6.6522417068481445, 4.9762372970581055, -8.83183479309082, -7.232987880706787, -4.92236852645874, -11.697629928588867, -1.453526496887207, 15.86865520477295, -20.26337432861328, -4.053828239440918, 14.96238899230957, 5.189215660095215, 9.362566947937012, -3.545013666152954, -11.75772476196289, -8.648184776306152, 2.755577325820923, -5.36797571182251, 13.07834529876709, 7.025815010070801, 5.431681156158447, -5.604262828826904, 11.998858451843262, 0.4245087206363678, 4.312943935394287, 17.238052368164062, 6.080533981323242, 15.382688522338867, 9.373159408569336, 4.667158126831055, 16.1428165435791, 15.293024063110352, 14.487286567687988, 3.0239739418029785, 12.444771766662598, -6.032868385314941, -4.613112926483154, -0.4164639711380005, -5.71743106842041, 27.889385223388672, 0.33948814868927, 8.89959716796875, -1.152760624885559, 4.992552280426025, 7.824203968048096, 10.76201057434082, -20.517417907714844, 0.6748831868171692, -9.512948036193848, -14.104219436645508, 10.971654891967773, -7.679636001586914, -5.352914810180664, -4.356760025024414, -5.859048366546631, -14.03695297241211, -0.08250121027231216, 9.535957336425781, 3.8233590126037598, 1.6920204162597656, 1.2175594568252563, -0.8092100024223328, -2.684600591659546, -3.7073006629943848, -6.71856689453125, -15.540887832641602, 4.154609680175781, 11.736299514770508, -23.021995544433594, 1.658707618713379, 3.2070305347442627, 2.2580406665802, -12.364568710327148, -8.488747596740723, 1.1629562377929688, 9.288251876831055, 12.687896728515625, -12.136991500854492, 10.123945236206055, -8.413256645202637, -6.634191513061523, 4.755984306335449, -12.654121398925781, -13.420351028442383, -18.476110458374023, -11.020919799804688, 0.03442418575286865, 10.274338722229004, -16.7470703125, -2.3539068698883057, 6.061378002166748, -18.99734878540039, -5.564542293548584, -4.92496919631958, 1.2993674278259277, 10.473784446716309, -8.600516319274902, 2.9529521465301514, -0.12989282608032227, -3.1768150329589844, -12.727705001831055, -1.2198865413665771, 21.063879013061523, 12.393527030944824, 1.2221041917800903, -10.115943908691406, 11.09315299987793, 20.340288162231445, -5.600526332855225, 3.05364727973938, -2.0437984466552734, -11.425760269165039, -17.681171417236328, -0.9754377603530884, 13.728080749511719, 6.596331596374512, 8.06978702545166, -10.874028205871582, 2.27777361869812, -9.880044937133789, -3.4914727210998535, 1.687037467956543, 0.1526830792427063, -15.037679672241211, 12.599714279174805, 7.137835502624512, 4.535198211669922, -9.532556533813477, 2.150860548019409, -5.517678260803223, -6.666579723358154, -8.511962890625, -5.980114459991455, -6.53479528427124, -0.20325076580047607, 2.600371837615967, 8.560995101928711, 6.289446830749512, 1.7041361331939697, -1.9397337436676025, -2.7217020988464355, -1.4501737356185913, 6.914617538452148, 7.1486496925354, 11.383484840393066, -0.8782089352607727, 6.6173553466796875, 16.759674072265625, -9.101557731628418, 2.8266119956970215, 13.289766311645508, 6.138903617858887, 15.119138717651367, -1.651228427886963, -16.702350616455078, 9.434532165527344, -4.690858364105225, 12.779443740844727, -11.480707168579102, 2.0225272178649902, -7.583227157592773, -14.794827461242676, -5.762633800506592, -4.593967914581299, -1.0580555200576782, -1.1066557168960571, -10.165977478027344, 0.0821930468082428, -20.40510368347168, -3.160027503967285, -2.5663349628448486, -0.7973530888557434, 19.91208267211914, 1.6949354410171509, -10.761157035827637, 2.00193190574646, 22.267501831054688, -21.107860565185547, -3.327167272567749, -0.21245278418064117, 4.494338512420654, 0.2588333487510681, -2.3076117038726807, 14.6121826171875, 13.841306686401367, -19.460336685180664, 9.675187110900879, -15.72811508178711, 3.0845375061035156, -10.461067199707031, -2.8703298568725586, 5.473381996154785, -12.236743927001953, -0.07727232575416565, -5.288199424743652, -2.2647366523742676, 1.355199933052063, 5.87955379486084, 1.224831223487854, 15.425734519958496, 4.382134437561035, -5.7452778816223145, 32.641136169433594, -16.82762336730957, 1.3555169105529785, 15.691332817077637, 2.864091634750366, 21.469755172729492, -5.352144241333008, -12.028474807739258, 1.4286118745803833, -13.343280792236328, 17.537948608398438, 1.7583709955215454, 1.552256464958191, 12.827157974243164, 1.233514428138733, 6.583711624145508, -10.073918342590332, -0.7568659782409668, -8.408817291259766, 20.648395538330078, 5.0203328132629395, -3.2044055461883545, -12.495126724243164, -8.1741943359375, 0.9729433059692383, 16.586833953857422, -33.58789825439453, -10.529269218444824, 8.03127670288086, 2.3186824321746826, 20.62759780883789, 19.11928939819336, 1.9335943460464478, 1.6561155319213867, 6.252200126647949, 6.510670185089111, -2.769010543823242, 1.9705729484558105, 10.703686714172363, -4.971639156341553, -11.520244598388672, 11.185297012329102, -6.406587600708008, -8.672897338867188, -16.001344680786133, -11.287732124328613, 3.620195150375366, -4.845152378082275, 9.413994789123535, -3.3264338970184326, 0.3976057767868042, -1.7547274827957153, -14.290379524230957, -15.81871509552002, -14.775656700134277, -11.731280326843262, 12.081174850463867, -8.302785873413086, -4.978646278381348, -7.367985248565674, 11.06191635131836, 6.2686004638671875, 1.5210916996002197, 2.7558398246765137, 11.574956893920898, 12.776894569396973, -8.752983093261719, -2.198779344558716, -11.348651885986328, -0.7971120476722717, 5.800257682800293, 5.9813551902771, -8.538250923156738, 2.6949386596679688, 9.685765266418457, -9.682018280029297, -4.53336238861084, 2.807624340057373, 14.355687141418457, -2.3451719284057617, -2.0067355632781982, 8.67531681060791, -21.08942985534668, 0.0459369458258152, 8.937164306640625, -3.5643794536590576, 1.239896297454834, 3.284766674041748, 11.509164810180664, 14.008520126342773, -2.746828556060791, 9.700408935546875, 10.984695434570312, -6.243622779846191, -6.9500041007995605, -7.403069496154785, -5.810677528381348, 9.95324993133545, 4.136806964874268, -9.961193084716797, -5.515241622924805, 3.3942601680755615, -8.881025314331055, -9.376913070678711, -0.4047715961933136, 0.6651698350906372, -13.913893699645996, -1.3512811660766602, 17.269039154052734, 6.89267110824585, -4.175045967102051, 7.736874103546143, 4.735854148864746, 0.1596388965845108, -10.501333236694336, 23.696802139282227, -23.63762092590332, -2.184098243713379, -5.802485466003418, -2.092848062515259, -6.264671802520752, -17.747102737426758, 2.510952949523926, 10.061253547668457, -4.288665771484375, 5.126865386962891, 6.936489105224609, 0.6880784630775452, -3.106102705001831, -23.76987648010254, -3.7088229656219482, -2.513366222381592, -4.196102619171143, 6.31044340133667, 1.6574654579162598, 9.186700820922852, -3.1706573963165283, -0.5850791335105896, 3.667827844619751, -6.563173294067383, 9.37173080444336, -10.518451690673828, -1.1332379579544067, 2.9329135417938232, 12.527810096740723, 4.246753692626953, 8.237692832946777, -13.41380500793457, 0.7456904053688049, -3.2917399406433105, 29.039386749267578, 6.289678573608398, -1.7348212003707886, -4.001889705657959, 11.213675498962402, 23.96291732788086, -9.684606552124023, 2.405508041381836, -1.2450016736984253, -7.134123802185059, -12.68912410736084, 11.424026489257812, 19.663660049438477, -8.112913131713867, 7.026715278625488, -2.6778573989868164, 15.710033416748047, -12.945676803588867, 23.477161407470703, 8.584036827087402, -13.678994178771973, -6.715217113494873, -9.28028678894043, -1.2102283239364624, 15.708121299743652, 3.586601972579956, 8.715290069580078, 8.765445709228516, -2.8651461601257324, 9.126666069030762, 7.245966911315918, -0.6458024978637695, -3.2263851165771484, -2.4059901237487793, -5.684306621551514, -1.7429219484329224, -6.720456123352051, -12.170853614807129, -3.868628740310669, 17.555639266967773, -4.411182880401611, -0.23119419813156128, 1.242069125175476, -12.23066520690918, -2.3115718364715576, 11.221441268920898, -3.917229175567627, 5.8306450843811035, -3.4267311096191406, 4.365394592285156, 0.006473730318248272, -10.862960815429688, 2.4349420070648193, -11.824317932128906, -8.97299575805664, 14.22380256652832, -10.684823036193848, 3.615544319152832, 9.859270095825195, -3.767690896987915, -6.313589096069336, 1.3273015022277832, -12.14048957824707, -2.9829840660095215, -4.2572712898254395, 19.40679359436035, -11.643699645996094, -9.42929458618164, -2.226011276245117, -11.559475898742676, -4.014808654785156, 7.547430038452148, -8.948578834533691, -1.0794318914413452, 14.758995056152344, -0.6222254633903503, -2.705142021179199, -11.588754653930664, 10.742258071899414, 8.797839164733887, 12.473477363586426, -6.612000465393066, 6.948959827423096, 15.309713363647461, -15.735106468200684, 4.070248603820801, -5.372186660766602, -8.479305267333984, 8.17791748046875, -19.18159294128418, -11.077288627624512, 4.489922046661377, 4.952199459075928, 4.781357765197754, -1.8401795625686646, 14.209161758422852, -12.303799629211426, 8.376799583435059, -2.8663110733032227, 5.835875511169434, -0.12224693596363068, -3.0526962280273438, 6.113515853881836, -14.75139045715332, 0.031755782663822174, 11.451004981994629, 4.782320022583008, 0.8407634496688843, 4.61603307723999, 16.154245376586914, -8.959379196166992, -20.93719482421875, -5.389678478240967, -0.8882623910903931, 3.440284252166748, 9.150760650634766, 2.2020137310028076, 6.160941123962402, -2.3746960163116455, -10.848145484924316, 0.851059079170227, 15.499478340148926, -14.623589515686035, 10.88853931427002, -9.172821998596191, -13.25274658203125, 5.126976013183594, -20.456239700317383, -10.563448905944824, -0.5039561986923218, -5.882227420806885, -13.960126876831055, 5.780682563781738, 2.5070319175720215, 10.692378044128418, 2.3207716941833496, 9.57568645477295, 0.8232343792915344, 2.8586909770965576, 7.801198959350586, 8.566448211669922, -15.008069038391113, -12.772459030151367, -10.495931625366211, -19.242198944091797, -6.453115940093994, 9.038456916809082, 21.447172164916992, 7.000875949859619, 6.769597053527832, -3.2283968925476074, 1.83785080909729, 4.3924713134765625, 1.4136396646499634, -0.7837291955947876, -10.776689529418945, 7.144735336303711, 0.6263465881347656, 1.9853453636169434, 6.395054817199707, -3.2188825607299805, -1.101162314414978, -6.981524467468262, 6.290457725524902, 6.0004377365112305, -1.9116389751434326, 16.73897933959961, 22.043941497802734, 2.0145668983459473, -4.880720138549805, -17.340187072753906, 29.254467010498047, 0.042890697717666626, -7.197690963745117, -0.35024401545524597, 5.193429470062256, -1.989890217781067, -1.4729511737823486, -1.4203882217407227, 6.832200050354004, -0.2570739686489105, 0.15622016787528992, 3.5195422172546387, -13.819160461425781, 16.739667892456055, 23.34246826171875, -12.219610214233398, -7.017167091369629, 7.061725616455078, -15.25448226928711, 4.824525833129883, -1.4277477264404297, -22.5681095123291, 10.77736759185791, -7.281980037689209, -2.717902898788452, 15.334324836730957, -8.735457420349121, 5.89466667175293, 16.502735137939453, -4.343311786651611, -9.84862232208252, 0.9211676120758057, -2.9750075340270996, -7.195720672607422, 3.420203924179077, -11.326501846313477, -11.878131866455078, -18.54967498779297, 16.729656219482422, -21.238983154296875, -6.139543533325195, -9.138365745544434, 1.3083196878433228, -12.23583984375, -4.750479221343994, 6.491568088531494, -1.8899345397949219, 14.390292167663574, 6.972480297088623, -16.732751846313477, -7.22464656829834, 3.8511338233947754, -5.8148345947265625, -7.659831523895264, -1.7217625379562378, 3.4260735511779785, 15.66678237915039, 14.22381591796875, 10.755425453186035, -15.61954402923584, 1.7989383935928345, 4.029016017913818, 10.475836753845215, 4.7146453857421875, -1.8910226821899414, 2.2107431888580322, -0.3818737268447876, 4.670705318450928, 10.424824714660645, -3.4294419288635254, -12.81704330444336, 0.39395952224731445, -2.430217742919922, 4.06459379196167, -6.740595817565918, 2.339031219482422, -2.353592872619629, 3.661036491394043, 13.835836410522461, -14.479167938232422, -10.272470474243164, 0.6719670295715332, -18.13832664489746, -10.542400360107422, 1.9277398586273193, 0.8805977702140808, -4.912028789520264, -1.3500107526779175, 3.6750733852386475, 3.3806087970733643, 17.729427337646484, -2.361260414123535, -22.539138793945312, -0.0019612833857536316, 21.370744705200195, 4.155737400054932, -9.006719589233398, 7.7942633628845215, 2.4318315982818604, 8.772834777832031, -8.005594253540039, 3.8583931922912598, -12.20534896850586, -2.843935489654541, -16.53793716430664, 5.945334434509277, 21.079925537109375, 13.197699546813965, -3.798556327819824, -8.910869598388672, -11.253850936889648, 8.831952095031738, -2.8170549869537354, 4.29780387878418, -16.060630798339844, 3.5079762935638428, -7.179460525512695, -1.7882425785064697, -0.15077155828475952, 7.037841796875, -3.2871341705322266, 10.462045669555664, -2.921764612197876, -7.36123514175415, -18.642868041992188, -2.098938226699829, -5.941546440124512, 11.621957778930664, -4.524195671081543, -4.499381065368652, 9.43592357635498, 1.3560993671417236, -12.730475425720215, -1.2765015363693237, -9.304288864135742, -8.781407356262207, 8.103282928466797, -8.789098739624023, -7.166234016418457, 5.648965358734131, 0.23010072112083435, -0.07740655541419983, 2.4040675163269043, 2.8006813526153564, -18.90074348449707, 10.87295913696289, 6.266114711761475, -5.580554485321045, 3.3587212562561035, 13.089432716369629, 22.365015029907227, 0.7563146352767944, -0.5145823955535889, 0.5170499682426453, -11.253686904907227, 1.0940476655960083, 7.208234786987305, 7.074587345123291, 10.157299041748047, -4.580533981323242, -5.420472621917725, 0.5736315250396729, -7.289194583892822, 6.774518966674805, 15.973869323730469, -3.024995803833008, 24.27884864807129, -27.502349853515625, -3.585381507873535, 1.1540892124176025, -7.782174110412598, 6.919517517089844, 0.052332885563373566, 15.9325590133667, 0.26300010085105896, -1.0573054552078247, 0.2967827320098877, 17.496803283691406, 13.197335243225098, 3.511784553527832, -7.255359172821045, 17.44368553161621, -8.816350936889648, 1.7864404916763306, 13.269671440124512, 3.1654796600341797, 20.725086212158203, -8.258027076721191, -2.394472122192383, 7.735226631164551, -14.71574592590332, 20.90247344970703, 5.168900012969971, 0.8880993127822876, -3.1146440505981445, -1.5516400337219238, 1.0699082612991333, -2.404649257659912, 24.40740203857422, 0.7007997632026672, -10.997308731079102, -9.447525024414062, -8.310575485229492, -3.728097677230835, -14.549396514892578, 12.160938262939453, 3.6958518028259277, -8.013679504394531, 5.353702545166016, -13.734339714050293, -3.851548433303833, 1.5965889692306519, -0.299595445394516, 3.128575086593628, -11.003530502319336, 1.6571670770645142, 8.129146575927734, 0.8964220285415649, 4.445037364959717, -10.686741828918457, -14.260082244873047, -9.07598876953125, 1.1747832298278809, 14.828511238098145, -0.831561803817749, -10.97485065460205, 8.669696807861328, 6.6972975730896, 7.885514259338379, 3.032839775085449, 0.283495157957077, 9.97638988494873, 0.871736466884613, -6.694626331329346, -2.6170012950897217, 10.396541595458984, -12.580554962158203, -5.36541748046875, -3.490349054336548, -3.3240184783935547, -21.71544647216797, -18.94843292236328, -3.703667163848877, 4.633183479309082, 6.153499603271484, -14.989665985107422, -6.561699390411377, -13.05388069152832, -0.8285852670669556, 8.46268081665039, 4.082294464111328, -9.60727310180664, 11.736754417419434, -3.550081729888916, -1.219523549079895, 2.564723491668701, 6.947854995727539, -2.4173543453216553, -6.068887233734131, -1.5748709440231323, 7.5601606369018555, -13.20901870727539, -1.9404913187026978, -20.967082977294922, 3.8419790267944336, -2.098095655441284, -8.845483779907227, -12.316645622253418, 13.909063339233398, -5.693672180175781, -0.4217870533466339, -12.138288497924805, 16.075838088989258, 5.510744094848633, -1.488420009613037, -12.965632438659668, 6.5996856689453125, 8.493101119995117, -2.7545597553253174, -12.89739990234375, -6.338349342346191, -0.13365203142166138, -16.494670867919922, -14.621576309204102, 12.107683181762695, -7.320347309112549, -7.4138288497924805, 5.295401096343994, -4.410092830657959, 19.652851104736328, -2.2922050952911377, 0.8922066688537598, -20.50716781616211, 9.275160789489746, -9.49332046508789, -6.32474946975708, 13.147909164428711, -6.997384071350098, 4.533802509307861, -19.250349044799805, -7.290778160095215, -3.3024251461029053, 12.11016845703125, 5.587143898010254, 26.139202117919922, 0.3671737313270569, 29.708900451660156, -7.792721748352051, -24.32319450378418, -4.338668346405029, -1.8068692684173584, 15.155399322509766, 2.1665332317352295, 5.708505630493164, 9.219614028930664, 8.833298683166504, 2.2348968982696533, 4.129817008972168, -23.75934600830078, -13.041040420532227, -10.76898193359375, -2.4216341972351074, 2.5796308517456055, -6.853463649749756, -7.104737758636475, -11.494599342346191, 2.620405673980713, -3.4427738189697266, 4.4843950271606445, 7.644427299499512, 7.961488246917725, 7.942488670349121, -21.586515426635742, -11.708145141601562, -3.4932260513305664, -1.5775096416473389, 17.429792404174805, 6.667949676513672, -10.841367721557617, 13.357532501220703, -25.637903213500977, -4.120157241821289, -23.886423110961914, 17.92934799194336, 14.32274055480957, -9.694040298461914, -2.848879098892212, -3.008500337600708, 1.0075911283493042, -13.464323043823242, -6.473994255065918, -16.773605346679688, -7.255049705505371, -4.990206718444824, 5.8295512199401855, 13.565306663513184, -10.209516525268555, 17.504995346069336, -5.63453483581543, 13.575844764709473, -9.621354103088379, 4.66485595703125, 1.8511998653411865, -4.991281986236572, -24.47522735595703, 9.533477783203125, -8.900835990905762, -4.020157814025879, -17.29258155822754, 19.251638412475586, -1.1219643354415894, -16.440950393676758, 4.338963508605957, 12.666919708251953, 5.652510643005371, 9.175034523010254, -1.7153804302215576, 0.22224798798561096, -16.4132022857666, -4.329578876495361, 13.526450157165527, 6.110931396484375, 4.204878807067871, -19.71905517578125, 6.45447301864624, 11.961668968200684, -5.881749153137207, -4.5808234214782715, -7.479415416717529, 3.4402451515197754, -15.239788055419922, 18.38199234008789, -2.034998655319214, -0.24988605082035065, -5.058077812194824, -6.812991619110107, -1.8369888067245483, -7.979901313781738, -7.995880126953125, 2.0109641551971436, -5.454135417938232, 2.2829489707946777, 8.48345947265625, 25.172529220581055, -13.657036781311035, 0.5686898827552795, -2.6125736236572266, -11.674793243408203, -19.399616241455078, -10.792703628540039, 14.607690811157227, 0.8794183731079102, -15.374055862426758, 5.58206844329834, 12.979452133178711, -0.5436739921569824, -12.643527030944824, 1.0948832035064697, 3.377021312713623, -8.939820289611816, 7.446615695953369, 22.650251388549805, -4.435096740722656, 13.297836303710938, -26.2745361328125, -8.546639442443848, 1.0880038738250732, 8.914485931396484, -12.94385814666748, -2.9183812141418457, 5.416069030761719, 9.676372528076172, -5.673348903656006, 0.08007235825061798, 15.425429344177246, -8.911480903625488, -0.2316151261329651, -7.0296196937561035, 19.74555778503418, -14.175323486328125, -9.963854789733887, 9.211519241333008, 1.535318374633789, 17.526453018188477, 7.272657871246338, 8.939860343933105, -1.1602009534835815, -6.4555888175964355, 1.789031744003296, 12.653031349182129, -7.9455413818359375, 10.248876571655273, -11.632183074951172, 8.517189979553223, -8.969317436218262, -3.707697868347168, -14.68990421295166, 4.214282512664795, 16.038938522338867, 6.123974323272705, 14.275282859802246, -3.9372754096984863, -14.749754905700684, -7.5267653465271, 1.927445411682129, 16.43672752380371, 3.700758934020996, 2.0973398685455322, 7.150906562805176, -14.936077117919922, -2.2791759967803955, 5.301226615905762, 14.611137390136719, -12.840394973754883, -12.192927360534668, -5.478676795959473, -8.272120475769043, -9.024799346923828, 0.9648340940475464, 14.915989875793457, -5.750539779663086, 18.452608108520508, 9.34449577331543, 13.508411407470703, 9.461190223693848, 0.7335747480392456, 6.874814510345459, -7.589196681976318, -9.293898582458496, 1.162332534790039, -14.171092987060547, -4.798068523406982, 5.740403652191162, 21.32643699645996, 8.784573554992676, 0.012880353257060051, 0.33521509170532227, -0.6011925935745239, 16.219165802001953, -3.0026051998138428, -20.049880981445312, 8.283830642700195, 4.267918109893799, 8.162500381469727, -7.297217845916748, 2.1360340118408203, 4.372072696685791, -12.203402519226074, -1.0825921297073364, -6.223995685577393, -3.9536337852478027, -11.322038650512695, 14.219127655029297, 15.509260177612305, 4.13040828704834, -10.926565170288086, 3.0043838024139404, 4.2181830406188965, 9.323601722717285, 2.8282511234283447, 2.4962215423583984, 3.6633527278900146, 7.4508957862854, -13.279678344726562, 0.9815846681594849, -5.093057632446289, 1.1925030946731567, -4.7290239334106445, -2.3300178050994873, 1.7427895069122314, 23.693208694458008, 4.2696123123168945, 0.18458770215511322, 17.243915557861328, 4.077292442321777, 3.1220481395721436, -10.306483268737793, 4.000704288482666, 0.49570128321647644, 6.308954238891602, 8.17353343963623, -15.82249641418457, -5.051218032836914, -20.08852195739746, -5.74204683303833, -4.906430721282959, -5.5239667892456055, -0.28285887837409973, 7.366146087646484, -6.487890243530273, 10.494250297546387, 5.010868072509766, -2.2592782974243164, -5.296141624450684, 0.2932860851287842, 0.7459217309951782, 4.535531044006348, -8.957059860229492, -16.599201202392578, 8.283062934875488, 2.5867855548858643, 0.7513303756713867, 2.2476866245269775, -8.200587272644043, 6.564867973327637, -5.01143217086792, 1.9936121702194214, -7.25886344909668, 12.453156471252441, -8.447818756103516, -7.835525989532471, -17.404277801513672, -9.958780288696289, 18.761207580566406, 2.727787494659424, -10.343985557556152, 5.915037155151367, 1.3965612649917603, 6.968421936035156, 2.2341294288635254, -23.6060791015625, -3.7926273345947266, -12.05980396270752, -2.45235538482666, -15.285229682922363, -5.999130725860596, -2.7226099967956543, 14.253335952758789, -12.155865669250488, 3.8934566974639893, -11.43805980682373, -1.990325689315796, -2.6270833015441895, 13.239922523498535, 2.481076240539551, 2.573272943496704, 10.855273246765137, -12.573873519897461, 0.02822987549006939, 14.354141235351562, -5.032458305358887, -5.108971118927002, 21.733356475830078, -12.205548286437988, 4.620883941650391, -1.666785717010498, 0.4375251531600952, 9.509309768676758, 8.522926330566406, -10.281370162963867, 0.6230343580245972, -2.142512798309326, 2.394260883331299, -3.024277448654175, -1.1388505697250366, -3.1403369903564453, 2.9254605770111084, -9.6215238571167, 7.513633728027344, -14.591351509094238, 9.716066360473633, 4.139869689941406, -3.4514684677124023, 10.211163520812988, -10.740180969238281, 11.85317611694336, 9.898866653442383, -10.178239822387695, -16.401643753051758, 0.6308234333992004, 3.312676191329956, 7.471530437469482, 4.42054557800293, 3.9817497730255127, -3.8739523887634277, -15.95505142211914, -3.39736008644104, 9.045619010925293, 5.428028106689453, -12.649599075317383, -3.7993857860565186, -3.366379737854004, -26.98299789428711, 23.33078384399414, 5.83634614944458, 11.148775100708008, 5.493302822113037, -7.033314228057861, 1.7098956108093262, 8.160075187683105, 0.558957040309906, 10.166617393493652, -8.964588165283203, 8.898076057434082, 13.332613945007324, 1.6728395223617554, 2.645726442337036]], "dtype": "float32"}], "e": {"__nd__": [40.60234451293945, 44.78474426269531, 33.16843795776367, 36.50949478149414], "dtype": "float32"}}, {"a": [{"__nd__": [[-50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0], [-50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, -50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0]], "dtype": "float32"}], "e": {"__nd__": [56.93147277832031, 56.93147277832031], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.8986411094665527, 0.4369370937347412, 1.3231847286224365, -0.35918208956718445, 0.48844480514526367, -1.404679298400879, -0.42791539430618286, 2.1320786476135254, -0.1798620969057083, -0.5008790493011475, -2.0721230506896973, -0.5957087874412537, -0.9467950463294983, -1.2094472646713257, 1.2698447704315186, 0.4660685360431671, -0.6561843752861023, -1.1930478811264038, 0.757473349571228, 1.1321194171905518, 0.5997649431228638, -0.13889294862747192, -1.4871959686279297, 0.9258396625518799, 1.396248698234558, 1.8029879331588745, 0.816933274269104, 0.9733282327651978, 0.8811883330345154, -1.8567560911178589, -0.6078856587409973, -1.3371517658233643, 0.708961009979248, -0.5021227598190308, 0.5407149791717529, 0.4851299822330475, -0.5773194432258606, -0.32157179713249207, -1.0779321193695068, 0.49689981341362, -0.8663799166679382, -0.2029028832912445, -1.1066023111343384, -0.3816002309322357, -0.20739616453647614, -1.6225099563598633, -0.24231933057308197, 2.0580761432647705, 1.8599731922149658, 1.0305002927780151, -0.007640504278242588, -0.9095337986946106, -0.37206199765205383, 0.47439029812812805, -0.3426821231842041, 2.2575881481170654, -0.6369973421096802, 0.7954803705215454, -0.348544716835022, 0.3226528763771057, 1.1318626403808594, -1.626709222793579, 1.415611982345581, 0.08857368677854538, -0.13040558993816376, -1.9193229675292969, 1.3820966482162476, 0.9330084919929504, -0.2599014639854431, 0.34468451142311096, -1.434018850326538, -0.06960636377334595, -2.0957181453704834, 0.0064918892458081245, -0.33140265941619873, -0.5803782343864441, -0.4862339496612549, 0.46306896209716797, 1.6405868530273438, -1.558302402496338, -0.8712024092674255, -0.09610443562269211, -0.7554647922515869, -0.24513982236385345, -0.6084626913070679, -0.7869330644607544, 0.37733492255210876, 0.9679665565490723, -1.230954885482788, -0.033235106617212296, 0.24879612028598785, -0.2851821482181549, -0.48974841833114624, -0.6084405183792114, 0.24986107647418976, -0.46478283405303955, -0.7234101295471191, -3.007145404815674, 1.4868929386138916, -0.2844916582107544, 3.212488889694214, 0.1093035638332367, -1.0268363952636719, 0.3477654755115509, -0.16381454467773438, 0.4107085168361664, 0.3957579433917999, 2.3104543685913086, 0.13444846868515015, -0.6637935638427734, 1.2882952690124512, 1.584226131439209, 1.60960054397583, 0.808688759803772, -1.2892358303070068, -0.14078697562217712, 0.7988728284835815, -0.6176537275314331, -0.5510903000831604, -0.8265544176101685, 0.19371552765369415, -1.1014904975891113, -0.008814419619739056, -0.6323376893997192, 0.7500060200691223, 0.4622531831264496, -0.2052084505558014, -0.8236216306686401, -0.177498459815979], [0.126032754778862, 0.6976622343063354, 2.22156023979187, -1.4439430236816406, 1.3147411346435547, 0.46187928318977356, 1.299654483795166, 0.4444592297077179, -2.336676836013794, -0.389413058757782, -0.38853180408477783, 0.38927099108695984, 1.4358766078948975, 1.3601680994033813, 0.3161783516407013, 0.9168223738670349, 0.4218344986438751, -0.12435523420572281, -0.5745798945426941, 1.621666669845581, 0.015775445848703384, -0.7201951146125793, 1.1627180576324463, 0.35089921951293945, -1.3136560916900635, -2.3817126750946045, 0.7196120619773865, 0.12511850893497467, -0.8126963376998901, -0.9627700448036194, -0.9487626552581787, 0.6252588629722595, -1.0596917867660522, -0.07591373473405838, -0.06488095223903656, -1.1614128351211548, 1.1411443948745728, 2.336501359939575, 0.09208918362855911, 1.5559028387069702, 1.1919240951538086, 0.1797112673521042, 0.24305479228496552, 1.6231807470321655, 1.0837029218673706, 1.0692676305770874, -0.0028656672220677137, 0.9939029812812805, 0.6591871380805969, -0.05908022075891495, 0.1893211007118225, 0.3261382281780243, 0.5281419157981873, -0.8467780351638794, 1.1090341806411743, -1.3487498760223389, -0.9283835291862488, 0.7969383597373962, -1.6493185758590698, -1.0302131175994873, 0.048388056457042694, 1.5501958131790161, 1.0994701385498047, -1.4521372318267822, -0.6076932549476624, -1.4848555326461792, -0.5224135518074036, -0.7560510039329529, -0.003474815748631954, 0.07159389555454254, 2.596097946166992, 0.6514841914176941, -0.9857624173164368, -0.10081854462623596, -0.5523437261581421, 1.2464656829833984, 0.24771273136138916, -0.7168921232223511, -1.3692378997802734, 2.176870822906494, -0.004886817652732134, -2.4664158821105957, -0.996299147605896, 0.6285021305084229, -0.4434276521205902, -1.8703898191452026, 0.6304798126220703, -1.5184900760650635, 0.7969703078269958, 0.344127893447876, 1.2845560312271118, 0.5601432919502258, -0.821505606174469, -0.10334360599517822, 0.10337583720684052, 1.535261869430542, -0.05195067077875137, 0.6908513307571411, -0.7171348929405212, 0.4471643269062042, 1.055708646774292, -0.755218505859375, -0.12847094237804413, 0.17773756384849548, -2.203049659729004, -1.824753999710083, 0.6239057779312134, -0.6265007257461548, -0.054152712225914, 0.49109068512916565, -0.5821321606636047, 1.1183745861053467, -2.5197219848632812, -0.31196606159210205, 0.12588675320148468, 1.7177428007125854, -0.28590700030326843, 0.962881326675415, -1.0298651456832886, -1.0393109321594238, -0.7396529316902161, -1.5969144105911255, -1.0724530220031738, -0.37240687012672424, -0.8597215414047241, -0.5465540885925293, 0.21663974225521088, -1.107486367225647, -0.10604167729616165], [-0.05667528882622719, -0.2682129442691803, 1.8138799667358398, -0.686637818813324, 0.41747191548347473, 0.2727731764316559, -1.7107164859771729, -1.1154237985610962, -1.4902393817901611, -1.185624599456787, -0.27570000290870667, -0.8639243245124817, 0.3838616907596588, 0.08003313839435577, -0.9802206158638, 0.2556127905845642, -1.9568778276443481, 1.7533050775527954, 0.6272474527359009, -0.6179015040397644, -1.382835030555725, -0.41021203994750977, 0.4157525300979614, -1.2669544219970703, -0.18895727396011353, 0.7350655198097229, 1.2801921367645264, -1.0618433952331543, 1.9380180835723877, -0.393669456243515, 0.7451169490814209, 0.19472354650497437, -1.7342551946640015, -1.0585606098175049, -0.8423290252685547, 0.00330679421313107, -0.6424781680107117, 0.6633136868476868, 0.3631804287433624, 1.7046881914138794, 0.4262172281742096, -1.550285816192627, 0.41577810049057007, -1.2276029586791992, 0.2886606454849243, 0.413232684135437, -0.5545918941497803, -0.2188323736190796, -0.32633382081985474, -1.6728559732437134, -1.1341187953948975, 0.8618358969688416, -0.0010196862276643515, 1.0180649757385254, -0.2580658793449402, -0.29341694712638855, -2.608532190322876, -0.4771675169467926, -0.1541854739189148, -0.5795466303825378, 0.3468129634857178, 0.8712512850761414, -1.6109604835510254, -0.3010103404521942, -0.4378609359264374, -1.4703857898712158, -1.3699475526809692, -0.12724071741104126, 2.0354650020599365, 0.8508949875831604, 2.0197529792785645, 0.08644901216030121, -1.5709525346755981, -0.5842607617378235, 0.14532135426998138, 0.09745101630687714, 1.42482590675354, 0.7959886193275452, 1.1363474130630493, -0.2453460693359375, -0.8846982717514038, -0.5520006418228149, 0.11977500468492508, 1.9803403615951538, 2.049788236618042, 0.8016088604927063, 0.44415611028671265, -0.7224884033203125, 0.050523899495601654, 1.4471057653427124, -0.1894390732049942, -1.3070000410079956, -1.1412454843521118, -0.7785659432411194, -0.6041193604469299, -0.11385732144117355, -0.20032529532909393, -0.12436449527740479, 0.17017218470573425, -1.1575493812561035, -0.7661092281341553, -0.33163440227508545, 0.4871954023838043, 0.026699166744947433, -0.07273571193218231, -1.2293130159378052, -0.5443890690803528, -0.5297354459762573, 0.5490471720695496, -0.005123422481119633, -0.8761075735092163, -1.0725160837173462, -1.36112380027771, 1.4202431440353394, -0.03226387873291969, 2.9380784034729004, 0.2799678444862366, -1.6524044275283813, -0.5424692034721375, -0.34895309805870056, 0.3719663918018341, -0.6494342088699341, 1.138493537902832, 0.5430102348327637, -1.0477018356323242, 1.693524956703186, 0.845778226852417, -0.43729591369628906, 0.806435227394104], [0.488667368888855, 0.15691488981246948, -1.2068365812301636, 0.4705390930175781, 1.0572508573532104, -1.4594333171844482, 0.7901896834373474, 1.2085832357406616, -1.1437608003616333, 1.3157846927642822, 0.6153044700622559, -0.318318635225296, -0.9816127419471741, -1.0051283836364746, 0.7047473192214966, 0.6191866397857666, 0.27425819635391235, 0.10867857187986374, -0.8854721188545227, 0.5666592717170715, 0.4775482416152954, -0.550778865814209, -2.2498507499694824, -0.5824248790740967, -2.0640451908111572, -1.6962251663208008, -0.881545901298523, 0.35955286026000977, 0.4345262944698334, -0.5708794593811035, -0.05660753697156906, 2.3881354331970215, -1.145972490310669, 0.3867175877094269, 0.31786659359931946, 0.5238234400749207, -0.9573954939842224, 1.2183276414871216, 0.24657388031482697, -0.4596969783306122, 0.1902201771736145, -0.33596011996269226, 0.1694207638502121, 0.14695870876312256, -0.943871796131134, -0.9192057251930237, -0.07800324261188507, 0.4080279767513275, -1.9929108619689941, 0.15708887577056885, -0.9323222041130066, 0.38242262601852417, 0.5868449807167053, -0.7215371131896973, -0.3879137933254242, 0.3478815257549286, -0.18547402322292328, -0.5604398250579834, -0.2010631263256073, 0.08931072801351547, -1.2853875160217285, -0.01642420142889023, 0.33400946855545044, 0.22885295748710632, -0.528620719909668, -0.20460839569568634, -1.0854586362838745, -1.5344469547271729, -0.21966683864593506, -1.5517642498016357, 0.22931793332099915, -0.7180019617080688, 1.7155498266220093, -1.4889447689056396, 1.9649430513381958, -0.776848554611206, 1.4897923469543457, -0.6766719818115234, 0.9859576225280762, -0.4596225321292877, -2.3411314487457275, 1.8010656833648682, 0.09687592089176178, -0.7449878454208374, -0.38067421317100525, -1.2479400634765625, -1.1898027658462524, 2.244605302810669, 1.0724079608917236, 1.0680701732635498, -0.591184139251709, 0.20127934217453003, -0.44501975178718567, 1.3120765686035156, 2.9413013458251953, 0.4367939233779907, 0.2790919542312622, -2.301041603088379, -0.6957332491874695, 0.5001959800720215, -0.41385313868522644, 0.3905954360961914, -0.8189995884895325, -1.1145670413970947, 0.17222684621810913, 0.7842850089073181, 0.21634379029273987, -1.1572238206863403, -1.3155778646469116, 0.6504511833190918, 0.6205227971076965, 2.197354555130005, -0.7816449403762817, 0.09189140051603317, -0.45604372024536133, 0.6692376136779785, 1.4243417978286743, 0.7206915020942688, 0.7966704368591309, 0.20209215581417084, 1.5590177774429321, 0.10693582147359848, -0.2212023138999939, -0.2801855504512787, 0.763861358165741, 0.9079435467720032, -1.2346348762512207, -0.9433349370956421, -1.2693305015563965], [-1.2325278520584106, -1.3505398035049438, 0.03948803246021271, -1.03294038772583, -0.2607578635215759, 0.7453573346138, -1.3611044883728027, -0.15958668291568756, 0.7960166931152344, -0.5314368605613708, 1.819712519645691, 0.345033198595047, -1.632982611656189, -0.3395238220691681, 0.08617474138736725, -1.7705291509628296, -0.3703017234802246, -1.0062592029571533, -1.1767717599868774, -0.009450707584619522, 0.10513655841350555, -0.7939847707748413, 0.7765681743621826, -0.5227534770965576, -0.49192485213279724, -2.44350266456604, 0.8582727313041687, 0.4255269765853882, -0.26900604367256165, -0.4239290654659271, 0.31022897362709045, 0.5997536778450012, 0.29995277523994446, 0.7676353454589844, 1.6912294626235962, 1.1640316247940063, 0.12295469641685486, 0.4384629726409912, 0.6467097997665405, 1.4206311702728271, -1.3339084386825562, 2.0072460174560547, -1.206850290298462, -1.042267918586731, 0.6611900925636292, -0.11539041250944138, 0.28753793239593506, -0.4095163345336914, 0.2130507528781891, -0.07391717284917831, -1.0826647281646729, 0.38582882285118103, -1.8819066286087036, 0.08217772841453552, -1.2803843021392822, -0.16251228749752045, -0.1761593371629715, -0.3446079194545746, 0.9380595088005066, -1.3070292472839355, -0.7046418190002441, -1.4672982692718506, 0.6193015575408936, 2.0152382850646973, -0.14099057018756866, -0.08958913385868073, -0.8736178874969482, -0.8737087249755859, -0.48547589778900146, -0.805811882019043, 0.9849227070808411, 0.916690468788147, -0.28711020946502686, -0.13397227227687836, 0.7092517614364624, 1.9977315664291382, 1.07333505153656, -1.5980900526046753, -1.122300386428833, 0.16868160665035248, 0.41471773386001587, -2.603425979614258, 2.005429983139038, 0.4365885555744171, 1.623834252357483, -1.1863338947296143, 1.5216206312179565, 1.9539589881896973, -0.39477238059043884, 0.11081445962190628, 0.018641872331500053, 0.4061067998409271, -1.3201526403427124, -0.1336485743522644, 0.6028925776481628, 1.2266048192977905, 0.8411593437194824, -0.7267999649047852, 0.751585841178894, -0.7469202280044556, 1.6866549253463745, 0.6021361947059631, 2.5534508228302, -0.6069477200508118, 1.1194287538528442, -0.2568413019180298, -0.0006042883615009487, -1.6482536792755127, 1.3565577268600464, -0.5977980494499207, -0.6011731028556824, 0.910197913646698, 0.16036401689052582, -0.5593590140342712, 0.8463504910469055, -0.17507940530776978, -0.18995706737041473, -0.7298546433448792, -0.7137212157249451, -1.0993160009384155, 0.3535284698009491, 0.9560720324516296, 2.350039005279541, 0.560646653175354, -0.8496109843254089, 0.6367887854576111, -0.4506169259548187, 0.6431133151054382, 1.0625693798065186], [-0.6445785164833069, 1.3225797414779663, -0.6023045182228088, -1.319674015045166, 1.4471770524978638, 0.36087918281555176, -0.6688058972358704, -2.3784894943237305, 0.09239768236875534, 1.8361386060714722, -0.44824233651161194, -0.7934833765029907, -0.32311922311782837, 0.12008063495159149, -2.399709463119507, -0.9977237582206726, 0.8761707544326782, 0.6384837627410889, -1.6977065801620483, -1.2466554641723633, 0.191671222448349, 0.8736436367034912, -1.389980673789978, 0.665259063243866, 0.04749526455998421, -1.6523298025131226, 0.8905444145202637, -0.32130083441734314, 0.2411336898803711, 1.1770260334014893, -0.8169651627540588, 0.08792325109243393, -0.6741226315498352, 1.380112648010254, -1.575546383857727, 0.7954340577125549, -2.2609317302703857, 0.30160269141197205, -1.2382802963256836, 1.832425594329834, -0.2884193956851959, 1.6563527584075928, 0.7225085496902466, -0.08699087798595428, -1.2877413034439087, 0.9800891280174255, -0.3773116171360016, -1.0348047018051147, 0.3626531660556793, -0.03373131901025772, 0.04109298437833786, 2.2813632488250732, -0.5797942876815796, -0.25984302163124084, 2.270120143890381, 2.8536376953125, -0.2394912987947464, 1.5045666694641113, -0.09830936044454575, 1.2970167398452759, 1.1011929512023926, -0.48773616552352905, -0.3354189693927765, 0.17987734079360962, -0.2837587893009186, 0.9593532085418701, 0.20272593200206757, 0.7689629197120667, -0.6915186047554016, -1.302807092666626, 0.06994008272886276, 0.30255916714668274, 0.7662297487258911, -0.4383780062198639, 0.5986958146095276, 1.1887526512145996, 0.5439198017120361, -1.2630729675292969, -0.2437984049320221, 0.6424716114997864, 0.049714360386133194, 1.7323176860809326, 0.24166861176490784, 0.5870978236198425, 0.9795157313346863, 0.7051073908805847, -1.7521671056747437, -0.2338150590658188, -0.46038883924484253, -1.3720877170562744, -0.5293381214141846, 0.006738307885825634, 0.14656654000282288, -0.17854250967502594, 0.348931223154068, 2.399916887283325, 1.1391491889953613, 0.9841930866241455, 0.224583700299263, -0.6500065326690674, 1.0804556608200073, -0.6524721384048462, -1.1485304832458496, -0.30034521222114563, 0.5453001260757446, 0.025421056896448135, 0.08959675580263138, -0.8778161406517029, 2.672706365585327, 0.2308732569217682, -1.9021795988082886, -0.3624911308288574, -0.4726100265979767, 0.3206072747707367, 0.3195187449455261, 0.5373223423957825, -1.3039226531982422, -0.4735446870326996, 0.14076967537403107, 2.217118501663208, -0.008905687369406223, 1.1436516046524048, 0.4022432565689087, -1.5997735261917114, 0.04002917557954788, 0.10785102099180222, 0.3493795096874237, -0.5446479916572571, -1.3442894220352173], [-1.4705030918121338, 0.007573871873319149, 0.7192694544792175, -0.8128805160522461, 1.5496621131896973, -1.9906682968139648, 1.4407620429992676, -0.9343315958976746, 0.5046364665031433, -0.3945558965206146, -1.408714771270752, 0.6522643566131592, -0.5449947118759155, 0.793178915977478, 0.5798801183700562, -1.2509106397628784, -0.2198563814163208, 0.11250918358564377, 1.2687742710113525, 0.441592276096344, -1.675158143043518, -0.34764477610588074, 0.1654987335205078, -0.43782493472099304, 1.039561152458191, 1.320846676826477, 1.5797297954559326, -0.4440584182739258, 0.9986405968666077, 1.5201308727264404, -0.8811549544334412, 1.7545689344406128, -0.04302458092570305, 1.0951852798461914, -0.018742376938462257, -2.007819652557373, 1.747875452041626, -0.2751086950302124, 1.1847232580184937, 1.3310739994049072, -0.13442020118236542, 1.1509370803833008, 0.546100378036499, -1.6028156280517578, -1.1191319227218628, 0.2802790403366089, 0.45493006706237793, -0.2735353410243988, 1.7309616804122925, 0.8848286867141724, 1.2235676050186157, 0.3288939595222473, -0.6716695427894592, -0.7066985964775085, -0.8542858362197876, -0.6287923455238342, 0.8418608903884888, -0.15944863855838776, -0.7763667702674866, -1.4584242105484009, 0.3400247395038605, 0.13781964778900146, -0.7913820743560791, -0.12138185650110245, 1.6172772645950317, -0.46963727474212646, -2.376309394836426, -0.7761696577072144, 1.261766791343689, 0.26569533348083496, -0.3659994602203369, 0.10217498242855072, -0.3656582534313202, -0.9555754661560059, -1.4566086530685425, 2.267669677734375, -1.4843344688415527, -0.7564805746078491, 1.6182383298873901, 1.3588861227035522, -1.05207359790802, 0.2804993689060211, -0.7564367055892944, 0.900618314743042, 0.618395209312439, 0.5186567306518555, -1.4482170343399048, 0.07471190392971039, 1.6130130290985107, -0.12223147600889206, -0.5134487748146057, -0.31833454966545105, -1.4975652694702148, -0.565118134021759, -1.195915699005127, 0.5535566806793213, 0.379935622215271, -0.6721076965332031, -0.6016996502876282, -0.04093995690345764, 0.8348261117935181, -0.45399415493011475, -1.2307831048965454, 0.60230553150177, -0.3886219263076782, -1.199734091758728, -0.8188099265098572, -0.543028712272644, -0.4985104501247406, 1.570351481437683, 0.5306434035301208, -0.2526998221874237, -0.4486687481403351, 1.209787130355835, 0.32237938046455383, 0.12741145491600037, 0.33097878098487854, 2.3129920959472656, 0.938085675239563, -0.753745436668396, -0.7306993007659912, -1.4659359455108643, 0.7598762512207031, -0.1214447095990181, 0.7003681063652039, 1.863134741783142, 0.8081151247024536, 0.10688818991184235, -1.370455026626587], [-1.5553845167160034, 1.1728938817977905, 0.3587808609008789, -1.105820655822754, 1.7645866870880127, -0.47017422318458557, -0.4879634380340576, 0.1703040599822998, 0.41415584087371826, -0.8675873279571533, -1.2922065258026123, -1.6960394382476807, 0.2641368806362152, -0.02018088661134243, -0.9260172247886658, -1.2463849782943726, -0.2736347019672394, 0.9059295058250427, -0.863145112991333, 0.027980297803878784, -0.4848100244998932, -0.46614205837249756, -0.9174177050590515, -1.0504671335220337, -1.2392246723175049, 0.4477306008338928, 0.642400860786438, 0.9895551204681396, -0.035172238945961, -1.6403334140777588, 0.8234243392944336, -0.7299412488937378, 1.5077348947525024, 0.6578781008720398, -0.8196861147880554, 1.1998136043548584, -0.3549008369445801, -0.44975173473358154, -0.030695006251335144, 1.2903170585632324, 0.2615205645561218, 1.0686722993850708, 0.5222404599189758, 0.03769233450293541, 0.2651687562465668, -1.03466796875, 0.09122700989246368, -1.6867471933364868, 1.5153404474258423, 0.4827175438404083, -1.115909218788147, -0.8721116185188293, -0.9364338517189026, -2.4305100440979004, 1.907069444656372, 0.9295154213905334, 0.1276891529560089, 0.9237675070762634, 0.7890884876251221, 0.9902276992797852, 1.4302335977554321, -1.0546694993972778, -0.9000931978225708, -0.13147777318954468, -0.3313223719596863, -0.4057701528072357, 1.373947262763977, -1.6974526643753052, -0.6291177272796631, -0.3686281442642212, 0.6248565912246704, -0.7553332448005676, 1.8025524616241455, 0.8848958611488342, 0.5707711577415466, 1.0111063718795776, -1.379392385482788, 0.42020466923713684, 0.6980116367340088, 0.4070274233818054, -0.5740498900413513, -0.4615575969219208, 0.7681456804275513, -1.109532356262207, 0.6589764356613159, -0.010767584666609764, 1.500011682510376, -0.7013270854949951, 1.559807538986206, -0.7175575494766235, -0.04268345609307289, 0.06753194332122803, 0.024705786257982254, 0.5837006568908691, 0.5372841358184814, 1.315516471862793, -1.5237911939620972, 1.4696009159088135, -0.09416832774877548, 0.16193030774593353, 0.4481034576892853, -1.1139458417892456, 0.04198798909783363, 0.43070071935653687, -0.46160486340522766, 0.9291746616363525, -0.3473856449127197, 0.2553413212299347, 0.9855117797851562, -0.0848994180560112, -0.813199520111084, -0.6616811752319336, -0.12076864391565323, 0.09088291227817535, -1.912493109703064, -0.845416784286499, -0.47328877449035645, 0.9086853861808777, -1.4332813024520874, 1.1041191816329956, 0.5038586258888245, -0.835699200630188, 0.13702978193759918, 1.533194899559021, 1.3985099792480469, 1.1938858032226562, 0.622065544128418, 1.8829059600830078, 0.5239217877388], [-0.5272707343101501, -0.6369988322257996, -1.1196106672286987, 0.09593973308801651, 1.1877633333206177, 0.31456053256988525, 2.062269926071167, -0.22160625457763672, 1.0380829572677612, -0.6595706939697266, 1.212154746055603, -1.3964173793792725, 0.6506643891334534, 0.8553762435913086, -0.962563693523407, -1.1541656255722046, 0.46203452348709106, 0.6157994270324707, -1.5090608596801758, 0.06335362046957016, 1.0908383131027222, -0.268512487411499, 0.009255177341401577, 1.140913963317871, 0.193744495511055, -0.06491409242153168, -0.934973955154419, 1.2689985036849976, -0.3945273160934448, -0.7392438054084778, 0.38856586813926697, -0.3147300183773041, -0.3503752648830414, 1.247962474822998, 1.71099054813385, 0.620981752872467, -2.4763362407684326, -0.5378692746162415, 1.3917540311813354, -0.5668628811836243, -0.5093990564346313, 0.260392427444458, -0.8239010572433472, -0.4319041967391968, -0.9928104281425476, -1.0887545347213745, -0.3764832615852356, 1.0482982397079468, 0.007701190188527107, 1.376745581626892, -0.4829764664173126, -1.1905847787857056, -0.16233643889427185, -1.106278419494629, 0.6045638918876648, 0.7939465641975403, 0.3263493478298187, -0.864865243434906, 0.7105026841163635, -1.1279631853103638, -0.6497830152511597, -0.12990079820156097, 0.7468883395195007, -0.8327356576919556, 0.6601842641830444, 0.83301842212677, -1.3445422649383545, -1.0266757011413574, 1.6156959533691406, 1.7115975618362427, 0.9735506772994995, -1.3861963748931885, -1.1340179443359375, -1.067585825920105, -1.0193859338760376, -1.7895411252975464, -1.3779923915863037, 0.5502603054046631, -0.9052507877349854, 1.1220637559890747, 0.9105782508850098, -0.5503340363502502, 1.2221593856811523, -1.0324716567993164, 0.295737624168396, 0.12554918229579926, 0.2667923867702484, -0.6104410290718079, 0.16696980595588684, -0.43079036474227905, 0.7926580309867859, 2.0268657207489014, 0.9239000678062439, 1.4337893724441528, -0.33819669485092163, 0.5665615797042847, 1.8007265329360962, -1.4737776517868042, -0.7517976760864258, -1.3208321332931519, -0.20564725995063782, -1.1621931791305542, -0.4598257541656494, -0.5321075320243835, -0.12367895990610123, -0.4354265332221985, -0.8164403438568115, -0.04077582061290741, -1.6418910026550293, 0.7491582632064819, -0.4345215857028961, 0.2743184566497803, -0.18798188865184784, 0.5559396147727966, -0.10116969048976898, -0.7318814992904663, -0.907956600189209, -0.3602583408355713, -0.2792896628379822, -1.6785718202590942, 1.2469507455825806, -1.2153962850570679, 0.30998939275741577, 1.015660047531128, 0.4067607820034027, -0.7182747721672058, 2.0498571395874023, 0.22796814143657684, -0.27213630080223083]], "dtype": "float32"}], "e": {"__nd__": [5.4209136962890625, 5.383999824523926, 5.3134965896606445, 5.344554901123047, 5.400788307189941, 5.490528583526611, 5.381019115447998, 5.310739994049072, 5.25916051864624], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(logsumexp_rows, json.loads(_TESTS), 'triton', *[1e-05, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")